In [1]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 1 — CONFIGURATION, STATE & INCREMENTAL WATERMARK
# ==================================================================================================
#
# Purpose
# -------
# Establish the immutable runtime configuration for one World Athletics
# incremental ingestion run.
#
# This cell:
#
#   1. Defines project/data directories.
#   2. Locates the current reconciled canonical javelin history.
#   3. Reads the canonical maximum result date.
#   4. Reads prior ingestion state if available.
#   5. Calculates a safe overlapping incremental download window.
#   6. Creates a run-specific raw/checkpoint directory.
#
#
# IMPORTANT
# ---------
# No World Athletics network calls occur in this cell.
#
# The existing canonical dataset is NEVER overwritten here.
# Promotion of an updated dataset happens only after final QA passes.
#
# ==================================================================================================

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import json
import os

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PROJECT CONFIGURATION
# ==================================================================================================

SINGAPORE_TZ = ZoneInfo(
    "Asia/Singapore"
)


@dataclass(frozen=True)
class WAIncrementalConfig:
    """
    Runtime configuration for one World Athletics incremental-update run.
    """

    # ----------------------------------------------------------------------------------------------
    # Project roots
    # ----------------------------------------------------------------------------------------------

    project_root: Path

    wa_root: Path

    final_dir: Path

    incremental_root: Path

    raw_root: Path

    checkpoint_root: Path

    candidate_root: Path

    manifest_root: Path

    state_root: Path


    # ----------------------------------------------------------------------------------------------
    # Existing canonical source
    # ----------------------------------------------------------------------------------------------

    canonical_dataset_path: Path


    # ----------------------------------------------------------------------------------------------
    # Historical extraction assets retained from previous work
    # ----------------------------------------------------------------------------------------------

    historical_backfill_root: Path

    historical_raw_athletes_dir: Path

    historical_manifest_path: Path


    # ----------------------------------------------------------------------------------------------
    # Incremental extraction policy
    # ----------------------------------------------------------------------------------------------

    lookback_days: int = 14

    request_timeout_seconds: float = 60.0

    graphql_retries: int = 4

    retry_base_seconds: float = 2.0

    request_pause_min_seconds: float = 0.25

    request_pause_max_seconds: float = 0.50


    # ----------------------------------------------------------------------------------------------
    # Production safety
    # ----------------------------------------------------------------------------------------------

    overwrite_canonical: bool = False

    enable_network_calls: bool = False


PROJECT_ROOT = Path(
    "/Users/veesheenyuen/Documents/code/veeyuen/SAA"
)


WA_ROOT = (
    PROJECT_ROOT
    / "world_athletics_global"
)


CONFIG = WAIncrementalConfig(

    project_root=
        PROJECT_ROOT,

    wa_root=
        WA_ROOT,

    final_dir=
        WA_ROOT
        / "final",

    incremental_root=
        WA_ROOT
        / "incremental",

    raw_root=
        WA_ROOT
        / "incremental"
        / "raw",

    checkpoint_root=
        WA_ROOT
        / "incremental"
        / "checkpoints",

    candidate_root=
        WA_ROOT
        / "incremental"
        / "candidates",

    manifest_root=
        WA_ROOT
        / "incremental"
        / "manifests",

    state_root=
        WA_ROOT
        / "incremental"
        / "state",


    # Latest confirmed reconciled historical source from the project.
    canonical_dataset_path=
        WA_ROOT
        / "final"
        / "javelin_reconciled_base_1986_2026.parquet",


    # Historical raw backfill retained for audit / recovery.
    historical_backfill_root=
        WA_ROOT
        / "historical_backfill",

    historical_raw_athletes_dir=
        WA_ROOT
        / "historical_backfill"
        / "raw_athletes",

    historical_manifest_path=
        WA_ROOT
        / "historical_backfill"
        / "global_backfill_manifest.parquet",


    lookback_days=
        14,

    request_timeout_seconds=
        60.0,

    graphql_retries=
        4,

    retry_base_seconds=
        2.0,

    request_pause_min_seconds=
        0.25,

    request_pause_max_seconds=
        0.50,


    # Safety defaults.
    overwrite_canonical=
        False,

    enable_network_calls=
        False,
)


# ==================================================================================================
# PART B — CREATE INCREMENTAL WORKING DIRECTORIES
# ==================================================================================================

for directory in [

    CONFIG.incremental_root,

    CONFIG.raw_root,

    CONFIG.checkpoint_root,

    CONFIG.candidate_root,

    CONFIG.manifest_root,

    CONFIG.state_root,

]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================================================================
# PART C — RUN IDENTIFIER
# ==================================================================================================

RUN_STARTED_AT = datetime.now(
    SINGAPORE_TZ
)


RUN_ID = RUN_STARTED_AT.strftime(
    "%Y%m%d_%H%M%S"
)


RUN_DATE = RUN_STARTED_AT.date()


RUN_RAW_DIR = (
    CONFIG.raw_root
    / RUN_ID
)


RUN_CHECKPOINT_DIR = (
    CONFIG.checkpoint_root
    / RUN_ID
)


RUN_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


RUN_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Individual raw layers.
RUN_CALENDAR_DIR = (
    RUN_RAW_DIR
    / "calendar"
)

RUN_COMPETITION_DIR = (
    RUN_RAW_DIR
    / "competitions"
)

RUN_ATHLETE_DIR = (
    RUN_RAW_DIR
    / "athletes"
)

RUN_EXCEPTION_DIR = (
    RUN_RAW_DIR
    / "exceptions"
)


for directory in [

    RUN_CALENDAR_DIR,

    RUN_COMPETITION_DIR,

    RUN_ATHLETE_DIR,

    RUN_EXCEPTION_DIR,

]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================================================================
# PART D — INGESTION STATE PATHS
# ==================================================================================================

INGESTION_STATE_PATH = (
    CONFIG.state_root
    / "wa_incremental_state.json"
)


RUN_MANIFEST_PATH = (
    CONFIG.manifest_root
    / f"wa_incremental_manifest_{RUN_ID}.json"
)


# Candidate output — deliberately NOT the production canonical path.
CANDIDATE_DATASET_PATH = (
    CONFIG.candidate_root
    / f"javelin_reconciled_candidate_{RUN_ID}.parquet"
)


# ==================================================================================================
# PART E — VERIFY CANONICAL DATASET
# ==================================================================================================

if not CONFIG.canonical_dataset_path.exists():

    raise FileNotFoundError(
        "Canonical reconciled javelin dataset was not found:\n\n"
        f"{CONFIG.canonical_dataset_path}\n\n"
        "Notebook 1 must start from the persisted reconciled historical dataset."
    )


# ==================================================================================================
# PART F — INSPECT PARQUET SCHEMA WITHOUT LOADING THE FULL TABLE
# ==================================================================================================

try:

    import pyarrow.parquet as pq

except ImportError as exc:

    raise ImportError(
        "pyarrow is required to inspect the canonical parquet dataset."
    ) from exc


canonical_parquet = pq.ParquetFile(
    CONFIG.canonical_dataset_path
)


CANONICAL_COLUMNS = list(
    canonical_parquet.schema.names
)


CANONICAL_ROW_COUNT = int(
    canonical_parquet.metadata.num_rows
)


# ==================================================================================================
# PART G — IDENTIFY CANONICAL DATE COLUMN
# ==================================================================================================

DATE_COLUMN_CANDIDATES = [

    "result_date_dt",

    "result_date",

    "date",

    "performance_date",

    "competition_date",

]


CANONICAL_DATE_COLUMN = next(

    (
        column

        for column
        in DATE_COLUMN_CANDIDATES

        if column
        in CANONICAL_COLUMNS
    ),

    None,
)


if CANONICAL_DATE_COLUMN is None:

    raise KeyError(
        "Could not identify a canonical result-date column.\n\n"
        f"Available columns:\n{CANONICAL_COLUMNS}"
    )


# ==================================================================================================
# PART H — READ ONLY DATE COLUMN TO DETERMINE CURRENT DATA WATERMARK
# ==================================================================================================

canonical_dates = pd.read_parquet(

    CONFIG.canonical_dataset_path,

    columns=[
        CANONICAL_DATE_COLUMN
    ],
)


canonical_dates[
    CANONICAL_DATE_COLUMN
] = pd.to_datetime(

    canonical_dates[
        CANONICAL_DATE_COLUMN
    ],

    errors="coerce",
)


if (
    canonical_dates[
        CANONICAL_DATE_COLUMN
    ]
    .notna()
    .sum()
    == 0
):

    raise RuntimeError(
        "Canonical dataset contains no valid result dates."
    )


CANONICAL_MIN_RESULT_DATE = pd.Timestamp(

    canonical_dates[
        CANONICAL_DATE_COLUMN
    ]
    .min()

).normalize()


CANONICAL_MAX_RESULT_DATE = pd.Timestamp(

    canonical_dates[
        CANONICAL_DATE_COLUMN
    ]
    .max()

).normalize()


del canonical_dates


# ==================================================================================================
# PART I — LOAD PRIOR INCREMENTAL STATE IF IT EXISTS
# ==================================================================================================

if INGESTION_STATE_PATH.exists():

    with INGESTION_STATE_PATH.open(
        "r",
        encoding="utf-8",
    ) as f:

        PRIOR_STATE = json.load(
            f
        )

else:

    PRIOR_STATE = {}


prior_state_max_date_raw = PRIOR_STATE.get(
    "max_result_date"
)


if prior_state_max_date_raw is not None:

    PRIOR_STATE_MAX_RESULT_DATE = pd.Timestamp(
        prior_state_max_date_raw
    ).normalize()

else:

    PRIOR_STATE_MAX_RESULT_DATE = None


# ==================================================================================================
# PART J — SAFE EFFECTIVE WATERMARK
# ==================================================================================================
#
# The canonical dataset is the primary source of truth.
#
# If the stored state and canonical dataset disagree, we deliberately choose
# the EARLIER date.
#
# This can cause us to redownload some already-known results, but it prevents
# us from skipping data after a partial or interrupted previous run.
# ==================================================================================================

watermark_candidates = [

    CANONICAL_MAX_RESULT_DATE
]


if PRIOR_STATE_MAX_RESULT_DATE is not None:

    watermark_candidates.append(
        PRIOR_STATE_MAX_RESULT_DATE
    )


EFFECTIVE_WATERMARK = min(
    watermark_candidates
)


# ==================================================================================================
# PART K — CALCULATE INCREMENTAL DOWNLOAD WINDOW
# ==================================================================================================

DOWNLOAD_START_DATE = (

    EFFECTIVE_WATERMARK
    - pd.Timedelta(
        days=CONFIG.lookback_days
    )
)


DOWNLOAD_END_DATE = pd.Timestamp(
    RUN_STARTED_AT.date()
)


if DOWNLOAD_START_DATE > DOWNLOAD_END_DATE:

    raise RuntimeError(
        "Calculated incremental download start is after the run date.\n\n"
        f"Download start : {DOWNLOAD_START_DATE.date()}\n"
        f"Download end   : {DOWNLOAD_END_DATE.date()}"
    )


# ==================================================================================================
# PART L — CURRENT RUN MANIFEST
# ==================================================================================================

RUN_MANIFEST = {

    "run_id":
        RUN_ID,

    "run_started_at":
        RUN_STARTED_AT.isoformat(),

    "timezone":
        "Asia/Singapore",

    "canonical_dataset_path":
        str(
            CONFIG.canonical_dataset_path
        ),

    "canonical_rows_before":
        CANONICAL_ROW_COUNT,

    "canonical_date_column":
        CANONICAL_DATE_COLUMN,

    "canonical_min_result_date":
        CANONICAL_MIN_RESULT_DATE.date().isoformat(),

    "canonical_max_result_date":
        CANONICAL_MAX_RESULT_DATE.date().isoformat(),

    "prior_state_max_result_date":
        (
            PRIOR_STATE_MAX_RESULT_DATE.date().isoformat()

            if PRIOR_STATE_MAX_RESULT_DATE
            is not None

            else None
        ),

    "effective_watermark":
        EFFECTIVE_WATERMARK.date().isoformat(),

    "lookback_days":
        CONFIG.lookback_days,

    "download_start_date":
        DOWNLOAD_START_DATE.date().isoformat(),

    "download_end_date":
        DOWNLOAD_END_DATE.date().isoformat(),

    "candidate_dataset_path":
        str(
            CANDIDATE_DATASET_PATH
        ),

    "network_calls_enabled":
        CONFIG.enable_network_calls,

    "status":
        "INITIALIZED",
}


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
    )


# ==================================================================================================
# PART M — SANITY CHECK HISTORICAL ASSETS
# ==================================================================================================

HISTORICAL_BACKFILL_STATUS = {

    "historical_raw_dir_exists":
        CONFIG.historical_raw_athletes_dir.exists(),

    "historical_manifest_exists":
        CONFIG.historical_manifest_path.exists(),
}


# ==================================================================================================
# PART N — OUTPUT
# ==================================================================================================

print()
print("=" * 120)
print("WORLD ATHLETICS INCREMENTAL UPDATE — INITIALIZATION")
print("=" * 120)


print()
print("RUN")
print("-" * 120)

print(
    f"Run ID                     : {RUN_ID}"
)

print(
    f"Started                     : {RUN_STARTED_AT.isoformat()}"
)

print(
    f"Network calls enabled       : {CONFIG.enable_network_calls}"
)


print()
print("CANONICAL DATASET")
print("-" * 120)

print(
    f"Path                        : {CONFIG.canonical_dataset_path}"
)

print(
    f"Rows                        : {CANONICAL_ROW_COUNT:,}"
)

print(
    f"Date column                 : {CANONICAL_DATE_COLUMN}"
)

print(
    f"Minimum result date         : {CANONICAL_MIN_RESULT_DATE.date()}"
)

print(
    f"Maximum result date         : {CANONICAL_MAX_RESULT_DATE.date()}"
)


print()
print("PRIOR STATE")
print("-" * 120)

print(
    f"State file exists           : {INGESTION_STATE_PATH.exists()}"
)

print(
    f"Prior state max result date : "
    f"{PRIOR_STATE_MAX_RESULT_DATE.date() if PRIOR_STATE_MAX_RESULT_DATE is not None else 'None'}"
)


print()
print("INCREMENTAL WINDOW")
print("-" * 120)

print(
    f"Effective watermark         : {EFFECTIVE_WATERMARK.date()}"
)

print(
    f"Lookback                    : {CONFIG.lookback_days} days"
)

print(
    f"Download start              : {DOWNLOAD_START_DATE.date()}"
)

print(
    f"Download end                : {DOWNLOAD_END_DATE.date()}"
)


print()
print("RUN OUTPUTS")
print("-" * 120)

print(
    f"Raw directory               : {RUN_RAW_DIR}"
)

print(
    f"Checkpoint directory        : {RUN_CHECKPOINT_DIR}"
)

print(
    f"Candidate dataset           : {CANDIDATE_DATASET_PATH}"
)

print(
    f"Run manifest                : {RUN_MANIFEST_PATH}"
)


print()
print("HISTORICAL BACKFILL")
print("-" * 120)

print(
    f"Raw athlete archive exists  : "
    f"{HISTORICAL_BACKFILL_STATUS['historical_raw_dir_exists']}"
)

print(
    f"Backfill manifest exists    : "
    f"{HISTORICAL_BACKFILL_STATUS['historical_manifest_exists']}"
)


print()
print("=" * 120)
print("CELL 1 STATUS: READY")
print("=" * 120)

print()
print(
    "No World Athletics requests have been made."
)

print(
    "The next cell will recover the LIVE World Athletics GraphQL "
    "endpoint and x-api-key using the existing Playwright bootstrap."
)


WORLD ATHLETICS INCREMENTAL UPDATE — INITIALIZATION

RUN
------------------------------------------------------------------------------------------------------------------------
Run ID                     : 20260919_225312
Started                     : 2026-09-19T22:53:12.045835+08:00
Network calls enabled       : False

CANONICAL DATASET
------------------------------------------------------------------------------------------------------------------------
Path                        : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/final/javelin_reconciled_base_1986_2026.parquet
Rows                        : 248,896
Date column                 : result_date_dt
Minimum result date         : 1986-07-17
Maximum result date         : 2026-09-13

PRIOR STATE
------------------------------------------------------------------------------------------------------------------------
State file exists           : False
Prior state max result date : None

INCREMENTAL WINDOW

In [2]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 2 — LIVE WORLD ATHLETICS GRAPHQL BOOTSTRAP
# ==================================================================================================
#
# Purpose
# -------
# Dynamically discover the CURRENT World Athletics GraphQL configuration.
#
# This cell:
#
#   1. Opens the live World Athletics website with Playwright.
#   2. Observes GraphQL network requests.
#   3. Captures:
#
#          GraphQL endpoint
#          x-api-key
#          relevant request headers
#
#   4. Creates the reusable asynchronous wa_graphql() client.
#   5. Sends a lightweight direct GraphQL probe.
#   6. Saves the discovered runtime configuration for audit/recovery.
#
#
# IMPORTANT
# ---------
# We DO NOT hard-code:
#
#       GraphQL URL
#       x-api-key
#
# These values have changed historically and must be discovered from the
# live World Athletics site.
#
#
# This cell DOES make a small number of network requests.
#
# It does NOT start the athlete/result crawl.
# ==================================================================================================

from __future__ import annotations

import asyncio
import json
import os
import random
import time

from datetime import datetime

import httpx


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "CONFIG",
    "RUN_ID",
    "RUN_STARTED_AT",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",
    "SINGAPORE_TZ",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cell 1 first. Missing:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_globals
        )
    )


# ==================================================================================================
# PART B — PLAYWRIGHT IMPORT
# ==================================================================================================

try:

    from playwright.async_api import (
        async_playwright,
        Error as PlaywrightError,
    )

except ImportError as exc:

    raise ImportError(
        "Playwright is not installed in this environment.\n\n"
        "Install it with:\n\n"
        "    pip install playwright\n"
        "    playwright install chromium"
    ) from exc


# ==================================================================================================
# PART C — BOOTSTRAP SETTINGS
# ==================================================================================================

# This enables ONLY the live API discovery/probe performed in this cell.
# The bulk incremental crawl is still not started.
ENABLE_WA_BOOTSTRAP = True


# Browser pages used only to trigger live WA GraphQL traffic.
#
# We try several pages because the precise front-end route can change while
# the network-discovery method remains valid.
WA_TRIGGER_URLS = [

    "https://worldathletics.org/",

    (
        "https://worldathletics.org/records/toplists/"
        "throws/javelin-throw/outdoor/men/senior/2026"
    ),

    "https://worldathletics.org/competition/calendar-results",
]


WA_BOOTSTRAP_PATH = (
    RUN_CHECKPOINT_DIR
    / f"wa_live_graphql_bootstrap_{RUN_ID}.json"
)


# ==================================================================================================
# PART D — HELPERS
# ==================================================================================================

def mask_secret(
    value,
    visible_start=6,
    visible_end=4,
):
    """
    Mask sensitive-looking runtime strings when printing them.
    """

    if value is None:

        return None


    value = str(
        value
    )


    if len(value) <= (
        visible_start
        + visible_end
    ):

        return "*" * len(
            value
        )


    return (
        value[
            :visible_start
        ]
        + "..."
        + value[
            -visible_end:
        ]
    )


def safe_post_payload(
    request,
):
    """
    Best-effort extraction of a request JSON body.

    Returns None when the body is absent or not JSON.
    """

    try:

        payload = (
            request.post_data_json
        )


        if callable(
            payload
        ):

            payload = payload()


        return payload


    except Exception:

        try:

            raw = request.post_data


            if raw:

                return json.loads(
                    raw
                )


        except Exception:

            pass


    return None


# ==================================================================================================
# PART E — LIVE PLAYWRIGHT DISCOVERY
# ==================================================================================================

async def discover_live_wa_graphql(
    trigger_urls=WA_TRIGGER_URLS,
):
    """
    Discover the current World Athletics GraphQL endpoint and API headers.

    Returns
    -------
    dict

        endpoint
        api_key
        headers
        source_page
        operation_name
        captured_at
        captures
    """

    captures = []

    pending_tasks = []


    async with async_playwright() as p:

        try:

            browser = await p.chromium.launch(
                headless=True
            )

        except Exception as exc:

            raise RuntimeError(
                "Playwright could not launch Chromium.\n\n"
                "If Chromium has not yet been installed, run:\n\n"
                "    playwright install chromium"
            ) from exc


        context = await browser.new_context(

            viewport={
                "width":
                    1440,

                "height":
                    1000,
            },

            locale=
                "en-GB",
        )


        page = await context.new_page()


        # ------------------------------------------------------------------------------------------
        # Capture relevant network requests
        # ------------------------------------------------------------------------------------------

        async def inspect_request(
            request,
        ):

            url = str(
                request.url
            )


            if "graphql" not in url.lower():

                return


            try:

                headers = await request.all_headers()

            except Exception:

                headers = dict(
                    request.headers
                )


            headers = {
                str(k).lower():
                    str(v)

                for k, v
                in headers.items()
            }


            api_key = headers.get(
                "x-api-key"
            )


            # We only care about requests carrying the live WA GraphQL key.
            if not api_key:

                return


            payload = safe_post_payload(
                request
            )


            operation_name = None

            query_text = None


            if isinstance(
                payload,
                dict,
            ):

                operation_name = (
                    payload.get(
                        "operationName"
                    )
                )

                query_text = (
                    payload.get(
                        "query"
                    )
                )


            captures.append(
                {
                    "url":
                        url,

                    "method":
                        str(
                            request.method
                        ),

                    "api_key":
                        api_key,

                    "headers":
                        headers,

                    "operation_name":
                        operation_name,

                    "query_text":
                        query_text,

                    "payload":
                        payload,
                }
            )


        def request_listener(
            request,
        ):

            task = asyncio.create_task(
                inspect_request(
                    request
                )
            )

            pending_tasks.append(
                task
            )


        page.on(
            "request",
            request_listener,
        )


        # ------------------------------------------------------------------------------------------
        # Visit trigger pages until a valid GraphQL request is observed
        # ------------------------------------------------------------------------------------------

        successful_page = None


        for trigger_url in trigger_urls:

            try:

                await page.goto(
                    trigger_url,
                    wait_until="domcontentloaded",
                    timeout=60_000,
                )


                # Allow front-end data requests to fire.
                await page.wait_for_timeout(
                    5_000
                )


                # Scroll once because some WA pages lazily request data.
                try:

                    await page.evaluate(
                        """
                        window.scrollTo(
                            0,
                            Math.max(
                                document.body.scrollHeight,
                                document.documentElement.scrollHeight
                            )
                        );
                        """
                    )


                    await page.wait_for_timeout(
                        3_000
                    )

                except Exception:

                    pass


                # Allow any pending interception tasks to finish.
                if pending_tasks:

                    await asyncio.gather(
                        *pending_tasks,
                        return_exceptions=True,
                    )


                if captures:

                    successful_page = (
                        trigger_url
                    )

                    break


            except Exception:

                # Try the next trigger page.
                continue


        # Final interception flush.
        if pending_tasks:

            await asyncio.gather(
                *pending_tasks,
                return_exceptions=True,
            )


        await context.close()

        await browser.close()


    if not captures:

        raise RuntimeError(
            "No World Athletics GraphQL request carrying an x-api-key "
            "was observed.\n\n"
            "The WA front-end may have changed, or the pages may not have "
            "fully loaded. No bulk crawl has been started."
        )


    # ==============================================================================================
    # Rank the captures.
    #
    # Prefer the operation that historically proved useful for complete
    # athlete result histories, but ANY live GraphQL request carrying the
    # endpoint/key is sufficient for bootstrap.
    # ==============================================================================================

    def capture_score(
        capture,
    ):

        operation = str(
            capture.get(
                "operation_name"
            )
            or ""
        ).lower()


        query_text = str(
            capture.get(
                "query_text"
            )
            or ""
        ).lower()


        score = 0


        if (
            "getsinglecompetitorresultsdate"
            in operation
            or
            "getsinglecompetitorresultsdate"
            in query_text
        ):

            score += 100


        if str(
            capture.get(
                "method"
            )
        ).upper() == "POST":

            score += 20


        if capture.get(
            "api_key"
        ):

            score += 10


        return score


    best_capture = max(
        captures,
        key=capture_score,
    )


    captured_headers = (
        best_capture[
            "headers"
        ]
    )


    # ==============================================================================================
    # Keep only headers useful for direct GraphQL POST calls.
    # ==============================================================================================

    allowed_header_names = [

        "accept",

        "accept-language",

        "content-type",

        "x-api-key",

        "x-amz-user-agent",

        "user-agent",

        "origin",

        "referer",
    ]


    direct_headers = {

        header:
            captured_headers[
                header
            ]

        for header
        in allowed_header_names

        if header
        in captured_headers
    }


    # Always ensure JSON content type.
    direct_headers[
        "content-type"
    ] = "application/json"


    return {

        "endpoint":
            best_capture[
                "url"
            ],

        "api_key":
            best_capture[
                "api_key"
            ],

        "headers":
            direct_headers,

        "source_page":
            successful_page,

        "operation_name":
            best_capture.get(
                "operation_name"
            ),

        "captured_at":
            datetime.now(
                SINGAPORE_TZ
            ).isoformat(),

        "captures":
            captures,
    }


# ==================================================================================================
# PART F — RUN LIVE DISCOVERY
# ==================================================================================================

if not ENABLE_WA_BOOTSTRAP:

    raise RuntimeError(
        "ENABLE_WA_BOOTSTRAP is False.\n"
        "Set it to True in Cell 2 when you are ready to perform "
        "the live API discovery."
    )


WA_BOOTSTRAP = await discover_live_wa_graphql()


WA_GRAPHQL_URL = str(
    WA_BOOTSTRAP[
        "endpoint"
    ]
)


WA_API_KEY = str(
    WA_BOOTSTRAP[
        "api_key"
    ]
)


WA_GRAPHQL_HEADERS = dict(
    WA_BOOTSTRAP[
        "headers"
    ]
)


# ==================================================================================================
# PART G — LOW-LEVEL ASYNC GRAPHQL CLIENT
# ==================================================================================================

WA_RETRYABLE_STATUS_CODES = {
    429,
    500,
    502,
    503,
    504,
}


async def _wa_graphql_request(
    query,
    variables=None,
    *,
    operation_name=None,
    retries=None,
):
    """
    Internal direct World Athletics GraphQL request.

    Used by the bootstrap probe and the public wa_graphql() wrapper.
    """

    if variables is None:

        variables = {}


    if retries is None:

        retries = int(
            CONFIG.graphql_retries
        )


    payload = {

        "query":
            query,

        "variables":
            variables,
    }


    if operation_name is not None:

        payload[
            "operationName"
        ] = operation_name


    timeout = httpx.Timeout(
        90.0,
        connect=30.0,
    )


    last_error = None


    async with httpx.AsyncClient(
        timeout=timeout,
        follow_redirects=True,
    ) as client:

        for attempt in range(
            retries
        ):

            try:

                response = await client.post(

                    WA_GRAPHQL_URL,

                    headers=
                        WA_GRAPHQL_HEADERS,

                    json=
                        payload,
                )


                # ----------------------------------------------------------------------------------
                # Retry temporary server/throttling responses
                # ----------------------------------------------------------------------------------

                if (
                    response.status_code
                    in WA_RETRYABLE_STATUS_CODES
                ):

                    raise httpx.HTTPStatusError(
                        (
                            "Retryable World Athletics HTTP status "
                            f"{response.status_code}"
                        ),
                        request=response.request,
                        response=response,
                    )


                response.raise_for_status()


                body = response.json()


                if body.get(
                    "errors"
                ):

                    raise RuntimeError(
                        "World Athletics GraphQL returned errors:\n"
                        + json.dumps(
                            body[
                                "errors"
                            ],
                            indent=2,
                        )[:5000]
                    )


                data = body.get(
                    "data"
                )


                # Polite pacing between successful direct requests.
                await asyncio.sleep(
                    random.uniform(
                        CONFIG.request_pause_min_seconds,
                        CONFIG.request_pause_max_seconds,
                    )
                )


                return data


            except (
                httpx.TransportError,
                httpx.TimeoutException,
                httpx.HTTPStatusError,
            ) as exc:

                last_error = exc


                # Do not keep retrying non-transient HTTP failures.
                if isinstance(
                    exc,
                    httpx.HTTPStatusError,
                ):

                    status = (
                        exc.response.status_code
                        if exc.response is not None
                        else None
                    )


                    if (
                        status is not None
                        and status
                        not in WA_RETRYABLE_STATUS_CODES
                    ):

                        raise


                if attempt >= (
                    retries - 1
                ):

                    break


                delay = (
                    min(
                        60.0,
                        2.0 ** attempt,
                    )
                    +
                    random.uniform(
                        0.5,
                        2.0,
                    )
                )


                await asyncio.sleep(
                    delay
                )


    raise RuntimeError(
        "World Athletics GraphQL request failed after retries."
    ) from last_error


# ==================================================================================================
# PART H — VERIFY THE CAPTURED ENDPOINT / KEY
# ==================================================================================================
#
# __typename is deliberately tiny.
#
# It verifies that:
#
#       endpoint works
#       key works
#       captured headers work
#       direct HTTPX access works
#
# without beginning any athlete crawl.
# ==================================================================================================

WA_BOOTSTRAP_PROBE_QUERY = """
query BootstrapProbe {
    __typename
}
"""


WA_BOOTSTRAP_PROBE = await _wa_graphql_request(

    query=
        WA_BOOTSTRAP_PROBE_QUERY,

    variables=
        {},

    operation_name=
        "BootstrapProbe",

    retries=
        CONFIG.graphql_retries,
)


if not isinstance(
    WA_BOOTSTRAP_PROBE,
    dict,
):

    raise RuntimeError(
        "World Athletics GraphQL probe returned an unexpected response."
    )


if "__typename" not in WA_BOOTSTRAP_PROBE:

    raise RuntimeError(
        "World Athletics GraphQL probe succeeded at HTTP level but "
        "did not return the expected GraphQL root typename."
    )


WA_NETWORK_READY = True


# ==================================================================================================
# PART I — PUBLIC GRAPHQL CLIENT
# ==================================================================================================

async def wa_graphql(
    query,
    variables=None,
    *,
    operation_name=None,
    retries=None,
):
    """
    Public World Athletics GraphQL client for later notebook cells.

    The client is usable only after Cell 2 has successfully discovered
    and validated the live endpoint/key.
    """

    if not globals().get(
        "WA_NETWORK_READY",
        False,
    ):

        raise RuntimeError(
            "World Athletics network client is not ready. "
            "Run Notebook 1 Cell 2 successfully first."
        )


    return await _wa_graphql_request(

        query=query,

        variables=variables,

        operation_name=operation_name,

        retries=retries,
    )


# ==================================================================================================
# PART J — SAVE BOOTSTRAP AUDIT FILE
# ==================================================================================================
#
# This file contains the live API key because later cells in the SAME local
# run may need to recover after a kernel interruption.
#
# Do not commit this checkpoint directory to source control.
# ==================================================================================================

BOOTSTRAP_AUDIT = {

    "run_id":
        RUN_ID,

    "captured_at":
        WA_BOOTSTRAP[
            "captured_at"
        ],

    "source_page":
        WA_BOOTSTRAP[
            "source_page"
        ],

    "graphql_url":
        WA_GRAPHQL_URL,

    "api_key":
        WA_API_KEY,

    "headers":
        WA_GRAPHQL_HEADERS,

    "selected_operation_name":
        WA_BOOTSTRAP.get(
            "operation_name"
        ),

    "n_graphql_requests_observed":
        len(
            WA_BOOTSTRAP[
                "captures"
            ]
        ),

    "probe_response":
        WA_BOOTSTRAP_PROBE,

    "status":
        "VALIDATED",
}


with WA_BOOTSTRAP_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        BOOTSTRAP_AUDIT,
        f,
        indent=2,
    )


# ==================================================================================================
# PART K — UPDATE RUN MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "graphql_bootstrap"
] = {

    "status":
        "VALIDATED",

    "captured_at":
        WA_BOOTSTRAP[
            "captured_at"
        ],

    "source_page":
        WA_BOOTSTRAP[
            "source_page"
        ],

    "graphql_url":
        WA_GRAPHQL_URL,

    "n_graphql_requests_observed":
        len(
            WA_BOOTSTRAP[
                "captures"
            ]
        ),

    "checkpoint_path":
        str(
            WA_BOOTSTRAP_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = "GRAPHQL_READY"


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
    )


# ==================================================================================================
# PART L — OUTPUT
# ==================================================================================================

print()
print("=" * 120)
print("WORLD ATHLETICS LIVE GRAPHQL BOOTSTRAP")
print("=" * 120)


print()
print("DISCOVERY")
print("-" * 120)

print(
    f"Source page                 : "
    f"{WA_BOOTSTRAP['source_page']}"
)

print(
    f"GraphQL requests observed   : "
    f"{len(WA_BOOTSTRAP['captures'])}"
)

print(
    f"Selected operation          : "
    f"{WA_BOOTSTRAP.get('operation_name')}"
)


print()
print("LIVE API")
print("-" * 120)

print(
    f"GraphQL endpoint            : "
    f"{WA_GRAPHQL_URL}"
)

print(
    f"x-api-key                   : "
    f"{mask_secret(WA_API_KEY)}"
)

print(
    f"Origin                      : "
    f"{WA_GRAPHQL_HEADERS.get('origin')}"
)

print(
    f"Referer                     : "
    f"{WA_GRAPHQL_HEADERS.get('referer')}"
)


print()
print("DIRECT GRAPHQL PROBE")
print("-" * 120)

print(
    f"Response                    : "
    f"{WA_BOOTSTRAP_PROBE}"
)

print(
    f"Direct client ready         : "
    f"{WA_NETWORK_READY}"
)


print()
print("CHECKPOINT")
print("-" * 120)

print(
    f"Bootstrap file              : "
    f"{WA_BOOTSTRAP_PATH}"
)


print()
print("=" * 120)
print("CELL 2 STATUS: LIVE WORLD ATHLETICS GRAPHQL CLIENT READY")
print("=" * 120)

print()
print(
    "No athlete/result crawl has started."
)

print(
    "The next cell will define and validate the existing "
    "getSingleCompetitorResultsDate result-history operation "
    "before we perform the incremental update."
)


WORLD ATHLETICS LIVE GRAPHQL BOOTSTRAP

DISCOVERY
------------------------------------------------------------------------------------------------------------------------
Source page                 : https://worldathletics.org/
GraphQL requests observed   : 9
Selected operation          : GetTopRankings

LIVE API
------------------------------------------------------------------------------------------------------------------------
GraphQL endpoint            : https://graphql-prod-4895.edge.aws.worldathletics.org/graphql
x-api-key                   : da2-q7...bk5u
Origin                      : https://worldathletics.org
Referer                     : https://worldathletics.org/

DIRECT GRAPHQL PROBE
------------------------------------------------------------------------------------------------------------------------
Response                    : {'__typename': 'Query'}
Direct client ready         : True

CHECKPOINT
-------------------------------------------------------------------

In [3]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 3 — VALIDATE getSingleCompetitorResultsDate
# ==================================================================================================
#
# Purpose
# -------
# Validate the previously proven World Athletics athlete/year results operation
# against the LIVE endpoint discovered in Cell 2.
#
# This cell:
#
#   1. Defines the proven getSingleCompetitorResultsDate query.
#   2. Creates fetch_athlete_year_results().
#   3. Tests one athlete/year only.
#   4. Validates the response structure.
#   5. Saves the raw test response.
#
#
# IMPORTANT
# ---------
# This is still NOT the bulk incremental crawl.
#
# The operation is year-scoped:
#
#       athlete ID + year
#
# and returns:
#
#       resultsByDate
#
# as the collection for that athlete/year.
#
# ==================================================================================================

from __future__ import annotations

import json

import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "CONFIG",
    "RUN_ID",
    "RUN_RAW_DIR",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",
    "WA_NETWORK_READY",
    "wa_graphql",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–2 first. Missing:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_globals
        )
    )


if not WA_NETWORK_READY:

    raise RuntimeError(
        "World Athletics GraphQL client is not ready."
    )


# ==================================================================================================
# PART B — PROVEN ATHLETE-YEAR RESULTS QUERY
# ==================================================================================================
#
# This is the operation previously validated in the historical extraction work.
#
# Notes
# -----
# - resultsByYear restricts the response to one year.
# - resultsByYearOrderBy="date" requests chronological ordering.
# - activeYears is useful later for profile/history inspection.
#
# ==================================================================================================

ATHLETE_YEAR_RESULTS_QUERY = """
query GetSingleCompetitorResultsDate(
    $id: Int!,
    $year: Int!
) {
    getSingleCompetitorResultsDate(
        id: $id,
        resultsByYear: $year,
        resultsByYearOrderBy: "date"
    ) {
        activeYears

        resultsByDate {
            date
            competition
            competitionId
            venue
            indoor

            discipline
            disciplineCode
            disciplineNameUrlSlug
            typeNameUrlSlug

            country
            category
            race
            place

            mark
            wind
            notLegal
            resultScore
            remark

            eventId
        }
    }
}
"""


# ==================================================================================================
# PART C — FETCH HELPER
# ==================================================================================================

async def fetch_athlete_year_results(
    athlete_id,
    year,
):
    """
    Fetch all World Athletics results returned for one athlete/year.

    Parameters
    ----------
    athlete_id : int
        World Athletics aaAthleteId.

    year : int
        Calendar year.

    Returns
    -------
    dict

        {
            "athlete_id": ...,
            "year": ...,
            "active_years": [...],
            "results": [...],
            "raw": {...}
        }
    """

    athlete_id = int(
        athlete_id
    )

    year = int(
        year
    )


    data = await wa_graphql(

        query=
            ATHLETE_YEAR_RESULTS_QUERY,

        variables={
            "id":
                athlete_id,

            "year":
                year,
        },

        operation_name=
            "GetSingleCompetitorResultsDate",
    )


    if not isinstance(
        data,
        dict,
    ):

        raise RuntimeError(
            "World Athletics returned an unexpected GraphQL data object."
        )


    payload = data.get(
        "getSingleCompetitorResultsDate"
    )


    if payload is None:

        raise RuntimeError(
            "GraphQL response did not contain "
            "'getSingleCompetitorResultsDate'."
        )


    active_years = (
        payload.get(
            "activeYears"
        )
        or []
    )


    results = (
        payload.get(
            "resultsByDate"
        )
        or []
    )


    if not isinstance(
        results,
        list,
    ):

        raise RuntimeError(
            "'resultsByDate' is not a list."
        )


    return {
        "athlete_id":
            athlete_id,

        "year":
            year,

        "active_years":
            active_years,

        "results":
            results,

        "raw":
            payload,
    }


# ==================================================================================================
# PART D — ONE LIVE VALIDATION CASE
# ==================================================================================================
#
# 15191874 was used in the earlier captured live athlete-results operation.
#
# We first request 2026.
#
# If that year happens to contain no results, but activeYears are returned,
# the cell automatically retries the latest active year.
#
# ==================================================================================================

TEST_ATHLETE_ID = 15191874

TEST_YEAR = 2026


ATHLETE_YEAR_TEST = await fetch_athlete_year_results(

    athlete_id=
        TEST_ATHLETE_ID,

    year=
        TEST_YEAR,
)


# --------------------------------------------------------------------------------------------------
# Fallback only if the chosen test year happens to be empty
# --------------------------------------------------------------------------------------------------

if (
    len(
        ATHLETE_YEAR_TEST[
            "results"
        ]
    )
    == 0
):

    active_years_numeric = sorted(

        {
            int(year)

            for year
            in ATHLETE_YEAR_TEST[
                "active_years"
            ]

            if str(
                year
            ).isdigit()
        }
    )


    if active_years_numeric:

        fallback_year = max(
            active_years_numeric
        )


        if (
            fallback_year
            != TEST_YEAR
        ):

            TEST_YEAR = (
                fallback_year
            )


            ATHLETE_YEAR_TEST = (
                await fetch_athlete_year_results(

                    athlete_id=
                        TEST_ATHLETE_ID,

                    year=
                        TEST_YEAR,
                )
            )


# ==================================================================================================
# PART E — VALIDATE RESPONSE STRUCTURE
# ==================================================================================================

EXPECTED_RESULT_FIELDS = [

    "date",

    "competition",

    "competitionId",

    "venue",

    "indoor",

    "discipline",

    "disciplineCode",

    "disciplineNameUrlSlug",

    "typeNameUrlSlug",

    "country",

    "category",

    "race",

    "place",

    "mark",

    "wind",

    "notLegal",

    "resultScore",

    "remark",

    "eventId",
]


ATHLETE_YEAR_RESULTS = (
    ATHLETE_YEAR_TEST[
        "results"
    ]
)


if len(
    ATHLETE_YEAR_RESULTS
) == 0:

    raise RuntimeError(
        "The live operation executed successfully but returned no "
        "test results. Choose another known athlete/year before "
        "starting the bulk crawl."
    )


first_result = (
    ATHLETE_YEAR_RESULTS[
        0
    ]
)


if not isinstance(
    first_result,
    dict,
):

    raise RuntimeError(
        "First resultsByDate element is not a dictionary."
    )


RESULT_FIELDS_RETURNED = list(
    first_result.keys()
)


MISSING_EXPECTED_FIELDS = [

    field

    for field
    in EXPECTED_RESULT_FIELDS

    if field
    not in first_result
]


if MISSING_EXPECTED_FIELDS:

    raise RuntimeError(
        "Live World Athletics results schema is missing expected fields:\n"
        + "\n".join(
            f"  - {field}"

            for field
            in MISSING_EXPECTED_FIELDS
        )
    )


# ==================================================================================================
# PART F — CONVERT TEST RESPONSE TO DATAFRAME FOR INSPECTION
# ==================================================================================================

ATHLETE_YEAR_TEST_DF = pd.DataFrame(
    ATHLETE_YEAR_RESULTS
)


if "date" in ATHLETE_YEAR_TEST_DF.columns:

    ATHLETE_YEAR_TEST_DF[
        "date_parsed"
    ] = pd.to_datetime(

        ATHLETE_YEAR_TEST_DF[
            "date"
        ],

        errors="coerce",
    )


# ==================================================================================================
# PART G — JAVELIN IDENTIFICATION DIAGNOSTIC
# ==================================================================================================
#
# We do NOT filter anything permanently yet.
#
# This simply shows which records from the test athlete/year look like
# javelin performances according to the WA discipline fields.
#
# ==================================================================================================

def looks_like_javelin_result(
    row,
):
    """
    Best-effort source-level javelin identification.

    Permanent filtering rules are defined in the next extraction/normalization
    cells. This helper is diagnostic only.
    """

    candidate_fields = [

        row.get(
            "discipline"
        ),

        row.get(
            "disciplineNameUrlSlug"
        ),

        row.get(
            "typeNameUrlSlug"
        ),

        row.get(
            "disciplineCode"
        ),
    ]


    text = " ".join(

        str(value)

        for value
        in candidate_fields

        if value is not None
    ).lower()


    return (
        "javelin"
        in text
    )


ATHLETE_YEAR_TEST_DF[
    "looks_like_javelin"
] = (

    ATHLETE_YEAR_TEST_DF.apply(

        looks_like_javelin_result,

        axis=1,
    )
)


# ==================================================================================================
# PART H — CHECK SAME-DATE MULTIPLE RESULTS
# ==================================================================================================
#
# We explicitly retain these later.
#
# This diagnostic confirms whether the test response contains cases where
# multiple result records share the same date.
# ==================================================================================================

if (
    "date_parsed"
    in ATHLETE_YEAR_TEST_DF.columns
):

    same_date_counts = (

        ATHLETE_YEAR_TEST_DF.groupby(
            "date_parsed",
            dropna=False,
        )
        .size()
        .reset_index(
            name="n_results"
        )
    )


    SAME_DATE_MULTIPLE_RESULTS = (

        same_date_counts.loc[
            same_date_counts[
                "n_results"
            ]
            > 1
        ]
        .copy()
    )


else:

    SAME_DATE_MULTIPLE_RESULTS = pd.DataFrame()


# ==================================================================================================
# PART I — SAVE RAW TEST RESPONSE
# ==================================================================================================

ATHLETE_YEAR_TEST_RAW_PATH = (

    RUN_RAW_DIR
    / "athletes"
    / (
        f"validation_athlete_{TEST_ATHLETE_ID}"
        f"_year_{TEST_YEAR}.json"
    )
)


with ATHLETE_YEAR_TEST_RAW_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(

        ATHLETE_YEAR_TEST,

        f,

        indent=2,

        default=str,
    )


# ==================================================================================================
# PART J — UPDATE RUN MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "athlete_year_operation_validation"
] = {

    "status":
        "VALIDATED",

    "operation":
        "getSingleCompetitorResultsDate",

    "test_athlete_id":
        TEST_ATHLETE_ID,

    "test_year":
        TEST_YEAR,

    "n_results":
        len(
            ATHLETE_YEAR_TEST_DF
        ),

    "n_javelin_like_results":
        int(
            ATHLETE_YEAR_TEST_DF[
                "looks_like_javelin"
            ]
            .sum()
        ),

    "active_years":
        ATHLETE_YEAR_TEST[
            "active_years"
        ],

    "raw_test_path":
        str(
            ATHLETE_YEAR_TEST_RAW_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = "ATHLETE_YEAR_OPERATION_READY"


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART K — OUTPUT
# ==================================================================================================

print()
print("=" * 120)
print("WORLD ATHLETICS ATHLETE-YEAR RESULTS OPERATION")
print("=" * 120)


print()
print("LIVE VALIDATION")
print("-" * 120)

print(
    f"Athlete ID                  : "
    f"{TEST_ATHLETE_ID}"
)

print(
    f"Year                        : "
    f"{TEST_YEAR}"
)

print(
    f"Active years returned       : "
    f"{ATHLETE_YEAR_TEST['active_years']}"
)

print(
    f"Results returned            : "
    f"{len(ATHLETE_YEAR_TEST_DF):,}"
)

print(
    f"Javelin-like results        : "
    f"{int(ATHLETE_YEAR_TEST_DF['looks_like_javelin'].sum()):,}"
)


print()
print("SCHEMA")
print("-" * 120)

print(
    f"Expected fields             : "
    f"{len(EXPECTED_RESULT_FIELDS)}"
)

print(
    f"Fields returned             : "
    f"{len(RESULT_FIELDS_RETURNED)}"
)

print(
    f"Missing expected fields     : "
    f"{MISSING_EXPECTED_FIELDS}"
)


print()
print("RAW CHECKPOINT")
print("-" * 120)

print(
    f"Saved to                    : "
    f"{ATHLETE_YEAR_TEST_RAW_PATH}"
)


print()
print("FIRST RESULTS")
print("-" * 120)

display(
    ATHLETE_YEAR_TEST_DF.head(
        10
    )
)


print()
print("SAME-DATE MULTIPLE RESULTS")
print("-" * 120)

if len(
    SAME_DATE_MULTIPLE_RESULTS
) > 0:

    display(
        SAME_DATE_MULTIPLE_RESULTS.head(
            20
        )
    )

else:

    print(
        "No same-date multiple results in this validation case."
    )


print()
print("=" * 120)
print("CELL 3 STATUS: getSingleCompetitorResultsDate VALIDATED")
print("=" * 120)

print()
print(
    "The next cell will identify the athlete IDs that actually need "
    "to be refreshed for the incremental window, so we do not "
    "redownload every historical athlete."
)


WORLD ATHLETICS ATHLETE-YEAR RESULTS OPERATION

LIVE VALIDATION
------------------------------------------------------------------------------------------------------------------------
Athlete ID                  : 15191874
Year                        : 2026
Active years returned       : ['2026', '2025', '2024']
Results returned            : 3
Javelin-like results        : 3

SCHEMA
------------------------------------------------------------------------------------------------------------------------
Expected fields             : 19
Fields returned             : 19
Missing expected fields     : []

RAW CHECKPOINT
------------------------------------------------------------------------------------------------------------------------
Saved to                    : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/raw/20260919_225312/athletes/validation_athlete_15191874_year_2026.json

FIRST RESULTS
---------------------------------------------------------

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_3760/2531914472.py:478: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ] = pd.to_datetime(


,date,competition,competitionId,venue,indoor,discipline,disciplineCode,disciplineNameUrlSlug,typeNameUrlSlug,country,...,race,place,mark,wind,notLegal,resultScore,remark,eventId,date_parsed,looks_like_javelin
0,07 FEB 2026,"SA All Comers Meet 2, Home of Athletics, Singa...",7236966,"Home of Athletics, Singapore (SGP)",None,Javelin Throw,None,None,None,SGP,...,F,2.,57.20,None,False,778,,10229636,2026-02-07,True
1,07 MAR 2026,"SA All Comers Meet 3, Home of Athletics, Singa...",7236967,"Home of Athletics, Singapore (SGP)",None,Javelin Throw,None,None,None,SGP,...,F,1.,57.64,None,False,784,,10229636,2026-03-07,True
2,15 APR 2026,"86th Singaporean Championships, The Kallang - ...",7240277,"The Kallang - Singapore Sports Hub, Singapore ...",None,Javelin Throw,None,None,None,SGP,...,F,3.,57.56,None,False,783,,10229636,2026-04-15,True



SAME-DATE MULTIPLE RESULTS
------------------------------------------------------------------------------------------------------------------------
No same-date multiple results in this validation case.

CELL 3 STATUS: getSingleCompetitorResultsDate VALIDATED

The next cell will identify the athlete IDs that actually need to be refreshed for the incremental window, so we do not redownload every historical athlete.


In [4]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 4 — DISCOVER COMPETITIONS IN THE INCREMENTAL WINDOW
# ==================================================================================================
#
# Purpose
# -------
# Retrieve the COMPLETE World Athletics competition list covering:
#
#       DOWNLOAD_START_DATE
#           →
#       DOWNLOAD_END_DATE
#
# This cell:
#
#   1. Calls getCalendarEvents.
#   2. Handles offset/limit pagination.
#   3. Keeps only competitions that potentially expose results.
#   4. Saves every raw calendar page.
#   5. Saves a consolidated competition discovery table.
#
#
# IMPORTANT
# ---------
# This cell does NOT download competition results yet.
#
# It only discovers the competitions that Cell 5 must inspect.
#
# ==================================================================================================

from __future__ import annotations

import json
import asyncio

from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "RUN_ID",
    "RUN_CALENDAR_DIR",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",
    "DOWNLOAD_START_DATE",
    "DOWNLOAD_END_DATE",
    "WA_NETWORK_READY",
    "wa_graphql",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–3 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not WA_NETWORK_READY:

    raise RuntimeError(
        "World Athletics GraphQL client is not ready."
    )


# ==================================================================================================
# PART B — CALENDAR QUERY
# ==================================================================================================

CALENDAR_EVENTS_QUERY = """
query getCalendarEvents(
    $startDate: String,
    $endDate: String,
    $query: String,
    $regionType: String,
    $regionId: Int,
    $currentSeason: Boolean,
    $disciplineId: Int,
    $rankingCategoryId: Int,
    $permitLevelId: Int,
    $competitionGroupId: Int,
    $competitionSubgroupId: Int,
    $competitionGroupSlug: String,
    $limit: Int,
    $offset: Int,
    $showOptionsWithNoHits: Boolean,
    $hideCompetitionsWithNoResults: Boolean,
    $orderDirection: OrderDirectionEnum
) {
    getCalendarEvents(
        startDate: $startDate,
        endDate: $endDate,
        query: $query,
        regionType: $regionType,
        regionId: $regionId,
        currentSeason: $currentSeason,
        disciplineId: $disciplineId,
        rankingCategoryId: $rankingCategoryId,
        permitLevelId: $permitLevelId,
        competitionGroupId: $competitionGroupId,
        competitionSubgroupId: $competitionSubgroupId,
        competitionGroupSlug: $competitionGroupSlug,
        limit: $limit,
        offset: $offset,
        showOptionsWithNoHits: $showOptionsWithNoHits,
        hideCompetitionsWithNoResults: $hideCompetitionsWithNoResults,
        orderDirection: $orderDirection
    ) {
        hits
        paginationPage
        defaultOffset

        parameters {
            startDate
            endDate
            query
            regionType
            regionId
            disciplineId
            rankingCategoryId
            permitLevelId
            competitionGroupId
            competitionSubgroupId
            limit
            offset
            showOptionsWithNoHits
            hideCompetitionsWithNoResults
        }

        results {
            id
            iaafId

            hasResults
            hasApiResults
            hasStartlist

            name
            venue
            area

            rankingCategory
            disciplines

            competitionGroup
            competitionSubgroup

            startDate
            endDate
            dateRange

            hasCompetitionInformation

            undeterminedCompetitionPeriod {
                status
                label
                remark
            }

            season
            wasUrl
        }
    }
}
"""


# ==================================================================================================
# PART C — DISCOVERY SETTINGS
# ==================================================================================================

ENABLE_CALENDAR_DISCOVERY = True


CALENDAR_PAGE_SIZE = 100


CALENDAR_START_DATE = (
    pd.Timestamp(
        DOWNLOAD_START_DATE
    )
    .date()
    .isoformat()
)


CALENDAR_END_DATE = (
    pd.Timestamp(
        DOWNLOAD_END_DATE
    )
    .date()
    .isoformat()
)


# ==================================================================================================
# PART D — ONE CALENDAR PAGE
# ==================================================================================================

async def fetch_calendar_page(
    start_date,
    end_date,
    *,
    limit=100,
    offset=0,
):
    """
    Fetch one World Athletics calendar page.
    """

    variables = {

        "startDate":
            str(
                start_date
            ),

        "endDate":
            str(
                end_date
            ),

        "query":
            None,

        "regionType":
            None,

        "regionId":
            None,

        "currentSeason":
            None,

        "disciplineId":
            None,

        "rankingCategoryId":
            None,

        "permitLevelId":
            None,

        "competitionGroupId":
            None,

        "competitionSubgroupId":
            None,

        "competitionGroupSlug":
            None,

        "limit":
            int(
                limit
            ),

        "offset":
            int(
                offset
            ),

        "showOptionsWithNoHits":
            False,

        # We only need competitions with published results for
        # incremental result discovery.
        "hideCompetitionsWithNoResults":
            True,

        "orderDirection":
            "Ascending",
    }


    data = await wa_graphql(

        query=
            CALENDAR_EVENTS_QUERY,

        variables=
            variables,

        operation_name=
            "getCalendarEvents",
    )


    if not isinstance(
        data,
        dict,
    ):

        raise RuntimeError(
            "Calendar query returned an unexpected GraphQL data object."
        )


    payload = data.get(
        "getCalendarEvents"
    )


    if payload is None:

        raise RuntimeError(
            "GraphQL response did not contain 'getCalendarEvents'."
        )


    results = (
        payload.get(
            "results"
        )
        or []
    )


    if not isinstance(
        results,
        list,
    ):

        raise RuntimeError(
            "getCalendarEvents.results is not a list."
        )


    return {
        "hits":
            int(
                payload.get(
                    "hits"
                )
                or 0
            ),

        "paginationPage":
            payload.get(
                "paginationPage"
            ),

        "defaultOffset":
            payload.get(
                "defaultOffset"
            ),

        "parameters":
            payload.get(
                "parameters"
            ),

        "results":
            results,
    }


# ==================================================================================================
# PART E — FETCH ALL CALENDAR PAGES
# ==================================================================================================

async def fetch_all_calendar_competitions(
    start_date,
    end_date,
    *,
    page_size=100,
):
    """
    Fetch every World Athletics calendar competition in the requested window.

    Raw page responses are checkpointed individually.
    """

    all_results = []

    page_summaries = []

    offset = 0

    page_number = 1

    expected_hits = None


    while True:

        page = await fetch_calendar_page(

            start_date=
                start_date,

            end_date=
                end_date,

            limit=
                page_size,

            offset=
                offset,
        )


        if expected_hits is None:

            expected_hits = int(
                page[
                    "hits"
                ]
            )


        results = (
            page[
                "results"
            ]
        )


        # ------------------------------------------------------------------------------------------
        # Save raw page immediately
        # ------------------------------------------------------------------------------------------

        raw_page_path = (

            RUN_CALENDAR_DIR
            / (
                f"calendar_"
                f"{start_date}_"
                f"{end_date}_"
                f"offset_{offset:06d}.json"
            )
        )


        with raw_page_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(

                page,

                f,

                indent=2,

                default=str,
            )


        page_summaries.append(
            {
                "page_number":
                    page_number,

                "offset":
                    offset,

                "n_results":
                    len(
                        results
                    ),

                "hits":
                    page[
                        "hits"
                    ],

                "raw_path":
                    str(
                        raw_page_path
                    ),
            }
        )


        all_results.extend(
            results
        )


        # ------------------------------------------------------------------------------------------
        # Pagination termination
        # ------------------------------------------------------------------------------------------

        if len(
            results
        ) == 0:

            break


        if len(
            all_results
        ) >= expected_hits:

            break


        if len(
            results
        ) < page_size:

            break


        offset += int(
            page_size
        )


        page_number += 1


    return {

        "start_date":
            start_date,

        "end_date":
            end_date,

        "expected_hits":
            expected_hits,

        "results":
            all_results,

        "pages":
            page_summaries,
    }


# ==================================================================================================
# PART F — RUN CALENDAR DISCOVERY
# ==================================================================================================

if not ENABLE_CALENDAR_DISCOVERY:

    raise RuntimeError(
        "ENABLE_CALENDAR_DISCOVERY is False."
    )


CALENDAR_DISCOVERY = await fetch_all_calendar_competitions(

    start_date=
        CALENDAR_START_DATE,

    end_date=
        CALENDAR_END_DATE,

    page_size=
        CALENDAR_PAGE_SIZE,
)


CALENDAR_RAW_RESULTS = (
    CALENDAR_DISCOVERY[
        "results"
    ]
)


# ==================================================================================================
# PART G — BUILD DISCOVERY TABLE
# ==================================================================================================

CALENDAR_COMPETITIONS = pd.DataFrame(
    CALENDAR_RAW_RESULTS
)


if len(
    CALENDAR_COMPETITIONS
) == 0:

    raise RuntimeError(
        "World Athletics calendar returned no competitions "
        "for the incremental window."
    )


# --------------------------------------------------------------------------------------------------
# Normalize competition ID
# --------------------------------------------------------------------------------------------------

if "id" not in CALENDAR_COMPETITIONS.columns:

    raise RuntimeError(
        "Calendar result does not contain competition id."
    )


CALENDAR_COMPETITIONS[
    "competition_id"
] = pd.to_numeric(

    CALENDAR_COMPETITIONS[
        "id"
    ],

    errors="coerce",
).astype(
    "Int64"
)


# --------------------------------------------------------------------------------------------------
# Parse calendar dates
# --------------------------------------------------------------------------------------------------

for column in [
    "startDate",
    "endDate",
]:

    if column in CALENDAR_COMPETITIONS.columns:

        CALENDAR_COMPETITIONS[
            f"{column}_dt"
        ] = pd.to_datetime(

            CALENDAR_COMPETITIONS[
                column
            ],

            errors="coerce",
        )


# ==================================================================================================
# PART H — REMOVE INVALID / DUPLICATE COMPETITION IDs
# ==================================================================================================

CALENDAR_COMPETITIONS = (

    CALENDAR_COMPETITIONS.dropna(
        subset=[
            "competition_id"
        ]
    )
    .copy()
)


CALENDAR_DUPLICATE_IDS = (

    CALENDAR_COMPETITIONS.loc[
        CALENDAR_COMPETITIONS[
            "competition_id"
        ]
        .duplicated(
            keep=False
        )
    ]
    .sort_values(
        "competition_id"
    )
    .copy()
)


CALENDAR_COMPETITIONS = (

    CALENDAR_COMPETITIONS.drop_duplicates(
        subset=[
            "competition_id"
        ],
        keep="last",
    )
    .sort_values(
        [
            "startDate_dt",
            "competition_id",
        ],
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART I — RESULT AVAILABILITY
# ==================================================================================================

for column in [
    "hasResults",
    "hasApiResults",
]:

    if column not in CALENDAR_COMPETITIONS.columns:

        CALENDAR_COMPETITIONS[
            column
        ] = False


    CALENDAR_COMPETITIONS[
        column
    ] = (

        CALENDAR_COMPETITIONS[
            column
        ]
        .fillna(
            False
        )
        .astype(
            bool
        )
    )


# We keep a broad candidate list.
#
# Some competitions expose calendar results even if the metadata flags are
# imperfect, therefore either result flag is sufficient here.
CALENDAR_COMPETITIONS[
    "candidate_for_result_scan"
] = (

    CALENDAR_COMPETITIONS[
        "hasResults"
    ]

    |
    CALENDAR_COMPETITIONS[
        "hasApiResults"
    ]
)


COMPETITIONS_TO_SCAN = (

    CALENDAR_COMPETITIONS.loc[
        CALENDAR_COMPETITIONS[
            "candidate_for_result_scan"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART J — QA
# ==================================================================================================

CALENDAR_EXPECTED_HITS = int(
    CALENDAR_DISCOVERY[
        "expected_hits"
    ]
    or 0
)


CALENDAR_FETCHED_ROWS = int(
    len(
        CALENDAR_RAW_RESULTS
    )
)


CALENDAR_UNIQUE_COMPETITIONS = int(
    len(
        CALENDAR_COMPETITIONS
    )
)


CALENDAR_SCAN_COUNT = int(
    len(
        COMPETITIONS_TO_SCAN
    )
)


CALENDAR_PAGINATION_COMPLETE = bool(

    CALENDAR_FETCHED_ROWS
    >= CALENDAR_EXPECTED_HITS
)


if not CALENDAR_PAGINATION_COMPLETE:

    raise RuntimeError(
        "Calendar pagination appears incomplete.\n\n"
        f"Expected hits : {CALENDAR_EXPECTED_HITS}\n"
        f"Fetched rows  : {CALENDAR_FETCHED_ROWS}"
    )


# ==================================================================================================
# PART K — SAVE CONSOLIDATED CHECKPOINTS
# ==================================================================================================

CALENDAR_TABLE_PATH = (

    RUN_CHECKPOINT_DIR
    / "calendar_competitions.parquet"
)


COMPETITIONS_TO_SCAN_PATH = (

    RUN_CHECKPOINT_DIR
    / "competitions_to_scan.parquet"
)


CALENDAR_PAGE_SUMMARY_PATH = (

    RUN_CHECKPOINT_DIR
    / "calendar_page_summary.parquet"
)


CALENDAR_COMPETITIONS.to_parquet(

    CALENDAR_TABLE_PATH,

    index=False,
)


COMPETITIONS_TO_SCAN.to_parquet(

    COMPETITIONS_TO_SCAN_PATH,

    index=False,
)


pd.DataFrame(
    CALENDAR_DISCOVERY[
        "pages"
    ]
).to_parquet(

    CALENDAR_PAGE_SUMMARY_PATH,

    index=False,
)


# ==================================================================================================
# PART L — UPDATE RUN MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "calendar_discovery"
] = {

    "status":
        "VALIDATED",

    "operation":
        "getCalendarEvents",

    "start_date":
        CALENDAR_START_DATE,

    "end_date":
        CALENDAR_END_DATE,

    "expected_hits":
        CALENDAR_EXPECTED_HITS,

    "rows_fetched":
        CALENDAR_FETCHED_ROWS,

    "unique_competitions":
        CALENDAR_UNIQUE_COMPETITIONS,

    "competitions_to_scan":
        CALENDAR_SCAN_COUNT,

    "n_pages":
        len(
            CALENDAR_DISCOVERY[
                "pages"
            ]
        ),

    "calendar_table_path":
        str(
            CALENDAR_TABLE_PATH
        ),

    "competitions_to_scan_path":
        str(
            COMPETITIONS_TO_SCAN_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = "CALENDAR_DISCOVERY_READY"


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART M — DISPLAY COLUMNS
# ==================================================================================================

preferred_display_columns = [

    "competition_id",

    "name",

    "venue",

    "area",

    "startDate",

    "endDate",

    "rankingCategory",

    "disciplines",

    "competitionGroup",

    "competitionSubgroup",

    "hasResults",

    "hasApiResults",
]


display_columns = [

    column

    for column
    in preferred_display_columns

    if column
    in COMPETITIONS_TO_SCAN.columns
]


# ==================================================================================================
# PART N — OUTPUT
# ==================================================================================================

print()
print("=" * 120)
print("WORLD ATHLETICS INCREMENTAL CALENDAR DISCOVERY")
print("=" * 120)


print()
print("DATE WINDOW")
print("-" * 120)

print(
    f"Start                       : "
    f"{CALENDAR_START_DATE}"
)

print(
    f"End                         : "
    f"{CALENDAR_END_DATE}"
)


print()
print("PAGINATION")
print("-" * 120)

print(
    f"WA reported hits            : "
    f"{CALENDAR_EXPECTED_HITS:,}"
)

print(
    f"Rows fetched                : "
    f"{CALENDAR_FETCHED_ROWS:,}"
)

print(
    f"Pages fetched               : "
    f"{len(CALENDAR_DISCOVERY['pages']):,}"
)

print(
    f"Pagination complete         : "
    f"{CALENDAR_PAGINATION_COMPLETE}"
)


print()
print("COMPETITIONS")
print("-" * 120)

print(
    f"Unique competitions         : "
    f"{CALENDAR_UNIQUE_COMPETITIONS:,}"
)

print(
    f"Duplicate ID rows           : "
    f"{len(CALENDAR_DUPLICATE_IDS):,}"
)

print(
    f"With hasResults             : "
    f"{int(CALENDAR_COMPETITIONS['hasResults'].sum()):,}"
)

print(
    f"With hasApiResults          : "
    f"{int(CALENDAR_COMPETITIONS['hasApiResults'].sum()):,}"
)

print(
    f"Selected for result scan    : "
    f"{CALENDAR_SCAN_COUNT:,}"
)


print()
print("CHECKPOINTS")
print("-" * 120)

print(
    f"Calendar table              : "
    f"{CALENDAR_TABLE_PATH}"
)

print(
    f"Result-scan list            : "
    f"{COMPETITIONS_TO_SCAN_PATH}"
)


print()
print("FIRST COMPETITIONS TO SCAN")
print("-" * 120)

display(
    COMPETITIONS_TO_SCAN[
        display_columns
    ]
    .head(
        20
    )
)


print()
print("=" * 120)
print("CELL 4 STATUS: INCREMENTAL COMPETITION LIST READY")
print("=" * 120)

print()
print(
    "No individual competition results have been downloaded yet."
)

print(
    "Cell 5 will inspect these competitions for javelin events, "
    "preserve same-day/race-level results, and collect only "
    "profile-backed athlete candidates for refresh."
)


WORLD ATHLETICS INCREMENTAL CALENDAR DISCOVERY

DATE WINDOW
------------------------------------------------------------------------------------------------------------------------
Start                       : 2026-08-30
End                         : 2026-09-19

PAGINATION
------------------------------------------------------------------------------------------------------------------------
WA reported hits            : 264
Rows fetched                : 264
Pages fetched               : 3
Pagination complete         : True

COMPETITIONS
------------------------------------------------------------------------------------------------------------------------
Unique competitions         : 264
Duplicate ID rows           : 0
With hasResults             : 264
With hasApiResults          : 264
Selected for result scan    : 264

CHECKPOINTS
------------------------------------------------------------------------------------------------------------------------
Calendar table              : /

,competition_id,name,venue,area,startDate,endDate,rankingCategory,disciplines,competitionGroup,competitionSubgroup,hasResults,hasApiResults
0,7240769,Norwegian U20 Championships,"Bislett Stadion, Oslo (NOR)",Europe,2026-08-28,2026-08-30,F,"Race Walking, Track and Field",None,None,True,True
1,7240770,Norwegian U23 Championships,"Bislett Stadion, Oslo (NOR)",Europe,2026-08-28,2026-08-30,F,"Race Walking, Track and Field",None,None,True,True
2,7242771,Hungarian Team Championships,"Lantos Mihály Sportközpont, Budapest (HUN)",Europe,2026-08-28,2026-08-30,E,"Race Walking, Track and Field",None,None,True,True
3,7245114,I^ Meeting Int.le Atl-Etica di Prove Multiple ...,"Stadio Comunale, Pieve del Grappa (ITA)",Europe,2026-08-28,2026-08-30,F,Combined Events,None,None,True,True
4,7245170,SA Age Group Championships,"Home of Athletics, Singapore (SGP)",Asia,2026-08-28,2026-09-06,F,"Combined Events, Race Walking, Track and Field",None,None,True,True
5,7234875,Internationales Hochsprungmeeting Heilbronn,"Marktplatz, Heilbronn (GER)",Europe,2026-08-29,2026-08-30,B,Track and Field,World Athletics Continental Tour – Silver,None,True,True
6,7237754,Finnkampen,"Olympiastadion, Helsinki (FIN)",Europe,2026-08-29,2026-08-30,D,"Combined Events, Race Walking, Track and Field",Traditional International Meeting,None,True,True
7,7239162,4J Studios scottishathletics National Senior &...,"Aberdeen Sports Village, Aberdeen (GBR)",Europe,2026-08-29,2026-08-30,F,Track and Field,None,None,True,True
8,7239383,Czech U23 Championships,"Stadion Míru, Tábor (CZE)",Europe,2026-08-29,2026-08-30,E,Track and Field,None,None,True,True
9,7240012,Keistad Meerkampen,"Atletiekbaan Altis, Amersfoort (NED)",Europe,2026-08-29,2026-08-30,F,Combined Events,None,None,True,True



CELL 4 STATUS: INCREMENTAL COMPETITION LIST READY

No individual competition results have been downloaded yet.
Cell 5 will inspect these competitions for javelin events, preserve same-day/race-level results, and collect only profile-backed athlete candidates for refresh.


In [5]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 5 — VALIDATE COMPETITION RESULT TREE + JAVELIN ATHLETE DISCOVERY
# ==================================================================================================
#
# Purpose
# -------
# Validate the live getCalendarCompetitionResults operation before scanning
# all 264 incremental competitions.
#
# This cell:
#
#   1. Introspects the LIVE GraphQL schema.
#   2. Finds getCalendarCompetitionResults and its exact argument types.
#   3. Dynamically constructs a safe nested result query.
#   4. Tests one known Track & Field competition.
#   5. Identifies javelin event branches.
#   6. Extracts profile-backed athlete candidates only.
#   7. Preserves no-profile competitors separately for audit.
#   8. Saves the raw response and validation tables.
#
#
# IMPORTANT
# ---------
# This is still NOT the 264-competition bulk crawl.
#
# We validate one competition first.
#
# ==================================================================================================

from __future__ import annotations

import json
import re

from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "RUN_ID",
    "RUN_RAW_DIR",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",
    "COMPETITIONS_TO_SCAN",
    "WA_NETWORK_READY",
    "wa_graphql",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–4 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not WA_NETWORK_READY:

    raise RuntimeError(
        "World Athletics GraphQL client is not ready."
    )


# ==================================================================================================
# PART B — GRAPHQL INTROSPECTION HELPERS
# ==================================================================================================

QUERY_SCHEMA_INTROSPECTION = """
query QuerySchemaForCompetitionResults {
    __type(name: "Query") {
        fields {
            name
            args {
                name
                type {
                    kind
                    name
                    ofType {
                        kind
                        name
                        ofType {
                            kind
                            name
                            ofType {
                                kind
                                name
                            }
                        }
                    }
                }
            }
            type {
                kind
                name
                ofType {
                    kind
                    name
                    ofType {
                        kind
                        name
                        ofType {
                            kind
                            name
                        }
                    }
                }
            }
        }
    }
}
"""


TYPE_SCHEMA_INTROSPECTION = """
query IntrospectOneType($name: String!) {
    __type(name: $name) {
        kind
        name

        fields {
            name

            type {
                kind
                name
                ofType {
                    kind
                    name
                    ofType {
                        kind
                        name
                        ofType {
                            kind
                            name
                        }
                    }
                }
            }
        }
    }
}
"""


def graphql_type_to_string(
    type_info,
):
    """
    Convert nested GraphQL introspection type representation to a GraphQL
    type expression such as:

        Int
        Int!
        [String]
        [SomeType!]
    """

    if type_info is None:

        return None


    kind = type_info.get(
        "kind"
    )

    name = type_info.get(
        "name"
    )

    of_type = type_info.get(
        "ofType"
    )


    if kind == "NON_NULL":

        inner = graphql_type_to_string(
            of_type
        )

        return (
            f"{inner}!"
        )


    if kind == "LIST":

        inner = graphql_type_to_string(
            of_type
        )

        return (
            f"[{inner}]"
        )


    return name


def graphql_base_named_type(
    type_info,
):
    """
    Return the innermost named GraphQL type.
    """

    current = type_info


    while (
        current is not None
        and current.get(
            "name"
        ) is None
    ):

        current = current.get(
            "ofType"
        )


    if current is None:

        return None


    return current.get(
        "name"
    )


def graphql_base_kind(
    type_info,
):
    """
    Return the innermost GraphQL kind.
    """

    current = type_info


    while (
        current is not None
        and current.get(
            "name"
        ) is None
    ):

        current = current.get(
            "ofType"
        )


    if current is None:

        return None


    return current.get(
        "kind"
    )


# ==================================================================================================
# PART C — DISCOVER LIVE OPERATION SIGNATURE
# ==================================================================================================

query_schema_data = await wa_graphql(

    query=
        QUERY_SCHEMA_INTROSPECTION,

    variables=
        {},

    operation_name=
        "QuerySchemaForCompetitionResults",
)


query_type = (
    query_schema_data.get(
        "__type"
    )
    or {}
)


query_fields = (
    query_type.get(
        "fields"
    )
    or []
)


competition_field = next(

    (
        field

        for field
        in query_fields

        if field.get(
            "name"
        )
        == "getCalendarCompetitionResults"
    ),

    None,
)


if competition_field is None:

    raise RuntimeError(
        "The live GraphQL schema does not expose "
        "getCalendarCompetitionResults."
    )


COMPETITION_RESULTS_RETURN_TYPE = (
    graphql_base_named_type(
        competition_field[
            "type"
        ]
    )
)


COMPETITION_RESULTS_ARGUMENTS = {

    arg[
        "name"
    ]:
        graphql_type_to_string(
            arg[
                "type"
            ]
        )

    for arg
    in competition_field.get(
        "args",
        []
    )
}


print(
    "Live getCalendarCompetitionResults arguments:"
)

display(
    pd.DataFrame(
        [
            {
                "argument":
                    name,

                "graphql_type":
                    type_string,
            }

            for name, type_string
            in COMPETITION_RESULTS_ARGUMENTS.items()
        ]
    )
)


# ==================================================================================================
# PART D — LIVE TYPE CACHE
# ==================================================================================================

GRAPHQL_TYPE_CACHE = {}


async def get_graphql_type_definition(
    type_name,
):
    """
    Retrieve one GraphQL object definition and cache it.
    """

    if type_name in GRAPHQL_TYPE_CACHE:

        return GRAPHQL_TYPE_CACHE[
            type_name
        ]


    data = await wa_graphql(

        query=
            TYPE_SCHEMA_INTROSPECTION,

        variables={
            "name":
                type_name
        },

        operation_name=
            "IntrospectOneType",
    )


    definition = data.get(
        "__type"
    )


    if definition is None:

        raise RuntimeError(
            f"Could not introspect GraphQL type {type_name!r}."
        )


    GRAPHQL_TYPE_CACHE[
        type_name
    ] = definition


    return definition


# ==================================================================================================
# PART E — DYNAMIC SELECTION-SET BUILDER
# ==================================================================================================
#
# We preserve all scalar fields available at each relevant level and descend
# only through the nested structures required for competition result discovery.
#
# This makes the query resistant to harmless schema additions/removals.
# ==================================================================================================

ALLOWED_CHILD_FIELDS = {

    # Root
    "competition",
    "options",
    "eventTitles",

    # Event structure
    "events",
    "summary",
    "races",

    # Result structures
    "results",
    "startList",

    # Athlete identity
    "competitor",
    "teamMembers",

    # Competition day options
    "days",
}


async def build_dynamic_selection(
    type_name,
    *,
    depth=0,
    max_depth=8,
):
    """
    Recursively build a GraphQL selection set.

    Rules
    -----
    - Include every scalar/enum field.
    - Descend only through ALLOWED_CHILD_FIELDS.
    """

    if depth > max_depth:

        return ""


    definition = await get_graphql_type_definition(
        type_name
    )


    fields = (
        definition.get(
            "fields"
        )
        or []
    )


    selections = []


    for field in fields:

        field_name = field[
            "name"
        ]

        field_type = field[
            "type"
        ]


        base_kind = graphql_base_kind(
            field_type
        )


        base_type = graphql_base_named_type(
            field_type
        )


        # ------------------------------------------------------------------------------------------
        # Scalar / enum
        # ------------------------------------------------------------------------------------------

        if base_kind in {
            "SCALAR",
            "ENUM",
        }:

            selections.append(
                field_name
            )

            continue


        # ------------------------------------------------------------------------------------------
        # Nested object/list
        # ------------------------------------------------------------------------------------------

        if (
            field_name
            not in ALLOWED_CHILD_FIELDS
        ):

            continue


        if base_type is None:

            continue


        child_selection = (
            await build_dynamic_selection(

                base_type,

                depth=
                    depth + 1,

                max_depth=
                    max_depth,
            )
        )


        if child_selection.strip():

            selections.append(
                (
                    f"{field_name} {{\n"
                    f"{child_selection}\n"
                    f"}}"
                )
            )


    return "\n".join(
        selections
    )


COMPETITION_RESULTS_SELECTION = (
    await build_dynamic_selection(
        COMPETITION_RESULTS_RETURN_TYPE
    )
)


if not COMPETITION_RESULTS_SELECTION.strip():

    raise RuntimeError(
        "Dynamic competition-result selection set is empty."
    )


# ==================================================================================================
# PART F — BUILD LIVE QUERY
# ==================================================================================================

required_arg_names = [
    "competitionId",
]


for required_arg in required_arg_names:

    if required_arg not in COMPETITION_RESULTS_ARGUMENTS:

        raise RuntimeError(
            "Live schema no longer exposes required argument:\n"
            f"{required_arg}"
        )


# We use only arguments that actually exist in the live schema.
query_arg_names = [

    name

    for name
    in [
        "competitionId",
        "day",
        "eventId",
    ]

    if name
    in COMPETITION_RESULTS_ARGUMENTS
]


variable_definitions = ", ".join(

    (
        f"${name}: "
        f"{COMPETITION_RESULTS_ARGUMENTS[name]}"
    )

    for name
    in query_arg_names
)


field_arguments = ", ".join(

    (
        f"{name}: ${name}"
    )

    for name
    in query_arg_names
)


COMPETITION_RESULTS_QUERY = f"""
query IncrementalCompetitionResults(
    {variable_definitions}
) {{
    getCalendarCompetitionResults(
        {field_arguments}
    ) {{
        {COMPETITION_RESULTS_SELECTION}
    }}
}}
"""


# ==================================================================================================
# PART G — TEST COMPETITION
# ==================================================================================================
#
# Prefer SA Age Group Championships because it appears in the current
# incremental calendar and is expected to contain javelin events.
#
# Otherwise fall back to the first Track & Field competition.
# ==================================================================================================

preferred_competition_id = (
    7245170
)


preferred_match = (

    COMPETITIONS_TO_SCAN.loc[
        COMPETITIONS_TO_SCAN[
            "competition_id"
        ]
        == preferred_competition_id
    ]
)


if len(
    preferred_match
) > 0:

    test_competition = (
        preferred_match.iloc[
            0
        ]
    )


else:

    track_field_mask = (

        COMPETITIONS_TO_SCAN[
            "disciplines"
        ]
        .astype(str)
        .str.contains(
            "Track and Field",
            case=False,
            na=False,
        )
    )


    if track_field_mask.any():

        test_competition = (

            COMPETITIONS_TO_SCAN.loc[
                track_field_mask
            ]
            .iloc[
                0
            ]
        )


    else:

        test_competition = (
            COMPETITIONS_TO_SCAN.iloc[
                0
            ]
        )


TEST_COMPETITION_ID = int(
    test_competition[
        "competition_id"
    ]
)


TEST_COMPETITION_NAME = str(
    test_competition.get(
        "name",
        ""
    )
)


# ==================================================================================================
# PART H — RESOLVE DAY ARGUMENT TYPE
# ==================================================================================================

def convert_day_value_for_graphql(
    day_number,
):
    """
    Match the live schema's day argument type.
    """

    day_type = (
        COMPETITION_RESULTS_ARGUMENTS.get(
            "day"
        )
    )


    if day_type is None:

        return None


    clean_type = (
        str(
            day_type
        )
        .replace(
            "!",
            "",
        )
    )


    if clean_type == "String":

        return str(
            day_number
        )


    return int(
        day_number
    )


# First call one explicit day, because earlier production work found that
# explicit-day retrieval is safer than relying on one whole-competition call.
TEST_DAY = 1


test_variables = {

    "competitionId":
        TEST_COMPETITION_ID,
}


if "day" in query_arg_names:

    test_variables[
        "day"
    ] = convert_day_value_for_graphql(
        TEST_DAY
    )


if "eventId" in query_arg_names:

    test_variables[
        "eventId"
    ] = None


# ==================================================================================================
# PART I — LIVE TEST CALL
# ==================================================================================================

test_data = await wa_graphql(

    query=
        COMPETITION_RESULTS_QUERY,

    variables=
        test_variables,

    operation_name=
        "IncrementalCompetitionResults",
)


TEST_COMPETITION_RESULT_TREE = (
    test_data.get(
        "getCalendarCompetitionResults"
    )
)


if TEST_COMPETITION_RESULT_TREE is None:

    raise RuntimeError(
        "getCalendarCompetitionResults returned null for the "
        "validation competition."
    )


# ==================================================================================================
# PART J — GENERIC TREE WALKERS
# ==================================================================================================

def walk_json_tree(
    obj,
    path="root",
):
    """
    Yield every dict/list node in a nested GraphQL response.
    """

    yield (
        path,
        obj,
    )


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            yield from walk_json_tree(

                value,

                path=
                    f"{path}.{key}",
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            yield from walk_json_tree(

                value,

                path=
                    f"{path}[{index}]",
            )


def object_contains_javelin(
    obj,
):
    """
    Determine whether an event/result branch visibly identifies itself
    as javelin.
    """

    if not isinstance(
        obj,
        dict,
    ):

        return False


    pieces = []


    for key, value in obj.items():

        key_lower = str(
            key
        ).lower()


        if any(
            token in key_lower

            for token
            in [
                "event",
                "discipline",
                "title",
                "name",
                "description",
            ]
        ):

            if isinstance(
                value,
                (
                    str,
                    int,
                    float,
                ),
            ):

                pieces.append(
                    str(
                        value
                    )
                )


    text = " ".join(
        pieces
    ).lower()


    return (
        "javelin"
        in text
    )


# ==================================================================================================
# PART K — FIND JAVELIN BRANCHES
# ==================================================================================================

JAVELIN_BRANCHES = []


for path, node in walk_json_tree(
    TEST_COMPETITION_RESULT_TREE
):

    if object_contains_javelin(
        node
    ):

        JAVELIN_BRANCHES.append(
            {
                "path":
                    path,

                "node":
                    node,
            }
        )


# ==================================================================================================
# PART L — ATHLETE PROFILE-ID EXTRACTION
# ==================================================================================================

ATHLETE_ID_PATTERNS = [

    # /athletes/country/name-12345678
    re.compile(
        r"-(\d{6,10})$"
    ),

    # /athletes/athlete=12345678
    re.compile(
        r"athlete=(\d{6,10})"
    ),

    # /athletes/_/12345678
    re.compile(
        r"/(\d{6,10})/?$"
    ),
]


def extract_wa_athlete_id_from_slug(
    value,
):
    """
    Extract a numeric World Athletics athlete ID candidate from a profile slug.

    IMPORTANT:
    Caller must separately require hasProfile=True.
    """

    if value is None:

        return None


    text = str(
        value
    ).strip()


    if not text:

        return None


    for pattern in ATHLETE_ID_PATTERNS:

        match = pattern.search(
            text
        )


        if match:

            return int(
                match.group(
                    1
                )
            )


    return None


# ==================================================================================================
# PART M — EXTRACT COMPETITORS FROM JAVELIN BRANCHES
# ==================================================================================================

javelin_competitor_rows = []


for branch in JAVELIN_BRANCHES:

    branch_path = (
        branch[
            "path"
        ]
    )


    branch_node = (
        branch[
            "node"
        ]
    )


    for path, node in walk_json_tree(
        branch_node,
        path=branch_path,
    ):

        if not isinstance(
            node,
            dict,
        ):

            continue


        # Competitor objects from this endpoint are recognizable by the
        # identity fields established in the previous extraction project.
        identity_field_count = sum(

            key in node

            for key
            in [
                "name",
                "urlSlug",
                "id",
                "iaafId",
                "birthDate",
                "hasProfile",
            ]
        )


        if identity_field_count < 2:

            continue


        # Require at least one athlete-like identity signal.
        if not any(
            key in node

            for key
            in [
                "urlSlug",
                "hasProfile",
                "birthDate",
            ]
        ):

            continue


        url_slug = node.get(
            "urlSlug"
        )


        has_profile = bool(
            node.get(
                "hasProfile",
                False,
            )
        )


        candidate_aa_id = (

            extract_wa_athlete_id_from_slug(
                url_slug
            )

            if has_profile

            else None
        )


        javelin_competitor_rows.append(
            {
                "competition_id":
                    TEST_COMPETITION_ID,

                "competition_name":
                    TEST_COMPETITION_NAME,

                "day":
                    TEST_DAY,

                "branch_path":
                    branch_path,

                "competitor_path":
                    path,

                "name":
                    node.get(
                        "name"
                    ),

                "urlSlug":
                    url_slug,

                "hasProfile":
                    has_profile,

                "candidate_aaAthleteId":
                    candidate_aa_id,

                "source_competitor_id":
                    node.get(
                        "id"
                    ),

                "iaafId":
                    node.get(
                        "iaafId"
                    ),

                "birthDate":
                    node.get(
                        "birthDate"
                    ),
            }
        )


JAVELIN_COMPETITORS = pd.DataFrame(
    javelin_competitor_rows
)


if len(
    JAVELIN_COMPETITORS
) > 0:

    JAVELIN_COMPETITORS = (

        JAVELIN_COMPETITORS.drop_duplicates(

            subset=[
                "competition_id",
                "day",
                "name",
                "urlSlug",
                "hasProfile",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ==================================================================================================
# PART N — SPLIT PROFILE CANDIDATES / NO-PROFILE AUDIT
# ==================================================================================================

if len(
    JAVELIN_COMPETITORS
) > 0:

    PROFILE_BACKED_JAVELIN_CANDIDATES = (

        JAVELIN_COMPETITORS.loc[
            (
                JAVELIN_COMPETITORS[
                    "hasProfile"
                ]
                == True
            )
            &
            (
                JAVELIN_COMPETITORS[
                    "candidate_aaAthleteId"
                ]
                .notna()
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    NO_PROFILE_JAVELIN_COMPETITORS = (

        JAVELIN_COMPETITORS.loc[
            ~(
                (
                    JAVELIN_COMPETITORS[
                        "hasProfile"
                    ]
                    == True
                )
                &
                (
                    JAVELIN_COMPETITORS[
                        "candidate_aaAthleteId"
                    ]
                    .notna()
                )
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


else:

    PROFILE_BACKED_JAVELIN_CANDIDATES = pd.DataFrame()

    NO_PROFILE_JAVELIN_COMPETITORS = pd.DataFrame()


# ==================================================================================================
# PART O — AVAILABLE COMPETITION DAYS
# ==================================================================================================

def collect_day_values(
    obj,
):
    """
    Collect possible day values from any nested 'days' field.
    """

    found = []


    for path, node in walk_json_tree(
        obj
    ):

        if not isinstance(
            node,
            dict,
        ):

            continue


        if "days" not in node:

            continue


        value = node[
            "days"
        ]


        if isinstance(
            value,
            list,
        ):

            for item in value:

                if isinstance(
                    item,
                    dict,
                ):

                    found.append(
                        item
                    )

                else:

                    found.append(
                        {
                            "value":
                                item
                        }
                    )


    return found


COMPETITION_DAY_OPTIONS = pd.DataFrame(

    collect_day_values(
        TEST_COMPETITION_RESULT_TREE
    )
)


# ==================================================================================================
# PART P — SAVE RAW + VALIDATION TABLES
# ==================================================================================================

TEST_COMPETITION_RAW_PATH = (

    RUN_RAW_DIR
    / "competitions"
    / (
        f"validation_competition_"
        f"{TEST_COMPETITION_ID}_"
        f"day_{TEST_DAY}.json"
    )
)


with TEST_COMPETITION_RAW_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(

        TEST_COMPETITION_RESULT_TREE,

        f,

        indent=2,

        default=str,
    )


JAVELIN_CANDIDATE_PATH = (

    RUN_CHECKPOINT_DIR
    / "validation_javelin_profile_candidates.parquet"
)


NO_PROFILE_PATH = (

    RUN_CHECKPOINT_DIR
    / "validation_javelin_no_profile.parquet"
)


DAY_OPTIONS_PATH = (

    RUN_CHECKPOINT_DIR
    / "validation_competition_day_options.parquet"
)


PROFILE_BACKED_JAVELIN_CANDIDATES.to_parquet(

    JAVELIN_CANDIDATE_PATH,

    index=False,
)


NO_PROFILE_JAVELIN_COMPETITORS.to_parquet(

    NO_PROFILE_PATH,

    index=False,
)


COMPETITION_DAY_OPTIONS.to_parquet(

    DAY_OPTIONS_PATH,

    index=False,
)


# ==================================================================================================
# PART Q — MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "competition_result_validation"
] = {

    "status":
        "VALIDATED",

    "operation":
        "getCalendarCompetitionResults",

    "return_type":
        COMPETITION_RESULTS_RETURN_TYPE,

    "arguments":
        COMPETITION_RESULTS_ARGUMENTS,

    "test_competition_id":
        TEST_COMPETITION_ID,

    "test_competition_name":
        TEST_COMPETITION_NAME,

    "test_day":
        TEST_DAY,

    "javelin_branches_found":
        len(
            JAVELIN_BRANCHES
        ),

    "javelin_competitors_found":
        len(
            JAVELIN_COMPETITORS
        ),

    "profile_backed_candidates":
        len(
            PROFILE_BACKED_JAVELIN_CANDIDATES
        ),

    "no_profile_competitors":
        len(
            NO_PROFILE_JAVELIN_COMPETITORS
        ),

    "raw_path":
        str(
            TEST_COMPETITION_RAW_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = "COMPETITION_RESULTS_OPERATION_READY"


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART R — OUTPUT
# ==================================================================================================

print()
print("=" * 120)
print("WORLD ATHLETICS COMPETITION RESULT TREE VALIDATION")
print("=" * 120)


print()
print("LIVE OPERATION")
print("-" * 120)

print(
    f"Return type                 : "
    f"{COMPETITION_RESULTS_RETURN_TYPE}"
)

print(
    f"Arguments                   : "
    f"{COMPETITION_RESULTS_ARGUMENTS}"
)


print()
print("TEST COMPETITION")
print("-" * 120)

print(
    f"Competition ID              : "
    f"{TEST_COMPETITION_ID}"
)

print(
    f"Competition                 : "
    f"{TEST_COMPETITION_NAME}"
)

print(
    f"Day tested                  : "
    f"{TEST_DAY}"
)


print()
print("JAVELIN DISCOVERY")
print("-" * 120)

print(
    f"Javelin branches found      : "
    f"{len(JAVELIN_BRANCHES):,}"
)

print(
    f"Competitor records found    : "
    f"{len(JAVELIN_COMPETITORS):,}"
)

print(
    f"Profile-backed candidates   : "
    f"{len(PROFILE_BACKED_JAVELIN_CANDIDATES):,}"
)

print(
    f"No-profile/audit records    : "
    f"{len(NO_PROFILE_JAVELIN_COMPETITORS):,}"
)


print()
print("COMPETITION DAY OPTIONS")
print("-" * 120)

if len(
    COMPETITION_DAY_OPTIONS
) > 0:

    display(
        COMPETITION_DAY_OPTIONS
    )

else:

    print(
        "No explicit day-option table was exposed in this day response."
    )


print()
print("PROFILE-BACKED JAVELIN CANDIDATES")
print("-" * 120)

if len(
    PROFILE_BACKED_JAVELIN_CANDIDATES
) > 0:

    display(
        PROFILE_BACKED_JAVELIN_CANDIDATES
    )

else:

    print(
        "No profile-backed javelin candidates found on this test day."
    )


print()
print("NO-PROFILE / UNRESOLVED COMPETITORS")
print("-" * 120)

if len(
    NO_PROFILE_JAVELIN_COMPETITORS
) > 0:

    display(
        NO_PROFILE_JAVELIN_COMPETITORS
    )

else:

    print(
        "None."
    )


print()
print("RAW CHECKPOINT")
print("-" * 120)

print(
    f"Saved to                    : "
    f"{TEST_COMPETITION_RAW_PATH}"
)


print()
print("=" * 120)
print("CELL 5 STATUS: COMPETITION RESULT TREE VALIDATED")
print("=" * 120)

print()
print(
    "No bulk competition crawl has started."
)

print(
    "Cell 6 will use this validated operation to scan every relevant "
    "competition/day, retain only javelin branches, and produce the "
    "incremental athlete-refresh population."
)

Live getCalendarCompetitionResults arguments:


,argument,graphql_type
0,competitionId,Int
1,day,Int
2,eventId,Int



WORLD ATHLETICS COMPETITION RESULT TREE VALIDATION

LIVE OPERATION
------------------------------------------------------------------------------------------------------------------------
Return type                 : CalendarCompetitionResults
Arguments                   : {'competitionId': 'Int', 'day': 'Int', 'eventId': 'Int'}

TEST COMPETITION
------------------------------------------------------------------------------------------------------------------------
Competition ID              : 7245170
Competition                 : SA Age Group Championships
Day tested                  : 1

JAVELIN DISCOVERY
------------------------------------------------------------------------------------------------------------------------
Javelin branches found      : 2
Competitor records found    : 0
Profile-backed candidates   : 0
No-profile/audit records    : 0

COMPETITION DAY OPTIONS
------------------------------------------------------------------------------------------------------------

,date,day
0,28 AUG 2026,1
1,29 AUG 2026,2
2,30 AUG 2026,3
3,04 SEP 2026,8
4,05 SEP 2026,9
5,06 SEP 2026,10



PROFILE-BACKED JAVELIN CANDIDATES
------------------------------------------------------------------------------------------------------------------------
No profile-backed javelin candidates found on this test day.

NO-PROFILE / UNRESOLVED COMPETITORS
------------------------------------------------------------------------------------------------------------------------
None.

RAW CHECKPOINT
------------------------------------------------------------------------------------------------------------------------
Saved to                    : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/raw/20260919_225312/competitions/validation_competition_7245170_day_1.json

CELL 5 STATUS: COMPETITION RESULT TREE VALIDATED

No bulk competition crawl has started.
Cell 6 will use this validated operation to scan every relevant competition/day, retain only javelin branches, and produce the incremental athlete-refresh population.


In [6]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 6 — SCAN ALL DAYS OF VALIDATION COMPETITION
# ==================================================================================================
#
# Purpose
# -------
# Confirm that the competition-result parser can recover real javelin
# performances and profile-backed athlete candidates.
#
# This cell:
#
#   1. Uses the actual day list discovered in Cell 5.
#   2. Downloads every result day for the validation competition.
#   3. Parses the known World Athletics hierarchy:
#
#          eventTitles
#              ↓
#          events
#              ↓
#          summary
#          races
#              ↓
#          results
#              ↓
#          competitor
#
#   4. Keeps javelin events only.
#   5. Preserves individual result rows.
#   6. Extracts profile-backed athlete candidates from urlSlug.
#   7. Keeps no-profile competitors separately for audit.
#   8. Checkpoints every raw day response.
#
#
# IMPORTANT
# ---------
# This still scans ONLY ONE competition.
#
# The global 264-competition crawl begins only after this parser is validated.
#
# ==================================================================================================

from __future__ import annotations

import json
import re

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "RUN_ID",
    "RUN_RAW_DIR",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",
    "WA_NETWORK_READY",
    "wa_graphql",
    "COMPETITION_RESULTS_QUERY",
    "query_arg_names",
    "convert_day_value_for_graphql",
    "TEST_COMPETITION_ID",
    "TEST_COMPETITION_NAME",
    "COMPETITION_DAY_OPTIONS",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–5 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not WA_NETWORK_READY:

    raise RuntimeError(
        "World Athletics GraphQL client is not ready."
    )


# ==================================================================================================
# PART B — RESOLVE VALID COMPETITION DAYS
# ==================================================================================================

if (
    len(
        COMPETITION_DAY_OPTIONS
    )
    == 0
    or
    "day"
    not in COMPETITION_DAY_OPTIONS.columns
):

    raise RuntimeError(
        "Cell 5 did not return a usable competition-day table."
    )


VALIDATION_COMPETITION_DAYS = sorted(
    {
        int(day)

        for day
        in pd.to_numeric(
            COMPETITION_DAY_OPTIONS[
                "day"
            ],
            errors="coerce",
        ).dropna()
    }
)


if len(
    VALIDATION_COMPETITION_DAYS
) == 0:

    raise RuntimeError(
        "No valid competition days were discovered."
    )


# ==================================================================================================
# PART C — WA DATE PARSER
# ==================================================================================================

def parse_wa_result_date(
    value,
):
    """
    Parse World Athletics display dates such as:

        07 FEB 2026
        15 APR 2026

    without producing pandas format-inference warnings.
    """

    if value is None:

        return pd.NaT


    text = str(
        value
    ).strip()


    if not text:

        return pd.NaT


    for fmt in [
        "%d %b %Y",
        "%d %B %Y",
        "%Y-%m-%d",
    ]:

        parsed = pd.to_datetime(
            text,
            format=fmt,
            errors="coerce",
        )


        if not pd.isna(
            parsed
        ):

            return parsed


    return pd.to_datetime(
        text,
        errors="coerce",
    )


# ==================================================================================================
# PART D — PROFILE-SLUG ATHLETE-ID CANDIDATE
# ==================================================================================================
#
# IMPORTANT:
#
# The numeric suffix is only a CANDIDATE identity.
#
# It is accepted here only when:
#
#       hasProfile == True
#
# Cell 8/9 will later confirm the canonical aaAthleteId through
# getSingleCompetitor.basicData.aaId.
#
# ==================================================================================================

ATHLETE_SLUG_ID_PATTERN = re.compile(
    r"-(\d{7,10})(?:$|[/?#])"
)


def candidate_athlete_id_from_profile(
    competitor,
):
    """
    Extract a candidate WA athlete ID from a profile-backed competitor.
    """

    if not isinstance(
        competitor,
        dict,
    ):

        return None


    has_profile = bool(
        competitor.get(
            "hasProfile",
            False,
        )
    )


    if not has_profile:

        return None


    slug = competitor.get(
        "urlSlug"
    )


    if slug is None:

        return None


    match = ATHLETE_SLUG_ID_PATTERN.search(
        str(
            slug
        )
    )


    if match is None:

        return None


    return int(
        match.group(
            1
        )
    )


# ==================================================================================================
# PART E — JAVELIN EVENT IDENTIFICATION
# ==================================================================================================

def is_javelin_event(
    event_title,
    event_name,
):
    """
    Determine whether an event branch is a javelin event.
    """

    text = " ".join(
        [
            str(
                event_title
                or ""
            ),

            str(
                event_name
                or ""
            ),
        ]
    ).lower()


    return (
        "javelin"
        in text
    )


# ==================================================================================================
# PART F — FETCH ONE COMPETITION DAY
# ==================================================================================================

async def fetch_competition_day_results(
    competition_id,
    day,
):
    """
    Fetch one explicit World Athletics competition-result day.
    """

    variables = {
        "competitionId":
            int(
                competition_id
            ),
    }


    if "day" in query_arg_names:

        variables[
            "day"
        ] = convert_day_value_for_graphql(
            day
        )


    if "eventId" in query_arg_names:

        variables[
            "eventId"
        ] = None


    data = await wa_graphql(

        query=
            COMPETITION_RESULTS_QUERY,

        variables=
            variables,

        operation_name=
            "IncrementalCompetitionResults",
    )


    payload = data.get(
        "getCalendarCompetitionResults"
    )


    if payload is None:

        raise RuntimeError(
            "getCalendarCompetitionResults returned null for:\n"
            f"competition={competition_id}, day={day}"
        )


    return payload


# ==================================================================================================
# PART G — NORMALIZE ONE COMPETITOR
# ==================================================================================================

def normalize_competitor_identity(
    competitor,
):
    """
    Extract competitor identity information without assuming that
    competitor.id or iaafId is the canonical aaAthleteId.
    """

    if not isinstance(
        competitor,
        dict,
    ):

        competitor = {}


    has_profile = bool(
        competitor.get(
            "hasProfile",
            False,
        )
    )


    candidate_id = (
        candidate_athlete_id_from_profile(
            competitor
        )
    )


    return {
        "competitor_name":
            competitor.get(
                "name"
            ),

        "competitor_urlSlug":
            competitor.get(
                "urlSlug"
            ),

        "competitor_hasProfile":
            has_profile,

        "candidate_aaAthleteId":
            candidate_id,

        "source_competitor_id":
            competitor.get(
                "id"
            ),

        "source_iaafId":
            competitor.get(
                "iaafId"
            ),

        "competitor_birthDate":
            competitor.get(
                "birthDate"
            ),
    }


# ==================================================================================================
# PART H — FLATTEN ONE COMPETITION-DAY RESULT TREE
# ==================================================================================================

def flatten_javelin_competition_day(
    payload,
    *,
    competition_id,
    competition_name,
    requested_day,
):
    """
    Flatten all javelin result rows from one competition/day response.

    Both summary-level and race-level result structures are supported.

    Returns
    -------
    list[dict]
    """

    rows = []


    event_titles = (
        payload.get(
            "eventTitles"
        )
        or []
    )


    for event_title_obj in event_titles:

        if not isinstance(
            event_title_obj,
            dict,
        ):

            continue


        event_title = (
            event_title_obj.get(
                "eventTitle"
            )
        )


        ranking_category = (
            event_title_obj.get(
                "rankingCategory"
            )
        )


        events = (
            event_title_obj.get(
                "events"
            )
            or []
        )


        for event_obj in events:

            if not isinstance(
                event_obj,
                dict,
            ):

                continue


            event_name = (
                event_obj.get(
                    "event"
                )
            )


            if not is_javelin_event(
                event_title,
                event_name,
            ):

                continue


            event_id = (
                event_obj.get(
                    "eventId"
                )
            )


            gender = (
                event_obj.get(
                    "gender"
                )
            )


            is_relay = (
                event_obj.get(
                    "isRelay"
                )
            )


            # ======================================================================================
            # 1. SUMMARY RESULTS
            # ======================================================================================

            summaries = (
                event_obj.get(
                    "summary"
                )
                or []
            )


            for summary_index, summary in enumerate(
                summaries
            ):

                if not isinstance(
                    summary,
                    dict,
                ):

                    continue


                competitor = (
                    summary.get(
                        "competitor"
                    )
                    or {}
                )


                identity = (
                    normalize_competitor_identity(
                        competitor
                    )
                )


                row = {

                    "competition_id":
                        int(
                            competition_id
                        ),

                    "competition_name":
                        competition_name,

                    "requested_day":
                        int(
                            requested_day
                        ),

                    "source_layer":
                        "summary",

                    "event_title":
                        event_title,

                    "ranking_category":
                        ranking_category,

                    "event":
                        event_name,

                    "event_id":
                        event_id,

                    "gender":
                        gender,

                    "is_relay":
                        is_relay,

                    "race":
                        None,

                    "race_id":
                        None,

                    "race_number":
                        summary.get(
                            "raceNumber"
                        ),

                    "result_id":
                        None,

                    "date_raw":
                        None,

                    "mark":
                        summary.get(
                            "mark"
                        ),

                    "nationality":
                        summary.get(
                            "nationality"
                        ),

                    "place":
                        (
                            summary.get(
                                "placeInRace"
                            )
                            if summary.get(
                                "placeInRace"
                            )
                            is not None
                            else summary.get(
                                "placeInRound"
                            )
                        ),

                    "place_in_race":
                        summary.get(
                            "placeInRace"
                        ),

                    "place_in_round":
                        summary.get(
                            "placeInRound"
                        ),

                    "points":
                        summary.get(
                            "points"
                        ),

                    "wind":
                        summary.get(
                            "wind"
                        ),

                    "qualified":
                        None,

                    "remark":
                        None,

                    "details":
                        None,

                    "records":
                        summary.get(
                            "records"
                        ),

                    "summary_index":
                        summary_index,
                }


                row.update(
                    identity
                )


                rows.append(
                    row
                )


            # ======================================================================================
            # 2. RACE / RESULT ROWS
            # ======================================================================================

            races = (
                event_obj.get(
                    "races"
                )
                or []
            )


            for race_index, race in enumerate(
                races
            ):

                if not isinstance(
                    race,
                    dict,
                ):

                    continue


                race_date = (
                    race.get(
                        "date"
                    )
                )


                race_day = (
                    race.get(
                        "day"
                    )
                )


                race_name = (
                    race.get(
                        "race"
                    )
                )


                race_id = (
                    race.get(
                        "raceId"
                    )
                )


                race_number = (
                    race.get(
                        "raceNumber"
                    )
                )


                race_wind = (
                    race.get(
                        "wind"
                    )
                )


                results = (
                    race.get(
                        "results"
                    )
                    or []
                )


                for result_index, result in enumerate(
                    results
                ):

                    if not isinstance(
                        result,
                        dict,
                    ):

                        continue


                    competitor = (
                        result.get(
                            "competitor"
                        )
                        or {}
                    )


                    identity = (
                        normalize_competitor_identity(
                            competitor
                        )
                    )


                    row = {

                        "competition_id":
                            int(
                                competition_id
                            ),

                        "competition_name":
                            competition_name,

                        "requested_day":
                            int(
                                requested_day
                            ),

                        "source_layer":
                            "race_result",

                        "event_title":
                            event_title,

                        "ranking_category":
                            ranking_category,

                        "event":
                            event_name,

                        "event_id":
                            event_id,

                        "gender":
                            gender,

                        "is_relay":
                            is_relay,

                        "race":
                            race_name,

                        "race_id":
                            race_id,

                        "race_number":
                            race_number,

                        "result_id":
                            result.get(
                                "id"
                            ),

                        "date_raw":
                            race_date,

                        "race_day":
                            race_day,

                        "mark":
                            result.get(
                                "mark"
                            ),

                        "nationality":
                            result.get(
                                "nationality"
                            ),

                        "place":
                            result.get(
                                "place"
                            ),

                        "place_in_race":
                            None,

                        "place_in_round":
                            None,

                        "points":
                            result.get(
                                "points"
                            ),

                        "wind":
                            (
                                result.get(
                                    "wind"
                                )
                                if result.get(
                                    "wind"
                                )
                                is not None
                                else race_wind
                            ),

                        "qualified":
                            result.get(
                                "qualified"
                            ),

                        "remark":
                            result.get(
                                "remark"
                            ),

                        "details":
                            result.get(
                                "details"
                            ),

                        "records":
                            result.get(
                                "records"
                            ),

                        "race_index":
                            race_index,

                        "result_index":
                            result_index,
                    }


                    row.update(
                        identity
                    )


                    rows.append(
                        row
                    )


    return rows


# ==================================================================================================
# PART I — SCAN ALL DAYS
# ==================================================================================================

VALIDATION_DAY_SUMMARIES = []

validation_rows = []


for day in VALIDATION_COMPETITION_DAYS:

    try:

        payload = await fetch_competition_day_results(

            competition_id=
                TEST_COMPETITION_ID,

            day=
                day,
        )


        # ------------------------------------------------------------------------------------------
        # Raw checkpoint
        # ------------------------------------------------------------------------------------------

        raw_path = (

            RUN_RAW_DIR
            / "competitions"
            / (
                f"validation_competition_"
                f"{TEST_COMPETITION_ID}_"
                f"day_{day}.json"
            )
        )


        with raw_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(

                payload,

                f,

                indent=2,

                default=str,
            )


        # ------------------------------------------------------------------------------------------
        # Flatten javelin result rows
        # ------------------------------------------------------------------------------------------

        day_rows = (
            flatten_javelin_competition_day(

                payload,

                competition_id=
                    TEST_COMPETITION_ID,

                competition_name=
                    TEST_COMPETITION_NAME,

                requested_day=
                    day,
            )
        )


        validation_rows.extend(
            day_rows
        )


        VALIDATION_DAY_SUMMARIES.append(
            {
                "day":
                    day,

                "status":
                    "OK",

                "javelin_rows":
                    len(
                        day_rows
                    ),

                "raw_path":
                    str(
                        raw_path
                    ),
            }
        )


    except Exception as exc:

        VALIDATION_DAY_SUMMARIES.append(
            {
                "day":
                    day,

                "status":
                    "ERROR",

                "javelin_rows":
                    0,

                "error":
                    repr(
                        exc
                    ),
            }
        )


# ==================================================================================================
# PART J — RESULT TABLE
# ==================================================================================================

VALIDATION_JAVELIN_RESULTS = pd.DataFrame(
    validation_rows
)


if len(
    VALIDATION_JAVELIN_RESULTS
) == 0:

    raise RuntimeError(
        "No javelin result rows were recovered from ANY competition day.\n"
        "The parser requires review before the global crawl begins."
    )


# --------------------------------------------------------------------------------------------------
# Parse source result date
# --------------------------------------------------------------------------------------------------

VALIDATION_JAVELIN_RESULTS[
    "result_date_dt"
] = (

    VALIDATION_JAVELIN_RESULTS[
        "date_raw"
    ]
    .map(
        parse_wa_result_date
    )
)


# ==================================================================================================
# PART K — PROFILE-BACKED ATHLETE CANDIDATES
# ==================================================================================================

PROFILE_BACKED_MASK = (

    (
        VALIDATION_JAVELIN_RESULTS[
            "competitor_hasProfile"
        ]
        == True
    )

    &
    (
        VALIDATION_JAVELIN_RESULTS[
            "candidate_aaAthleteId"
        ]
        .notna()
    )
)


VALIDATION_PROFILE_CANDIDATES = (

    VALIDATION_JAVELIN_RESULTS.loc[
        PROFILE_BACKED_MASK
    ]
    [
        [
            "candidate_aaAthleteId",
            "competitor_name",
            "competitor_urlSlug",
            "competitor_birthDate",
            "source_competitor_id",
            "source_iaafId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "candidate_aaAthleteId",
            "competitor_name",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART L — NO-PROFILE / UNRESOLVED AUDIT
# ==================================================================================================

VALIDATION_NO_PROFILE = (

    VALIDATION_JAVELIN_RESULTS.loc[
        ~PROFILE_BACKED_MASK
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART M — SAME-DATE MULTIPLE-PERFORMANCE AUDIT
# ==================================================================================================
#
# We explicitly preserve these.
#
# ==================================================================================================

profile_result_rows = (

    VALIDATION_JAVELIN_RESULTS.loc[
        PROFILE_BACKED_MASK
    ]
    .copy()
)


if len(
    profile_result_rows
) > 0:

    SAME_DATE_RESULT_COUNTS = (

        profile_result_rows.dropna(
            subset=[
                "candidate_aaAthleteId",
                "result_date_dt",
            ]
        )
        .groupby(
            [
                "candidate_aaAthleteId",
                "result_date_dt",
            ],
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size":
                    "n_rows"
            }
        )
    )


    SAME_DATE_MULTIPLE_PERFORMANCES = (

        SAME_DATE_RESULT_COUNTS.loc[
            SAME_DATE_RESULT_COUNTS[
                "n_rows"
            ]
            > 1
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


else:

    SAME_DATE_RESULT_COUNTS = pd.DataFrame()

    SAME_DATE_MULTIPLE_PERFORMANCES = pd.DataFrame()


# ==================================================================================================
# PART N — EVENT / DAY SUMMARY
# ==================================================================================================

VALIDATION_DAY_SUMMARY = pd.DataFrame(
    VALIDATION_DAY_SUMMARIES
)


VALIDATION_EVENT_SUMMARY = (

    VALIDATION_JAVELIN_RESULTS.groupby(
        [
            "requested_day",
            "event_title",
            "event",
            "gender",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        rows=(
            "mark",
            "size",
        ),

        profile_rows=(
            "competitor_hasProfile",
            lambda s:
                int(
                    pd.Series(
                        s
                    )
                    .fillna(
                        False
                    )
                    .astype(
                        bool
                    )
                    .sum()
                ),
        ),

        athletes=(
            "candidate_aaAthleteId",
            lambda s:
                int(
                    pd.Series(
                        s
                    )
                    .dropna()
                    .nunique()
                ),
        ),
    )
)


# ==================================================================================================
# PART O — QA
# ==================================================================================================

N_VALIDATION_DAYS = int(
    len(
        VALIDATION_COMPETITION_DAYS
    )
)


N_DAY_ERRORS = int(
    (
        VALIDATION_DAY_SUMMARY[
            "status"
        ]
        == "ERROR"
    ).sum()
)


N_JAVELIN_ROWS = int(
    len(
        VALIDATION_JAVELIN_RESULTS
    )
)


N_PROFILE_RESULT_ROWS = int(
    PROFILE_BACKED_MASK.sum()
)


N_PROFILE_CANDIDATES = int(
    VALIDATION_PROFILE_CANDIDATES[
        "candidate_aaAthleteId"
    ]
    .nunique()
)


N_NO_PROFILE_ROWS = int(
    len(
        VALIDATION_NO_PROFILE
    )
)


VALIDATION_PARSER_PASS = bool(

    N_DAY_ERRORS == 0

    and
    N_JAVELIN_ROWS > 0

    and
    N_PROFILE_RESULT_ROWS > 0

    and
    N_PROFILE_CANDIDATES > 0
)


# ==================================================================================================
# PART P — CHECKPOINTS
# ==================================================================================================

VALIDATION_RESULTS_PATH = (

    RUN_CHECKPOINT_DIR
    / "validation_all_days_javelin_results.parquet"
)


VALIDATION_PROFILE_PATH = (

    RUN_CHECKPOINT_DIR
    / "validation_all_days_profile_candidates.parquet"
)


VALIDATION_NO_PROFILE_PATH = (

    RUN_CHECKPOINT_DIR
    / "validation_all_days_no_profile.parquet"
)


VALIDATION_DAY_SUMMARY_PATH = (

    RUN_CHECKPOINT_DIR
    / "validation_all_days_summary.parquet"
)


VALIDATION_JAVELIN_RESULTS.to_parquet(

    VALIDATION_RESULTS_PATH,

    index=False,
)


VALIDATION_PROFILE_CANDIDATES.to_parquet(

    VALIDATION_PROFILE_PATH,

    index=False,
)


VALIDATION_NO_PROFILE.to_parquet(

    VALIDATION_NO_PROFILE_PATH,

    index=False,
)


VALIDATION_DAY_SUMMARY.to_parquet(

    VALIDATION_DAY_SUMMARY_PATH,

    index=False,
)


# ==================================================================================================
# PART Q — UPDATE MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "competition_all_day_validation"
] = {

    "status":
        (
            "VALIDATED"
            if VALIDATION_PARSER_PASS
            else "REVIEW_REQUIRED"
        ),

    "competition_id":
        TEST_COMPETITION_ID,

    "competition_name":
        TEST_COMPETITION_NAME,

    "days_requested":
        VALIDATION_COMPETITION_DAYS,

    "n_days":
        N_VALIDATION_DAYS,

    "day_errors":
        N_DAY_ERRORS,

    "javelin_rows":
        N_JAVELIN_ROWS,

    "profile_result_rows":
        N_PROFILE_RESULT_ROWS,

    "unique_profile_candidates":
        N_PROFILE_CANDIDATES,

    "no_profile_rows":
        N_NO_PROFILE_ROWS,

    "results_path":
        str(
            VALIDATION_RESULTS_PATH
        ),

    "profile_candidates_path":
        str(
            VALIDATION_PROFILE_PATH
        ),

    "no_profile_path":
        str(
            VALIDATION_NO_PROFILE_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "JAVELIN_RESULT_PARSER_READY"

    if VALIDATION_PARSER_PASS

    else "JAVELIN_RESULT_PARSER_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART R — OUTPUT
# ==================================================================================================

print()
print("=" * 120)
print("WORLD ATHLETICS JAVELIN RESULT PARSER — ALL-DAY VALIDATION")
print("=" * 120)


print()
print("COMPETITION")
print("-" * 120)

print(
    f"Competition ID              : "
    f"{TEST_COMPETITION_ID}"
)

print(
    f"Competition                 : "
    f"{TEST_COMPETITION_NAME}"
)

print(
    f"Days discovered             : "
    f"{VALIDATION_COMPETITION_DAYS}"
)


print()
print("DAY SCAN")
print("-" * 120)

display(
    VALIDATION_DAY_SUMMARY
)


print()
print("JAVELIN EVENTS")
print("-" * 120)

display(
    VALIDATION_EVENT_SUMMARY
)


print()
print("RESULT DISCOVERY")
print("-" * 120)

print(
    f"Javelin result rows         : "
    f"{N_JAVELIN_ROWS:,}"
)

print(
    f"Profile-backed result rows  : "
    f"{N_PROFILE_RESULT_ROWS:,}"
)

print(
    f"Unique profile candidates   : "
    f"{N_PROFILE_CANDIDATES:,}"
)

print(
    f"No-profile/audit rows       : "
    f"{N_NO_PROFILE_ROWS:,}"
)


print()
print("PROFILE-BACKED ATHLETE CANDIDATES")
print("-" * 120)

display(
    VALIDATION_PROFILE_CANDIDATES.head(
        50
    )
)


print()
print("SAME-DATE MULTIPLE PERFORMANCES")
print("-" * 120)

if len(
    SAME_DATE_MULTIPLE_PERFORMANCES
) > 0:

    display(
        SAME_DATE_MULTIPLE_PERFORMANCES.head(
            50
        )
    )

else:

    print(
        "No profile-backed athlete has multiple recovered rows "
        "on the same result date in this validation competition."
    )


print()
print("NO-PROFILE / UNRESOLVED AUDIT")
print("-" * 120)

if len(
    VALIDATION_NO_PROFILE
) > 0:

    display(
        VALIDATION_NO_PROFILE[
            [
                column
                for column
                in [
                    "requested_day",
                    "event_title",
                    "event",
                    "competitor_name",
                    "competitor_urlSlug",
                    "competitor_hasProfile",
                    "mark",
                    "result_date_dt",
                ]
                if column
                in VALIDATION_NO_PROFILE.columns
            ]
        ]
        .head(
            50
        )
    )


else:

    print(
        "None."
    )


print()
print("CHECKPOINTS")
print("-" * 120)

print(
    f"Javelin results             : "
    f"{VALIDATION_RESULTS_PATH}"
)

print(
    f"Profile candidates          : "
    f"{VALIDATION_PROFILE_PATH}"
)

print(
    f"No-profile audit            : "
    f"{VALIDATION_NO_PROFILE_PATH}"
)


print()
print("=" * 120)


if VALIDATION_PARSER_PASS:

    print(
        "CELL 6 STATUS: JAVELIN RESULT PARSER VALIDATED"
    )

    print()
    print(
        "The competition parser has recovered real javelin "
        "performances and profile-backed athlete candidates."
    )

    print(
        "The next cell can safely scale this exact parser to "
        "the 264 incremental competitions."
    )


else:

    print(
        "CELL 6 STATUS: REVIEW REQUIRED"
    )

    print()
    print(
        "Do not start the 264-competition crawl yet."
    )


print("=" * 120)


WORLD ATHLETICS JAVELIN RESULT PARSER — ALL-DAY VALIDATION

COMPETITION
------------------------------------------------------------------------------------------------------------------------
Competition ID              : 7245170
Competition                 : SA Age Group Championships
Days discovered             : [1, 2, 3, 8, 9, 10]

DAY SCAN
------------------------------------------------------------------------------------------------------------------------


,day,status,javelin_rows,raw_path
0,1,OK,0,/Users/veesheenyuen/Documents/code/veeyuen/SAA...
1,2,OK,1,/Users/veesheenyuen/Documents/code/veeyuen/SAA...
2,3,OK,4,/Users/veesheenyuen/Documents/code/veeyuen/SAA...
3,8,OK,0,/Users/veesheenyuen/Documents/code/veeyuen/SAA...
4,9,OK,1,/Users/veesheenyuen/Documents/code/veeyuen/SAA...
5,10,OK,0,/Users/veesheenyuen/Documents/code/veeyuen/SAA...



JAVELIN EVENTS
------------------------------------------------------------------------------------------------------------------------


,requested_day,event_title,event,gender,rows,profile_rows,athletes
0,2,NaN,Women's Javelin Throw (500g),W,1,1,1
1,3,Combined Events,Men's Javelin Throw,M,2,2,1
2,3,Combined Events - U20,Men's Javelin Throw,M,2,2,1
3,9,NaN,Men's Javelin Throw,M,1,1,1



RESULT DISCOVERY
------------------------------------------------------------------------------------------------------------------------
Javelin result rows         : 6
Profile-backed result rows  : 6
Unique profile candidates   : 4
No-profile/audit rows       : 0

PROFILE-BACKED ATHLETE CANDIDATES
------------------------------------------------------------------------------------------------------------------------


,candidate_aaAthleteId,competitor_name,competitor_urlSlug,competitor_birthDate,source_competitor_id,source_iaafId
0,15024858,Lucas WONG,singapore/lucas-wong-15024858,05 MAY 2004,lavvkdi2s2dbyux29pz9yi,None
1,15044864,Le Cong Lucas FUN,singapore/le-cong-lucas-fun-15044864,11 JAN 2006,15044864,None
2,15208273,Jayden NG,singapore/jayden-ng-15208273,2008,15208273,None
3,15347518,Raejia LIAM,singapore/raejia-liam-15347518,None,b2kyx0arq1e32zi1lfstmy,None



SAME-DATE MULTIPLE PERFORMANCES
------------------------------------------------------------------------------------------------------------------------
No profile-backed athlete has multiple recovered rows on the same result date in this validation competition.

NO-PROFILE / UNRESOLVED AUDIT
------------------------------------------------------------------------------------------------------------------------
None.

CHECKPOINTS
------------------------------------------------------------------------------------------------------------------------
Javelin results             : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/validation_all_days_javelin_results.parquet
Profile candidates          : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/validation_all_days_profile_candidates.parquet
No-profile audit            : /Users/veesheenyuen/Documents/code/veeyuen/

In [7]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 7 — BULK INCREMENTAL COMPETITION SCAN
# ==================================================================================================
#
# Purpose
# -------
# Scan every World Athletics competition discovered in Cell 4 and retain only:
#
#       javelin result rows
#       profile-backed athlete candidates
#       unresolved/no-profile javelin rows
#
#
# Design
# ------
#
# For each competition:
#
#       request day 1
#           ↓
#       discover actual competition days
#           ↓
#       request every actual day explicitly
#           ↓
#       flatten javelin branches using validated Cell 6 parser
#           ↓
#       retain rows in incremental date window
#           ↓
#       checkpoint competition independently
#
#
# Resume behaviour
# ----------------
#
# A competition is considered complete only when its own summary checkpoint exists.
#
# Therefore, if this cell is interrupted:
#
#       rerun Cell 7
#
# and completed competitions will be skipped.
#
#
# IMPORTANT
# ---------
# This cell does NOT yet:
#
#       confirm canonical aaAthleteId via getSingleCompetitor
#       download complete athlete-year histories
#       merge with the historical canonical dataset
#
# Those happen in subsequent cells.
#
# ==================================================================================================

from __future__ import annotations

import json
import traceback

from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "RUN_ID",
    "RUN_RAW_DIR",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "DOWNLOAD_START_DATE",
    "DOWNLOAD_END_DATE",

    "COMPETITIONS_TO_SCAN",

    "WA_NETWORK_READY",

    "COMPETITION_RESULTS_QUERY",
    "query_arg_names",
    "convert_day_value_for_graphql",

    "fetch_competition_day_results",
    "flatten_javelin_competition_day",
    "parse_wa_result_date",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–6 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not WA_NETWORK_READY:

    raise RuntimeError(
        "World Athletics GraphQL client is not ready."
    )


# ==================================================================================================
# PART B — BULK SCAN DIRECTORIES
# ==================================================================================================

BULK_ROOT = (
    RUN_CHECKPOINT_DIR
    / "bulk_competition_scan"
)


BULK_COMPETITION_DIR = (
    BULK_ROOT
    / "competitions"
)


BULK_RESULT_DIR = (
    BULK_ROOT
    / "javelin_results"
)


BULK_NO_PROFILE_DIR = (
    BULK_ROOT
    / "no_profile"
)


BULK_SUMMARY_DIR = (
    BULK_ROOT
    / "summaries"
)


BULK_ERROR_DIR = (
    BULK_ROOT
    / "errors"
)


for directory in [
    BULK_ROOT,
    BULK_COMPETITION_DIR,
    BULK_RESULT_DIR,
    BULK_NO_PROFILE_DIR,
    BULK_SUMMARY_DIR,
    BULK_ERROR_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================================================================
# PART C — RUN SETTINGS
# ==================================================================================================

ENABLE_BULK_COMPETITION_SCAN = True


INCREMENTAL_START = pd.Timestamp(
    DOWNLOAD_START_DATE
).normalize()


INCREMENTAL_END = pd.Timestamp(
    DOWNLOAD_END_DATE
).normalize()


N_COMPETITIONS_TO_SCAN = int(
    len(
        COMPETITIONS_TO_SCAN
    )
)


# ==================================================================================================
# PART D — COMPETITION DAY DISCOVERY
# ==================================================================================================

def extract_day_map_from_payload(
    payload,
):
    """
    Recover World Athletics competition day numbers and calendar dates.

    Example
    -------
        {
            1: Timestamp("2026-08-28"),
            2: Timestamp("2026-08-29"),
            ...
        }

    The day list normally lives under:

        options -> days

    but we search recursively so harmless schema nesting changes do not break
    the incremental crawler.
    """

    day_rows = []


    def walk(
        obj,
    ):

        if isinstance(
            obj,
            dict,
        ):

            if "days" in obj:

                days_obj = obj.get(
                    "days"
                )


                if isinstance(
                    days_obj,
                    list,
                ):

                    for item in days_obj:

                        if isinstance(
                            item,
                            dict,
                        ):

                            day_rows.append(
                                item
                            )


            for value in obj.values():

                walk(
                    value
                )


        elif isinstance(
            obj,
            list,
        ):

            for value in obj:

                walk(
                    value
                )


    walk(
        payload
    )


    day_map = {}


    for item in day_rows:

        raw_day = item.get(
            "day"
        )


        raw_date = item.get(
            "date"
        )


        try:

            day_number = int(
                raw_day
            )

        except Exception:

            continue


        parsed_date = (
            parse_wa_result_date(
                raw_date
            )
        )


        day_map[
            day_number
        ] = parsed_date


    return dict(
        sorted(
            day_map.items()
        )
    )


# ==================================================================================================
# PART E — ADD REQUESTED-DAY DATE TO SUMMARY ROWS
# ==================================================================================================

def assign_result_dates_from_day_map(
    df,
    day_map,
):
    """
    Fill result_date_dt.

    Race-level WA rows often carry their own source date.

    Summary-level field-event rows may not.

    For summary rows, the explicitly requested competition day provides the
    correct calendar date.
    """

    if len(
        df
    ) == 0:

        return df.copy()


    out = df.copy()


    if "date_raw" in out.columns:

        out[
            "result_date_dt"
        ] = (
            out[
                "date_raw"
            ]
            .map(
                parse_wa_result_date
            )
        )


    else:

        out[
            "result_date_dt"
        ] = pd.NaT


    def requested_day_date(
        row,
    ):

        current = row.get(
            "result_date_dt"
        )


        if not pd.isna(
            current
        ):

            return current


        requested_day = row.get(
            "requested_day"
        )


        try:

            requested_day = int(
                requested_day
            )

        except Exception:

            return pd.NaT


        return day_map.get(
            requested_day,
            pd.NaT,
        )


    out[
        "result_date_dt"
    ] = out.apply(
        requested_day_date,
        axis=1,
    )


    out[
        "result_date_dt"
    ] = pd.to_datetime(
        out[
            "result_date_dt"
        ],
        errors="coerce",
    )


    return out


# ==================================================================================================
# PART F — PATH HELPERS
# ==================================================================================================

def competition_paths(
    competition_id,
):
    """
    All run-specific paths for one competition.
    """

    competition_id = int(
        competition_id
    )


    return {

        "raw_dir":
            RUN_RAW_DIR
            / "competitions"
            / str(
                competition_id
            ),

        "result_path":
            BULK_RESULT_DIR
            / (
                f"competition_{competition_id}_"
                f"javelin.parquet"
            ),

        "no_profile_path":
            BULK_NO_PROFILE_DIR
            / (
                f"competition_{competition_id}_"
                f"no_profile.parquet"
            ),

        "summary_path":
            BULK_SUMMARY_DIR
            / (
                f"competition_{competition_id}_"
                f"summary.json"
            ),

        "error_path":
            BULK_ERROR_DIR
            / (
                f"competition_{competition_id}_"
                f"error.json"
            ),
    }


# ==================================================================================================
# PART G — ONE COMPETITION SCANNER
# ==================================================================================================

async def scan_one_competition(
    competition_row,
):
    """
    Scan one competition completely and checkpoint the result.

    Returns
    -------
    dict
        Competition summary.
    """

    competition_id = int(
        competition_row[
            "competition_id"
        ]
    )


    competition_name = str(
        competition_row.get(
            "name",
            ""
        )
    )


    paths = competition_paths(
        competition_id
    )


    paths[
        "raw_dir"
    ].mkdir(
        parents=True,
        exist_ok=True,
    )


    # ----------------------------------------------------------------------------------------------
    # Resume
    # ----------------------------------------------------------------------------------------------

    if paths[
        "summary_path"
    ].exists():

        with paths[
            "summary_path"
        ].open(
            "r",
            encoding="utf-8",
        ) as f:

            prior_summary = json.load(
                f
            )


        prior_summary[
            "resumed"
        ] = True


        return prior_summary


    # ----------------------------------------------------------------------------------------------
    # Day 1 bootstrap
    # ----------------------------------------------------------------------------------------------

    first_payload = await fetch_competition_day_results(

        competition_id=
            competition_id,

        day=
            1,
    )


    first_raw_path = (

        paths[
            "raw_dir"
        ]
        / "day_1.json"
    )


    with first_raw_path.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            first_payload,
            f,
            indent=2,
            default=str,
        )


    day_map = extract_day_map_from_payload(
        first_payload
    )


    # ----------------------------------------------------------------------------------------------
    # Most WA competitions expose day options.
    #
    # If not, day 1 is still a valid explicitly fetched result payload.
    # ----------------------------------------------------------------------------------------------

    if day_map:

        days = sorted(
            day_map.keys()
        )

    else:

        days = [
            1
        ]


    competition_rows = []

    day_summaries = []


    # ----------------------------------------------------------------------------------------------
    # Scan each explicit competition day
    # ----------------------------------------------------------------------------------------------

    for day in days:

        try:

            if day == 1:

                payload = first_payload


            else:

                payload = await fetch_competition_day_results(

                    competition_id=
                        competition_id,

                    day=
                        day,
                )


                raw_path = (

                    paths[
                        "raw_dir"
                    ]
                    / (
                        f"day_{day}.json"
                    )
                )


                with raw_path.open(
                    "w",
                    encoding="utf-8",
                ) as f:

                    json.dump(
                        payload,
                        f,
                        indent=2,
                        default=str,
                    )


            day_rows = flatten_javelin_competition_day(

                payload,

                competition_id=
                    competition_id,

                competition_name=
                    competition_name,

                requested_day=
                    day,
            )


            competition_rows.extend(
                day_rows
            )


            day_summaries.append(
                {
                    "day":
                        int(
                            day
                        ),

                    "date":
                        (
                            day_map[
                                day
                            ].date().isoformat()

                            if (
                                day in day_map
                                and
                                not pd.isna(
                                    day_map[
                                        day
                                    ]
                                )
                            )

                            else None
                        ),

                    "status":
                        "OK",

                    "javelin_rows":
                        len(
                            day_rows
                        ),
                }
            )


        except Exception as exc:

            day_summaries.append(
                {
                    "day":
                        int(
                            day
                        ),

                    "date":
                        (
                            day_map[
                                day
                            ].date().isoformat()

                            if (
                                day in day_map
                                and
                                not pd.isna(
                                    day_map[
                                        day
                                    ]
                                )
                            )

                            else None
                        ),

                    "status":
                        "ERROR",

                    "javelin_rows":
                        0,

                    "error":
                        repr(
                            exc
                        ),
                }
            )


    # ----------------------------------------------------------------------------------------------
    # Convert competition rows
    # ----------------------------------------------------------------------------------------------

    result_df = pd.DataFrame(
        competition_rows
    )


    if len(
        result_df
    ) > 0:

        result_df = assign_result_dates_from_day_map(

            result_df,

            day_map,
        )


        # ------------------------------------------------------------------------------------------
        # Restrict to the incremental extraction window.
        #
        # This matters for competitions beginning before DOWNLOAD_START_DATE.
        # ------------------------------------------------------------------------------------------

        in_window = (

            result_df[
                "result_date_dt"
            ]
            .notna()

            &
            (
                result_df[
                    "result_date_dt"
                ]
                >= INCREMENTAL_START
            )

            &
            (
                result_df[
                    "result_date_dt"
                ]
                <= INCREMENTAL_END
            )
        )


        result_df[
            "inside_incremental_window"
        ] = in_window


        result_df = (

            result_df.loc[
                in_window
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


    # ----------------------------------------------------------------------------------------------
    # Profile-backed vs unresolved
    # ----------------------------------------------------------------------------------------------

    if len(
        result_df
    ) > 0:

        profile_mask = (

            (
                result_df[
                    "competitor_hasProfile"
                ]
                == True
            )

            &
            (
                result_df[
                    "candidate_aaAthleteId"
                ]
                .notna()
            )
        )


        profile_df = (

            result_df.loc[
                profile_mask
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        no_profile_df = (

            result_df.loc[
                ~profile_mask
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


    else:

        profile_df = pd.DataFrame()

        no_profile_df = pd.DataFrame()


    # ----------------------------------------------------------------------------------------------
    # Persist normalized competition rows
    # ----------------------------------------------------------------------------------------------

    if len(
        result_df
    ) > 0:

        result_df.to_parquet(

            paths[
                "result_path"
            ],

            index=False,
        )


    else:

        # Empty marker file is represented by an empty parquet with
        # competition ID/name columns.
        pd.DataFrame(
            columns=[
                "competition_id",
                "competition_name",
            ]
        ).to_parquet(

            paths[
                "result_path"
            ],

            index=False,
        )


    if len(
        no_profile_df
    ) > 0:

        no_profile_df.to_parquet(

            paths[
                "no_profile_path"
            ],

            index=False,
        )


    else:

        pd.DataFrame(
            columns=[
                "competition_id",
                "competition_name",
            ]
        ).to_parquet(

            paths[
                "no_profile_path"
            ],

            index=False,
        )


    # ----------------------------------------------------------------------------------------------
    # Competition-level summary
    # ----------------------------------------------------------------------------------------------

    day_errors = sum(

        row[
            "status"
        ]
        == "ERROR"

        for row
        in day_summaries
    )


    unique_profile_candidates = (

        int(
            profile_df[
                "candidate_aaAthleteId"
            ]
            .nunique()
        )

        if (
            len(
                profile_df
            ) > 0
            and
            "candidate_aaAthleteId"
            in profile_df.columns
        )

        else 0
    )


    summary = {

        "competition_id":
            competition_id,

        "competition_name":
            competition_name,

        "n_days":
            len(
                days
            ),

        "days":
            [
                int(
                    x
                )
                for x in days
            ],

        "day_errors":
            int(
                day_errors
            ),

        "javelin_rows_in_window":
            int(
                len(
                    result_df
                )
            ),

        "profile_rows":
            int(
                len(
                    profile_df
                )
            ),

        "unique_profile_candidates":
            unique_profile_candidates,

        "no_profile_rows":
            int(
                len(
                    no_profile_df
                )
            ),

        "day_summaries":
            day_summaries,

        "result_path":
            str(
                paths[
                    "result_path"
            ]
            ),

        "no_profile_path":
            str(
                paths[
                    "no_profile_path"
            ]
            ),

        "status":
            (
                "COMPLETE"

                if day_errors == 0

                else "COMPLETE_WITH_DAY_ERRORS"
            ),

        "resumed":
            False,
    }


    # Summary is written LAST.
    #
    # Its existence therefore means the competition scan is complete enough
    # to resume safely.
    with paths[
        "summary_path"
    ].open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            summary,
            f,
            indent=2,
            default=str,
        )


    return summary


# ==================================================================================================
# PART H — BULK SCAN
# ==================================================================================================

if not ENABLE_BULK_COMPETITION_SCAN:

    raise RuntimeError(
        "ENABLE_BULK_COMPETITION_SCAN is False."
    )


bulk_summaries = []


for position, (_, competition_row) in enumerate(

    COMPETITIONS_TO_SCAN.iterrows(),

    start=1,
):

    competition_id = int(
        competition_row[
            "competition_id"
        ]
    )


    competition_name = str(
        competition_row.get(
            "name",
            ""
        )
    )


    try:

        summary = await scan_one_competition(
            competition_row
        )


        bulk_summaries.append(
            summary
        )


        # ------------------------------------------------------------------------------------------
        # Lightweight progress output
        # ------------------------------------------------------------------------------------------

        if (
            position == 1

            or
            position % 10 == 0

            or
            position == N_COMPETITIONS_TO_SCAN
        ):

            print(
                f"[{position:>3}/{N_COMPETITIONS_TO_SCAN}] "
                f"{competition_id} | "
                f"javelin rows={summary['javelin_rows_in_window']} | "
                f"athletes={summary['unique_profile_candidates']} | "
                f"{summary['status']}"
            )


    except Exception as exc:

        paths = competition_paths(
            competition_id
        )


        error_record = {

            "competition_id":
                competition_id,

            "competition_name":
                competition_name,

            "status":
                "ERROR",

            "error":
                repr(
                    exc
                ),

            "traceback":
                traceback.format_exc(),
        }


        with paths[
            "error_path"
        ].open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                error_record,
                f,
                indent=2,
                default=str,
            )


        bulk_summaries.append(
            {
                "competition_id":
                    competition_id,

                "competition_name":
                    competition_name,

                "n_days":
                    0,

                "day_errors":
                    0,

                "javelin_rows_in_window":
                    0,

                "profile_rows":
                    0,

                "unique_profile_candidates":
                    0,

                "no_profile_rows":
                    0,

                "status":
                    "ERROR",

                "error":
                    repr(
                        exc
                    ),

                "resumed":
                    False,
            }
        )


# ==================================================================================================
# PART I — REBUILD SUMMARY FROM ALL COMPETITION CHECKPOINTS
# ==================================================================================================
#
# We rebuild from files on disk rather than relying only on in-memory objects.
#
# This makes resumed runs deterministic.
# ==================================================================================================

summary_records = []


for path in sorted(
    BULK_SUMMARY_DIR.glob(
        "competition_*_summary.json"
    )
):

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        summary_records.append(
            json.load(
                f
            )
        )


# Include any top-level errors that do not have successful summaries.
completed_ids = {

    int(
        record[
            "competition_id"
        ]
    )

    for record
    in summary_records
}


for path in sorted(
    BULK_ERROR_DIR.glob(
        "competition_*_error.json"
    )
):

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        error_record = json.load(
            f
        )


    competition_id = int(
        error_record[
            "competition_id"
        ]
    )


    if competition_id not in completed_ids:

        summary_records.append(
            {
                "competition_id":
                    competition_id,

                "competition_name":
                    error_record.get(
                        "competition_name"
                    ),

                "n_days":
                    0,

                "day_errors":
                    0,

                "javelin_rows_in_window":
                    0,

                "profile_rows":
                    0,

                "unique_profile_candidates":
                    0,

                "no_profile_rows":
                    0,

                "status":
                    "ERROR",

                "error":
                    error_record.get(
                        "error"
                    ),
            }
        )


BULK_COMPETITION_SUMMARY = pd.DataFrame(
    summary_records
)


if len(
    BULK_COMPETITION_SUMMARY
) > 0:

    BULK_COMPETITION_SUMMARY = (

        BULK_COMPETITION_SUMMARY.sort_values(
            "competition_id"
        )
        .reset_index(
            drop=True
        )
    )


# ==================================================================================================
# PART J — CONSOLIDATE ALL JAVELIN RESULT CHECKPOINTS
# ==================================================================================================

result_frames = []


for path in sorted(
    BULK_RESULT_DIR.glob(
        "competition_*_javelin.parquet"
    )
):

    frame = pd.read_parquet(
        path
    )


    if len(
        frame
    ) > 0:

        result_frames.append(
            frame
        )


if result_frames:

    INCREMENTAL_JAVELIN_RESULTS = pd.concat(

        result_frames,

        ignore_index=True,

        sort=False,
    )


else:

    INCREMENTAL_JAVELIN_RESULTS = pd.DataFrame()


# ==================================================================================================
# PART K — CONSOLIDATE NO-PROFILE AUDIT
# ==================================================================================================

no_profile_frames = []


for path in sorted(
    BULK_NO_PROFILE_DIR.glob(
        "competition_*_no_profile.parquet"
    )
):

    frame = pd.read_parquet(
        path
    )


    if len(
        frame
    ) > 0:

        no_profile_frames.append(
            frame
        )


if no_profile_frames:

    INCREMENTAL_NO_PROFILE_RESULTS = pd.concat(

        no_profile_frames,

        ignore_index=True,

        sort=False,
    )


else:

    INCREMENTAL_NO_PROFILE_RESULTS = pd.DataFrame()


# ==================================================================================================
# PART L — PROFILE-BACKED ATHLETE CANDIDATES
# ==================================================================================================

if len(
    INCREMENTAL_JAVELIN_RESULTS
) > 0:

    profile_mask = (

        (
            INCREMENTAL_JAVELIN_RESULTS[
                "competitor_hasProfile"
            ]
            == True
        )

        &
        (
            INCREMENTAL_JAVELIN_RESULTS[
                "candidate_aaAthleteId"
            ]
            .notna()
        )
    )


    INCREMENTAL_PROFILE_CANDIDATES = (

        INCREMENTAL_JAVELIN_RESULTS.loc[
            profile_mask
        ]
        [
            [
                "candidate_aaAthleteId",
                "competitor_name",
                "competitor_urlSlug",
                "competitor_birthDate",
                "source_competitor_id",
                "source_iaafId",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "candidate_aaAthleteId",
                "competitor_name",
            ]
        )
        .reset_index(
            drop=True
        )
    )


else:

    INCREMENTAL_PROFILE_CANDIDATES = pd.DataFrame(
        columns=[
            "candidate_aaAthleteId",
            "competitor_name",
            "competitor_urlSlug",
            "competitor_birthDate",
            "source_competitor_id",
            "source_iaafId",
        ]
    )


# ==================================================================================================
# PART M — DUPLICATE / SAME-DATE DIAGNOSTICS
# ==================================================================================================

if len(
    INCREMENTAL_JAVELIN_RESULTS
) > 0:

    candidate_rows = (

        INCREMENTAL_JAVELIN_RESULTS.loc[
            INCREMENTAL_JAVELIN_RESULTS[
                "candidate_aaAthleteId"
            ]
            .notna()
        ]
        .copy()
    )


    INCREMENTAL_SAME_DATE_COUNTS = (

        candidate_rows.dropna(
            subset=[
                "result_date_dt"
            ]
        )
        .groupby(
            [
                "candidate_aaAthleteId",
                "result_date_dt",
            ],
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size":
                    "n_result_rows"
            }
        )
    )


    INCREMENTAL_SAME_DATE_MULTIPLE = (

        INCREMENTAL_SAME_DATE_COUNTS.loc[
            INCREMENTAL_SAME_DATE_COUNTS[
                "n_result_rows"
            ]
            > 1
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


else:

    INCREMENTAL_SAME_DATE_COUNTS = pd.DataFrame()

    INCREMENTAL_SAME_DATE_MULTIPLE = pd.DataFrame()


# ==================================================================================================
# PART N — QA COUNTS
# ==================================================================================================

N_COMPLETE = int(

    BULK_COMPETITION_SUMMARY[
        "status"
    ]
    .isin(
        [
            "COMPLETE",
            "COMPLETE_WITH_DAY_ERRORS",
        ]
    )
    .sum()
)


N_TOP_LEVEL_ERRORS = int(

    (
        BULK_COMPETITION_SUMMARY[
            "status"
        ]
        == "ERROR"
    )
    .sum()
)


N_WITH_DAY_ERRORS = int(

    (
        BULK_COMPETITION_SUMMARY[
            "status"
        ]
        == "COMPLETE_WITH_DAY_ERRORS"
    )
    .sum()
)


N_COMPETITIONS_WITH_JAVELIN = int(

    (
        pd.to_numeric(
            BULK_COMPETITION_SUMMARY[
                "javelin_rows_in_window"
            ],
            errors="coerce",
        )
        .fillna(
            0
        )
        > 0
    )
    .sum()
)


N_INCREMENTAL_JAVELIN_ROWS = int(
    len(
        INCREMENTAL_JAVELIN_RESULTS
    )
)


N_INCREMENTAL_PROFILE_ATHLETES = int(

    INCREMENTAL_PROFILE_CANDIDATES[
        "candidate_aaAthleteId"
    ]
    .nunique()
)


N_INCREMENTAL_NO_PROFILE = int(
    len(
        INCREMENTAL_NO_PROFILE_RESULTS
    )
)


# ==================================================================================================
# PART O — SAVE AGGREGATED CHECKPOINTS
# ==================================================================================================

BULK_SUMMARY_PATH = (
    RUN_CHECKPOINT_DIR
    / "bulk_competition_scan_summary.parquet"
)


INCREMENTAL_JAVELIN_PATH = (
    RUN_CHECKPOINT_DIR
    / "incremental_javelin_results_raw.parquet"
)


INCREMENTAL_PROFILE_CANDIDATES_PATH = (
    RUN_CHECKPOINT_DIR
    / "incremental_profile_candidates.parquet"
)


INCREMENTAL_NO_PROFILE_PATH = (
    RUN_CHECKPOINT_DIR
    / "incremental_no_profile_results.parquet"
)


BULK_COMPETITION_SUMMARY.to_parquet(
    BULK_SUMMARY_PATH,
    index=False,
)


INCREMENTAL_JAVELIN_RESULTS.to_parquet(
    INCREMENTAL_JAVELIN_PATH,
    index=False,
)


INCREMENTAL_PROFILE_CANDIDATES.to_parquet(
    INCREMENTAL_PROFILE_CANDIDATES_PATH,
    index=False,
)


INCREMENTAL_NO_PROFILE_RESULTS.to_parquet(
    INCREMENTAL_NO_PROFILE_PATH,
    index=False,
)


# ==================================================================================================
# PART P — MANIFEST
# ==================================================================================================

BULK_SCAN_PASS = bool(

    N_COMPLETE
    + N_TOP_LEVEL_ERRORS
    == N_COMPETITIONS_TO_SCAN

    and
    N_INCREMENTAL_JAVELIN_ROWS
    >= 0
)


RUN_MANIFEST[
    "bulk_competition_scan"
] = {

    "status":
        (
            "COMPLETE"

            if BULK_SCAN_PASS

            else "INCOMPLETE"
        ),

    "competitions_requested":
        N_COMPETITIONS_TO_SCAN,

    "competitions_completed":
        N_COMPLETE,

    "competitions_with_top_level_errors":
        N_TOP_LEVEL_ERRORS,

    "competitions_with_day_errors":
        N_WITH_DAY_ERRORS,

    "competitions_with_javelin":
        N_COMPETITIONS_WITH_JAVELIN,

    "javelin_rows_in_window":
        N_INCREMENTAL_JAVELIN_ROWS,

    "unique_profile_candidates":
        N_INCREMENTAL_PROFILE_ATHLETES,

    "no_profile_rows":
        N_INCREMENTAL_NO_PROFILE,

    "summary_path":
        str(
            BULK_SUMMARY_PATH
        ),

    "javelin_path":
        str(
            INCREMENTAL_JAVELIN_PATH
        ),

    "profile_candidates_path":
        str(
            INCREMENTAL_PROFILE_CANDIDATES_PATH
        ),

    "no_profile_path":
        str(
            INCREMENTAL_NO_PROFILE_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "BULK_JAVELIN_SCAN_COMPLETE"

    if BULK_SCAN_PASS

    else "BULK_JAVELIN_SCAN_INCOMPLETE"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART Q — OUTPUT
# ==================================================================================================

print()
print("=" * 125)
print("WORLD ATHLETICS BULK INCREMENTAL JAVELIN SCAN")
print("=" * 125)


print()
print("COMPETITION COVERAGE")
print("-" * 125)

print(
    f"Competitions requested      : "
    f"{N_COMPETITIONS_TO_SCAN:,}"
)

print(
    f"Competitions completed      : "
    f"{N_COMPLETE:,}"
)

print(
    f"Top-level errors            : "
    f"{N_TOP_LEVEL_ERRORS:,}"
)

print(
    f"Competitions with day errors: "
    f"{N_WITH_DAY_ERRORS:,}"
)

print(
    f"Competitions with javelin   : "
    f"{N_COMPETITIONS_WITH_JAVELIN:,}"
)


print()
print("INCREMENTAL JAVELIN DISCOVERY")
print("-" * 125)

print(
    f"Javelin result rows         : "
    f"{N_INCREMENTAL_JAVELIN_ROWS:,}"
)

print(
    f"Profile-backed athletes     : "
    f"{N_INCREMENTAL_PROFILE_ATHLETES:,}"
)

print(
    f"No-profile result rows      : "
    f"{N_INCREMENTAL_NO_PROFILE:,}"
)

print(
    f"Date window                 : "
    f"{INCREMENTAL_START.date()} → "
    f"{INCREMENTAL_END.date()}"
)


print()
print("COMPETITIONS WITH JAVELIN RESULTS")
print("-" * 125)

display(
    BULK_COMPETITION_SUMMARY.loc[
        pd.to_numeric(
            BULK_COMPETITION_SUMMARY[
                "javelin_rows_in_window"
            ],
            errors="coerce",
        )
        .fillna(
            0
        )
        > 0
    ]
    [
        [
            column
            for column
            in [
                "competition_id",
                "competition_name",
                "n_days",
                "javelin_rows_in_window",
                "profile_rows",
                "unique_profile_candidates",
                "no_profile_rows",
                "status",
            ]
            if column
            in BULK_COMPETITION_SUMMARY.columns
        ]
    ]
    .sort_values(
        [
            "competition_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("PROFILE-BACKED ATHLETE CANDIDATES")
print("-" * 125)

display(
    INCREMENTAL_PROFILE_CANDIDATES.head(
        100
    )
)


print()
print("SAME-DATE MULTIPLE-PERFORMANCE AUDIT")
print("-" * 125)

if len(
    INCREMENTAL_SAME_DATE_MULTIPLE
) > 0:

    display(
        INCREMENTAL_SAME_DATE_MULTIPLE.head(
            100
        )
    )

else:

    print(
        "No athlete/date has multiple raw result rows in this "
        "incremental extraction."
    )


print()
print("UNRESOLVED / NO-PROFILE RESULTS")
print("-" * 125)

if len(
    INCREMENTAL_NO_PROFILE_RESULTS
) > 0:

    display(
        INCREMENTAL_NO_PROFILE_RESULTS[
            [
                column
                for column
                in [
                    "competition_id",
                    "competition_name",
                    "result_date_dt",
                    "event_title",
                    "event",
                    "competitor_name",
                    "competitor_urlSlug",
                    "mark",
                ]
                if column
                in INCREMENTAL_NO_PROFILE_RESULTS.columns
            ]
        ]
        .head(
            100
        )
    )

else:

    print(
        "None."
    )


print()
print("CHECKPOINTS")
print("-" * 125)

print(
    f"Competition summary         : "
    f"{BULK_SUMMARY_PATH}"
)

print(
    f"Raw javelin results         : "
    f"{INCREMENTAL_JAVELIN_PATH}"
)

print(
    f"Athlete candidates          : "
    f"{INCREMENTAL_PROFILE_CANDIDATES_PATH}"
)

print(
    f"No-profile audit            : "
    f"{INCREMENTAL_NO_PROFILE_PATH}"
)


print()
print("=" * 125)


if BULK_SCAN_PASS:

    print(
        "CELL 7 STATUS: BULK INCREMENTAL JAVELIN SCAN COMPLETE"
    )

    print()
    print(
        "The next cell will resolve every profile candidate against "
        "getSingleCompetitor.basicData.aaId before we trust the athlete identity."
    )


else:

    print(
        "CELL 7 STATUS: BULK SCAN INCOMPLETE"
    )


print("=" * 125)

[  1/264] 7240769 | javelin rows=1 | athletes=1 | COMPLETE
[ 10/264] 7240012 | javelin rows=54 | athletes=26 | COMPLETE
[ 20/264] 7245037 | javelin rows=2 | athletes=2 | COMPLETE
[ 30/264] 7235589 | javelin rows=0 | athletes=0 | COMPLETE
[ 40/264] 7241684 | javelin rows=0 | athletes=0 | COMPLETE
[ 50/264] 7246156 | javelin rows=0 | athletes=0 | COMPLETE
[ 60/264] 7238646 | javelin rows=0 | athletes=0 | COMPLETE
[ 70/264] 7239942 | javelin rows=0 | athletes=0 | COMPLETE
[ 80/264] 7236716 | javelin rows=16 | athletes=10 | COMPLETE
[ 90/264] 7238421 | javelin rows=0 | athletes=0 | COMPLETE
[100/264] 7240847 | javelin rows=0 | athletes=0 | COMPLETE
[110/264] 7244472 | javelin rows=0 | athletes=0 | COMPLETE
[120/264] 7244844 | javelin rows=1 | athletes=1 | COMPLETE
[130/264] 7236223 | javelin rows=0 | athletes=0 | COMPLETE
[140/264] 7243834 | javelin rows=0 | athletes=0 | COMPLETE
[150/264] 7245364 | javelin rows=7 | athletes=7 | COMPLETE
[160/264] 7245394 | javelin rows=0 | athletes=0 | CO

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_3760/4257420784.py:1428: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  INCREMENTAL_JAVELIN_RESULTS = pd.concat(



WORLD ATHLETICS BULK INCREMENTAL JAVELIN SCAN

COMPETITION COVERAGE
-----------------------------------------------------------------------------------------------------------------------------
Competitions requested      : 264
Competitions completed      : 264
Top-level errors            : 0
Competitions with day errors: 0
Competitions with javelin   : 104

INCREMENTAL JAVELIN DISCOVERY
-----------------------------------------------------------------------------------------------------------------------------
Javelin result rows         : 1,304
Profile-backed athletes     : 867
No-profile result rows      : 13
Date window                 : 2026-08-30 → 2026-09-19

COMPETITIONS WITH JAVELIN RESULTS
-----------------------------------------------------------------------------------------------------------------------------


,competition_id,competition_name,n_days,javelin_rows_in_window,profile_rows,unique_profile_candidates,no_profile_rows,status
0,7212925,World Athletics Ultimate Championship,3,16,16,16,0,COMPLETE
1,7214029,Allianz Memorial van Damme,2,13,13,13,0,COMPLETE
2,7233421,XX Mediterranean Games,5,22,22,22,0,COMPLETE
3,7234714,ISTAF 2026,1,9,9,9,0,COMPLETE
4,7234811,Velká cena Chebu,1,17,17,17,0,COMPLETE
...,...,...,...,...,...,...,...,...
99,7245513,SPAR NK Teams - 1e divisie Promotie/Degradatie,1,10,10,10,0,COMPLETE
100,7245514,SPAR Competitie 2e - 3e Divisie Promotie/Degra...,1,1,1,1,0,COMPLETE
101,7245555,Mistrovství Čech družstev dorostenců a doroste...,1,2,2,2,0,COMPLETE
102,7245947,Campionati Provinciali U18 - U20 -U23,2,6,6,6,0,COMPLETE



PROFILE-BACKED ATHLETE CANDIDATES
-----------------------------------------------------------------------------------------------------------------------------


...


SAME-DATE MULTIPLE-PERFORMANCE AUDIT
-----------------------------------------------------------------------------------------------------------------------------


,candidate_aaAthleteId,result_date_dt,n_result_rows
0,14373225.0,2026-09-02,2
1,14450089.0,2026-08-30,2
2,14451604.0,2026-09-06,2
3,14534754.0,2026-08-30,2
4,14597174.0,2026-08-30,2
...,...,...,...
95,14981914.0,2026-08-30,2
96,14982578.0,2026-09-07,2
97,14984695.0,2026-08-30,2
98,14992001.0,2026-08-30,2



UNRESOLVED / NO-PROFILE RESULTS
-----------------------------------------------------------------------------------------------------------------------------


,competition_id,competition_name,result_date_dt,event_title,event,competitor_name,competitor_urlSlug,mark
0,7237674,Swedish U18 Team Championships,2026-09-13,U18 Events,Women's Javelin Throw (500g),Alma CARLSON,sweden/alma-carlson-15322465,36.86
1,7237949,Swedish U18 Combined Events Championships,2026-09-06,Combined Events - U17,Men's Javelin Throw (700g),Gabriel PRESELJ,sweden/gabriel-preselj-15291162,33.07
2,7237949,Swedish U18 Combined Events Championships,2026-09-06,Combined Events - U17,Men's Javelin Throw (700g),Gabriel PRESELJ,sweden/gabriel-preselj-15291162,33.07
3,7237949,Swedish U18 Combined Events Championships,2026-09-06,Combined Events - U17,Women's Javelin Throw (500g),Leia GIMLANDER,sweden/leia-gimlander-15326984,20.79
4,7237949,Swedish U18 Combined Events Championships,2026-09-06,Combined Events - U17,Women's Javelin Throw (500g),Leia GIMLANDER,sweden/leia-gimlander-15326984,20.79
5,7239162,4J Studios scottishathletics National Senior &...,2026-08-30,None,Men's Javelin Throw,David LEES,great-britain-ni/david-lees-15176055,52.47
6,7239941,Serbian U18 Combined Events Championships,2026-09-05,Combined Events,Women's Javelin Throw (500g),Elena ĐURIĆ,serbia/elena-duric-15299026,23.12
7,7239941,Serbian U18 Combined Events Championships,2026-09-05,Combined Events,Women's Javelin Throw (500g),Elena ĐURIĆ,serbia/elena-duric-15299026,23.12
8,7239944,Serbian U20 Combined Events Championships,2026-09-06,Combined Events,Men's Javelin Throw,Marko KOVAČ,serbia/marko-kovac-15347617,33.85
9,7239944,Serbian U20 Combined Events Championships,2026-09-06,Combined Events,Men's Javelin Throw,Marko KOVAČ,serbia/marko-kovac-15347617,33.85



CHECKPOINTS
-----------------------------------------------------------------------------------------------------------------------------
Competition summary         : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/bulk_competition_scan_summary.parquet
Raw javelin results         : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/incremental_javelin_results_raw.parquet
Athlete candidates          : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/incremental_profile_candidates.parquet
No-profile audit            : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/incremental_no_profile_results.parquet

CELL 7 STATUS: BULK INCREMENTAL JAVELIN SCAN COMPLETE

The next cell will resolve every profile candidate against getSingleCompetitor.basicData.

In [8]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 8 — CANONICAL ATHLETE IDENTITY RESOLUTION
# ==================================================================================================
#
# Purpose
# -------
# Convert competition-result profile candidates into trusted canonical
# World Athletics athlete identities.
#
#
# Efficiency rule
# ---------------
#
# If candidate_aaAthleteId already exists in the canonical historical dataset:
#
#       NO profile download is required.
#
#       candidate ID is already a trusted historical aaAthleteId.
#
#
# If candidate_aaAthleteId is NEW:
#
#       getSingleCompetitor(candidate ID)
#              ↓
#       basicData.aaId
#              ↓
#       trusted canonical aaAthleteId
#
#
# This cell:
#
#   1. Loads existing canonical athlete IDs only.
#   2. Splits candidates into KNOWN vs NEW.
#   3. Introspects the live getSingleCompetitor schema.
#   4. Downloads profiles ONLY for genuinely new athlete IDs.
#   5. Verifies candidate slug ID against basicData.aaId.
#   6. Checkpoints every new profile independently.
#   7. Creates the canonical incremental athlete mapping.
#
#
# No-profile competition results remain separate and are NOT silently discarded.
# They will be addressed in the next cell.
#
# ==================================================================================================

from __future__ import annotations

import json
import traceback

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "CONFIG",
    "RUN_ID",
    "RUN_ATHLETE_DIR",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",
    "WA_NETWORK_READY",
    "wa_graphql",

    "INCREMENTAL_PROFILE_CANDIDATES",
    "INCREMENTAL_NO_PROFILE_RESULTS",

    "query_fields",
    "graphql_type_to_string",
    "graphql_base_named_type",
    "graphql_base_kind",
    "get_graphql_type_definition",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–7 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not WA_NETWORK_READY:

    raise RuntimeError(
        "World Athletics GraphQL client is not ready."
    )


# ==================================================================================================
# PART B — ID NORMALIZATION
# ==================================================================================================

def normalize_wa_id(
    value,
):
    """
    Normalize World Athletics numeric identifiers.

    Handles values such as:

        14177510
        14177510.0
        "14177510"
        "14177510.0"
    """

    if value is None:

        return None


    try:

        if pd.isna(
            value
        ):

            return None

    except Exception:

        pass


    try:

        numeric = float(
            value
        )


        if not np.isfinite(
            numeric
        ):

            return None


        rounded = int(
            round(
                numeric
            )
        )


        if not np.isclose(
            numeric,
            rounded,
            atol=1e-9,
            rtol=0,
        ):

            return None


        return rounded


    except Exception:

        return None


# ==================================================================================================
# PART C — EXISTING CANONICAL ATHLETE IDS
# ==================================================================================================

canonical_schema = pq.ParquetFile(
    CONFIG.canonical_dataset_path
)


canonical_columns = list(
    canonical_schema.schema.names
)


CANONICAL_ID_COLUMN = next(
    (
        column

        for column
        in [
            "aaAthleteId",
            "aaathleteid",
            "athlete_id",
            "athleteId",
        ]

        if column
        in canonical_columns
    ),
    None,
)


if CANONICAL_ID_COLUMN is None:

    raise RuntimeError(
        "Could not identify aaAthleteId in canonical historical dataset."
    )


canonical_ids_df = pd.read_parquet(

    CONFIG.canonical_dataset_path,

    columns=[
        CANONICAL_ID_COLUMN
    ],
)


canonical_ids = (
    canonical_ids_df[
        CANONICAL_ID_COLUMN
    ]
    .map(
        normalize_wa_id
    )
    .dropna()
    .astype(
        int
    )
)


KNOWN_CANONICAL_ATHLETE_IDS = set(
    canonical_ids.unique()
)


del canonical_ids_df
del canonical_ids


# ==================================================================================================
# PART D — NORMALIZE CURRENT CANDIDATE LIST
# ==================================================================================================

PROFILE_CANDIDATES = (
    INCREMENTAL_PROFILE_CANDIDATES.copy()
)


PROFILE_CANDIDATES[
    "candidate_aaAthleteId"
] = (
    PROFILE_CANDIDATES[
        "candidate_aaAthleteId"
    ]
    .map(
        normalize_wa_id
    )
)


PROFILE_CANDIDATES = (
    PROFILE_CANDIDATES.dropna(
        subset=[
            "candidate_aaAthleteId"
        ]
    )
    .copy()
)


PROFILE_CANDIDATES[
    "candidate_aaAthleteId"
] = (
    PROFILE_CANDIDATES[
        "candidate_aaAthleteId"
    ]
    .astype(
        int
    )
)


PROFILE_CANDIDATES = (
    PROFILE_CANDIDATES.sort_values(
        [
            "candidate_aaAthleteId",
            "competitor_name",
        ]
    )
    .drop_duplicates(
        subset=[
            "candidate_aaAthleteId"
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


PROFILE_CANDIDATES[
    "already_in_canonical"
] = (
    PROFILE_CANDIDATES[
        "candidate_aaAthleteId"
    ]
    .isin(
        KNOWN_CANONICAL_ATHLETE_IDS
    )
)


KNOWN_PROFILE_CANDIDATES = (
    PROFILE_CANDIDATES.loc[
        PROFILE_CANDIDATES[
            "already_in_canonical"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


NEW_PROFILE_CANDIDATES = (
    PROFILE_CANDIDATES.loc[
        ~PROFILE_CANDIDATES[
            "already_in_canonical"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART E — DISCOVER LIVE getSingleCompetitor SIGNATURE
# ==================================================================================================

single_competitor_field = next(
    (
        field

        for field
        in query_fields

        if field.get(
            "name"
        )
        == "getSingleCompetitor"
    ),
    None,
)


if single_competitor_field is None:

    raise RuntimeError(
        "Live GraphQL schema does not expose getSingleCompetitor."
    )


SINGLE_COMPETITOR_RETURN_TYPE = (
    graphql_base_named_type(
        single_competitor_field[
            "type"
        ]
    )
)


SINGLE_COMPETITOR_ARGUMENTS = {
    arg[
        "name"
    ]:
        graphql_type_to_string(
            arg[
                "type"
            ]
        )

    for arg
    in single_competitor_field.get(
        "args",
        []
    )
}


if "id" not in SINGLE_COMPETITOR_ARGUMENTS:

    raise RuntimeError(
        "Live getSingleCompetitor operation has no 'id' argument."
    )


# ==================================================================================================
# PART F — DISCOVER basicData SCHEMA
# ==================================================================================================

single_competitor_definition = (
    await get_graphql_type_definition(
        SINGLE_COMPETITOR_RETURN_TYPE
    )
)


single_competitor_fields = (
    single_competitor_definition.get(
        "fields"
    )
    or []
)


basic_data_field = next(
    (
        field

        for field
        in single_competitor_fields

        if field.get(
            "name"
        )
        == "basicData"
    ),
    None,
)


if basic_data_field is None:

    raise RuntimeError(
        "getSingleCompetitor return type does not expose basicData."
    )


BASIC_DATA_TYPE = graphql_base_named_type(
    basic_data_field[
        "type"
    ]
)


basic_data_definition = (
    await get_graphql_type_definition(
        BASIC_DATA_TYPE
    )
)


basic_data_fields = (
    basic_data_definition.get(
        "fields"
    )
    or []
)


# --------------------------------------------------------------------------------------------------
# Keep all scalar / enum basicData fields.
# --------------------------------------------------------------------------------------------------

BASIC_DATA_SCALAR_FIELDS = []


for field in basic_data_fields:

    base_kind = graphql_base_kind(
        field[
            "type"
        ]
    )


    if base_kind in {
        "SCALAR",
        "ENUM",
    }:

        BASIC_DATA_SCALAR_FIELDS.append(
            field[
                "name"
            ]
        )


if "aaId" not in BASIC_DATA_SCALAR_FIELDS:

    raise RuntimeError(
        "Live basicData schema does not expose aaId."
    )


# ==================================================================================================
# PART G — BUILD LIVE PROFILE QUERY
# ==================================================================================================

basic_data_selection = "\n".join(
    BASIC_DATA_SCALAR_FIELDS
)


id_graphql_type = (
    SINGLE_COMPETITOR_ARGUMENTS[
        "id"
    ]
)


SINGLE_COMPETITOR_QUERY = f"""
query IncrementalSingleCompetitor(
    $id: {id_graphql_type}
) {{
    getSingleCompetitor(
        id: $id
    ) {{
        basicData {{
            {basic_data_selection}
        }}
    }}
}}
"""


# ==================================================================================================
# PART H — PROFILE FETCHER
# ==================================================================================================

async def fetch_single_competitor_profile(
    candidate_id,
):
    """
    Retrieve canonical World Athletics basicData for one athlete.
    """

    candidate_id = int(
        candidate_id
    )


    data = await wa_graphql(

        query=
            SINGLE_COMPETITOR_QUERY,

        variables={
            "id":
                candidate_id
        },

        operation_name=
            "IncrementalSingleCompetitor",
    )


    payload = data.get(
        "getSingleCompetitor"
    )


    if payload is None:

        raise RuntimeError(
            f"getSingleCompetitor returned null for {candidate_id}."
        )


    basic_data = (
        payload.get(
            "basicData"
        )
        or {}
    )


    if not isinstance(
        basic_data,
        dict,
    ):

        raise RuntimeError(
            f"basicData is not a dictionary for {candidate_id}."
        )


    canonical_id = normalize_wa_id(
        basic_data.get(
            "aaId"
        )
    )


    if canonical_id is None:

        raise RuntimeError(
            f"basicData.aaId is missing for {candidate_id}."
        )


    return {
        "candidate_id":
            candidate_id,

        "canonical_id":
            int(
                canonical_id
            ),

        "basicData":
            basic_data,

        "raw":
            payload,
    }


# ==================================================================================================
# PART I — PROFILE CHECKPOINT DIRECTORIES
# ==================================================================================================

PROFILE_RESOLUTION_ROOT = (
    RUN_CHECKPOINT_DIR
    / "profile_resolution"
)


PROFILE_RAW_DIR = (
    RUN_ATHLETE_DIR
    / "canonical_profiles"
)


PROFILE_ERROR_DIR = (
    PROFILE_RESOLUTION_ROOT
    / "errors"
)


for directory in [
    PROFILE_RESOLUTION_ROOT,
    PROFILE_RAW_DIR,
    PROFILE_ERROR_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================================================================
# PART J — RESOLVE NEW ATHLETES ONLY
# ==================================================================================================

new_resolution_records = []


N_NEW_TO_RESOLVE = int(
    len(
        NEW_PROFILE_CANDIDATES
    )
)


for position, (_, candidate) in enumerate(

    NEW_PROFILE_CANDIDATES.iterrows(),

    start=1,
):

    candidate_id = int(
        candidate[
            "candidate_aaAthleteId"
        ]
    )


    raw_path = (
        PROFILE_RAW_DIR
        / f"athlete_{candidate_id}.json"
    )


    error_path = (
        PROFILE_ERROR_DIR
        / f"athlete_{candidate_id}_error.json"
    )


    try:

        # ------------------------------------------------------------------------------------------
        # Resume from raw profile checkpoint
        # ------------------------------------------------------------------------------------------

        if raw_path.exists():

            with raw_path.open(
                "r",
                encoding="utf-8",
            ) as f:

                saved = json.load(
                    f
                )


            basic_data = (
                saved.get(
                    "basicData"
                )
                or {}
            )


            canonical_id = normalize_wa_id(
                basic_data.get(
                    "aaId"
                )
            )


            if canonical_id is None:

                raise RuntimeError(
                    "Saved profile checkpoint contains no valid aaId."
                )


            result = {
                "candidate_id":
                    candidate_id,

                "canonical_id":
                    int(
                        canonical_id
                    ),

                "basicData":
                    basic_data,

                "raw":
                    saved,
            }


            resumed = True


        else:

            result = (
                await fetch_single_competitor_profile(
                    candidate_id
                )
            )


            with raw_path.open(
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    result[
                        "raw"
                    ],
                    f,
                    indent=2,
                    default=str,
                )


            resumed = False


        canonical_id = int(
            result[
                "canonical_id"
            ]
        )


        basic_data = (
            result[
                "basicData"
            ]
        )


        record = {
            "candidate_aaAthleteId":
                candidate_id,

            "canonical_aaAthleteId":
                canonical_id,

            "id_match":
                bool(
                    candidate_id
                    == canonical_id
                ),

            "resolution_source":
                (
                    "PROFILE_CHECKPOINT"
                    if resumed
                    else "GET_SINGLE_COMPETITOR"
                ),

            "status":
                "RESOLVED",

            "raw_profile_path":
                str(
                    raw_path
                ),
        }


        # ------------------------------------------------------------------------------------------
        # Preserve all scalar basicData fields with a prefix
        # ------------------------------------------------------------------------------------------

        for key, value in basic_data.items():

            if isinstance(
                value,
                (
                    str,
                    int,
                    float,
                    bool,
                    type(None),
                )
            ):

                record[
                    f"profile_{key}"
                ] = value


        new_resolution_records.append(
            record
        )


        if (
            position == 1
            or position % 25 == 0
            or position == N_NEW_TO_RESOLVE
        ):

            print(
                f"[{position:>4}/{N_NEW_TO_RESOLVE}] "
                f"candidate={candidate_id} "
                f"canonical={canonical_id} "
                f"match={candidate_id == canonical_id}"
            )


    except Exception as exc:

        error_record = {
            "candidate_aaAthleteId":
                candidate_id,

            "competitor_name":
                candidate.get(
                    "competitor_name"
                ),

            "competitor_urlSlug":
                candidate.get(
                    "competitor_urlSlug"
                ),

            "status":
                "ERROR",

            "error":
                repr(
                    exc
                ),

            "traceback":
                traceback.format_exc(),
        }


        with error_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                error_record,
                f,
                indent=2,
                default=str,
            )


        new_resolution_records.append(
            error_record
        )


NEW_PROFILE_RESOLUTION = pd.DataFrame(
    new_resolution_records
)


# ==================================================================================================
# PART K — CREATE KNOWN-CANONICAL RESOLUTION RECORDS
# ==================================================================================================

known_records = []


for _, candidate in (
    KNOWN_PROFILE_CANDIDATES.iterrows()
):

    candidate_id = int(
        candidate[
            "candidate_aaAthleteId"
        ]
    )


    known_records.append(
        {
            "candidate_aaAthleteId":
                candidate_id,

            "canonical_aaAthleteId":
                candidate_id,

            "id_match":
                True,

            "resolution_source":
                "EXISTING_CANONICAL_DATASET",

            "status":
                "RESOLVED",

            "raw_profile_path":
                None,
        }
    )


KNOWN_PROFILE_RESOLUTION = pd.DataFrame(
    known_records
)


# ==================================================================================================
# PART L — COMBINE ALL PROFILE-BACKED RESOLUTIONS
# ==================================================================================================

resolution_frames = []


if len(
    KNOWN_PROFILE_RESOLUTION
) > 0:

    resolution_frames.append(
        KNOWN_PROFILE_RESOLUTION
    )


if len(
    NEW_PROFILE_RESOLUTION
) > 0:

    resolution_frames.append(
        NEW_PROFILE_RESOLUTION
    )


if resolution_frames:

    CANONICAL_PROFILE_RESOLUTION = pd.concat(

        resolution_frames,

        ignore_index=True,

        sort=False,
    )


else:

    CANONICAL_PROFILE_RESOLUTION = pd.DataFrame()


# ==================================================================================================
# PART M — JOIN BACK TO COMPETITION CANDIDATE METADATA
# ==================================================================================================

CANONICAL_PROFILE_CANDIDATES = (
    PROFILE_CANDIDATES.merge(

        CANONICAL_PROFILE_RESOLUTION,

        on=
            "candidate_aaAthleteId",

        how=
            "left",

        validate=
            "one_to_one",
    )
)


# ==================================================================================================
# PART N — QA
# ==================================================================================================

N_PROFILE_CANDIDATES = int(
    len(
        PROFILE_CANDIDATES
    )
)


N_KNOWN_CANONICAL = int(
    len(
        KNOWN_PROFILE_CANDIDATES
    )
)


N_NEW_CANDIDATES = int(
    len(
        NEW_PROFILE_CANDIDATES
    )
)


N_RESOLVED = int(
    (
        CANONICAL_PROFILE_CANDIDATES[
            "status"
        ]
        == "RESOLVED"
    )
    .sum()
)


N_PROFILE_ERRORS = int(
    (
        CANONICAL_PROFILE_CANDIDATES[
            "status"
        ]
        == "ERROR"
    )
    .sum()
)


N_ID_MISMATCHES = int(
    (
        (
            CANONICAL_PROFILE_CANDIDATES[
                "status"
            ]
            == "RESOLVED"
        )
        &
        (
            CANONICAL_PROFILE_CANDIDATES[
                "id_match"
            ]
            == False
        )
    )
    .sum()
)


ID_MISMATCHES = (
    CANONICAL_PROFILE_CANDIDATES.loc[
        (
            CANONICAL_PROFILE_CANDIDATES[
                "status"
            ]
            == "RESOLVED"
        )
        &
        (
            CANONICAL_PROFILE_CANDIDATES[
                "id_match"
            ]
            == False
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


PROFILE_RESOLUTION_ERRORS = (
    CANONICAL_PROFILE_CANDIDATES.loc[
        CANONICAL_PROFILE_CANDIDATES[
            "status"
        ]
        != "RESOLVED"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


PROFILE_RESOLUTION_PASS = bool(

    N_PROFILE_CANDIDATES
    == N_RESOLVED

    and
    N_PROFILE_ERRORS == 0
)


# ==================================================================================================
# PART O — RESULT-ROW CANONICAL ID MAPPING
# ==================================================================================================

candidate_to_canonical = {

    int(
        row[
            "candidate_aaAthleteId"
        ]
    ):
        int(
            row[
                "canonical_aaAthleteId"
        ]
    )

    for _, row
    in CANONICAL_PROFILE_CANDIDATES.loc[
        CANONICAL_PROFILE_CANDIDATES[
            "status"
        ]
        == "RESOLVED"
    ].iterrows()
}


INCREMENTAL_JAVELIN_RESULTS_CANONICAL = (
    INCREMENTAL_JAVELIN_RESULTS.copy()
)


INCREMENTAL_JAVELIN_RESULTS_CANONICAL[
    "candidate_aaAthleteId"
] = (
    INCREMENTAL_JAVELIN_RESULTS_CANONICAL[
        "candidate_aaAthleteId"
    ]
    .map(
        normalize_wa_id
    )
)


INCREMENTAL_JAVELIN_RESULTS_CANONICAL[
    "aaAthleteId"
] = (
    INCREMENTAL_JAVELIN_RESULTS_CANONICAL[
        "candidate_aaAthleteId"
    ]
    .map(
        candidate_to_canonical
    )
)


# No-profile rows deliberately remain null here.
# They are handled separately in Cell 9.


# ==================================================================================================
# PART P — SAVE CHECKPOINTS
# ==================================================================================================

PROFILE_RESOLUTION_PATH = (
    RUN_CHECKPOINT_DIR
    / "canonical_profile_resolution.parquet"
)


CANONICAL_PROFILE_CANDIDATES_PATH = (
    RUN_CHECKPOINT_DIR
    / "canonical_profile_candidates.parquet"
)


CANONICAL_INCREMENTAL_RESULTS_PATH = (
    RUN_CHECKPOINT_DIR
    / "incremental_javelin_results_identity_resolved.parquet"
)


CANONICAL_PROFILE_RESOLUTION.to_parquet(
    PROFILE_RESOLUTION_PATH,
    index=False,
)


CANONICAL_PROFILE_CANDIDATES.to_parquet(
    CANONICAL_PROFILE_CANDIDATES_PATH,
    index=False,
)


INCREMENTAL_JAVELIN_RESULTS_CANONICAL.to_parquet(
    CANONICAL_INCREMENTAL_RESULTS_PATH,
    index=False,
)


# ==================================================================================================
# PART Q — UPDATE MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "profile_identity_resolution"
] = {

    "status":
        (
            "VALIDATED"

            if PROFILE_RESOLUTION_PASS

            else "REVIEW_REQUIRED"
        ),

    "profile_candidates":
        N_PROFILE_CANDIDATES,

    "already_in_canonical":
        N_KNOWN_CANONICAL,

    "new_candidates_requiring_profile_lookup":
        N_NEW_CANDIDATES,

    "resolved":
        N_RESOLVED,

    "errors":
        N_PROFILE_ERRORS,

    "candidate_to_canonical_id_mismatches":
        N_ID_MISMATCHES,

    "no_profile_rows_pending":
        int(
            len(
                INCREMENTAL_NO_PROFILE_RESULTS
            )
        ),

    "resolution_path":
        str(
            PROFILE_RESOLUTION_PATH
        ),

    "resolved_results_path":
        str(
            CANONICAL_INCREMENTAL_RESULTS_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "PROFILE_IDENTITIES_RESOLVED"

    if PROFILE_RESOLUTION_PASS

    else "PROFILE_IDENTITY_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART R — OUTPUT
# ==================================================================================================

print()
print("=" * 125)
print("WORLD ATHLETICS CANONICAL ATHLETE IDENTITY RESOLUTION")
print("=" * 125)


print()
print("INCREMENTAL PROFILE POPULATION")
print("-" * 125)

print(
    f"Profile-backed candidates       : "
    f"{N_PROFILE_CANDIDATES:,}"
)

print(
    f"Already in canonical history    : "
    f"{N_KNOWN_CANONICAL:,}"
)

print(
    f"New athlete IDs                 : "
    f"{N_NEW_CANDIDATES:,}"
)

print(
    f"Profile API calls required      : "
    f"{N_NEW_CANDIDATES:,}"
)


print()
print("IDENTITY RESOLUTION")
print("-" * 125)

print(
    f"Successfully resolved           : "
    f"{N_RESOLVED:,}"
)

print(
    f"Resolution errors               : "
    f"{N_PROFILE_ERRORS:,}"
)

print(
    f"Candidate/canonical mismatches  : "
    f"{N_ID_MISMATCHES:,}"
)

print(
    f"No-profile rows still pending   : "
    f"{len(INCREMENTAL_NO_PROFILE_RESULTS):,}"
)


print()
print("LIVE PROFILE SCHEMA")
print("-" * 125)

print(
    f"Return type                     : "
    f"{SINGLE_COMPETITOR_RETURN_TYPE}"
)

print(
    f"basicData type                  : "
    f"{BASIC_DATA_TYPE}"
)

print(
    f"basicData scalar fields         : "
    f"{BASIC_DATA_SCALAR_FIELDS}"
)


if N_ID_MISMATCHES > 0:

    print()
    print("CANDIDATE → CANONICAL ID MISMATCHES")
    print("-" * 125)

    display(
        ID_MISMATCHES
    )


if N_PROFILE_ERRORS > 0:

    print()
    print("PROFILE RESOLUTION ERRORS")
    print("-" * 125)

    display(
        PROFILE_RESOLUTION_ERRORS
    )


print()
print("NEWLY RESOLVED ATHLETES")
print("-" * 125)

new_display = (
    CANONICAL_PROFILE_CANDIDATES.loc[
        CANONICAL_PROFILE_CANDIDATES[
            "resolution_source"
        ]
        .isin(
            [
                "GET_SINGLE_COMPETITOR",
                "PROFILE_CHECKPOINT",
            ]
        )
    ]
    .copy()
)


if len(
    new_display
) > 0:

    display(
        new_display.head(
            100
        )
    )

else:

    print(
        "All profile-backed candidates already existed in "
        "the canonical historical dataset."
    )


print()
print("CHECKPOINTS")
print("-" * 125)

print(
    f"Identity resolution             : "
    f"{PROFILE_RESOLUTION_PATH}"
)

print(
    f"Candidate/profile table         : "
    f"{CANONICAL_PROFILE_CANDIDATES_PATH}"
)

print(
    f"Identity-resolved results       : "
    f"{CANONICAL_INCREMENTAL_RESULTS_PATH}"
)


print()
print("=" * 125)


if PROFILE_RESOLUTION_PASS:

    print(
        "CELL 8 STATUS: PROFILE-BACKED ATHLETE IDENTITIES RESOLVED"
    )

    print()
    print(
        "Existing athletes were reused from the canonical history; "
        "only genuinely new IDs required World Athletics profile calls."
    )

    print()
    print(
        "Cell 9 will recover or adjudicate the small no-profile population "
        "and then define the final athlete-refresh list."
    )


else:

    print(
        "CELL 8 STATUS: IDENTITY REVIEW REQUIRED"
    )


print("=" * 125)

[   1/75] candidate=14228196 canonical=14228196 match=True
[  25/75] candidate=15212843 canonical=15212843 match=True
[  50/75] candidate=15346836 canonical=15346836 match=True
[  75/75] candidate=15347931 canonical=15347931 match=True

WORLD ATHLETICS CANONICAL ATHLETE IDENTITY RESOLUTION

INCREMENTAL PROFILE POPULATION
-----------------------------------------------------------------------------------------------------------------------------
Profile-backed candidates       : 867
Already in canonical history    : 792
New athlete IDs                 : 75
Profile API calls required      : 75

IDENTITY RESOLUTION
-----------------------------------------------------------------------------------------------------------------------------
Successfully resolved           : 867
Resolution errors               : 0
Candidate/canonical mismatches  : 0
No-profile rows still pending   : 13

LIVE PROFILE SCHEMA
--------------------------------------------------------------------------------------

,candidate_aaAthleteId,competitor_name,competitor_urlSlug,competitor_birthDate,source_competitor_id,source_iaafId,already_in_canonical,canonical_aaAthleteId,id_match,resolution_source,...,profile_aaId,profile_countryFullName,profile_familyName,profile_givenName,profile_birthDateStr,profile_facebookUsername,profile_twitterUsername,profile_instagramUsername,profile_male,profile_mediaIds
3,14228196,Nicklas WIBERG,sweden/nicklas-wiberg-14228196,16 APR 1985,edotalsn26v2yfj1z15le8,198427.0,False,14228196,True,GET_SINGLE_COMPETITOR,...,14228196.0,Sweden,WIBERG,Nicklas,None,None,None,None,True,None
17,14376821,Eikka PANKKONEN,finland/eikka-pankkonen-14376821,11 MAY 1989,6vovrazd8oxpdo8m0wti,NaN,False,14376821,True,GET_SINGLE_COMPETITOR,...,14376821.0,Finland,PANKKONEN,Eikka,None,None,None,None,True,None
52,14547867,Yuki MORI,japan/yuki-mori-14547867,14 MAY 1991,jt8u4c8qq2org7mh7dy5jf,NaN,False,14547867,True,GET_SINGLE_COMPETITOR,...,14547867.0,Japan,MORI,Yuki,None,None,None,None,True,None
66,14646249,Anastasia DAVIES,great-britain-ni/anastasia-davies-14646249,09 APR 1999,353dadlpfbjvq2kzx6o5id,340955.0,False,14646249,True,GET_SINGLE_COMPETITOR,...,14646249.0,Great Britain & N.I.,DAVIES,Anastasia,None,None,None,None,False,None
142,14800681,Florence BAULK,great-britain-ni/florence-baulk-14800681,None,14800681,491685.0,False,14800681,True,GET_SINGLE_COMPETITOR,...,14800681.0,Great Britain & N.I.,BAULK,Florence,None,None,None,None,False,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
862,15347573,Kouki KAGIYAMA,japan/kouki-kagiyama-15347573,17 APR 2004,15347573,NaN,False,15347573,True,GET_SINGLE_COMPETITOR,...,15347573.0,Japan,KAGIYAMA,Kouki,None,None,None,None,True,None
863,15347574,Yudai FURUSAWA,japan/yudai-furusawa-15347574,07 NOV 2004,15347574,NaN,False,15347574,True,GET_SINGLE_COMPETITOR,...,15347574.0,Japan,FURUSAWA,Yudai,None,None,None,None,True,None
864,15347651,Ivar THONÉRFELTH,sweden/ivar-thonerfelth-15347651,2009,15347651,NaN,False,15347651,True,GET_SINGLE_COMPETITOR,...,15347651.0,Sweden,THONÉRFELTH,Ivar,None,None,None,None,True,None
865,15347866,Cristian RUBILAR,chile/cristian-rubilar-15347866,04 JAN 2011,rl2s3zwb8vja9yeksvlkwr,NaN,False,15347866,True,GET_SINGLE_COMPETITOR,...,15347866.0,Chile,RUBILAR,Cristian,None,None,None,None,True,None



CHECKPOINTS
-----------------------------------------------------------------------------------------------------------------------------
Identity resolution             : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/canonical_profile_resolution.parquet
Candidate/profile table         : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/canonical_profile_candidates.parquet
Identity-resolved results       : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/incremental_javelin_results_identity_resolved.parquet

CELL 8 STATUS: PROFILE-BACKED ATHLETE IDENTITIES RESOLVED

Existing athletes were reused from the canonical history; only genuinely new IDs required World Athletics profile calls.

Cell 9 will recover or adjudicate the small no-profile population and then define the final athlete-refresh list.


In [9]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 9 — RECOVER NO-PROFILE ATHLETES + BUILD FINAL REFRESH PLAN
# ==================================================================================================
#
# Purpose
# -------
# Resolve the small residual population where competition results returned:
#
#       hasProfile = False
#
# even though a numeric World Athletics athlete ID appears in the urlSlug.
#
#
# Recovery hierarchy
# ------------------
#
#       numeric ID from urlSlug
#              ↓
#       already present in canonical history?
#              │
#              ├── YES → trust existing canonical aaAthleteId
#              │
#              └── NO
#                    ↓
#             getSingleCompetitor(id)
#                    ↓
#             basicData.aaId
#                    ↓
#             exact ID match required
#
#
# After identity resolution, this cell creates the FINAL athlete refresh plan:
#
#       EXISTING ATHLETE
#           → fetch only year(s) touched by incremental window
#
#       NEW ATHLETE
#           → fetch FULL active-year history
#
#
# IMPORTANT
# ---------
# This cell still does NOT download athlete histories.
#
# Cell 10 will execute the refresh plan.
# ==================================================================================================

from __future__ import annotations

import json
import re
import traceback

from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "RUN_ID",
    "RUN_ATHLETE_DIR",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "INCREMENTAL_START",
    "INCREMENTAL_END",

    "INCREMENTAL_NO_PROFILE_RESULTS",
    "INCREMENTAL_JAVELIN_RESULTS_CANONICAL",

    "KNOWN_CANONICAL_ATHLETE_IDS",

    "normalize_wa_id",
    "fetch_single_competitor_profile",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–8 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


# ==================================================================================================
# PART B — SLUG-ID EXTRACTION
# ==================================================================================================

NO_PROFILE_SLUG_PATTERN = re.compile(
    r"-(\d{7,10})(?:$|[/?#])"
)


def provisional_id_from_slug(
    slug,
):
    """
    Extract a provisional numeric WA athlete ID from a profile slug.

    Examples
    --------
    sweden/alma-carlson-15322465
        -> 15322465

    sri-lanka/example-014661567
        -> 14661567

    IMPORTANT
    ---------
    This is a recovery probe only.

    It does NOT become canonical until corroborated by:
        - existing historical canonical data, or
        - getSingleCompetitor.basicData.aaId
    """

    if slug is None:

        return None


    text = str(
        slug
    ).strip()


    if not text:

        return None


    match = NO_PROFILE_SLUG_PATTERN.search(
        text
    )


    if match is None:

        return None


    try:

        return int(
            match.group(
                1
            )
        )

    except Exception:

        return None


# ==================================================================================================
# PART C — BUILD UNIQUE NO-PROFILE CANDIDATE POPULATION
# ==================================================================================================

NO_PROFILE_SOURCE = (
    INCREMENTAL_NO_PROFILE_RESULTS.copy()
)


if len(
    NO_PROFILE_SOURCE
) > 0:

    NO_PROFILE_SOURCE[
        "provisional_aaAthleteId"
    ] = (

        NO_PROFILE_SOURCE[
            "competitor_urlSlug"
        ]
        .map(
            provisional_id_from_slug
        )
    )


    NO_PROFILE_CANDIDATES = (

        NO_PROFILE_SOURCE[
            [
                column
                for column
                in [
                    "provisional_aaAthleteId",
                    "competitor_name",
                    "competitor_urlSlug",
                    "competitor_birthDate",
                ]
                if column
                in NO_PROFILE_SOURCE.columns
            ]
        ]
        .drop_duplicates()
        .reset_index(
            drop=True
        )
    )


else:

    NO_PROFILE_SOURCE[
        "provisional_aaAthleteId"
    ] = pd.Series(
        dtype="Int64"
    )


    NO_PROFILE_CANDIDATES = pd.DataFrame(
        columns=[
            "provisional_aaAthleteId",
            "competitor_name",
            "competitor_urlSlug",
            "competitor_birthDate",
        ]
    )


# ==================================================================================================
# PART D — RECOVERY CHECKPOINT DIRECTORIES
# ==================================================================================================

NO_PROFILE_RECOVERY_ROOT = (
    RUN_CHECKPOINT_DIR
    / "no_profile_recovery"
)


NO_PROFILE_PROFILE_DIR = (
    RUN_ATHLETE_DIR
    / "no_profile_recovery"
)


NO_PROFILE_ERROR_DIR = (
    NO_PROFILE_RECOVERY_ROOT
    / "errors"
)


for directory in [
    NO_PROFILE_RECOVERY_ROOT,
    NO_PROFILE_PROFILE_DIR,
    NO_PROFILE_ERROR_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================================================================
# PART E — RESOLVE ONE NO-PROFILE CANDIDATE
# ==================================================================================================

async def resolve_no_profile_candidate(
    row,
):
    """
    Resolve one no-profile competitor conservatively.
    """

    provisional_id = normalize_wa_id(
        row.get(
            "provisional_aaAthleteId"
        )
    )


    competitor_name = row.get(
        "competitor_name"
    )


    competitor_slug = row.get(
        "competitor_urlSlug"
    )


    # ----------------------------------------------------------------------------------------------
    # No usable numeric slug
    # ----------------------------------------------------------------------------------------------

    if provisional_id is None:

        return {
            "provisional_aaAthleteId":
                None,

            "canonical_aaAthleteId":
                None,

            "competitor_name":
                competitor_name,

            "competitor_urlSlug":
                competitor_slug,

            "status":
                "UNRESOLVED",

            "resolution_source":
                "NO_NUMERIC_SLUG",

            "id_match":
                None,
        }


    provisional_id = int(
        provisional_id
    )


    # ----------------------------------------------------------------------------------------------
    # Existing canonical history corroborates this ID
    # ----------------------------------------------------------------------------------------------

    if provisional_id in KNOWN_CANONICAL_ATHLETE_IDS:

        return {
            "provisional_aaAthleteId":
                provisional_id,

            "canonical_aaAthleteId":
                provisional_id,

            "competitor_name":
                competitor_name,

            "competitor_urlSlug":
                competitor_slug,

            "status":
                "RESOLVED",

            "resolution_source":
                "EXISTING_CANONICAL_SLUG_CORROBORATION",

            "id_match":
                True,

            "raw_profile_path":
                None,
        }


    # ----------------------------------------------------------------------------------------------
    # New ID — verify against canonical WA profile endpoint
    # ----------------------------------------------------------------------------------------------

    raw_path = (
        NO_PROFILE_PROFILE_DIR
        / f"athlete_{provisional_id}.json"
    )


    try:

        if raw_path.exists():

            with raw_path.open(
                "r",
                encoding="utf-8",
            ) as f:

                saved_payload = json.load(
                    f
                )


            basic_data = (
                saved_payload.get(
                    "basicData"
                )
                or {}
            )


            canonical_id = normalize_wa_id(
                basic_data.get(
                    "aaId"
                )
            )


            resolution_source = (
                "NO_PROFILE_CHECKPOINT"
            )


        else:

            fetched = (
                await fetch_single_competitor_profile(
                    provisional_id
                )
            )


            canonical_id = normalize_wa_id(
                fetched[
                    "canonical_id"
                ]
            )


            saved_payload = (
                fetched[
                    "raw"
                ]
            )


            with raw_path.open(
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    saved_payload,
                    f,
                    indent=2,
                    default=str,
                )


            basic_data = (
                fetched[
                    "basicData"
                ]
            )


            resolution_source = (
                "GET_SINGLE_COMPETITOR"
            )


        # ------------------------------------------------------------------------------------------
        # Exact match required
        # ------------------------------------------------------------------------------------------

        if canonical_id is None:

            return {
                "provisional_aaAthleteId":
                    provisional_id,

                "canonical_aaAthleteId":
                    None,

                "competitor_name":
                    competitor_name,

                "competitor_urlSlug":
                    competitor_slug,

                "status":
                    "UNRESOLVED",

                "resolution_source":
                    resolution_source,

                "id_match":
                    None,

                "raw_profile_path":
                    str(
                        raw_path
                    ),
            }


        canonical_id = int(
            canonical_id
        )


        if canonical_id != provisional_id:

            return {
                "provisional_aaAthleteId":
                    provisional_id,

                "canonical_aaAthleteId":
                    canonical_id,

                "competitor_name":
                    competitor_name,

                "competitor_urlSlug":
                    competitor_slug,

                "status":
                    "REVIEW_REQUIRED",

                "resolution_source":
                    resolution_source,

                "id_match":
                    False,

                "raw_profile_path":
                    str(
                        raw_path
                    ),
            }


        result = {
            "provisional_aaAthleteId":
                provisional_id,

            "canonical_aaAthleteId":
                canonical_id,

            "competitor_name":
                competitor_name,

            "competitor_urlSlug":
                competitor_slug,

            "status":
                "RESOLVED",

            "resolution_source":
                resolution_source,

            "id_match":
                True,

            "raw_profile_path":
                str(
                    raw_path
                ),
        }


        # ------------------------------------------------------------------------------------------
        # Preserve useful profile fields
        # ------------------------------------------------------------------------------------------

        for field in [
            "fullName",
            "givenName",
            "familyName",
            "sexCode",
            "sexName",
            "countryCode",
            "countryFullName",
            "birthDate",
            "birthDateStr",
            "urlSlug",
        ]:

            if field in basic_data:

                result[
                    f"profile_{field}"
                ] = basic_data.get(
                    field
                )


        return result


    except Exception as exc:

        error_path = (
            NO_PROFILE_ERROR_DIR
            / f"athlete_{provisional_id}_error.json"
        )


        error_record = {
            "provisional_aaAthleteId":
                provisional_id,

            "competitor_name":
                competitor_name,

            "competitor_urlSlug":
                competitor_slug,

            "status":
                "ERROR",

            "resolution_source":
                "GET_SINGLE_COMPETITOR",

            "id_match":
                None,

            "error":
                repr(
                    exc
                ),

            "traceback":
                traceback.format_exc(),
        }


        with error_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                error_record,
                f,
                indent=2,
                default=str,
            )


        return error_record


# ==================================================================================================
# PART F — RESOLVE ALL UNIQUE NO-PROFILE CANDIDATES
# ==================================================================================================

no_profile_resolution_records = []


N_NO_PROFILE_CANDIDATES = int(
    len(
        NO_PROFILE_CANDIDATES
    )
)


for position, (_, candidate) in enumerate(

    NO_PROFILE_CANDIDATES.iterrows(),

    start=1,
):

    resolution = await resolve_no_profile_candidate(
        candidate
    )


    no_profile_resolution_records.append(
        resolution
    )


    print(
        f"[{position:>3}/{N_NO_PROFILE_CANDIDATES}] "
        f"{candidate.get('competitor_name')} | "
        f"candidate={resolution.get('provisional_aaAthleteId')} | "
        f"canonical={resolution.get('canonical_aaAthleteId')} | "
        f"{resolution.get('status')} | "
        f"{resolution.get('resolution_source')}"
    )


NO_PROFILE_RESOLUTION = pd.DataFrame(
    no_profile_resolution_records
)


# ==================================================================================================
# PART G — QA THE NO-PROFILE RECOVERY
# ==================================================================================================

if len(
    NO_PROFILE_RESOLUTION
) > 0:

    N_NO_PROFILE_RESOLVED = int(
        (
            NO_PROFILE_RESOLUTION[
                "status"
            ]
            == "RESOLVED"
        )
        .sum()
    )


    N_NO_PROFILE_UNRESOLVED = int(
        (
            NO_PROFILE_RESOLUTION[
                "status"
            ]
            == "UNRESOLVED"
        )
        .sum()
    )


    N_NO_PROFILE_REVIEW = int(
        (
            NO_PROFILE_RESOLUTION[
                "status"
            ]
            == "REVIEW_REQUIRED"
        )
        .sum()
    )


    N_NO_PROFILE_ERRORS = int(
        (
            NO_PROFILE_RESOLUTION[
                "status"
            ]
            == "ERROR"
        )
        .sum()
    )


else:

    N_NO_PROFILE_RESOLVED = 0
    N_NO_PROFILE_UNRESOLVED = 0
    N_NO_PROFILE_REVIEW = 0
    N_NO_PROFILE_ERRORS = 0


NO_PROFILE_RECOVERY_PASS = bool(

    N_NO_PROFILE_UNRESOLVED == 0

    and
    N_NO_PROFILE_REVIEW == 0

    and
    N_NO_PROFILE_ERRORS == 0
)


# ==================================================================================================
# PART H — BUILD PROVISIONAL → CANONICAL MAP
# ==================================================================================================

NO_PROFILE_ID_MAP = {}


if len(
    NO_PROFILE_RESOLUTION
) > 0:

    for _, row in (

        NO_PROFILE_RESOLUTION.loc[
            NO_PROFILE_RESOLUTION[
                "status"
            ]
            == "RESOLVED"
        ]
        .iterrows()
    ):

        provisional_id = normalize_wa_id(
            row.get(
                "provisional_aaAthleteId"
            )
        )


        canonical_id = normalize_wa_id(
            row.get(
                "canonical_aaAthleteId"
            )
        )


        if (
            provisional_id is not None
            and canonical_id is not None
        ):

            NO_PROFILE_ID_MAP[
                int(
                    provisional_id
                )
            ] = int(
                canonical_id
            )


# ==================================================================================================
# PART I — APPLY RECOVERED IDENTITIES TO THE COMPLETE INCREMENTAL RESULT TABLE
# ==================================================================================================

INCREMENTAL_RESULTS_IDENTITY_FINAL = (
    INCREMENTAL_JAVELIN_RESULTS_CANONICAL.copy()
)


# Every row gets a slug-derived provisional ID for recovery/audit.
INCREMENTAL_RESULTS_IDENTITY_FINAL[
    "slug_provisional_aaAthleteId"
] = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL[
        "competitor_urlSlug"
    ]
    .map(
        provisional_id_from_slug
    )
)


# Existing profile-backed rows already have aaAthleteId from Cell 8.
#
# Only fill currently-null identities.
missing_identity_mask = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL[
        "aaAthleteId"
    ]
    .isna()
)


INCREMENTAL_RESULTS_IDENTITY_FINAL.loc[
    missing_identity_mask,
    "aaAthleteId"
] = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL.loc[
        missing_identity_mask,
        "slug_provisional_aaAthleteId"
    ]
    .map(
        NO_PROFILE_ID_MAP
    )
)


INCREMENTAL_RESULTS_IDENTITY_FINAL[
    "aaAthleteId"
] = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL[
        "aaAthleteId"
    ]
    .map(
        normalize_wa_id
    )
)


# ==================================================================================================
# PART J — FINAL UNRESOLVED RESULT ROWS
# ==================================================================================================

FINAL_UNRESOLVED_RESULT_ROWS = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL.loc[
        INCREMENTAL_RESULTS_IDENTITY_FINAL[
            "aaAthleteId"
        ]
        .isna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


N_FINAL_UNRESOLVED_ROWS = int(
    len(
        FINAL_UNRESOLVED_RESULT_ROWS
    )
)


# ==================================================================================================
# PART K — BUILD FINAL UNIQUE ATHLETE POPULATION
# ==================================================================================================

resolved_incremental_ids = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL[
        "aaAthleteId"
    ]
    .dropna()
    .map(
        normalize_wa_id
    )
    .dropna()
    .astype(
        int
    )
    .unique()
)


FINAL_INCREMENTAL_ATHLETES = pd.DataFrame(
    {
        "aaAthleteId":
            sorted(
                resolved_incremental_ids
            )
    }
)


FINAL_INCREMENTAL_ATHLETES[
    "existed_before_run"
] = (

    FINAL_INCREMENTAL_ATHLETES[
        "aaAthleteId"
    ]
    .isin(
        KNOWN_CANONICAL_ATHLETE_IDS
    )
)


# ==================================================================================================
# PART L — DETERMINE YEARS TO REFRESH
# ==================================================================================================

WINDOW_YEARS = list(
    range(
        int(
            INCREMENTAL_START.year
        ),
        int(
            INCREMENTAL_END.year
        )
        + 1,
    )
)


FINAL_INCREMENTAL_ATHLETES[
    "refresh_scope"
] = np.where(

    FINAL_INCREMENTAL_ATHLETES[
        "existed_before_run"
    ],

    "WINDOW_YEARS_ONLY",

    "FULL_ACTIVE_YEAR_HISTORY",
)


FINAL_INCREMENTAL_ATHLETES[
    "window_years"
] = [

    WINDOW_YEARS.copy()

    for _ in range(
        len(
            FINAL_INCREMENTAL_ATHLETES
        )
    )
]


# ==================================================================================================
# PART M — RESULT-LEVEL DISCOVERY COUNTS
# ==================================================================================================

ATHLETE_DISCOVERY_COUNTS = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL.dropna(
        subset=[
            "aaAthleteId"
        ]
    )
    .groupby(
        "aaAthleteId",
        as_index=False,
    )
    .agg(
        incremental_result_rows=(
            "aaAthleteId",
            "size",
        ),

        competitions_seen=(
            "competition_id",
            "nunique",
        ),

        first_incremental_date=(
            "result_date_dt",
            "min",
        ),

        last_incremental_date=(
            "result_date_dt",
            "max",
        ),
    )
)


ATHLETE_DISCOVERY_COUNTS[
    "aaAthleteId"
] = (

    ATHLETE_DISCOVERY_COUNTS[
        "aaAthleteId"
    ]
    .map(
        normalize_wa_id
    )
    .astype(
        int
    )
)


FINAL_ATHLETE_REFRESH_PLAN = (

    FINAL_INCREMENTAL_ATHLETES.merge(

        ATHLETE_DISCOVERY_COUNTS,

        on=
            "aaAthleteId",

        how=
            "left",

        validate=
            "one_to_one",
    )
    .sort_values(
        [
            "existed_before_run",
            "aaAthleteId",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART N — REFRESH PLAN QA
# ==================================================================================================

N_FINAL_ATHLETES = int(
    len(
        FINAL_ATHLETE_REFRESH_PLAN
    )
)


N_EXISTING_REFRESH = int(
    FINAL_ATHLETE_REFRESH_PLAN[
        "existed_before_run"
    ]
    .sum()
)


N_NEW_FULL_HISTORY = int(
    (
        ~FINAL_ATHLETE_REFRESH_PLAN[
            "existed_before_run"
        ]
    )
    .sum()
)


REFRESH_PLAN_PASS = bool(

    NO_PROFILE_RECOVERY_PASS

    and
    N_FINAL_UNRESOLVED_ROWS == 0

    and
    N_FINAL_ATHLETES > 0
)


# ==================================================================================================
# PART O — SAVE CHECKPOINTS
# ==================================================================================================

NO_PROFILE_RESOLUTION_PATH = (
    RUN_CHECKPOINT_DIR
    / "no_profile_identity_resolution.parquet"
)


FINAL_IDENTITY_RESULTS_PATH = (
    RUN_CHECKPOINT_DIR
    / "incremental_javelin_results_identity_final.parquet"
)


FINAL_UNRESOLVED_PATH = (
    RUN_CHECKPOINT_DIR
    / "incremental_unresolved_results.parquet"
)


FINAL_REFRESH_PLAN_PATH = (
    RUN_CHECKPOINT_DIR
    / "final_athlete_refresh_plan.parquet"
)


NO_PROFILE_RESOLUTION.to_parquet(
    NO_PROFILE_RESOLUTION_PATH,
    index=False,
)


INCREMENTAL_RESULTS_IDENTITY_FINAL.to_parquet(
    FINAL_IDENTITY_RESULTS_PATH,
    index=False,
)


FINAL_UNRESOLVED_RESULT_ROWS.to_parquet(
    FINAL_UNRESOLVED_PATH,
    index=False,
)


FINAL_ATHLETE_REFRESH_PLAN.to_parquet(
    FINAL_REFRESH_PLAN_PATH,
    index=False,
)


# ==================================================================================================
# PART P — UPDATE MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "no_profile_recovery"
] = {

    "status":
        (
            "VALIDATED"
            if NO_PROFILE_RECOVERY_PASS
            else "REVIEW_REQUIRED"
        ),

    "source_result_rows":
        int(
            len(
                INCREMENTAL_NO_PROFILE_RESULTS
            )
        ),

    "unique_candidates":
        N_NO_PROFILE_CANDIDATES,

    "resolved":
        N_NO_PROFILE_RESOLVED,

    "unresolved":
        N_NO_PROFILE_UNRESOLVED,

    "review_required":
        N_NO_PROFILE_REVIEW,

    "errors":
        N_NO_PROFILE_ERRORS,

    "final_unresolved_result_rows":
        N_FINAL_UNRESOLVED_ROWS,

    "resolution_path":
        str(
            NO_PROFILE_RESOLUTION_PATH
        ),
}


RUN_MANIFEST[
    "athlete_refresh_plan"
] = {

    "status":
        (
            "VALIDATED"
            if REFRESH_PLAN_PASS
            else "REVIEW_REQUIRED"
        ),

    "total_athletes":
        N_FINAL_ATHLETES,

    "existing_athletes":
        N_EXISTING_REFRESH,

    "new_athletes_full_history":
        N_NEW_FULL_HISTORY,

    "window_years":
        WINDOW_YEARS,

    "refresh_plan_path":
        str(
            FINAL_REFRESH_PLAN_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "ATHLETE_REFRESH_PLAN_READY"

    if REFRESH_PLAN_PASS

    else "ATHLETE_REFRESH_PLAN_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART Q — OUTPUT
# ==================================================================================================

print()
print("=" * 125)
print("WORLD ATHLETICS NO-PROFILE RECOVERY + FINAL ATHLETE REFRESH PLAN")
print("=" * 125)


print()
print("NO-PROFILE RECOVERY")
print("-" * 125)

print(
    f"Source result rows              : "
    f"{len(INCREMENTAL_NO_PROFILE_RESULTS):,}"
)

print(
    f"Unique slug candidates          : "
    f"{N_NO_PROFILE_CANDIDATES:,}"
)

print(
    f"Resolved                        : "
    f"{N_NO_PROFILE_RESOLVED:,}"
)

print(
    f"Unresolved                      : "
    f"{N_NO_PROFILE_UNRESOLVED:,}"
)

print(
    f"Review required                 : "
    f"{N_NO_PROFILE_REVIEW:,}"
)

print(
    f"Errors                          : "
    f"{N_NO_PROFILE_ERRORS:,}"
)

print(
    f"Final unresolved result rows    : "
    f"{N_FINAL_UNRESOLVED_ROWS:,}"
)


print()
print("NO-PROFILE IDENTITY RESULTS")
print("-" * 125)

if len(
    NO_PROFILE_RESOLUTION
) > 0:

    display(
        NO_PROFILE_RESOLUTION
    )

else:

    print(
        "No no-profile candidates required recovery."
    )


print()
print("FINAL ATHLETE REFRESH POPULATION")
print("-" * 125)

print(
    f"Total athletes                  : "
    f"{N_FINAL_ATHLETES:,}"
)

print(
    f"Existing canonical athletes     : "
    f"{N_EXISTING_REFRESH:,}"
)

print(
    f"New athletes — full history     : "
    f"{N_NEW_FULL_HISTORY:,}"
)

print(
    f"Existing-athlete refresh years  : "
    f"{WINDOW_YEARS}"
)


print()
print("REFRESH SCOPE")
print("-" * 125)

display(
    FINAL_ATHLETE_REFRESH_PLAN.groupby(
        [
            "refresh_scope",
            "existed_before_run",
        ],
        as_index=False,
    )
    .agg(
        athletes=(
            "aaAthleteId",
            "nunique",
        ),

        incremental_result_rows=(
            "incremental_result_rows",
            "sum",
        ),
    )
)


print()
print("NEW ATHLETES REQUIRING FULL CAREER BACKFILL")
print("-" * 125)

new_athletes_display = (

    FINAL_ATHLETE_REFRESH_PLAN.loc[
        FINAL_ATHLETE_REFRESH_PLAN[
            "refresh_scope"
        ]
        == "FULL_ACTIVE_YEAR_HISTORY"
    ]
    .copy()
)


if len(
    new_athletes_display
) > 0:

    display(
        new_athletes_display.head(
            100
        )
    )

else:

    print(
        "None."
    )


if N_FINAL_UNRESOLVED_ROWS > 0:

    print()
    print("UNRESOLVED RESULT ROWS")
    print("-" * 125)

    display(
        FINAL_UNRESOLVED_RESULT_ROWS
    )


print()
print("CHECKPOINTS")
print("-" * 125)

print(
    f"No-profile resolution           : "
    f"{NO_PROFILE_RESOLUTION_PATH}"
)

print(
    f"Final identity-resolved results : "
    f"{FINAL_IDENTITY_RESULTS_PATH}"
)

print(
    f"Final athlete refresh plan      : "
    f"{FINAL_REFRESH_PLAN_PATH}"
)


print()
print("=" * 125)


if REFRESH_PLAN_PASS:

    print(
        "CELL 9 STATUS: FINAL ATHLETE REFRESH PLAN READY"
    )

    print()
    print(
        "Cell 10 will execute this plan:"
    )

    print(
        "  • existing athletes → refresh only incremental-window years"
    )

    print(
        "  • new athletes      → download all active years"
    )


else:

    print(
        "CELL 9 STATUS: IDENTITY REVIEW REQUIRED"
    )

    print()
    print(
        "Do not start athlete-history retrieval until unresolved "
        "identities have been reviewed."
    )


print("=" * 125)

[  1/8] Alma CARLSON | candidate=15322465 | canonical=None | ERROR | GET_SINGLE_COMPETITOR
[  2/8] Gabriel PRESELJ | candidate=15291162 | canonical=None | ERROR | GET_SINGLE_COMPETITOR
[  3/8] Leia GIMLANDER | candidate=15326984 | canonical=None | ERROR | GET_SINGLE_COMPETITOR
[  4/8] David LEES | candidate=15176055 | canonical=None | ERROR | GET_SINGLE_COMPETITOR
[  5/8] Elena ĐURIĆ | candidate=15299026 | canonical=None | ERROR | GET_SINGLE_COMPETITOR
[  6/8] Marko KOVAČ | candidate=15347617 | canonical=None | ERROR | GET_SINGLE_COMPETITOR
[  7/8] Lars BEUGELINK | candidate=15329741 | canonical=None | ERROR | GET_SINGLE_COMPETITOR
[  8/8] Róza DION | candidate=15301665 | canonical=None | ERROR | GET_SINGLE_COMPETITOR

WORLD ATHLETICS NO-PROFILE RECOVERY + FINAL ATHLETE REFRESH PLAN

NO-PROFILE RECOVERY
-----------------------------------------------------------------------------------------------------------------------------
Source result rows              : 13
Unique slug candidates

,provisional_aaAthleteId,competitor_name,competitor_urlSlug,status,resolution_source,id_match,error,traceback
0,15322465,Alma CARLSON,sweden/alma-carlson-15322465,ERROR,GET_SINGLE_COMPETITOR,None,RuntimeError('getSingleCompetitor returned nul...,"Traceback (most recent call last):\n File ""/v..."
1,15291162,Gabriel PRESELJ,sweden/gabriel-preselj-15291162,ERROR,GET_SINGLE_COMPETITOR,None,RuntimeError('getSingleCompetitor returned nul...,"Traceback (most recent call last):\n File ""/v..."
2,15326984,Leia GIMLANDER,sweden/leia-gimlander-15326984,ERROR,GET_SINGLE_COMPETITOR,None,RuntimeError('getSingleCompetitor returned nul...,"Traceback (most recent call last):\n File ""/v..."
3,15176055,David LEES,great-britain-ni/david-lees-15176055,ERROR,GET_SINGLE_COMPETITOR,None,RuntimeError('getSingleCompetitor returned nul...,"Traceback (most recent call last):\n File ""/v..."
4,15299026,Elena ĐURIĆ,serbia/elena-duric-15299026,ERROR,GET_SINGLE_COMPETITOR,None,RuntimeError('getSingleCompetitor returned nul...,"Traceback (most recent call last):\n File ""/v..."
5,15347617,Marko KOVAČ,serbia/marko-kovac-15347617,ERROR,GET_SINGLE_COMPETITOR,None,RuntimeError('getSingleCompetitor returned nul...,"Traceback (most recent call last):\n File ""/v..."
6,15329741,Lars BEUGELINK,netherlands/lars-beugelink-15329741,ERROR,GET_SINGLE_COMPETITOR,None,RuntimeError('getSingleCompetitor returned nul...,"Traceback (most recent call last):\n File ""/v..."
7,15301665,Róza DION,hungary/roza-dion-15301665,ERROR,GET_SINGLE_COMPETITOR,None,RuntimeError('getSingleCompetitor returned nul...,"Traceback (most recent call last):\n File ""/v..."



FINAL ATHLETE REFRESH POPULATION
-----------------------------------------------------------------------------------------------------------------------------
Total athletes                  : 867
Existing canonical athletes     : 792
New athletes — full history     : 75
Existing-athlete refresh years  : [2026]

REFRESH SCOPE
-----------------------------------------------------------------------------------------------------------------------------


,refresh_scope,existed_before_run,athletes,incremental_result_rows
0,FULL_ACTIVE_YEAR_HISTORY,False,75,126
1,WINDOW_YEARS_ONLY,True,792,1165



NEW ATHLETES REQUIRING FULL CAREER BACKFILL
-----------------------------------------------------------------------------------------------------------------------------


,aaAthleteId,existed_before_run,refresh_scope,window_years,incremental_result_rows,competitions_seen,first_incremental_date,last_incremental_date
792,14228196,False,FULL_ACTIVE_YEAR_HISTORY,[2026],1,1,2026-09-12,2026-09-12
793,14376821,False,FULL_ACTIVE_YEAR_HISTORY,[2026],1,1,2026-09-12,2026-09-12
794,14547867,False,FULL_ACTIVE_YEAR_HISTORY,[2026],1,1,2026-09-13,2026-09-13
795,14646249,False,FULL_ACTIVE_YEAR_HISTORY,[2026],1,1,2026-09-12,2026-09-12
796,14800681,False,FULL_ACTIVE_YEAR_HISTORY,[2026],2,1,2026-09-06,2026-09-06
...,...,...,...,...,...,...,...,...
862,15347573,False,FULL_ACTIVE_YEAR_HISTORY,[2026],2,1,2026-09-06,2026-09-06
863,15347574,False,FULL_ACTIVE_YEAR_HISTORY,[2026],2,1,2026-09-06,2026-09-06
864,15347651,False,FULL_ACTIVE_YEAR_HISTORY,[2026],2,1,2026-09-06,2026-09-06
865,15347866,False,FULL_ACTIVE_YEAR_HISTORY,[2026],1,1,2026-09-12,2026-09-12



UNRESOLVED RESULT ROWS
-----------------------------------------------------------------------------------------------------------------------------


,competition_id,competition_name,requested_day,source_layer,event_title,ranking_category,event,event_id,gender,is_relay,...,competitor_hasProfile,candidate_aaAthleteId,source_competitor_id,source_iaafId,competitor_birthDate,result_date_dt,inside_incremental_window,summary_index,aaAthleteId,slug_provisional_aaAthleteId
0,7237674,Swedish U18 Team Championships,1,race_result,U18 Events,F,Women's Javelin Throw (500g),10229927,W,False,...,False,NaN,lbqfcn7y2f9gevoypk04cu,NaN,01 APR 2010,2026-09-13,True,NaN,NaN,15322465
1,7237949,Swedish U18 Combined Events Championships,2,summary,Combined Events - U17,F,Men's Javelin Throw (700g),10229828,M,False,...,False,NaN,15291162,NaN,05 JUN 2010,2026-09-06,True,6.0,NaN,15291162
2,7237949,Swedish U18 Combined Events Championships,2,race_result,Combined Events - U17,F,Men's Javelin Throw (700g),10229828,M,False,...,False,NaN,cyscmz5xs1e5a08n0nwxva,NaN,05 JUN 2010,2026-09-06,True,NaN,NaN,15291162
3,7237949,Swedish U18 Combined Events Championships,2,summary,Combined Events - U17,F,Women's Javelin Throw (500g),10229927,W,False,...,False,NaN,15326984,NaN,09 APR 2010,2026-09-06,True,8.0,NaN,15326984
4,7237949,Swedish U18 Combined Events Championships,2,race_result,Combined Events - U17,F,Women's Javelin Throw (500g),10229927,W,False,...,False,NaN,r17ttl8i8zpm84jf1ss6da,NaN,09 APR 2010,2026-09-06,True,NaN,NaN,15326984
5,7239162,4J Studios scottishathletics National Senior &...,2,race_result,None,F,Men's Javelin Throw,10229636,M,False,...,False,NaN,g7fpxx2c33lmfh0k5ztoh,NaN,None,2026-08-30,True,NaN,NaN,15176055
6,7239941,Serbian U18 Combined Events Championships,2,summary,Combined Events,F,Women's Javelin Throw (500g),10229927,W,False,...,False,NaN,15299026,NaN,2010,2026-09-05,True,2.0,NaN,15299026
7,7239941,Serbian U18 Combined Events Championships,2,race_result,Combined Events,F,Women's Javelin Throw (500g),10229927,W,False,...,False,NaN,15299026,NaN,2010,2026-09-05,True,NaN,NaN,15299026
8,7239944,Serbian U20 Combined Events Championships,2,summary,Combined Events,F,Men's Javelin Throw,10229636,M,False,...,False,NaN,15347617,NaN,None,2026-09-06,True,1.0,NaN,15347617
9,7239944,Serbian U20 Combined Events Championships,2,race_result,Combined Events,F,Men's Javelin Throw,10229636,M,False,...,False,NaN,15347617,NaN,None,2026-09-06,True,NaN,NaN,15347617



CHECKPOINTS
-----------------------------------------------------------------------------------------------------------------------------
No-profile resolution           : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/no_profile_identity_resolution.parquet
Final identity-resolved results : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/incremental_javelin_results_identity_final.parquet
Final athlete refresh plan      : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/final_athlete_refresh_plan.parquet

CELL 9 STATUS: IDENTITY REVIEW REQUIRED

Do not start athlete-history retrieval until unresolved identities have been reviewed.


In [10]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 9A — QUARANTINE WORLD ATHLETICS RESULT-ONLY PARTICIPANTS
# ==================================================================================================
#
# Why this cell exists
# --------------------
# Cell 9 identified a legitimate World Athletics source-system condition:
#
#       competition result exists
#       hasProfile = False
#       numeric slug may exist
#       getSingleCompetitor(id) = null
#
# These rows cannot safely receive a canonical aaAthleteId.
#
# Production policy:
#
#       DO NOT invent an athlete identity
#       DO NOT discard the raw result
#       DO NOT block the entire incremental ingestion
#
# Instead:
#
#       quarantine result
#       exclude from longitudinal model data
#       preserve for audit
#       retry in later incremental runs
#
# ==================================================================================================

from __future__ import annotations

import json

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "RUN_ID",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "NO_PROFILE_RESOLUTION",
    "FINAL_UNRESOLVED_RESULT_ROWS",
    "INCREMENTAL_RESULTS_IDENTITY_FINAL",

    "FINAL_ATHLETE_REFRESH_PLAN",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cell 9 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


# ==================================================================================================
# PART B — DEFINE THE KNOWN QUARANTINE CONDITION
# ==================================================================================================

def is_expected_no_profile_null(
    row,
):
    """
    Identify the specific, acceptable WA condition:

        hasProfile=False
        getSingleCompetitor returns null

    We only quarantine this explicit condition.

    Other profile-resolution failures remain genuine errors.
    """

    status = str(
        row.get(
            "status",
            ""
        )
    )


    error_text = str(
        row.get(
            "error",
            ""
        )
    )


    return (
        status == "ERROR"

        and
        "getSingleCompetitor returned null"
        in error_text
    )


# ==================================================================================================
# PART C — RECLASSIFY NO-PROFILE RESOLUTION RECORDS
# ==================================================================================================

NO_PROFILE_RESOLUTION_FINAL = (
    NO_PROFILE_RESOLUTION.copy()
)


NO_PROFILE_RESOLUTION_FINAL[
    "quarantinable_source_condition"
] = (

    NO_PROFILE_RESOLUTION_FINAL.apply(
        is_expected_no_profile_null,
        axis=1,
    )
)


quarantine_resolution_mask = (

    NO_PROFILE_RESOLUTION_FINAL[
        "quarantinable_source_condition"
    ]
    == True
)


NO_PROFILE_RESOLUTION_FINAL.loc[
    quarantine_resolution_mask,
    "status"
] = "QUARANTINED_NO_PROFILE"


NO_PROFILE_RESOLUTION_FINAL.loc[
    quarantine_resolution_mask,
    "resolution_source"
] = "WA_RESULT_ONLY_PARTICIPANT"


NO_PROFILE_RESOLUTION_FINAL[
    "quarantine_reason"
] = None


NO_PROFILE_RESOLUTION_FINAL.loc[
    quarantine_resolution_mask,
    "quarantine_reason"
] = (
    "World Athletics result participant has hasProfile=False "
    "and getSingleCompetitor returned null."
)


NO_PROFILE_RESOLUTION_FINAL[
    "eligible_for_canonical_model"
] = (

    NO_PROFILE_RESOLUTION_FINAL[
        "status"
    ]
    == "RESOLVED"
)


NO_PROFILE_RESOLUTION_FINAL[
    "retry_future_incremental_run"
] = (

    NO_PROFILE_RESOLUTION_FINAL[
        "status"
    ]
    == "QUARANTINED_NO_PROFILE"
)


# ==================================================================================================
# PART D — QA ANY NON-QUARANTINABLE FAILURES
# ==================================================================================================

NON_QUARANTINABLE_IDENTITY_FAILURES = (

    NO_PROFILE_RESOLUTION_FINAL.loc[
        ~NO_PROFILE_RESOLUTION_FINAL[
            "status"
        ]
        .isin(
            [
                "RESOLVED",
                "QUARANTINED_NO_PROFILE",
            ]
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


N_NON_QUARANTINABLE_FAILURES = int(
    len(
        NON_QUARANTINABLE_IDENTITY_FAILURES
    )
)


# ==================================================================================================
# PART E — CREATE QUARANTINED RESULT TABLE
# ==================================================================================================

QUARANTINED_NO_PROFILE_RESULTS = (

    FINAL_UNRESOLVED_RESULT_ROWS.copy()
)


if len(
    QUARANTINED_NO_PROFILE_RESULTS
) > 0:

    QUARANTINED_NO_PROFILE_RESULTS[
        "identity_status"
    ] = "QUARANTINED_NO_PROFILE"


    QUARANTINED_NO_PROFILE_RESULTS[
        "quarantine_reason"
    ] = (
        "World Athletics result participant has no canonical athlete profile."
    )


    QUARANTINED_NO_PROFILE_RESULTS[
        "eligible_for_canonical_model"
    ] = False


    QUARANTINED_NO_PROFILE_RESULTS[
        "retry_future_incremental_run"
    ] = True


# ==================================================================================================
# PART F — LABEL EVERY INCREMENTAL RESULT ROW
# ==================================================================================================

INCREMENTAL_RESULTS_IDENTITY_FINAL = (
    INCREMENTAL_RESULTS_IDENTITY_FINAL.copy()
)


INCREMENTAL_RESULTS_IDENTITY_FINAL[
    "identity_status"
] = np.where(

    INCREMENTAL_RESULTS_IDENTITY_FINAL[
        "aaAthleteId"
    ]
    .notna(),

    "CANONICAL_ID_RESOLVED",

    "QUARANTINED_NO_PROFILE",
)


INCREMENTAL_RESULTS_IDENTITY_FINAL[
    "eligible_for_canonical_model"
] = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL[
        "aaAthleteId"
    ]
    .notna()
)


# ==================================================================================================
# PART G — MODEL-ELIGIBLE DISCOVERY TABLE
# ==================================================================================================

INCREMENTAL_RESULTS_MODEL_ELIGIBLE = (

    INCREMENTAL_RESULTS_IDENTITY_FINAL.loc[
        INCREMENTAL_RESULTS_IDENTITY_FINAL[
            "eligible_for_canonical_model"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


N_MODEL_ELIGIBLE_RESULT_ROWS = int(
    len(
        INCREMENTAL_RESULTS_MODEL_ELIGIBLE
    )
)


N_QUARANTINED_RESULT_ROWS = int(
    len(
        QUARANTINED_NO_PROFILE_RESULTS
    )
)


# ==================================================================================================
# PART H — VALIDATE FINAL REFRESH PLAN
# ==================================================================================================

N_REFRESH_ATHLETES = int(
    len(
        FINAL_ATHLETE_REFRESH_PLAN
    )
)


N_EXISTING_REFRESH = int(
    FINAL_ATHLETE_REFRESH_PLAN[
        "existed_before_run"
    ]
    .sum()
)


N_NEW_FULL_HISTORY = int(
    (
        ~FINAL_ATHLETE_REFRESH_PLAN[
            "existed_before_run"
        ]
    )
    .sum()
)


FINAL_REFRESH_PLAN_PASS = bool(

    N_NON_QUARANTINABLE_FAILURES == 0

    and
    N_REFRESH_ATHLETES > 0

    and
    FINAL_ATHLETE_REFRESH_PLAN[
        "aaAthleteId"
    ]
    .notna()
    .all()
)


# ==================================================================================================
# PART I — SAVE FINAL CHECKPOINTS
# ==================================================================================================

NO_PROFILE_RESOLUTION_FINAL_PATH = (

    RUN_CHECKPOINT_DIR
    / "no_profile_identity_resolution_final.parquet"
)


QUARANTINE_PATH = (

    RUN_CHECKPOINT_DIR
    / "quarantined_no_profile_results.parquet"
)


MODEL_ELIGIBLE_DISCOVERY_PATH = (

    RUN_CHECKPOINT_DIR
    / "incremental_javelin_results_model_eligible.parquet"
)


FINAL_IDENTITY_RESULTS_V2_PATH = (

    RUN_CHECKPOINT_DIR
    / "incremental_javelin_results_identity_final_v2.parquet"
)


NO_PROFILE_RESOLUTION_FINAL.to_parquet(

    NO_PROFILE_RESOLUTION_FINAL_PATH,

    index=False,
)


QUARANTINED_NO_PROFILE_RESULTS.to_parquet(

    QUARANTINE_PATH,

    index=False,
)


INCREMENTAL_RESULTS_MODEL_ELIGIBLE.to_parquet(

    MODEL_ELIGIBLE_DISCOVERY_PATH,

    index=False,
)


INCREMENTAL_RESULTS_IDENTITY_FINAL.to_parquet(

    FINAL_IDENTITY_RESULTS_V2_PATH,

    index=False,
)


# ==================================================================================================
# PART J — UPDATE MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "no_profile_recovery"
] = {

    "status":
        "QUARANTINED",

    "source_result_rows":
        int(
            len(
                FINAL_UNRESOLVED_RESULT_ROWS
            )
        ),

    "quarantined_result_rows":
        N_QUARANTINED_RESULT_ROWS,

    "non_quarantinable_identity_failures":
        N_NON_QUARANTINABLE_FAILURES,

    "policy":
        (
            "Preserve raw result, exclude from canonical longitudinal model, "
            "retry on future incremental run."
        ),

    "resolution_path":
        str(
            NO_PROFILE_RESOLUTION_FINAL_PATH
        ),

    "quarantine_path":
        str(
            QUARANTINE_PATH
        ),
}


RUN_MANIFEST[
    "athlete_refresh_plan"
] = {

    "status":
        (
            "VALIDATED"

            if FINAL_REFRESH_PLAN_PASS

            else "REVIEW_REQUIRED"
        ),

    "total_athletes":
        N_REFRESH_ATHLETES,

    "existing_athletes":
        N_EXISTING_REFRESH,

    "new_athletes_full_history":
        N_NEW_FULL_HISTORY,

    "quarantined_result_rows":
        N_QUARANTINED_RESULT_ROWS,

    "model_eligible_discovery_rows":
        N_MODEL_ELIGIBLE_RESULT_ROWS,

    "refresh_plan_path":
        str(
            FINAL_REFRESH_PLAN_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "ATHLETE_REFRESH_PLAN_READY"

    if FINAL_REFRESH_PLAN_PASS

    else "ATHLETE_REFRESH_PLAN_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART K — OUTPUT
# ==================================================================================================

print()
print("=" * 125)
print("WORLD ATHLETICS RESULT-ONLY PARTICIPANT QUARANTINE")
print("=" * 125)


print()
print("IDENTITY ADJUDICATION")
print("-" * 125)

print(
    f"No-profile result rows          : "
    f"{len(FINAL_UNRESOLVED_RESULT_ROWS):,}"
)

print(
    f"Quarantined result rows         : "
    f"{N_QUARANTINED_RESULT_ROWS:,}"
)

print(
    f"Non-quarantinable failures     : "
    f"{N_NON_QUARANTINABLE_FAILURES:,}"
)


print()
print("CANONICAL PIPELINE")
print("-" * 125)

print(
    f"Model-eligible discovery rows   : "
    f"{N_MODEL_ELIGIBLE_RESULT_ROWS:,}"
)

print(
    f"Athletes in refresh plan        : "
    f"{N_REFRESH_ATHLETES:,}"
)

print(
    f"Existing athletes               : "
    f"{N_EXISTING_REFRESH:,}"
)

print(
    f"New athletes — full history     : "
    f"{N_NEW_FULL_HISTORY:,}"
)


print()
print("QUARANTINED IDENTITIES")
print("-" * 125)

display(
    NO_PROFILE_RESOLUTION_FINAL[
        [
            column
            for column
            in [
                "provisional_aaAthleteId",
                "competitor_name",
                "competitor_urlSlug",
                "status",
                "quarantine_reason",
                "retry_future_incremental_run",
            ]
            if column
            in NO_PROFILE_RESOLUTION_FINAL.columns
        ]
    ]
)


if N_NON_QUARANTINABLE_FAILURES > 0:

    print()
    print("NON-QUARANTINABLE FAILURES")
    print("-" * 125)

    display(
        NON_QUARANTINABLE_IDENTITY_FAILURES
    )


print()
print("CHECKPOINTS")
print("-" * 125)

print(
    f"Final no-profile resolution     : "
    f"{NO_PROFILE_RESOLUTION_FINAL_PATH}"
)

print(
    f"Quarantined raw results         : "
    f"{QUARANTINE_PATH}"
)

print(
    f"Model-eligible discovery rows   : "
    f"{MODEL_ELIGIBLE_DISCOVERY_PATH}"
)


print()
print("=" * 125)


if FINAL_REFRESH_PLAN_PASS:

    print(
        "CELL 9A STATUS: ATHLETE REFRESH PLAN RELEASED"
    )

    print()
    print(
        "The 867 canonical athletes may proceed to athlete-history refresh."
    )

    print(
        "The result-only participants remain preserved in quarantine and "
        "will be retried on future incremental runs."
    )

    print()
    print(
        "Cell 10 will now download:"
    )

    print(
        "  • 2026 history only for existing canonical athletes"
    )

    print(
        "  • all active years for newly discovered athletes"
    )


else:

    print(
        "CELL 9A STATUS: REVIEW REQUIRED"
    )


print("=" * 125)


WORLD ATHLETICS RESULT-ONLY PARTICIPANT QUARANTINE

IDENTITY ADJUDICATION
-----------------------------------------------------------------------------------------------------------------------------
No-profile result rows          : 13
Quarantined result rows         : 13
Non-quarantinable failures     : 0

CANONICAL PIPELINE
-----------------------------------------------------------------------------------------------------------------------------
Model-eligible discovery rows   : 1,291
Athletes in refresh plan        : 867
Existing athletes               : 792
New athletes — full history     : 75

QUARANTINED IDENTITIES
-----------------------------------------------------------------------------------------------------------------------------


,provisional_aaAthleteId,competitor_name,competitor_urlSlug,status,quarantine_reason,retry_future_incremental_run
0,15322465,Alma CARLSON,sweden/alma-carlson-15322465,QUARANTINED_NO_PROFILE,World Athletics result participant has hasProf...,True
1,15291162,Gabriel PRESELJ,sweden/gabriel-preselj-15291162,QUARANTINED_NO_PROFILE,World Athletics result participant has hasProf...,True
2,15326984,Leia GIMLANDER,sweden/leia-gimlander-15326984,QUARANTINED_NO_PROFILE,World Athletics result participant has hasProf...,True
3,15176055,David LEES,great-britain-ni/david-lees-15176055,QUARANTINED_NO_PROFILE,World Athletics result participant has hasProf...,True
4,15299026,Elena ĐURIĆ,serbia/elena-duric-15299026,QUARANTINED_NO_PROFILE,World Athletics result participant has hasProf...,True
5,15347617,Marko KOVAČ,serbia/marko-kovac-15347617,QUARANTINED_NO_PROFILE,World Athletics result participant has hasProf...,True
6,15329741,Lars BEUGELINK,netherlands/lars-beugelink-15329741,QUARANTINED_NO_PROFILE,World Athletics result participant has hasProf...,True
7,15301665,Róza DION,hungary/roza-dion-15301665,QUARANTINED_NO_PROFILE,World Athletics result participant has hasProf...,True



CHECKPOINTS
-----------------------------------------------------------------------------------------------------------------------------
Final no-profile resolution     : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/no_profile_identity_resolution_final.parquet
Quarantined raw results         : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/quarantined_no_profile_results.parquet
Model-eligible discovery rows   : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/incremental_javelin_results_model_eligible.parquet

CELL 9A STATUS: ATHLETE REFRESH PLAN RELEASED

The 867 canonical athletes may proceed to athlete-history refresh.
The result-only participants remain preserved in quarantine and will be retried on future incremental runs.

Cell 10 will now download:
  • 2026 history only for existing canonica

In [11]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 10 — EXECUTE ATHLETE-HISTORY REFRESH
# ==================================================================================================
#
# Purpose
# -------
# Execute the final athlete refresh plan from Cells 9/9A.
#
#
# EXISTING ATHLETES
# -----------------
# Fetch only years touched by the incremental update window.
#
# Example for this run:
#
#       2026
#
#
# NEW ATHLETES
# ------------
# A newly discovered athlete must not enter the model with only their latest
# result.
#
# Therefore:
#
#       fetch 2026
#           ↓
#       read activeYears
#           ↓
#       fetch every active year
#
#
# CHECKPOINTING
# -------------
# One compressed JSON file is written per:
#
#       athlete × year
#
# Therefore an interrupted run is safely resumable.
#
#
# IMPORTANT
# ---------
# getSingleCompetitorResultsDate returns the athlete's complete athletics
# history for the requested year, not only javelin.
#
# Cell 11 will:
#
#       filter javelin
#       normalize marks
#       resolve implement groups
#       remove summary/race duplicates
#       reconcile against the historical canonical dataset
#
# ==================================================================================================

from __future__ import annotations

import gzip
import json
import traceback

from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "RUN_ID",
    "RUN_ATHLETE_DIR",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "FINAL_ATHLETE_REFRESH_PLAN",
    "WINDOW_YEARS",

    "fetch_athlete_year_results",

    "FINAL_REFRESH_PLAN_PASS",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–9A first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not FINAL_REFRESH_PLAN_PASS:

    raise RuntimeError(
        "The athlete refresh plan has not passed validation."
    )


# ==================================================================================================
# PART B — HISTORY REFRESH DIRECTORIES
# ==================================================================================================

HISTORY_REFRESH_ROOT = (
    RUN_CHECKPOINT_DIR
    / "athlete_history_refresh"
)


HISTORY_RAW_ROOT = (
    RUN_ATHLETE_DIR
    / "history"
)


HISTORY_ERROR_ROOT = (
    HISTORY_REFRESH_ROOT
    / "errors"
)


HISTORY_MANIFEST_ROOT = (
    HISTORY_REFRESH_ROOT
    / "athlete_manifests"
)


for directory in [
    HISTORY_REFRESH_ROOT,
    HISTORY_RAW_ROOT,
    HISTORY_ERROR_ROOT,
    HISTORY_MANIFEST_ROOT,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================================================================
# PART C — RUNTIME SETTINGS
# ==================================================================================================

ENABLE_ATHLETE_HISTORY_REFRESH = True


# For new athletes we need one known/current year from which to obtain activeYears.
#
# The latest year touched by the incremental window is the safest bootstrap.
ACTIVE_YEAR_BOOTSTRAP_YEAR = int(
    max(
        WINDOW_YEARS
    )
)


# ==================================================================================================
# PART D — YEAR NORMALIZATION
# ==================================================================================================

def normalize_year_list(
    values,
):
    """
    Convert World Athletics activeYears values to sorted unique integer years.
    """

    output = []


    for value in (
        values
        or []
    ):

        try:

            year = int(
                value
            )

        except Exception:

            continue


        # Broad sanity constraint only.
        if (
            year >= 1900
            and
            year <= 2100
        ):

            output.append(
                year
            )


    return sorted(
        set(
            output
        )
    )


# ==================================================================================================
# PART E — CHECKPOINT PATH HELPERS
# ==================================================================================================

def athlete_history_paths(
    athlete_id,
):
    """
    Return history-refresh paths for one athlete.
    """

    athlete_id = int(
        athlete_id
    )


    raw_dir = (
        HISTORY_RAW_ROOT
        / str(
            athlete_id
        )
    )


    raw_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    return {

        "raw_dir":
            raw_dir,

        "manifest_path":
            HISTORY_MANIFEST_ROOT
            / f"athlete_{athlete_id}.json",

        "error_dir":
            HISTORY_ERROR_ROOT
            / str(
                athlete_id
            ),
    }


def athlete_year_path(
    athlete_id,
    year,
):
    """
    Compressed raw athlete-year checkpoint.
    """

    paths = athlete_history_paths(
        athlete_id
    )


    return (
        paths[
            "raw_dir"
        ]
        / f"year_{int(year)}.json.gz"
    )


# ==================================================================================================
# PART F — READ/WRITE COMPRESSED JSON
# ==================================================================================================

def write_json_gzip(
    path,
    obj,
):
    """
    Write JSON using gzip compression.
    """

    path = Path(
        path
    )


    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    with gzip.open(
        path,
        "wt",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            default=str,
        )


def read_json_gzip(
    path,
):
    """
    Read compressed JSON checkpoint.
    """

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
    ) as f:

        return json.load(
            f
        )


# ==================================================================================================
# PART G — CACHED ATHLETE-YEAR FETCHER
# ==================================================================================================

async def fetch_athlete_year_cached(
    athlete_id,
    year,
):
    """
    Fetch or reload one athlete/year.

    Returns
    -------
    dict
        Same structure as fetch_athlete_year_results(), plus:
            checkpoint_path
            from_checkpoint
    """

    athlete_id = int(
        athlete_id
    )

    year = int(
        year
    )


    checkpoint_path = athlete_year_path(
        athlete_id,
        year,
    )


    # ----------------------------------------------------------------------------------------------
    # Resume
    # ----------------------------------------------------------------------------------------------

    if checkpoint_path.exists():

        saved = read_json_gzip(
            checkpoint_path
        )


        return {
            **saved,

            "checkpoint_path":
                str(
                    checkpoint_path
                ),

            "from_checkpoint":
                True,
        }


    # ----------------------------------------------------------------------------------------------
    # Live fetch
    # ----------------------------------------------------------------------------------------------

    fetched = await fetch_athlete_year_results(

        athlete_id=
            athlete_id,

        year=
            year,
    )


    saved = {

        "athlete_id":
            int(
                fetched[
                    "athlete_id"
                ]
            ),

        "year":
            int(
                fetched[
                    "year"
                ]
            ),

        "active_years":
            normalize_year_list(
                fetched.get(
                    "active_years"
                )
            ),

        "results":
            fetched.get(
                "results"
            )
            or [],

        "raw":
            fetched.get(
                "raw"
            ),
    }


    write_json_gzip(
        checkpoint_path,
        saved,
    )


    return {
        **saved,

        "checkpoint_path":
            str(
                checkpoint_path
            ),

        "from_checkpoint":
            False,
    }


# ==================================================================================================
# PART H — RAW JAVELIN IDENTIFICATION
# ==================================================================================================

def is_javelin_history_result(
    row,
):
    """
    Identify javelin observations in getSingleCompetitorResultsDate output.

    We use multiple WA fields because their exact population varies by record.
    """

    if not isinstance(
        row,
        dict,
    ):

        return False


    values = [
        row.get(
            "discipline"
        ),

        row.get(
            "disciplineCode"
        ),

        row.get(
            "disciplineNameUrlSlug"
        ),

        row.get(
            "typeNameUrlSlug"
        ),
    ]


    text = " ".join(
        str(
            value
        )

        for value
        in values

        if value is not None
    ).lower()


    return (
        "javelin"
        in text
    )


# ==================================================================================================
# PART I — EXECUTE ONE ATHLETE REFRESH
# ==================================================================================================

async def refresh_one_athlete(
    refresh_row,
):
    """
    Execute one athlete's refresh strategy.

    Existing athlete:
        WINDOW_YEARS_ONLY

    New athlete:
        FULL_ACTIVE_YEAR_HISTORY
    """

    athlete_id = int(
        refresh_row[
            "aaAthleteId"
        ]
    )


    existed_before = bool(
        refresh_row[
            "existed_before_run"
        ]
    )


    refresh_scope = str(
        refresh_row[
            "refresh_scope"
        ]
    )


    paths = athlete_history_paths(
        athlete_id
    )


    paths[
        "error_dir"
    ].mkdir(
        parents=True,
        exist_ok=True,
    )


    # ==============================================================================================
    # RESUME COMPLETED ATHLETE
    # ==============================================================================================

    if paths[
        "manifest_path"
    ].exists():

        with paths[
            "manifest_path"
        ].open(
            "r",
            encoding="utf-8",
        ) as f:

            prior_manifest = json.load(
                f
            )


        if prior_manifest.get(
            "status"
        ) == "COMPLETE":

            prior_manifest[
                "resumed"
            ] = True


            return prior_manifest


    # ==============================================================================================
    # DETERMINE YEARS
    # ==============================================================================================

    year_results = {}

    bootstrap_active_years = []


    if refresh_scope == "WINDOW_YEARS_ONLY":

        years_to_fetch = normalize_year_list(
            refresh_row.get(
                "window_years"
            )
        )


        if not years_to_fetch:

            years_to_fetch = normalize_year_list(
                WINDOW_YEARS
            )


    elif refresh_scope == "FULL_ACTIVE_YEAR_HISTORY":

        # ------------------------------------------------------------------------------------------
        # Bootstrap using latest incremental year
        # ------------------------------------------------------------------------------------------

        bootstrap = await fetch_athlete_year_cached(

            athlete_id=
                athlete_id,

            year=
                ACTIVE_YEAR_BOOTSTRAP_YEAR,
        )


        year_results[
            ACTIVE_YEAR_BOOTSTRAP_YEAR
        ] = bootstrap


        bootstrap_active_years = normalize_year_list(
            bootstrap.get(
                "active_years"
            )
        )


        # If activeYears is unexpectedly empty, do not discard the year we
        # know contains the athlete's newly discovered result.
        years_to_fetch = sorted(
            set(
                bootstrap_active_years
                +
                [
                    ACTIVE_YEAR_BOOTSTRAP_YEAR
                ]
            )
        )


    else:

        raise ValueError(
            f"Unknown refresh_scope {refresh_scope!r} "
            f"for athlete {athlete_id}."
        )


    # ==============================================================================================
    # FETCH YEARS
    # ==============================================================================================

    year_summaries = []

    athlete_errors = []


    for year in years_to_fetch:

        try:

            if year in year_results:

                result = (
                    year_results[
                        year
                    ]
                )


            else:

                result = (
                    await fetch_athlete_year_cached(

                        athlete_id=
                            athlete_id,

                        year=
                            year,
                    )
                )


                year_results[
                    year
                ] = result


            raw_results = (
                result.get(
                    "results"
                )
                or []
            )


            n_javelin = sum(

                is_javelin_history_result(
                    row
                )

                for row
                in raw_results
            )


            year_summaries.append(
                {
                    "year":
                        int(
                            year
                        ),

                    "status":
                        "OK",

                    "n_results":
                        int(
                            len(
                                raw_results
                            )
                        ),

                    "n_javelin_results":
                        int(
                            n_javelin
                        ),

                    "active_years":
                        normalize_year_list(
                            result.get(
                                "active_years"
                            )
                        ),

                    "from_checkpoint":
                        bool(
                            result.get(
                                "from_checkpoint",
                                False,
                            )
                        ),

                    "checkpoint_path":
                        result.get(
                            "checkpoint_path"
                        ),
                }
            )


        except Exception as exc:

            error_record = {

                "athlete_id":
                    athlete_id,

                "year":
                    int(
                        year
                    ),

                "error":
                    repr(
                        exc
                    ),

                "traceback":
                    traceback.format_exc(),
            }


            error_path = (

                paths[
                    "error_dir"
                ]
                / f"year_{year}_error.json"
            )


            with error_path.open(
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    error_record,
                    f,
                    indent=2,
                    default=str,
                )


            athlete_errors.append(
                error_record
            )


            year_summaries.append(
                {
                    "year":
                        int(
                            year
                        ),

                    "status":
                        "ERROR",

                    "n_results":
                        0,

                    "n_javelin_results":
                        0,

                    "active_years":
                        [],

                    "from_checkpoint":
                        False,

                    "checkpoint_path":
                        None,

                    "error":
                        repr(
                            exc
                        ),
                }
            )


    # ==============================================================================================
    # ATHLETE MANIFEST
    # ==============================================================================================

    n_results = int(
        sum(
            item[
                "n_results"
            ]

            for item
            in year_summaries
        )
    )


    n_javelin_results = int(
        sum(
            item[
                "n_javelin_results"
            ]

            for item
            in year_summaries
        )
    )


    manifest = {

        "aaAthleteId":
            athlete_id,

        "existed_before_run":
            existed_before,

        "refresh_scope":
            refresh_scope,

        "years_requested":
            [
                int(
                    year
                )
                for year
                in years_to_fetch
            ],

        "active_years":
            bootstrap_active_years,

        "n_years_requested":
            int(
                len(
                    years_to_fetch
                )
            ),

        "n_year_errors":
            int(
                len(
                    athlete_errors
                )
            ),

        "n_results":
            n_results,

        "n_javelin_results":
            n_javelin_results,

        "year_summaries":
            year_summaries,

        "status":
            (
                "COMPLETE"

                if len(
                    athlete_errors
                )
                == 0

                else "COMPLETE_WITH_ERRORS"
            ),

        "resumed":
            False,
    }


    # Manifest written LAST.
    with paths[
        "manifest_path"
    ].open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            manifest,
            f,
            indent=2,
            default=str,
        )


    return manifest


# ==================================================================================================
# PART J — RUN FULL REFRESH PLAN
# ==================================================================================================

if not ENABLE_ATHLETE_HISTORY_REFRESH:

    raise RuntimeError(
        "ENABLE_ATHLETE_HISTORY_REFRESH is False."
    )


athlete_manifests = []


N_ATHLETES_TO_REFRESH = int(
    len(
        FINAL_ATHLETE_REFRESH_PLAN
    )
)


for position, (_, refresh_row) in enumerate(

    FINAL_ATHLETE_REFRESH_PLAN.iterrows(),

    start=1,
):

    athlete_id = int(
        refresh_row[
            "aaAthleteId"
        ]
    )


    try:

        manifest = await refresh_one_athlete(
            refresh_row
        )


        athlete_manifests.append(
            manifest
        )


        if (
            position == 1
            or position % 25 == 0
            or position == N_ATHLETES_TO_REFRESH
        ):

            print(
                f"[{position:>4}/{N_ATHLETES_TO_REFRESH}] "
                f"athlete={athlete_id} | "
                f"scope={manifest['refresh_scope']} | "
                f"years={manifest['n_years_requested']} | "
                f"results={manifest['n_results']} | "
                f"javelin={manifest['n_javelin_results']} | "
                f"{manifest['status']}"
            )


    except Exception as exc:

        paths = athlete_history_paths(
            athlete_id
        )


        top_error = {

            "aaAthleteId":
                athlete_id,

            "refresh_scope":
                refresh_row.get(
                    "refresh_scope"
                ),

            "status":
                "ERROR",

            "error":
                repr(
                    exc
                ),

            "traceback":
                traceback.format_exc(),
        }


        error_path = (

            HISTORY_ERROR_ROOT
            / f"athlete_{athlete_id}_error.json"
        )


        with error_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                top_error,
                f,
                indent=2,
                default=str,
            )


        athlete_manifests.append(
            top_error
        )


# ==================================================================================================
# PART K — REBUILD ATHLETE MANIFEST TABLE FROM DISK
# ==================================================================================================

manifest_records = []


for path in sorted(
    HISTORY_MANIFEST_ROOT.glob(
        "athlete_*.json"
    )
):

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        manifest_records.append(
            json.load(
                f
            )
        )


completed_athlete_ids = {

    int(
        record[
            "aaAthleteId"
        ]
    )

    for record
    in manifest_records
}


# Add top-level failures that have no completed manifest.
for path in sorted(
    HISTORY_ERROR_ROOT.glob(
        "athlete_*_error.json"
    )
):

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        error_record = json.load(
            f
        )


    athlete_id = int(
        error_record[
            "aaAthleteId"
        ]
    )


    if athlete_id not in completed_athlete_ids:

        manifest_records.append(
            error_record
        )


ATHLETE_REFRESH_SUMMARY = pd.DataFrame(
    manifest_records
)


if len(
    ATHLETE_REFRESH_SUMMARY
) > 0:

    ATHLETE_REFRESH_SUMMARY = (

        ATHLETE_REFRESH_SUMMARY.sort_values(
            "aaAthleteId"
        )
        .reset_index(
            drop=True
        )
    )


# ==================================================================================================
# PART L — REBUILD ATHLETE-YEAR SUMMARY
# ==================================================================================================

year_summary_rows = []


for _, manifest in (
    ATHLETE_REFRESH_SUMMARY.iterrows()
):

    athlete_id = normalize_year_list(
        []
    )


    try:

        athlete_id = int(
            manifest[
                "aaAthleteId"
            ]
        )

    except Exception:

        continue


    year_summaries = manifest.get(
        "year_summaries"
    )


    if not isinstance(
        year_summaries,
        list,
    ):

        continue


    for year_summary in year_summaries:

        year_summary_rows.append(
            {
                "aaAthleteId":
                    athlete_id,

                "refresh_scope":
                    manifest.get(
                        "refresh_scope"
                    ),

                **year_summary,
            }
        )


ATHLETE_YEAR_REFRESH_SUMMARY = pd.DataFrame(
    year_summary_rows
)


# ==================================================================================================
# PART M — REBUILD RAW RESULT TABLE FROM SUCCESSFUL CHECKPOINTS
# ==================================================================================================

history_result_rows = []


for athlete_dir in sorted(
    HISTORY_RAW_ROOT.glob(
        "*"
    )
):

    if not athlete_dir.is_dir():

        continue


    try:

        athlete_id = int(
            athlete_dir.name
        )

    except Exception:

        continue


    # Determine whether athlete was existing/new from final refresh plan.
    plan_match = (

        FINAL_ATHLETE_REFRESH_PLAN.loc[
            FINAL_ATHLETE_REFRESH_PLAN[
                "aaAthleteId"
            ]
            == athlete_id
        ]
    )


    if len(
        plan_match
    ) > 0:

        refresh_scope = str(
            plan_match.iloc[
                0
            ][
                "refresh_scope"
            ]
        )


        existed_before = bool(
            plan_match.iloc[
                0
            ][
                "existed_before_run"
            ]
        )


    else:

        refresh_scope = None
        existed_before = None


    for year_path in sorted(
        athlete_dir.glob(
            "year_*.json.gz"
        )
    ):

        saved = read_json_gzip(
            year_path
        )


        refresh_year = int(
            saved[
                "year"
            ]
        )


        active_years = normalize_year_list(
            saved.get(
                "active_years"
            )
        )


        for result_index, result in enumerate(
            saved.get(
                "results"
            )
            or []
        ):

            if not isinstance(
                result,
                dict,
            ):

                continue


            row = dict(
                result
            )


            row[
                "aaAthleteId"
            ] = athlete_id


            row[
                "refresh_year"
            ] = refresh_year


            row[
                "refresh_scope"
            ] = refresh_scope


            row[
                "existed_before_run"
            ] = existed_before


            row[
                "athlete_active_years"
            ] = active_years


            row[
                "source_result_index"
            ] = result_index


            row[
                "raw_checkpoint_path"
            ] = str(
                year_path
            )


            history_result_rows.append(
                row
            )


REFRESHED_ATHLETE_RESULTS_RAW = pd.DataFrame(
    history_result_rows
)


# ==================================================================================================
# PART N — JAVELIN-ONLY RAW VIEW
# ==================================================================================================

if len(
    REFRESHED_ATHLETE_RESULTS_RAW
) > 0:

    REFRESHED_ATHLETE_RESULTS_RAW[
        "looks_like_javelin"
    ] = (

        REFRESHED_ATHLETE_RESULTS_RAW.apply(

            lambda row:
                is_javelin_history_result(
                    row.to_dict()
                ),

            axis=1,
        )
    )


    REFRESHED_JAVELIN_RESULTS_RAW = (

        REFRESHED_ATHLETE_RESULTS_RAW.loc[
            REFRESHED_ATHLETE_RESULTS_RAW[
                "looks_like_javelin"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


else:

    REFRESHED_JAVELIN_RESULTS_RAW = pd.DataFrame()


# ==================================================================================================
# PART O — PARSE WA RESULT DATES EXPLICITLY
# ==================================================================================================

def parse_wa_history_date(
    value,
):
    """
    Explicitly parse WA dates without the warning seen in Cell 3.
    """

    if value is None:

        return pd.NaT


    text = str(
        value
    ).strip()


    if not text:

        return pd.NaT


    for fmt in [
        "%d %b %Y",
        "%d %B %Y",
        "%Y-%m-%d",
    ]:

        value_parsed = pd.to_datetime(
            text,
            format=fmt,
            errors="coerce",
        )


        if not pd.isna(
            value_parsed
        ):

            return value_parsed


    return pd.to_datetime(
        text,
        errors="coerce",
    )


if (
    len(
        REFRESHED_ATHLETE_RESULTS_RAW
    )
    > 0
    and
    "date"
    in REFRESHED_ATHLETE_RESULTS_RAW.columns
):

    REFRESHED_ATHLETE_RESULTS_RAW[
        "result_date_dt"
    ] = (

        REFRESHED_ATHLETE_RESULTS_RAW[
            "date"
        ]
        .map(
            parse_wa_history_date
        )
    )


if (
    len(
        REFRESHED_JAVELIN_RESULTS_RAW
    )
    > 0
    and
    "date"
    in REFRESHED_JAVELIN_RESULTS_RAW.columns
):

    REFRESHED_JAVELIN_RESULTS_RAW[
        "result_date_dt"
    ] = (

        REFRESHED_JAVELIN_RESULTS_RAW[
            "date"
        ]
        .map(
            parse_wa_history_date
        )
    )


# ==================================================================================================
# PART P — QA COUNTS
# ==================================================================================================

if len(
    ATHLETE_REFRESH_SUMMARY
) > 0:

    N_ATHLETE_COMPLETE = int(
        (
            ATHLETE_REFRESH_SUMMARY[
                "status"
            ]
            == "COMPLETE"
        )
        .sum()
    )


    N_ATHLETE_WITH_ERRORS = int(
        (
            ATHLETE_REFRESH_SUMMARY[
                "status"
            ]
            == "COMPLETE_WITH_ERRORS"
        )
        .sum()
    )


    N_ATHLETE_FATAL_ERRORS = int(
        (
            ATHLETE_REFRESH_SUMMARY[
                "status"
            ]
            == "ERROR"
        )
        .sum()
    )


else:

    N_ATHLETE_COMPLETE = 0
    N_ATHLETE_WITH_ERRORS = 0
    N_ATHLETE_FATAL_ERRORS = 0


if len(
    ATHLETE_YEAR_REFRESH_SUMMARY
) > 0:

    N_YEAR_REQUESTS = int(
        len(
            ATHLETE_YEAR_REFRESH_SUMMARY
        )
    )


    N_YEAR_SUCCESS = int(
        (
            ATHLETE_YEAR_REFRESH_SUMMARY[
                "status"
            ]
            == "OK"
        )
        .sum()
    )


    N_YEAR_ERRORS = int(
        (
            ATHLETE_YEAR_REFRESH_SUMMARY[
                "status"
            ]
            == "ERROR"
        )
        .sum()
    )


    N_YEAR_FROM_CHECKPOINT = int(
        ATHLETE_YEAR_REFRESH_SUMMARY[
            "from_checkpoint"
        ]
        .fillna(
            False
        )
        .astype(
            bool
        )
        .sum()
    )


else:

    N_YEAR_REQUESTS = 0
    N_YEAR_SUCCESS = 0
    N_YEAR_ERRORS = 0
    N_YEAR_FROM_CHECKPOINT = 0


N_RAW_HISTORY_RESULTS = int(
    len(
        REFRESHED_ATHLETE_RESULTS_RAW
    )
)


N_RAW_JAVELIN_RESULTS = int(
    len(
        REFRESHED_JAVELIN_RESULTS_RAW
    )
)


if (
    len(
        REFRESHED_JAVELIN_RESULTS_RAW
    )
    > 0
):

    N_JAVELIN_ATHLETES = int(
        REFRESHED_JAVELIN_RESULTS_RAW[
            "aaAthleteId"
        ]
        .nunique()
    )


else:

    N_JAVELIN_ATHLETES = 0


# ==================================================================================================
# PART Q — NEW-ATHLETE HISTORY DEPTH
# ==================================================================================================

NEW_ATHLETE_YEAR_SUMMARY = pd.DataFrame()


if len(
    ATHLETE_REFRESH_SUMMARY
) > 0:

    NEW_ATHLETE_YEAR_SUMMARY = (

        ATHLETE_REFRESH_SUMMARY.loc[
            ATHLETE_REFRESH_SUMMARY[
                "refresh_scope"
            ]
            == "FULL_ACTIVE_YEAR_HISTORY"
        ]
        [
            [
                column
                for column
                in [
                    "aaAthleteId",
                    "active_years",
                    "n_years_requested",
                    "n_results",
                    "n_javelin_results",
                    "status",
                ]
                if column
                in ATHLETE_REFRESH_SUMMARY.columns
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


# ==================================================================================================
# PART R — FINAL REFRESH VALIDATION
# ==================================================================================================

HISTORY_REFRESH_PASS = bool(

    N_ATHLETE_COMPLETE
    == N_ATHLETES_TO_REFRESH

    and
    N_ATHLETE_WITH_ERRORS == 0

    and
    N_ATHLETE_FATAL_ERRORS == 0

    and
    N_YEAR_ERRORS == 0

    and
    N_JAVELIN_ATHLETES > 0
)


# ==================================================================================================
# PART S — SAVE AGGREGATED CHECKPOINTS
# ==================================================================================================

ATHLETE_REFRESH_SUMMARY_PATH = (
    RUN_CHECKPOINT_DIR
    / "athlete_history_refresh_summary.parquet"
)


ATHLETE_YEAR_REFRESH_SUMMARY_PATH = (
    RUN_CHECKPOINT_DIR
    / "athlete_year_refresh_summary.parquet"
)


REFRESHED_RESULTS_RAW_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_athlete_results_raw.parquet"
)


REFRESHED_JAVELIN_RAW_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_results_raw.parquet"
)


ATHLETE_REFRESH_SUMMARY.to_parquet(

    ATHLETE_REFRESH_SUMMARY_PATH,

    index=False,
)


ATHLETE_YEAR_REFRESH_SUMMARY.to_parquet(

    ATHLETE_YEAR_REFRESH_SUMMARY_PATH,

    index=False,
)


REFRESHED_ATHLETE_RESULTS_RAW.to_parquet(

    REFRESHED_RESULTS_RAW_PATH,

    index=False,
)


REFRESHED_JAVELIN_RESULTS_RAW.to_parquet(

    REFRESHED_JAVELIN_RAW_PATH,

    index=False,
)


# ==================================================================================================
# PART T — UPDATE RUN MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "athlete_history_refresh"
] = {

    "status":
        (
            "VALIDATED"

            if HISTORY_REFRESH_PASS

            else "REVIEW_REQUIRED"
        ),

    "athletes_requested":
        N_ATHLETES_TO_REFRESH,

    "athletes_complete":
        N_ATHLETE_COMPLETE,

    "athletes_complete_with_errors":
        N_ATHLETE_WITH_ERRORS,

    "athlete_fatal_errors":
        N_ATHLETE_FATAL_ERRORS,

    "athlete_year_requests":
        N_YEAR_REQUESTS,

    "athlete_year_success":
        N_YEAR_SUCCESS,

    "athlete_year_errors":
        N_YEAR_ERRORS,

    "athlete_years_loaded_from_checkpoint":
        N_YEAR_FROM_CHECKPOINT,

    "raw_history_results":
        N_RAW_HISTORY_RESULTS,

    "raw_javelin_results":
        N_RAW_JAVELIN_RESULTS,

    "javelin_athletes":
        N_JAVELIN_ATHLETES,

    "athlete_summary_path":
        str(
            ATHLETE_REFRESH_SUMMARY_PATH
        ),

    "athlete_year_summary_path":
        str(
            ATHLETE_YEAR_REFRESH_SUMMARY_PATH
        ),

    "raw_results_path":
        str(
            REFRESHED_RESULTS_RAW_PATH
        ),

    "raw_javelin_path":
        str(
            REFRESHED_JAVELIN_RAW_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "ATHLETE_HISTORY_REFRESH_COMPLETE"

    if HISTORY_REFRESH_PASS

    else "ATHLETE_HISTORY_REFRESH_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART U — OUTPUT
# ==================================================================================================

print()
print("=" * 125)
print("WORLD ATHLETICS ATHLETE-HISTORY REFRESH")
print("=" * 125)


print()
print("ATHLETES")
print("-" * 125)

print(
    f"Athletes requested               : "
    f"{N_ATHLETES_TO_REFRESH:,}"
)

print(
    f"Complete                         : "
    f"{N_ATHLETE_COMPLETE:,}"
)

print(
    f"Complete with year errors        : "
    f"{N_ATHLETE_WITH_ERRORS:,}"
)

print(
    f"Fatal athlete errors             : "
    f"{N_ATHLETE_FATAL_ERRORS:,}"
)


print()
print("ATHLETE-YEAR REQUESTS")
print("-" * 125)

print(
    f"Years requested                  : "
    f"{N_YEAR_REQUESTS:,}"
)

print(
    f"Successful                       : "
    f"{N_YEAR_SUCCESS:,}"
)

print(
    f"Errors                           : "
    f"{N_YEAR_ERRORS:,}"
)

print(
    f"Loaded from existing checkpoint  : "
    f"{N_YEAR_FROM_CHECKPOINT:,}"
)


print()
print("RAW RESULT RECOVERY")
print("-" * 125)

print(
    f"All athletics result rows        : "
    f"{N_RAW_HISTORY_RESULTS:,}"
)

print(
    f"Javelin-like result rows         : "
    f"{N_RAW_JAVELIN_RESULTS:,}"
)

print(
    f"Athletes with javelin history    : "
    f"{N_JAVELIN_ATHLETES:,}"
)


print()
print("NEW ATHLETES — FULL HISTORY")
print("-" * 125)

if len(
    NEW_ATHLETE_YEAR_SUMMARY
) > 0:

    display(
        NEW_ATHLETE_YEAR_SUMMARY
    )

else:

    print(
        "No newly discovered athletes required a full-history backfill."
    )


print()
print("YEAR REFRESH SUMMARY")
print("-" * 125)

if len(
    ATHLETE_YEAR_REFRESH_SUMMARY
) > 0:

    display(
        ATHLETE_YEAR_REFRESH_SUMMARY.groupby(
            [
                "refresh_scope",
                "year",
                "status",
            ],
            as_index=False,
        )
        .agg(
            athletes=(
                "aaAthleteId",
                "nunique",
            ),

            result_rows=(
                "n_results",
                "sum",
            ),

            javelin_rows=(
                "n_javelin_results",
                "sum",
            ),
        )
        .sort_values(
            [
                "refresh_scope",
                "year",
            ]
        )
    )


print()
print("CHECKPOINTS")
print("-" * 125)

print(
    f"Athlete summary                  : "
    f"{ATHLETE_REFRESH_SUMMARY_PATH}"
)

print(
    f"Athlete-year summary             : "
    f"{ATHLETE_YEAR_REFRESH_SUMMARY_PATH}"
)

print(
    f"All refreshed results            : "
    f"{REFRESHED_RESULTS_RAW_PATH}"
)

print(
    f"Raw refreshed javelin results    : "
    f"{REFRESHED_JAVELIN_RAW_PATH}"
)


print()
print("=" * 125)


if HISTORY_REFRESH_PASS:

    print(
        "CELL 10 STATUS: ATHLETE-HISTORY REFRESH COMPLETE"
    )

    print()
    print(
        "All planned athlete histories have been downloaded successfully."
    )

    print(
        "Cell 11 will normalize the refreshed javelin history and "
        "reconcile it against the 248,896-row historical canonical dataset."
    )


else:

    print(
        "CELL 10 STATUS: HISTORY REFRESH REVIEW REQUIRED"
    )

    print()
    print(
        "Review athlete/year failures before canonical reconciliation."
    )


print("=" * 125)

[   1/867] athlete=14177510 | scope=WINDOW_YEARS_ONLY | years=1 | results=9 | javelin=9 | COMPLETE
[  25/867] athlete=14422692 | scope=WINDOW_YEARS_ONLY | years=1 | results=8 | javelin=8 | COMPLETE
[  50/867] athlete=14534754 | scope=WINDOW_YEARS_ONLY | years=1 | results=47 | javelin=4 | COMPLETE
[  75/867] athlete=14661582 | scope=WINDOW_YEARS_ONLY | years=1 | results=11 | javelin=1 | COMPLETE
[ 100/867] athlete=14717516 | scope=WINDOW_YEARS_ONLY | years=1 | results=10 | javelin=1 | COMPLETE
[ 125/867] athlete=14766655 | scope=WINDOW_YEARS_ONLY | years=1 | results=9 | javelin=9 | COMPLETE
[ 150/867] athlete=14810456 | scope=WINDOW_YEARS_ONLY | years=1 | results=13 | javelin=13 | COMPLETE
[ 175/867] athlete=14849625 | scope=WINDOW_YEARS_ONLY | years=1 | results=9 | javelin=9 | COMPLETE
[ 200/867] athlete=14878260 | scope=WINDOW_YEARS_ONLY | years=1 | results=34 | javelin=2 | COMPLETE
[ 225/867] athlete=14919254 | scope=WINDOW_YEARS_ONLY | years=1 | results=8 | javelin=8 | COMPLETE
[ 25

,aaAthleteId,active_years,n_years_requested,n_results,n_javelin_results,status
0,14228196,"[2003, 2004, 2005, 2006, 2007, 2008, 2009, 201...",14,265,29,COMPLETE
1,14376821,"[2009, 2010, 2011, 2012, 2013, 2014, 2015, 201...",9,38,38,COMPLETE
2,14547867,"[2012, 2018, 2020, 2021, 2026]",5,5,5,COMPLETE
3,14646249,"[2014, 2015, 2016, 2017, 2018, 2019, 2020, 202...",13,267,2,COMPLETE
4,14800681,"[2017, 2019, 2021, 2022, 2026]",5,10,9,COMPLETE
...,...,...,...,...,...,...
70,15347573,[2026],1,15,1,COMPLETE
71,15347574,[2026],1,11,1,COMPLETE
72,15347651,[2026],1,11,1,COMPLETE
73,15347866,[2026],1,2,1,COMPLETE



YEAR REFRESH SUMMARY
-----------------------------------------------------------------------------------------------------------------------------


,refresh_scope,year,status,athletes,result_rows,javelin_rows
0,FULL_ACTIVE_YEAR_HISTORY,2003,OK,1,16,3
1,FULL_ACTIVE_YEAR_HISTORY,2004,OK,1,14,2
2,FULL_ACTIVE_YEAR_HISTORY,2005,OK,1,18,3
3,FULL_ACTIVE_YEAR_HISTORY,2006,OK,1,45,5
4,FULL_ACTIVE_YEAR_HISTORY,2007,OK,1,42,4
5,FULL_ACTIVE_YEAR_HISTORY,2008,OK,1,36,4
6,FULL_ACTIVE_YEAR_HISTORY,2009,OK,2,44,5
7,FULL_ACTIVE_YEAR_HISTORY,2010,OK,2,26,6
8,FULL_ACTIVE_YEAR_HISTORY,2011,OK,1,7,7
9,FULL_ACTIVE_YEAR_HISTORY,2012,OK,3,20,17



CHECKPOINTS
-----------------------------------------------------------------------------------------------------------------------------
Athlete summary                  : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/athlete_history_refresh_summary.parquet
Athlete-year summary             : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/athlete_year_refresh_summary.parquet
All refreshed results            : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_athlete_results_raw.parquet
Raw refreshed javelin results    : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_results_raw.parquet

CELL 10 STATUS: ATHLETE-HISTORY REFRESH COMPLETE

All planned athlete histories have been downloaded successfully.
Cell 11 will

In [12]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 11 — NORMALIZE REFRESHED JAVELIN HISTORY + CROSS-SOURCE QA
# ==================================================================================================
#
# Purpose
# -------
# Convert the refreshed athlete-history results into a clean staging table
# suitable for canonical reconciliation.
#
#
# This cell DOES NOT modify the historical canonical dataset.
#
# It:
#
#   1. Inspects the existing canonical schema.
#   2. Resolves athlete sex / DOB metadata.
#   3. Parses marks and dates.
#   4. Resolves javelin implement/model group.
#   5. Learns existing eventId → model_group mappings from canonical history.
#   6. Identifies unsupported/unresolved rows.
#   7. Removes only exact duplicated source observations.
#   8. Detects conflicting duplicate identities.
#   9. Cross-checks incremental competition discovery against athlete history.
#
#
# Canonical replacement happens in Cell 12 only after this QA passes.
# ==================================================================================================

from __future__ import annotations

import re
import json

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "CONFIG",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "REFRESHED_JAVELIN_RESULTS_RAW",
    "FINAL_ATHLETE_REFRESH_PLAN",

    "CANONICAL_PROFILE_CANDIDATES",
    "INCREMENTAL_RESULTS_MODEL_ELIGIBLE",

    "normalize_wa_id",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cells 1–10 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


# ==================================================================================================
# PART B — LOAD CANONICAL DATASET
# ==================================================================================================

CANONICAL_BASE = pd.read_parquet(
    CONFIG.canonical_dataset_path
)


print()
print(
    f"Canonical rows loaded       : {len(CANONICAL_BASE):,}"
)


# ==================================================================================================
# PART C — COLUMN DISCOVERY
# ==================================================================================================

def first_existing_column(
    df,
    candidates,
):
    """
    Return first candidate column present in df.
    """

    for column in candidates:

        if column in df.columns:

            return column


    return None


CAN_ID_COL = first_existing_column(
    CANONICAL_BASE,
    [
        "aaAthleteId",
        "athlete_id",
        "athleteId",
    ],
)


CAN_DATE_COL = first_existing_column(
    CANONICAL_BASE,
    [
        "result_date_dt",
        "result_date",
        "date",
        "performance_date",
    ],
)


CAN_SEX_COL = first_existing_column(
    CANONICAL_BASE,
    [
        "sex_group",
        "sex",
        "gender",
        "SEX",
        "GENDER",
    ],
)


CAN_DOB_COL = first_existing_column(
    CANONICAL_BASE,
    [
        "DOB",
        "dob",
        "birthDate",
        "birth_date",
        "date_of_birth",
    ],
)


CAN_EVENT_ID_COL = first_existing_column(
    CANONICAL_BASE,
    [
        "eventId",
        "event_id",
    ],
)


CAN_MODEL_GROUP_COL = first_existing_column(
    CANONICAL_BASE,
    [
        "model_group",
        "implement_group",
    ],
)


CAN_IMPLEMENT_COL = first_existing_column(
    CANONICAL_BASE,
    [
        "implement_weight_g",
        "implement_g",
        "implement",
    ],
)


CAN_MARK_NUMERIC_COL = first_existing_column(
    CANONICAL_BASE,
    [
        "mark_numeric_m",
        "result_numeric",
        "result_m",
        "mark_numeric",
    ],
)


if CAN_ID_COL is None:

    raise RuntimeError(
        "Canonical dataset has no identifiable athlete-ID column."
    )


if CAN_DATE_COL is None:

    raise RuntimeError(
        "Canonical dataset has no identifiable result-date column."
    )


# ==================================================================================================
# PART D — BASIC NORMALIZERS
# ==================================================================================================

def normalize_sex(
    value,
):
    """
    Normalize sex labels to M / W.
    """

    if value is None:

        return None


    try:

        if pd.isna(
            value
        ):

            return None

    except Exception:

        pass


    text = str(
        value
    ).strip().lower()


    male_values = {
        "m",
        "male",
        "men",
        "man",
        "boys",
        "boy",
        "1",
    }


    female_values = {
        "w",
        "f",
        "female",
        "women",
        "woman",
        "girls",
        "girl",
        "0",
    }


    if text in male_values:

        return "M"


    if text in female_values:

        return "W"


    return None


def parse_mark_m(
    value,
):
    """
    Parse a javelin distance into metres.

    Valid examples:
        68.42
        "68.42"
        "68.42m"

    Non-marks such as:
        NM
        DNS
        DNF
        DQ
        -
    return NaN.
    """

    if value is None:

        return np.nan


    try:

        if pd.isna(
            value
        ):

            return np.nan

    except Exception:

        pass


    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        numeric = float(
            value
        )


        return (
            numeric
            if np.isfinite(
                numeric
            )
            else np.nan
        )


    text = str(
        value
    ).strip().lower()


    if not text:

        return np.nan


    invalid_values = {
        "nm",
        "no mark",
        "dns",
        "dnf",
        "dq",
        "dsq",
        "nh",
        "x",
        "-",
        "--",
    }


    if text in invalid_values:

        return np.nan


    match = re.search(
        r"(?<!\d)(\d{1,3}(?:\.\d{1,3})?)(?!\d)",
        text,
    )


    if match is None:

        return np.nan


    try:

        return float(
            match.group(
                1
            )
        )

    except Exception:

        return np.nan


def parse_birth_information(
    value,
):
    """
    Preserve exact DOB only when WA supplies an exact date.

    Returns
    -------
    exact_dob : Timestamp or NaT
    birth_year : int or NaN
    precision : EXACT / YEAR / UNKNOWN
    """

    if value is None:

        return (
            pd.NaT,
            np.nan,
            "UNKNOWN",
        )


    try:

        if pd.isna(
            value
        ):

            return (
                pd.NaT,
                np.nan,
                "UNKNOWN",
            )

    except Exception:

        pass


    if isinstance(
        value,
        (
            pd.Timestamp,
            np.datetime64,
        ),
    ):

        parsed = pd.Timestamp(
            value
        )


        return (
            parsed,
            int(
                parsed.year
            ),
            "EXACT",
        )


    text = str(
        value
    ).strip()


    if not text:

        return (
            pd.NaT,
            np.nan,
            "UNKNOWN",
        )


    # ----------------------------------------------------------------------------------------------
    # Year only
    # ----------------------------------------------------------------------------------------------

    if re.fullmatch(
        r"\d{4}",
        text,
    ):

        return (
            pd.NaT,
            int(
                text
            ),
            "YEAR",
        )


    # ----------------------------------------------------------------------------------------------
    # Exact dates
    # ----------------------------------------------------------------------------------------------

    for fmt in [
        "%d %b %Y",
        "%d %B %Y",
        "%Y-%m-%d",
        "%Y/%m/%d",
    ]:

        parsed = pd.to_datetime(
            text,
            format=fmt,
            errors="coerce",
        )


        if not pd.isna(
            parsed
        ):

            return (
                parsed,
                int(
                    parsed.year
                ),
                "EXACT",
            )


    return (
        pd.NaT,
        np.nan,
        "UNKNOWN",
    )


def explicit_implement_weight_g(
    row,
):
    """
    Extract an explicitly stated implement weight from WA event text.

    Example:
        Men's Javelin Throw (700g) -> 700
    """

    fields = [
        row.get(
            "discipline"
        ),
        row.get(
            "disciplineNameUrlSlug"
        ),
        row.get(
            "typeNameUrlSlug"
        ),
    ]


    text = " ".join(
        str(
            value
        )

        for value
        in fields

        if value is not None
    ).lower()


    match = re.search(
        r"(?<!\d)(\d{3,4})\s*g\b",
        text,
    )


    if match is None:

        return np.nan


    return int(
        match.group(
            1
        )
    )


def model_group_from_sex_weight(
    sex,
    weight,
):
    """
    Supported model taxonomy only.
    """

    try:

        weight = int(
            weight
        )

    except Exception:

        return None


    mapping = {
        (
            "M",
            700,
        ):
            "M_700",

        (
            "M",
            800,
        ):
            "M_800",

        (
            "W",
            500,
        ):
            "W_500",

        (
            "W",
            600,
        ):
            "W_600",
    }


    return mapping.get(
        (
            sex,
            weight,
        )
    )


# ==================================================================================================
# PART E — NORMALIZE CANONICAL KEYS / METADATA
# ==================================================================================================

CANONICAL_BASE[
    "_aaAthleteId_norm"
] = (

    CANONICAL_BASE[
        CAN_ID_COL
    ]
    .map(
        normalize_wa_id
    )
)


CANONICAL_BASE[
    "_result_date_norm"
] = pd.to_datetime(

    CANONICAL_BASE[
        CAN_DATE_COL
    ],

    errors="coerce",
)


# ==================================================================================================
# PART F — CANONICAL SEX MAP
# ==================================================================================================

CANONICAL_SEX_MAP = {}


if CAN_SEX_COL is not None:

    sex_temp = (

        CANONICAL_BASE[
            [
                "_aaAthleteId_norm",
                CAN_SEX_COL,
            ]
        ]
        .copy()
    )


    sex_temp[
        "_sex_norm"
    ] = (

        sex_temp[
            CAN_SEX_COL
        ]
        .map(
            normalize_sex
        )
    )


    sex_temp = (

        sex_temp.dropna(
            subset=[
                "_aaAthleteId_norm",
                "_sex_norm",
            ]
        )
        .drop_duplicates(
            subset=[
                "_aaAthleteId_norm",
            ],
            keep="last",
        )
    )


    CANONICAL_SEX_MAP = dict(
        zip(
            sex_temp[
                "_aaAthleteId_norm"
            ]
            .astype(
                int
            ),

            sex_temp[
                "_sex_norm"
            ],
        )
    )


# ==================================================================================================
# PART G — CANONICAL DOB MAP
# ==================================================================================================

CANONICAL_DOB_MAP = {}


if CAN_DOB_COL is not None:

    dob_temp = (

        CANONICAL_BASE[
            [
                "_aaAthleteId_norm",
                CAN_DOB_COL,
            ]
        ]
        .dropna(
            subset=[
                "_aaAthleteId_norm",
                CAN_DOB_COL,
            ]
        )
        .drop_duplicates(
            subset=[
                "_aaAthleteId_norm",
            ],
            keep="last",
        )
    )


    CANONICAL_DOB_MAP = dict(
        zip(
            dob_temp[
                "_aaAthleteId_norm"
            ]
            .astype(
                int
            ),

            dob_temp[
                CAN_DOB_COL
            ],
        )
    )


# ==================================================================================================
# PART H — PROFILE METADATA MAPS
# ==================================================================================================

PROFILE_METADATA = (
    CANONICAL_PROFILE_CANDIDATES.copy()
)


PROFILE_METADATA[
    "candidate_aaAthleteId"
] = (

    PROFILE_METADATA[
        "candidate_aaAthleteId"
    ]
    .map(
        normalize_wa_id
    )
)


PROFILE_METADATA = (

    PROFILE_METADATA.dropna(
        subset=[
            "candidate_aaAthleteId"
        ]
    )
    .drop_duplicates(
        subset=[
            "candidate_aaAthleteId"
        ],
        keep="last",
    )
)


PROFILE_METADATA[
    "candidate_aaAthleteId"
] = (

    PROFILE_METADATA[
        "candidate_aaAthleteId"
    ]
    .astype(
        int
    )
)


PROFILE_SEX_MAP = {}

PROFILE_DOB_MAP = {}


for _, row in PROFILE_METADATA.iterrows():

    athlete_id = int(
        row[
            "candidate_aaAthleteId"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # Sex
    # ----------------------------------------------------------------------------------------------

    sex_candidates = [
        row.get(
            "profile_sexCode"
        ),
        row.get(
            "profile_sexName"
        ),
    ]


    if "profile_male" in row.index:

        male_flag = row.get(
            "profile_male"
        )


        if male_flag is True:

            sex_candidates.append(
                "M"
            )

        elif male_flag is False:

            sex_candidates.append(
                "W"
            )


    resolved_sex = None


    for candidate in sex_candidates:

        resolved_sex = normalize_sex(
            candidate
        )


        if resolved_sex is not None:

            break


    if resolved_sex is not None:

        PROFILE_SEX_MAP[
            athlete_id
        ] = resolved_sex


    # ----------------------------------------------------------------------------------------------
    # DOB
    # ----------------------------------------------------------------------------------------------

    dob_candidates = [
        row.get(
            "profile_birthDate"
        ),
        row.get(
            "profile_birthDateStr"
        ),
        row.get(
            "competitor_birthDate"
        ),
    ]


    for candidate in dob_candidates:

        if candidate is None:

            continue


        try:

            if pd.isna(
                candidate
            ):

                continue

        except Exception:

            pass


        if str(
            candidate
        ).strip():

            PROFILE_DOB_MAP[
                athlete_id
            ] = candidate

            break


# ==================================================================================================
# PART I — CURRENT DISCOVERY SEX MAP
# ==================================================================================================

DISCOVERY_SEX_MAP = {}


if (
    len(
        INCREMENTAL_RESULTS_MODEL_ELIGIBLE
    )
    > 0
    and
    "gender"
    in INCREMENTAL_RESULTS_MODEL_ELIGIBLE.columns
):

    discovery_sex = (

        INCREMENTAL_RESULTS_MODEL_ELIGIBLE[
            [
                "aaAthleteId",
                "gender",
            ]
        ]
        .copy()
    )


    discovery_sex[
        "aaAthleteId"
    ] = (

        discovery_sex[
            "aaAthleteId"
        ]
        .map(
            normalize_wa_id
        )
    )


    discovery_sex[
        "_sex_norm"
    ] = (

        discovery_sex[
            "gender"
        ]
        .map(
            normalize_sex
        )
    )


    discovery_sex = (

        discovery_sex.dropna(
            subset=[
                "aaAthleteId",
                "_sex_norm",
            ]
        )
        .drop_duplicates(
            subset=[
                "aaAthleteId",
            ],
            keep="last",
        )
    )


    DISCOVERY_SEX_MAP = dict(
        zip(
            discovery_sex[
                "aaAthleteId"
            ]
            .astype(
                int
            ),

            discovery_sex[
                "_sex_norm"
            ],
        )
    )


# ==================================================================================================
# PART J — LEARN HISTORICAL eventId → MODEL GROUP
# ==================================================================================================

EVENT_TO_GROUP_MAP = {}

EVENT_TO_WEIGHT_MAP = {}


if (
    CAN_EVENT_ID_COL is not None
    and
    CAN_MODEL_GROUP_COL is not None
):

    event_group = (

        CANONICAL_BASE[
            [
                CAN_EVENT_ID_COL,
                CAN_MODEL_GROUP_COL,
            ]
        ]
        .dropna()
        .copy()
    )


    event_group[
        "_event_id_norm"
    ] = (

        event_group[
            CAN_EVENT_ID_COL
        ]
        .map(
            normalize_wa_id
        )
    )


    event_group = event_group.dropna(
        subset=[
            "_event_id_norm"
        ]
    )


    for event_id, group in (
        event_group.groupby(
            "_event_id_norm"
        )
    ):

        values = (

            group[
                CAN_MODEL_GROUP_COL
            ]
            .dropna()
            .astype(
                str
            )
            .unique()
        )


        supported_values = [
            value
            for value
            in values
            if value
            in {
                "M_700",
                "M_800",
                "W_500",
                "W_600",
            }
        ]


        if len(
            supported_values
        ) == 1:

            EVENT_TO_GROUP_MAP[
                int(
                    event_id
                )
            ] = supported_values[
                0
            ]


if (
    CAN_EVENT_ID_COL is not None
    and
    CAN_IMPLEMENT_COL is not None
):

    event_weight = (

        CANONICAL_BASE[
            [
                CAN_EVENT_ID_COL,
                CAN_IMPLEMENT_COL,
            ]
        ]
        .dropna()
        .copy()
    )


    event_weight[
        "_event_id_norm"
    ] = (

        event_weight[
            CAN_EVENT_ID_COL
        ]
        .map(
            normalize_wa_id
        )
    )


    for event_id, group in (
        event_weight.dropna(
            subset=[
                "_event_id_norm"
            ]
        )
        .groupby(
            "_event_id_norm"
        )
    ):

        values = (

            pd.to_numeric(
                group[
                    CAN_IMPLEMENT_COL
                ],
                errors="coerce",
            )
            .dropna()
            .astype(
                int
            )
            .unique()
        )


        if len(
            values
        ) == 1:

            EVENT_TO_WEIGHT_MAP[
                int(
                    event_id
                )
            ] = int(
                values[
                    0
                ]
            )


# ==================================================================================================
# PART K — BUILD STANDARDIZED STAGING TABLE
# ==================================================================================================

REFRESHED_STAGING = (
    REFRESHED_JAVELIN_RESULTS_RAW.copy()
)


REFRESHED_STAGING[
    "aaAthleteId"
] = (

    REFRESHED_STAGING[
        "aaAthleteId"
    ]
    .map(
        normalize_wa_id
    )
)


REFRESHED_STAGING = (

    REFRESHED_STAGING.dropna(
        subset=[
            "aaAthleteId"
        ]
    )
    .copy()
)


REFRESHED_STAGING[
    "aaAthleteId"
] = (

    REFRESHED_STAGING[
        "aaAthleteId"
    ]
    .astype(
        int
    )
)


# --------------------------------------------------------------------------------------------------
# Dates
# --------------------------------------------------------------------------------------------------

REFRESHED_STAGING[
    "result_date_dt"
] = pd.to_datetime(

    REFRESHED_STAGING[
        "result_date_dt"
    ],

    errors="coerce",
)


# --------------------------------------------------------------------------------------------------
# Numeric mark
# --------------------------------------------------------------------------------------------------

REFRESHED_STAGING[
    "mark_numeric_m"
] = (

    REFRESHED_STAGING[
        "mark"
    ]
    .map(
        parse_mark_m
    )
)


# --------------------------------------------------------------------------------------------------
# Normalize event ID
# --------------------------------------------------------------------------------------------------

if "eventId" in REFRESHED_STAGING.columns:

    REFRESHED_STAGING[
        "event_id_norm"
    ] = (

        REFRESHED_STAGING[
            "eventId"
        ]
        .map(
            normalize_wa_id
        )
    )


else:

    REFRESHED_STAGING[
        "event_id_norm"
    ] = np.nan


# ==================================================================================================
# PART L — RESOLVE SEX
# ==================================================================================================

def resolve_staging_sex(
    athlete_id,
):
    """
    Resolution order:
        existing canonical
        live profile
        current competition discovery
    """

    athlete_id = int(
        athlete_id
    )


    for mapping in [
        CANONICAL_SEX_MAP,
        PROFILE_SEX_MAP,
        DISCOVERY_SEX_MAP,
    ]:

        value = mapping.get(
            athlete_id
        )


        if value is not None:

            return value


    return None


REFRESHED_STAGING[
    "sex"
] = (

    REFRESHED_STAGING[
        "aaAthleteId"
    ]
    .map(
        resolve_staging_sex
    )
)


# ==================================================================================================
# PART M — RESOLVE DOB / AGE INFORMATION
# ==================================================================================================

def resolve_birth_value(
    athlete_id,
):
    athlete_id = int(
        athlete_id
    )


    if athlete_id in CANONICAL_DOB_MAP:

        return CANONICAL_DOB_MAP[
            athlete_id
        ]


    return PROFILE_DOB_MAP.get(
        athlete_id
    )


birth_info = (

    REFRESHED_STAGING[
        "aaAthleteId"
    ]
    .map(
        resolve_birth_value
    )
    .map(
        parse_birth_information
    )
)


REFRESHED_STAGING[
    "DOB"
] = [
    item[
        0
    ]
    for item
    in birth_info
]


REFRESHED_STAGING[
    "birth_year"
] = [
    item[
        1
    ]
    for item
    in birth_info
]


REFRESHED_STAGING[
    "birth_date_precision"
] = [
    item[
        2
    ]
    for item
    in birth_info
]


# Exact age only when exact DOB is available.
REFRESHED_STAGING[
    "age_years"
] = np.where(

    REFRESHED_STAGING[
        "DOB"
    ]
    .notna()
    &
    REFRESHED_STAGING[
        "result_date_dt"
    ]
    .notna(),

    (
        REFRESHED_STAGING[
            "result_date_dt"
        ]
        -
        REFRESHED_STAGING[
            "DOB"
        ]
    )
    .dt.days
    /
    365.25,

    np.nan,
)


# Age interval for year-only DOB.
REFRESHED_STAGING[
    "age_lower_years"
] = np.nan


REFRESHED_STAGING[
    "age_upper_years"
] = np.nan


year_only_mask = (

    REFRESHED_STAGING[
        "birth_date_precision"
    ]
    == "YEAR"
)


REFRESHED_STAGING.loc[
    year_only_mask,
    "age_lower_years"
] = (

    REFRESHED_STAGING.loc[
        year_only_mask,
        "result_date_dt"
    ]
    .dt.year

    -
    REFRESHED_STAGING.loc[
        year_only_mask,
        "birth_year"
    ]

    -
    1
)


REFRESHED_STAGING.loc[
    year_only_mask,
    "age_upper_years"
] = (

    REFRESHED_STAGING.loc[
        year_only_mask,
        "result_date_dt"
    ]
    .dt.year

    -
    REFRESHED_STAGING.loc[
        year_only_mask,
        "birth_year"
    ]
)


# ==================================================================================================
# PART N — IMPLEMENT / MODEL GROUP RESOLUTION
# ==================================================================================================

REFRESHED_STAGING[
    "explicit_implement_weight_g"
] = (

    REFRESHED_STAGING.apply(
        explicit_implement_weight_g,
        axis=1,
    )
)


def resolve_model_group(
    row,
):
    """
    Resolution hierarchy:
        1. explicit implement weight in current WA result text
        2. historical canonical eventId → model_group mapping
        3. unresolved

    We deliberately do NOT infer youth/adult implement purely from age.
    """

    sex = row.get(
        "sex"
    )


    explicit_weight = row.get(
        "explicit_implement_weight_g"
    )


    if not pd.isna(
        explicit_weight
    ):

        group = model_group_from_sex_weight(
            sex,
            explicit_weight,
        )


        if group is not None:

            return (
                group,
                int(
                    explicit_weight
                ),
                "EXPLICIT_WEIGHT",
            )


        return (
            None,
            int(
                explicit_weight
            ),
            "EXPLICIT_UNSUPPORTED_WEIGHT",
        )


    event_id = normalize_wa_id(
        row.get(
            "event_id_norm"
        )
    )


    if event_id is not None:

        event_id = int(
            event_id
        )


        group = EVENT_TO_GROUP_MAP.get(
            event_id
        )


        weight = EVENT_TO_WEIGHT_MAP.get(
            event_id
        )


        if group is not None:

            return (
                group,
                weight,
                "CANONICAL_EVENT_ID_MAP",
            )


    return (
        None,
        np.nan,
        "UNRESOLVED",
    )


group_resolution = (

    REFRESHED_STAGING.apply(
        resolve_model_group,
        axis=1,
    )
)


REFRESHED_STAGING[
    "model_group"
] = [
    item[
        0
    ]
    for item
    in group_resolution
]


REFRESHED_STAGING[
    "implement_weight_g"
] = [
    item[
        1
    ]
    for item
    in group_resolution
]


REFRESHED_STAGING[
    "group_resolution_source"
] = [
    item[
        2
    ]
    for item
    in group_resolution
]


# ==================================================================================================
# PART O — ELIGIBILITY CLASSIFICATION
# ==================================================================================================

REFRESHED_STAGING[
    "has_valid_date"
] = (

    REFRESHED_STAGING[
        "result_date_dt"
    ]
    .notna()
)


REFRESHED_STAGING[
    "has_numeric_mark"
] = (

    REFRESHED_STAGING[
        "mark_numeric_m"
    ]
    .notna()
)


REFRESHED_STAGING[
    "has_resolved_sex"
] = (

    REFRESHED_STAGING[
        "sex"
    ]
    .notna()
)


REFRESHED_STAGING[
    "has_supported_model_group"
] = (

    REFRESHED_STAGING[
        "model_group"
    ]
    .isin(
        [
            "M_700",
            "M_800",
            "W_500",
            "W_600",
        ]
    )
)


REFRESHED_STAGING[
    "canonical_candidate"
] = (

    REFRESHED_STAGING[
        "has_valid_date"
    ]

    &
    REFRESHED_STAGING[
        "has_numeric_mark"
    ]

    &
    REFRESHED_STAGING[
        "has_resolved_sex"
    ]

    &
    REFRESHED_STAGING[
        "has_supported_model_group"
    ]
)


# ==================================================================================================
# PART P — DUPLICATE SOURCE OBSERVATION CHECK
# ==================================================================================================
#
# Athlete-history endpoint should normally contain one row per performance.
#
# Identity key deliberately excludes mark so a contradictory pair of marks
# is detected as a conflict.
# ==================================================================================================

def normalized_string(
    series,
):
    return (
        series
        .fillna(
            ""
        )
        .astype(
            str
        )
        .str.strip()
    )


REFRESHED_STAGING[
    "_competition_id_key"
] = (

    REFRESHED_STAGING[
        "competitionId"
    ]
    .map(
        normalize_wa_id
    )
    if "competitionId"
    in REFRESHED_STAGING.columns
    else None
)


REFRESHED_STAGING[
    "_event_id_key"
] = (

    REFRESHED_STAGING[
        "eventId"
    ]
    .map(
        normalize_wa_id
    )
    if "eventId"
    in REFRESHED_STAGING.columns
    else None
)


REFRESHED_STAGING[
    "_race_key"
] = (

    normalized_string(
        REFRESHED_STAGING[
            "race"
        ]
    )

    if "race"
    in REFRESHED_STAGING.columns

    else ""
)


SOURCE_IDENTITY_COLUMNS = [
    "aaAthleteId",
    "result_date_dt",
    "_competition_id_key",
    "_event_id_key",
    "_race_key",
]


SOURCE_IDENTITY_DUPLICATES = (

    REFRESHED_STAGING.loc[
        REFRESHED_STAGING[
            "canonical_candidate"
        ]
    ]
    .groupby(
        SOURCE_IDENTITY_COLUMNS,
        dropna=False,
    )
    .agg(
        n_rows=(
            "mark_numeric_m",
            "size",
        ),

        n_marks=(
            "mark_numeric_m",
            "nunique",
        ),

        marks=(
            "mark_numeric_m",
            lambda s:
                sorted(
                    set(
                        float(x)
                        for x
                        in s.dropna()
                    )
                ),
        ),
    )
    .reset_index()
)


DUPLICATE_IDENTITY_ROWS = (

    SOURCE_IDENTITY_DUPLICATES.loc[
        SOURCE_IDENTITY_DUPLICATES[
            "n_rows"
        ]
        > 1
    ]
    .copy()
)


CONFLICTING_IDENTITY_ROWS = (

    SOURCE_IDENTITY_DUPLICATES.loc[
        SOURCE_IDENTITY_DUPLICATES[
            "n_marks"
        ]
        > 1
    ]
    .copy()
)


# Exact duplicated source observations may safely collapse.
dedupe_columns = (
    SOURCE_IDENTITY_COLUMNS
    +
    [
        "mark_numeric_m"
    ]
)


REFRESHED_STAGING_DEDUPED = (

    REFRESHED_STAGING.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .drop_duplicates(
        subset=
            dedupe_columns,

        keep=
            "last",
    )
    .reset_index(
        drop=True
    )
)


REFRESHED_CANONICAL_CANDIDATES = (

    REFRESHED_STAGING_DEDUPED.loc[
        REFRESHED_STAGING_DEDUPED[
            "canonical_candidate"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


REFRESHED_EXCLUDED_ROWS = (

    REFRESHED_STAGING_DEDUPED.loc[
        ~REFRESHED_STAGING_DEDUPED[
            "canonical_candidate"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART Q — CROSS-SOURCE COVERAGE
# ==================================================================================================
#
# Compare:
#
#       competition scan
#           vs
#       refreshed athlete history
#
# inside the incremental date window.
#
# We use a result fingerprint:
#
#       athlete
#       date
#       competition
#       event
#       numeric mark
#
# Summary/race duplicates from the competition tree therefore collapse.
# ==================================================================================================

def build_result_fingerprint(
    athlete_id,
    result_date,
    competition_id,
    event_id,
    mark,
):
    athlete_id = normalize_wa_id(
        athlete_id
    )


    competition_id = normalize_wa_id(
        competition_id
    )


    event_id = normalize_wa_id(
        event_id
    )


    result_date = pd.to_datetime(
        result_date,
        errors="coerce",
    )


    mark = parse_mark_m(
        mark
    )


    if (
        athlete_id is None
        or
        pd.isna(
            result_date
        )
        or
        pd.isna(
            mark
        )
    ):

        return None


    return "|".join(
        [
            str(
                int(
                    athlete_id
                )
            ),

            result_date.strftime(
                "%Y-%m-%d"
            ),

            (
                str(
                    int(
                        competition_id
                    )
                )
                if competition_id is not None
                else ""
            ),

            (
                str(
                    int(
                        event_id
                    )
                )
                if event_id is not None
                else ""
            ),

            f"{float(mark):.3f}",
        ]
    )


DISCOVERY_QA = (

    INCREMENTAL_RESULTS_MODEL_ELIGIBLE.copy()
)


DISCOVERY_QA[
    "_fingerprint"
] = [

    build_result_fingerprint(
        athlete_id=
            row.get(
                "aaAthleteId"
            ),

        result_date=
            row.get(
                "result_date_dt"
            ),

        competition_id=
            row.get(
                "competition_id"
            ),

        event_id=
            row.get(
                "event_id"
            ),

        mark=
            row.get(
                "mark"
            ),
    )

    for _, row
    in DISCOVERY_QA.iterrows()
]


DISCOVERY_FINGERPRINTS = set(

    DISCOVERY_QA[
        "_fingerprint"
    ]
    .dropna()
    .unique()
)


REFRESH_QA = (

    REFRESHED_CANONICAL_CANDIDATES.loc[
        (
            REFRESHED_CANONICAL_CANDIDATES[
                "result_date_dt"
            ]
            >= INCREMENTAL_START
        )
        &
        (
            REFRESHED_CANONICAL_CANDIDATES[
                "result_date_dt"
            ]
            <= INCREMENTAL_END
        )
    ]
    .copy()
)


REFRESH_QA[
    "_fingerprint"
] = [

    build_result_fingerprint(
        athlete_id=
            row.get(
                "aaAthleteId"
            ),

        result_date=
            row.get(
                "result_date_dt"
            ),

        competition_id=
            row.get(
                "competitionId"
            ),

        event_id=
            row.get(
                "eventId"
            ),

        mark=
            row.get(
                "mark_numeric_m"
            ),
    )

    for _, row
    in REFRESH_QA.iterrows()
]


REFRESH_FINGERPRINTS = set(

    REFRESH_QA[
        "_fingerprint"
    ]
    .dropna()
    .unique()
)


DISCOVERY_MISSING_FROM_HISTORY = (

    DISCOVERY_FINGERPRINTS
    -
    REFRESH_FINGERPRINTS
)


DISCOVERY_MATCHED = (

    DISCOVERY_FINGERPRINTS
    &
    REFRESH_FINGERPRINTS
)


CROSS_SOURCE_COVERAGE = (

    len(
        DISCOVERY_MATCHED
    )
    /
    len(
        DISCOVERY_FINGERPRINTS
    )

    if len(
        DISCOVERY_FINGERPRINTS
    )
    > 0

    else np.nan
)


# ==================================================================================================
# PART R — REFRESH YEAR COVERAGE
# ==================================================================================================

REFRESHED_CANONICAL_CANDIDATES[
    "result_year"
] = (

    REFRESHED_CANONICAL_CANDIDATES[
        "result_date_dt"
    ]
    .dt.year
)


REFRESH_YEAR_COUNTS = (

    REFRESHED_CANONICAL_CANDIDATES.groupby(
        [
            "refresh_scope",
            "result_year",
            "model_group",
        ],
        as_index=False,
    )
    .agg(
        rows=(
            "mark_numeric_m",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),
    )
)


# ==================================================================================================
# PART S — QA COUNTS
# ==================================================================================================

N_RAW_JAVELIN = int(
    len(
        REFRESHED_STAGING
    )
)


N_VALID_DATE = int(
    REFRESHED_STAGING[
        "has_valid_date"
    ]
    .sum()
)


N_NUMERIC_MARK = int(
    REFRESHED_STAGING[
        "has_numeric_mark"
    ]
    .sum()
)


N_RESOLVED_SEX = int(
    REFRESHED_STAGING[
        "has_resolved_sex"
    ]
    .sum()
)


N_SUPPORTED_GROUP = int(
    REFRESHED_STAGING[
        "has_supported_model_group"
    ]
    .sum()
)


N_CANONICAL_CANDIDATES = int(
    len(
        REFRESHED_CANONICAL_CANDIDATES
    )
)


N_EXCLUDED = int(
    len(
        REFRESHED_EXCLUDED_ROWS
    )
)


N_DUPLICATE_IDENTITIES = int(
    len(
        DUPLICATE_IDENTITY_ROWS
    )
)


N_CONFLICTING_IDENTITIES = int(
    len(
        CONFLICTING_IDENTITY_ROWS
    )
)


N_DISCOVERY_FP = int(
    len(
        DISCOVERY_FINGERPRINTS
    )
)


N_DISCOVERY_MATCHED = int(
    len(
        DISCOVERY_MATCHED
    )
)


N_DISCOVERY_MISSING = int(
    len(
        DISCOVERY_MISSING_FROM_HISTORY
    )
)


# ==================================================================================================
# PART T — NORMALIZATION PASS / REVIEW
# ==================================================================================================
#
# We require:
#
#   no contradictory duplicate source identities
#
# We DO NOT yet require every raw javelin row to be model eligible:
# unsupported implements / non-marks are legitimate exclusions.
#
# Cross-source misses are surfaced for review rather than silently ignored.
# ==================================================================================================

NORMALIZATION_CORE_PASS = bool(

    N_CONFLICTING_IDENTITIES == 0

    and
    N_CANONICAL_CANDIDATES > 0
)


CROSS_SOURCE_PASS = bool(

    N_DISCOVERY_MISSING == 0
)


CELL_11_PASS = bool(

    NORMALIZATION_CORE_PASS

    and
    CROSS_SOURCE_PASS
)


# ==================================================================================================
# PART U — SAVE STAGING TABLES
# ==================================================================================================

REFRESHED_STAGING_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_normalized_staging.parquet"
)


REFRESHED_CANONICAL_CANDIDATES_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_canonical_candidates.parquet"
)


REFRESHED_EXCLUDED_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_excluded_rows.parquet"
)


DISCOVERY_MISSING_PATH = (
    RUN_CHECKPOINT_DIR
    / "competition_discovery_missing_from_athlete_history.parquet"
)


REFRESHED_STAGING_DEDUPED.to_parquet(
    REFRESHED_STAGING_PATH,
    index=False,
)


REFRESHED_CANONICAL_CANDIDATES.to_parquet(
    REFRESHED_CANONICAL_CANDIDATES_PATH,
    index=False,
)


REFRESHED_EXCLUDED_ROWS.to_parquet(
    REFRESHED_EXCLUDED_PATH,
    index=False,
)


if N_DISCOVERY_MISSING > 0:

    DISCOVERY_QA.loc[
        DISCOVERY_QA[
            "_fingerprint"
        ]
        .isin(
            DISCOVERY_MISSING_FROM_HISTORY
        )
    ].to_parquet(
        DISCOVERY_MISSING_PATH,
        index=False,
    )


else:

    pd.DataFrame().to_parquet(
        DISCOVERY_MISSING_PATH,
        index=False,
    )


# ==================================================================================================
# PART V — MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "refreshed_history_normalization"
] = {

    "status":
        (
            "VALIDATED"
            if CELL_11_PASS
            else "REVIEW_REQUIRED"
        ),

    "raw_javelin_rows":
        N_RAW_JAVELIN,

    "rows_with_valid_date":
        N_VALID_DATE,

    "rows_with_numeric_mark":
        N_NUMERIC_MARK,

    "rows_with_resolved_sex":
        N_RESOLVED_SEX,

    "rows_with_supported_model_group":
        N_SUPPORTED_GROUP,

    "canonical_candidate_rows":
        N_CANONICAL_CANDIDATES,

    "excluded_rows":
        N_EXCLUDED,

    "duplicate_source_identities":
        N_DUPLICATE_IDENTITIES,

    "conflicting_source_identities":
        N_CONFLICTING_IDENTITIES,

    "competition_discovery_fingerprints":
        N_DISCOVERY_FP,

    "competition_discovery_matched":
        N_DISCOVERY_MATCHED,

    "competition_discovery_missing":
        N_DISCOVERY_MISSING,

    "cross_source_coverage":
        (
            float(
                CROSS_SOURCE_COVERAGE
            )
            if np.isfinite(
                CROSS_SOURCE_COVERAGE
            )
            else None
        ),

    "staging_path":
        str(
            REFRESHED_STAGING_PATH
        ),

    "canonical_candidate_path":
        str(
            REFRESHED_CANONICAL_CANDIDATES_PATH
        ),

    "excluded_path":
        str(
            REFRESHED_EXCLUDED_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "REFRESHED_HISTORY_NORMALIZED"

    if CELL_11_PASS

    else "REFRESHED_HISTORY_NORMALIZATION_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART W — OUTPUT
# ==================================================================================================

print()
print("=" * 130)
print("WORLD ATHLETICS REFRESHED HISTORY — NORMALIZATION & CROSS-SOURCE QA")
print("=" * 130)


print()
print("CANONICAL SCHEMA DISCOVERY")
print("-" * 130)

print(
    f"Athlete ID column              : {CAN_ID_COL}"
)

print(
    f"Result date column             : {CAN_DATE_COL}"
)

print(
    f"Sex column                     : {CAN_SEX_COL}"
)

print(
    f"DOB column                     : {CAN_DOB_COL}"
)

print(
    f"Event ID column                : {CAN_EVENT_ID_COL}"
)

print(
    f"Model group column             : {CAN_MODEL_GROUP_COL}"
)

print(
    f"Implement column               : {CAN_IMPLEMENT_COL}"
)

print(
    f"Numeric mark column            : {CAN_MARK_NUMERIC_COL}"
)

print(
    f"Canonical columns              : {len(CANONICAL_BASE.columns):,}"
)


print()
print("REFRESHED HISTORY")
print("-" * 130)

print(
    f"Raw javelin-like rows          : {N_RAW_JAVELIN:,}"
)

print(
    f"Valid result date              : {N_VALID_DATE:,}"
)

print(
    f"Numeric mark                   : {N_NUMERIC_MARK:,}"
)

print(
    f"Resolved sex                   : {N_RESOLVED_SEX:,}"
)

print(
    f"Supported model group          : {N_SUPPORTED_GROUP:,}"
)

print(
    f"Canonical candidate rows       : {N_CANONICAL_CANDIDATES:,}"
)

print(
    f"Excluded rows                  : {N_EXCLUDED:,}"
)


print()
print("MODEL GROUP RESOLUTION")
print("-" * 130)

display(
    REFRESHED_STAGING_DEDUPED.groupby(
        [
            "group_resolution_source",
            "model_group",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        rows=(
            "aaAthleteId",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),
    )
    .sort_values(
        "rows",
        ascending=False,
    )
)


print()
print("DUPLICATE SOURCE CHECK")
print("-" * 130)

print(
    f"Duplicate identity groups      : {N_DUPLICATE_IDENTITIES:,}"
)

print(
    f"Conflicting identity groups    : {N_CONFLICTING_IDENTITIES:,}"
)


if N_DUPLICATE_IDENTITIES > 0:

    display(
        DUPLICATE_IDENTITY_ROWS.head(
            100
        )
    )


print()
print("COMPETITION ↔ ATHLETE-HISTORY CROSS-CHECK")
print("-" * 130)

print(
    f"Competition fingerprints       : {N_DISCOVERY_FP:,}"
)

print(
    f"Matched in athlete history     : {N_DISCOVERY_MATCHED:,}"
)

print(
    f"Missing from athlete history   : {N_DISCOVERY_MISSING:,}"
)

print(
    f"Coverage                       : "
    f"{CROSS_SOURCE_COVERAGE:.4%}"
    if np.isfinite(
        CROSS_SOURCE_COVERAGE
    )
    else "Coverage                       : N/A"
)


if N_DISCOVERY_MISSING > 0:

    print()
    print("DISCOVERY ROWS MISSING FROM ATHLETE HISTORY")
    print("-" * 130)

    display(
        DISCOVERY_QA.loc[
            DISCOVERY_QA[
                "_fingerprint"
            ]
            .isin(
                DISCOVERY_MISSING_FROM_HISTORY
            )
        ]
        .head(
            100
        )
    )


print()
print("REFRESHED MODEL-ELIGIBLE HISTORY")
print("-" * 130)

display(
    REFRESH_YEAR_COUNTS
)


print()
print("EXCLUDED ROW REASONS")
print("-" * 130)

if len(
    REFRESHED_EXCLUDED_ROWS
) > 0:

    exclusion_summary = (

        REFRESHED_EXCLUDED_ROWS.groupby(
            [
                "has_valid_date",
                "has_numeric_mark",
                "has_resolved_sex",
                "has_supported_model_group",
                "group_resolution_source",
            ],
            dropna=False,
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size":
                    "rows"
            }
        )
        .sort_values(
            "rows",
            ascending=False,
        )
    )


    display(
        exclusion_summary
    )


    print()
    print("FIRST EXCLUDED ROWS")
    print("-" * 130)

    display(
        REFRESHED_EXCLUDED_ROWS[
            [
                column
                for column
                in [
                    "aaAthleteId",
                    "result_date_dt",
                    "discipline",
                    "eventId",
                    "mark",
                    "mark_numeric_m",
                    "sex",
                    "explicit_implement_weight_g",
                    "model_group",
                    "group_resolution_source",
                ]
                if column
                in REFRESHED_EXCLUDED_ROWS.columns
            ]
        ]
        .head(
            100
        )
    )


print()
print("CHECKPOINTS")
print("-" * 130)

print(
    f"Normalized staging             : {REFRESHED_STAGING_PATH}"
)

print(
    f"Canonical candidates           : {REFRESHED_CANONICAL_CANDIDATES_PATH}"
)

print(
    f"Excluded rows                  : {REFRESHED_EXCLUDED_PATH}"
)

print(
    f"Cross-source missing           : {DISCOVERY_MISSING_PATH}"
)


print()
print("=" * 130)


if CELL_11_PASS:

    print(
        "CELL 11 STATUS: NORMALIZATION & CROSS-SOURCE QA PASSED"
    )

    print()
    print(
        "Cell 12 may reconcile refreshed athlete-year slices into "
        "the historical canonical dataset."
    )


else:

    print(
        "CELL 11 STATUS: REVIEW REQUIRED"
    )

    print()
    print(
        "Do not replace canonical athlete-year slices yet."
    )


print("=" * 130)


Canonical rows loaded       : 248,896

WORLD ATHLETICS REFRESHED HISTORY — NORMALIZATION & CROSS-SOURCE QA

CANONICAL SCHEMA DISCOVERY
----------------------------------------------------------------------------------------------------------------------------------
Athlete ID column              : aaAthleteId
Result date column             : result_date_dt
Sex column                     : None
DOB column                     : None
Event ID column                : event_id
Model group column             : model_group
Implement column               : implement_weight_g
Numeric mark column            : mark_numeric_m
Canonical columns              : 20

REFRESHED HISTORY
----------------------------------------------------------------------------------------------------------------------------------
Raw javelin-like rows          : 4,187
Valid result date              : 4,187
Numeric mark                   : 4,157
Resolved sex                   : 4,187
Supported model group          : 4,

,group_resolution_source,model_group,rows,athletes
0,CANONICAL_EVENT_ID_MAP,M_800,1731,351
1,CANONICAL_EVENT_ID_MAP,W_600,1701,321
3,EXPLICIT_WEIGHT,W_500,421,135
2,EXPLICIT_WEIGHT,M_700,259,103
4,UNRESOLVED,NaN,75,9



DUPLICATE SOURCE CHECK
----------------------------------------------------------------------------------------------------------------------------------
Duplicate identity groups      : 0
Conflicting identity groups    : 0

COMPETITION ↔ ATHLETE-HISTORY CROSS-CHECK
----------------------------------------------------------------------------------------------------------------------------------
Competition fingerprints       : 976
Matched in athlete history     : 973
Missing from athlete history   : 3
Coverage                       : 99.6926%

DISCOVERY ROWS MISSING FROM ATHLETE HISTORY
----------------------------------------------------------------------------------------------------------------------------------


,competition_id,competition_name,requested_day,source_layer,event_title,ranking_category,event,event_id,gender,is_relay,...,source_iaafId,competitor_birthDate,result_date_dt,inside_incremental_window,summary_index,aaAthleteId,slug_provisional_aaAthleteId,identity_status,eligible_for_canonical_model,_fingerprint
123,7236717,Chilean U16 Championships,1,race_result,None,F,Men's Javelin Throw (600gr),10229872,M,False,...,NaN,15 FEB 2011,2026-09-12,True,NaN,15332265.0,15332265,CANONICAL_ID_RESOLVED,True,15332265|2026-09-12|7236717|10229872|56.680
124,7236717,Chilean U16 Championships,1,race_result,None,F,Men's Javelin Throw (600gr),10229872,M,False,...,NaN,04 JAN 2011,2026-09-12,True,NaN,15347866.0,15347866,CANONICAL_ID_RESOLVED,True,15347866|2026-09-12|7236717|10229872|55.810
1105,7244867,Slovenian U16 Championships,2,race_result,None,F,Women's Javelin Throw (400gr),10229958,W,False,...,NaN,2011,2026-09-13,True,NaN,15227826.0,15227826,CANONICAL_ID_RESOLVED,True,15227826|2026-09-13|7244867|10229958|39.200



REFRESHED MODEL-ELIGIBLE HISTORY
----------------------------------------------------------------------------------------------------------------------------------


...


EXCLUDED ROW REASONS
----------------------------------------------------------------------------------------------------------------------------------


,has_valid_date,has_numeric_mark,has_resolved_sex,has_supported_model_group,group_resolution_source,rows
3,True,True,True,False,UNRESOLVED,69
1,True,False,True,True,CANONICAL_EVENT_ID_MAP,20
0,True,False,True,False,UNRESOLVED,6
2,True,False,True,True,EXPLICIT_WEIGHT,4



FIRST EXCLUDED ROWS
----------------------------------------------------------------------------------------------------------------------------------


,aaAthleteId,result_date_dt,discipline,eventId,mark,mark_numeric_m,sex,explicit_implement_weight_g,model_group,group_resolution_source
0,14228196,2003-08-01,Javelin Throw,None,58.72,58.72,M,NaN,None,UNRESOLVED
1,14228196,2003-08-17,Javelin Throw,None,63.66,63.66,M,NaN,None,UNRESOLVED
2,14228196,2004-06-19,Javelin Throw,None,65.02,65.02,M,NaN,None,UNRESOLVED
3,14228196,2004-08-21,Javelin Throw,None,69.38,69.38,M,NaN,None,UNRESOLVED
4,14228196,2005-08-19,Javelin Throw,None,66.52,66.52,M,NaN,None,UNRESOLVED
...,...,...,...,...,...,...,...,...,...,...
94,15281782,2026-07-26,Javelin Throw (500g),10229927,NM,NaN,W,500.0,W_500,EXPLICIT_WEIGHT
95,15323024,2026-09-06,Javelin Throw,10229636,NM,NaN,M,NaN,M_800,CANONICAL_EVENT_ID_MAP
96,15324270,2026-09-06,Javelin Throw (700g),10229828,NM,NaN,M,700.0,M_700,EXPLICIT_WEIGHT
97,15332265,2026-09-12,Javelin Throw (600gr),10229872,56.68,56.68,M,NaN,None,UNRESOLVED



CHECKPOINTS
----------------------------------------------------------------------------------------------------------------------------------
Normalized staging             : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_normalized_staging.parquet
Canonical candidates           : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_canonical_candidates.parquet
Excluded rows                  : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_excluded_rows.parquet
Cross-source missing           : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/competition_discovery_missing_from_athlete_history.parquet

CELL 11 STATUS: REVIEW REQUIRED

Do not replace canonical athlete-year slices yet.


In [13]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 11A — IMPLEMENT-RESOLUTION PATCH + CORRECTED CROSS-SOURCE QA
# ==================================================================================================
#
# Fixes two issues exposed by Cell 11:
#
#   1. Explicit weights such as "(600gr)" and "(400gr)" were not parsed.
#
#   2. Cross-source QA incorrectly compared competition discovery only against
#      SUPPORTED model rows. Unsupported javelin implements therefore appeared
#      to be "missing" even though they existed in athlete history.
#
#
# Conservative generic-event fallback
# -----------------------------------
#
# For an unweighted generic "Javelin Throw" with no usable eventId:
#
#       definitely age >= 18 + Male   -> M_800
#       definitely age >= 18 + Female -> W_600
#
# We DO NOT infer youth implement groups from age alone.
#
# ==================================================================================================

from __future__ import annotations

import json
import re

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "REFRESHED_STAGING",
    "DISCOVERY_QA",
    "INCREMENTAL_START",
    "INCREMENTAL_END",

    "EVENT_TO_GROUP_MAP",
    "EVENT_TO_WEIGHT_MAP",

    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "normalize_wa_id",
    "parse_mark_m",
    "model_group_from_sex_weight",
    "build_result_fingerprint",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cell 11 first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


# ==================================================================================================
# PART B — IMPROVED EXPLICIT IMPLEMENT PARSER
# ==================================================================================================

def explicit_implement_weight_g_v2(
    row,
):
    """
    Extract explicit javelin implement weight.

    Supports examples such as:

        (700g)
        (700gr)
        700 g
        700 gram
        700 grams

        0.7kg
        0.8 kg

    Returns
    -------
    float
        Weight in grams, or NaN.
    """

    candidate_fields = [
        row.get("discipline"),
        row.get("disciplineNameUrlSlug"),
        row.get("typeNameUrlSlug"),
    ]


    text = " ".join(
        str(value)
        for value in candidate_fields
        if value is not None
    ).lower()


    # ------------------------------------------------------------------------------------------------
    # Grams
    # ------------------------------------------------------------------------------------------------

    gram_match = re.search(
        r"(?<!\d)(\d{3,4})\s*(?:g|gr|gram|grams)\b",
        text,
    )


    if gram_match is not None:

        return float(
            gram_match.group(1)
        )


    # ------------------------------------------------------------------------------------------------
    # Kilograms
    # ------------------------------------------------------------------------------------------------

    kg_match = re.search(
        r"(?<!\d)(\d(?:\.\d+)?)\s*kg\b",
        text,
    )


    if kg_match is not None:

        return float(
            kg_match.group(1)
        ) * 1000.0


    return np.nan


# ==================================================================================================
# PART C — DEFINITELY-ADULT TEST
# ==================================================================================================

def definitely_age_18_plus(
    row,
):
    """
    Return True only when we can establish that the athlete is at least 18.

    Uses:
        exact age first
        age interval lower bound second

    Does NOT use uncertain birth-year midpoint assumptions.
    """

    age_exact = row.get(
        "age_years"
    )


    if (
        age_exact is not None
        and
        not pd.isna(age_exact)
    ):

        return bool(
            float(age_exact) >= 18.0
        )


    age_lower = row.get(
        "age_lower_years"
    )


    if (
        age_lower is not None
        and
        not pd.isna(age_lower)
    ):

        return bool(
            float(age_lower) >= 18.0
        )


    return False


# ==================================================================================================
# PART D — GENERIC JAVELIN CHECK
# ==================================================================================================

def is_generic_unweighted_javelin(
    row,
):
    """
    True when WA identifies the performance only as generic javelin,
    with no explicit implement weight.
    """

    discipline = str(
        row.get(
            "discipline"
        )
        or ""
    ).strip().lower()


    if "javelin" not in discipline:

        return False


    explicit_weight = (
        explicit_implement_weight_g_v2(
            row
        )
    )


    return bool(
        pd.isna(
            explicit_weight
        )
    )


# ==================================================================================================
# PART E — REBUILD IMPLEMENT / GROUP RESOLUTION
# ==================================================================================================

STAGING_V2 = (
    REFRESHED_STAGING.copy()
)


STAGING_V2[
    "explicit_implement_weight_g"
] = (

    STAGING_V2.apply(
        explicit_implement_weight_g_v2,
        axis=1,
    )
)


def resolve_model_group_v2(
    row,
):
    """
    Resolution hierarchy
    --------------------

    1. Explicit implement weight in WA discipline text.
    2. Existing canonical eventId -> model_group mapping.
    3. Conservative adult fallback for generic unweighted javelin.
    4. Leave unresolved.

    Unsupported explicit implements are retained and clearly classified.
    """

    sex = row.get(
        "sex"
    )


    explicit_weight = row.get(
        "explicit_implement_weight_g"
    )


    # ==============================================================================================
    # 1. EXPLICIT WEIGHT
    # ==============================================================================================

    if (
        explicit_weight is not None
        and
        not pd.isna(explicit_weight)
    ):

        explicit_weight = int(
            round(
                float(
                    explicit_weight
                )
            )
        )


        group = model_group_from_sex_weight(
            sex,
            explicit_weight,
        )


        if group is not None:

            return (
                group,
                explicit_weight,
                "EXPLICIT_WEIGHT",
            )


        # Valid javelin performance, but outside this model's supported groups.
        return (
            None,
            explicit_weight,
            "EXPLICIT_UNSUPPORTED_WEIGHT",
        )


    # ==============================================================================================
    # 2. HISTORICAL EVENT-ID MAP
    # ==============================================================================================

    event_id = normalize_wa_id(
        row.get(
            "event_id_norm"
        )
    )


    if event_id is not None:

        event_id = int(
            event_id
        )


        mapped_group = EVENT_TO_GROUP_MAP.get(
            event_id
        )


        mapped_weight = EVENT_TO_WEIGHT_MAP.get(
            event_id
        )


        if mapped_group is not None:

            return (
                mapped_group,
                mapped_weight,
                "CANONICAL_EVENT_ID_MAP",
            )


    # ==============================================================================================
    # 3. CONSERVATIVE ADULT FALLBACK
    # ==============================================================================================
    #
    # At age 18+, generic senior/U20 javelin aligns with:
    #
    #       Men   800g
    #       Women 600g
    #
    # We deliberately DO NOT assign youth weights from age alone.
    # ==============================================================================================

    if (
        is_generic_unweighted_javelin(
            row
        )
        and
        definitely_age_18_plus(
            row
        )
    ):

        if sex == "M":

            return (
                "M_800",
                800,
                "ADULT_GENERIC_AGE_FALLBACK",
            )


        if sex == "W":

            return (
                "W_600",
                600,
                "ADULT_GENERIC_AGE_FALLBACK",
            )


    # ==============================================================================================
    # 4. UNRESOLVED
    # ==============================================================================================

    return (
        None,
        np.nan,
        "UNRESOLVED",
    )


group_resolution_v2 = (

    STAGING_V2.apply(
        resolve_model_group_v2,
        axis=1,
    )
)


STAGING_V2[
    "model_group"
] = [
    value[0]
    for value
    in group_resolution_v2
]


STAGING_V2[
    "implement_weight_g"
] = [
    value[1]
    for value
    in group_resolution_v2
]


STAGING_V2[
    "group_resolution_source"
] = [
    value[2]
    for value
    in group_resolution_v2
]


# ==================================================================================================
# PART F — REBUILD ELIGIBILITY
# ==================================================================================================

STAGING_V2[
    "has_valid_date"
] = (

    STAGING_V2[
        "result_date_dt"
    ]
    .notna()
)


STAGING_V2[
    "has_numeric_mark"
] = (

    STAGING_V2[
        "mark_numeric_m"
    ]
    .notna()
)


STAGING_V2[
    "has_resolved_sex"
] = (

    STAGING_V2[
        "sex"
    ]
    .notna()
)


STAGING_V2[
    "has_supported_model_group"
] = (

    STAGING_V2[
        "model_group"
    ]
    .isin(
        [
            "M_700",
            "M_800",
            "W_500",
            "W_600",
        ]
    )
)


STAGING_V2[
    "canonical_candidate"
] = (

    STAGING_V2[
        "has_valid_date"
    ]

    &
    STAGING_V2[
        "has_numeric_mark"
    ]

    &
    STAGING_V2[
        "has_resolved_sex"
    ]

    &
    STAGING_V2[
        "has_supported_model_group"
    ]
)


# ==================================================================================================
# PART G — SOURCE DUPLICATE / CONFLICT CHECK
# ==================================================================================================

STAGING_V2[
    "_competition_id_key"
] = (

    STAGING_V2[
        "competitionId"
    ]
    .map(
        normalize_wa_id
    )
    if "competitionId"
    in STAGING_V2.columns
    else None
)


STAGING_V2[
    "_event_id_key"
] = (

    STAGING_V2[
        "eventId"
    ]
    .map(
        normalize_wa_id
    )
    if "eventId"
    in STAGING_V2.columns
    else None
)


if "race" in STAGING_V2.columns:

    STAGING_V2[
        "_race_key"
    ] = (

        STAGING_V2[
            "race"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )


else:

    STAGING_V2[
        "_race_key"
    ] = ""


SOURCE_IDENTITY_COLUMNS_V2 = [
    "aaAthleteId",
    "result_date_dt",
    "_competition_id_key",
    "_event_id_key",
    "_race_key",
]


SOURCE_IDENTITY_QA_V2 = (

    STAGING_V2.loc[
        STAGING_V2[
            "has_valid_date"
        ]
        &
        STAGING_V2[
            "has_numeric_mark"
        ]
    ]
    .groupby(
        SOURCE_IDENTITY_COLUMNS_V2,
        dropna=False,
    )
    .agg(
        n_rows=(
            "mark_numeric_m",
            "size",
        ),

        n_marks=(
            "mark_numeric_m",
            "nunique",
        ),

        marks=(
            "mark_numeric_m",
            lambda s:
                sorted(
                    set(
                        float(value)
                        for value
                        in s.dropna()
                    )
                ),
        ),
    )
    .reset_index()
)


DUPLICATE_SOURCE_IDENTITIES_V2 = (

    SOURCE_IDENTITY_QA_V2.loc[
        SOURCE_IDENTITY_QA_V2[
            "n_rows"
        ]
        > 1
    ]
    .copy()
)


CONFLICTING_SOURCE_IDENTITIES_V2 = (

    SOURCE_IDENTITY_QA_V2.loc[
        SOURCE_IDENTITY_QA_V2[
            "n_marks"
        ]
        > 1
    ]
    .copy()
)


# ==================================================================================================
# PART H — EXACT SOURCE DEDUPLICATION
# ==================================================================================================

dedupe_columns_v2 = (
    SOURCE_IDENTITY_COLUMNS_V2
    +
    [
        "mark_numeric_m"
    ]
)


STAGING_V2_DEDUPED = (

    STAGING_V2.sort_values(
        [
            "aaAthleteId",
            "result_date_dt",
        ]
    )
    .drop_duplicates(
        subset=
            dedupe_columns_v2,

        keep=
            "last",
    )
    .reset_index(
        drop=True
    )
)


CANONICAL_CANDIDATES_V2 = (

    STAGING_V2_DEDUPED.loc[
        STAGING_V2_DEDUPED[
            "canonical_candidate"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


EXCLUDED_ROWS_V2 = (

    STAGING_V2_DEDUPED.loc[
        ~STAGING_V2_DEDUPED[
            "canonical_candidate"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART I — DISTINGUISH UNSUPPORTED FROM TRULY UNRESOLVED
# ==================================================================================================

UNSUPPORTED_IMPLEMENT_ROWS = (

    STAGING_V2_DEDUPED.loc[
        STAGING_V2_DEDUPED[
            "group_resolution_source"
        ]
        == "EXPLICIT_UNSUPPORTED_WEIGHT"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


UNRESOLVED_GROUP_ROWS = (

    STAGING_V2_DEDUPED.loc[
        (
            STAGING_V2_DEDUPED[
                "group_resolution_source"
            ]
            == "UNRESOLVED"
        )
        &
        (
            STAGING_V2_DEDUPED[
                "has_numeric_mark"
            ]
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART J — CORRECTED CROSS-SOURCE QA
# ==================================================================================================
#
# IMPORTANT:
#
# Cross-source QA asks:
#
#       "Was the competition-discovered performance also present in the
#        athlete-history endpoint?"
#
# It does NOT ask:
#
#       "Is the performance supported by the Bayesian model?"
#
# Therefore ALL valid numeric javelin results must participate, including:
#
#       M_600
#       W_400
#       other unsupported implements
#
# ==================================================================================================

ATHLETE_HISTORY_QA_ALL = (

    STAGING_V2_DEDUPED.loc[
        (
            STAGING_V2_DEDUPED[
                "has_valid_date"
            ]
        )
        &
        (
            STAGING_V2_DEDUPED[
                "has_numeric_mark"
            ]
        )
        &
        (
            STAGING_V2_DEDUPED[
                "result_date_dt"
            ]
            >= INCREMENTAL_START
        )
        &
        (
            STAGING_V2_DEDUPED[
                "result_date_dt"
            ]
            <= INCREMENTAL_END
        )
    ]
    .copy()
)


ATHLETE_HISTORY_QA_ALL[
    "_fingerprint"
] = [

    build_result_fingerprint(

        athlete_id=
            row.get(
                "aaAthleteId"
            ),

        result_date=
            row.get(
                "result_date_dt"
            ),

        competition_id=
            row.get(
                "competitionId"
            ),

        event_id=
            row.get(
                "eventId"
            ),

        mark=
            row.get(
                "mark_numeric_m"
            ),
    )

    for _, row
    in ATHLETE_HISTORY_QA_ALL.iterrows()
]


ATHLETE_HISTORY_ALL_FINGERPRINTS = set(

    ATHLETE_HISTORY_QA_ALL[
        "_fingerprint"
    ]
    .dropna()
    .unique()
)


# Reuse the competition fingerprints generated by Cell 11.
DISCOVERY_FINGERPRINTS_V2 = set(

    DISCOVERY_QA[
        "_fingerprint"
    ]
    .dropna()
    .unique()
)


DISCOVERY_MATCHED_V2 = (

    DISCOVERY_FINGERPRINTS_V2
    &
    ATHLETE_HISTORY_ALL_FINGERPRINTS
)


DISCOVERY_MISSING_V2 = (

    DISCOVERY_FINGERPRINTS_V2
    -
    ATHLETE_HISTORY_ALL_FINGERPRINTS
)


HISTORY_EXTRA_V2 = (

    ATHLETE_HISTORY_ALL_FINGERPRINTS
    -
    DISCOVERY_FINGERPRINTS_V2
)


CROSS_SOURCE_COVERAGE_V2 = (

    len(
        DISCOVERY_MATCHED_V2
    )
    /
    len(
        DISCOVERY_FINGERPRINTS_V2
    )

    if len(
        DISCOVERY_FINGERPRINTS_V2
    )
    > 0

    else np.nan
)


# ==================================================================================================
# PART K — QA COUNTS
# ==================================================================================================

N_RAW_ROWS_V2 = int(
    len(
        STAGING_V2
    )
)


N_CANONICAL_CANDIDATES_V2 = int(
    len(
        CANONICAL_CANDIDATES_V2
    )
)


N_UNSUPPORTED_ROWS = int(
    len(
        UNSUPPORTED_IMPLEMENT_ROWS
    )
)


N_UNRESOLVED_NUMERIC_GROUP_ROWS = int(
    len(
        UNRESOLVED_GROUP_ROWS
    )
)


N_CONFLICTS_V2 = int(
    len(
        CONFLICTING_SOURCE_IDENTITIES_V2
    )
)


N_DISCOVERY_FP_V2 = int(
    len(
        DISCOVERY_FINGERPRINTS_V2
    )
)


N_MATCHED_V2 = int(
    len(
        DISCOVERY_MATCHED_V2
    )
)


N_MISSING_V2 = int(
    len(
        DISCOVERY_MISSING_V2
    )
)


N_HISTORY_EXTRA_V2 = int(
    len(
        HISTORY_EXTRA_V2
    )
)


# ==================================================================================================
# PART L — RELEASE CRITERIA
# ==================================================================================================
#
# We require:
#
#       no conflicting source identities
#       complete competition -> athlete-history reconciliation
#       no numeric supported-history rows left with unknown implement
#
# Explicit unsupported implements are acceptable exclusions.
# Non-numeric marks such as NM are acceptable exclusions.
#
# ==================================================================================================

CELL_11A_PASS = bool(

    N_CONFLICTS_V2 == 0

    and
    N_MISSING_V2 == 0

    and
    N_UNRESOLVED_NUMERIC_GROUP_ROWS == 0

    and
    N_CANONICAL_CANDIDATES_V2 > 0
)


# ==================================================================================================
# PART M — SAVE PATCHED STAGING OUTPUTS
# ==================================================================================================

STAGING_V2_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_normalized_staging_v2.parquet"
)


CANONICAL_CANDIDATES_V2_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_canonical_candidates_v2.parquet"
)


EXCLUDED_ROWS_V2_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_excluded_rows_v2.parquet"
)


UNSUPPORTED_IMPLEMENT_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_unsupported_implements.parquet"
)


UNRESOLVED_GROUP_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_unresolved_groups_v2.parquet"
)


DISCOVERY_MISSING_V2_PATH = (
    RUN_CHECKPOINT_DIR
    / "competition_discovery_missing_from_history_v2.parquet"
)


STAGING_V2_DEDUPED.to_parquet(
    STAGING_V2_PATH,
    index=False,
)


CANONICAL_CANDIDATES_V2.to_parquet(
    CANONICAL_CANDIDATES_V2_PATH,
    index=False,
)


EXCLUDED_ROWS_V2.to_parquet(
    EXCLUDED_ROWS_V2_PATH,
    index=False,
)


UNSUPPORTED_IMPLEMENT_ROWS.to_parquet(
    UNSUPPORTED_IMPLEMENT_PATH,
    index=False,
)


UNRESOLVED_GROUP_ROWS.to_parquet(
    UNRESOLVED_GROUP_PATH,
    index=False,
)


if N_MISSING_V2 > 0:

    DISCOVERY_QA.loc[
        DISCOVERY_QA[
            "_fingerprint"
        ]
        .isin(
            DISCOVERY_MISSING_V2
        )
    ].to_parquet(
        DISCOVERY_MISSING_V2_PATH,
        index=False,
    )


else:

    pd.DataFrame().to_parquet(
        DISCOVERY_MISSING_V2_PATH,
        index=False,
    )


# ==================================================================================================
# PART N — PROMOTE PATCHED VARIABLES FOR CELL 12
# ==================================================================================================
#
# Cell 12 should use these patched staging objects.
# ==================================================================================================

REFRESHED_STAGING_FINAL = (
    STAGING_V2_DEDUPED
)


REFRESHED_CANONICAL_CANDIDATES_FINAL = (
    CANONICAL_CANDIDATES_V2
)


REFRESHED_EXCLUDED_ROWS_FINAL = (
    EXCLUDED_ROWS_V2
)


NORMALIZATION_FINAL_PASS = (
    CELL_11A_PASS
)


# ==================================================================================================
# PART O — UPDATE MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "refreshed_history_normalization_v2"
] = {

    "status":
        (
            "VALIDATED"
            if CELL_11A_PASS
            else "REVIEW_REQUIRED"
        ),

    "raw_rows":
        N_RAW_ROWS_V2,

    "canonical_candidate_rows":
        N_CANONICAL_CANDIDATES_V2,

    "unsupported_implement_rows":
        N_UNSUPPORTED_ROWS,

    "unresolved_numeric_group_rows":
        N_UNRESOLVED_NUMERIC_GROUP_ROWS,

    "conflicting_source_identities":
        N_CONFLICTS_V2,

    "competition_fingerprints":
        N_DISCOVERY_FP_V2,

    "matched_fingerprints":
        N_MATCHED_V2,

    "missing_fingerprints":
        N_MISSING_V2,

    "history_only_fingerprints":
        N_HISTORY_EXTRA_V2,

    "cross_source_coverage":
        (
            float(
                CROSS_SOURCE_COVERAGE_V2
            )
            if np.isfinite(
                CROSS_SOURCE_COVERAGE_V2
            )
            else None
        ),

    "staging_path":
        str(
            STAGING_V2_PATH
        ),

    "canonical_candidates_path":
        str(
            CANONICAL_CANDIDATES_V2_PATH
        ),

    "unsupported_implements_path":
        str(
            UNSUPPORTED_IMPLEMENT_PATH
        ),

    "unresolved_group_path":
        str(
            UNRESOLVED_GROUP_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "REFRESHED_HISTORY_NORMALIZED"

    if CELL_11A_PASS

    else "REFRESHED_HISTORY_NORMALIZATION_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART P — OUTPUT
# ==================================================================================================

print()
print("=" * 130)
print("WORLD ATHLETICS REFRESHED HISTORY — IMPLEMENT PATCH & CORRECTED QA")
print("=" * 130)


print()
print("MODEL GROUP RESOLUTION — V2")
print("-" * 130)

display(
    STAGING_V2_DEDUPED.groupby(
        [
            "group_resolution_source",
            "model_group",
            "implement_weight_g",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        rows=(
            "aaAthleteId",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),
    )
    .sort_values(
        "rows",
        ascending=False,
    )
)


print()
print("CANONICAL / EXCLUSION COUNTS")
print("-" * 130)

print(
    f"Raw javelin rows               : {N_RAW_ROWS_V2:,}"
)

print(
    f"Canonical candidate rows        : {N_CANONICAL_CANDIDATES_V2:,}"
)

print(
    f"Explicit unsupported implements : {N_UNSUPPORTED_ROWS:,}"
)

print(
    f"Unresolved numeric group rows   : {N_UNRESOLVED_NUMERIC_GROUP_ROWS:,}"
)


print()
print("SOURCE DUPLICATE CHECK")
print("-" * 130)

print(
    f"Duplicate source identities     : {len(DUPLICATE_SOURCE_IDENTITIES_V2):,}"
)

print(
    f"Conflicting source identities   : {N_CONFLICTS_V2:,}"
)


print()
print("CORRECTED COMPETITION ↔ ATHLETE-HISTORY QA")
print("-" * 130)

print(
    f"Competition fingerprints        : {N_DISCOVERY_FP_V2:,}"
)

print(
    f"Matched in athlete history      : {N_MATCHED_V2:,}"
)

print(
    f"Missing from athlete history    : {N_MISSING_V2:,}"
)

print(
    f"History-only fingerprints       : {N_HISTORY_EXTRA_V2:,}"
)

print(
    f"Coverage                        : "
    f"{CROSS_SOURCE_COVERAGE_V2:.4%}"
    if np.isfinite(
        CROSS_SOURCE_COVERAGE_V2
    )
    else "Coverage                        : N/A"
)


if N_MISSING_V2 > 0:

    print()
    print("REMAINING CROSS-SOURCE MISSES")
    print("-" * 130)

    display(
        DISCOVERY_QA.loc[
            DISCOVERY_QA[
                "_fingerprint"
            ]
            .isin(
                DISCOVERY_MISSING_V2
            )
        ]
    )


print()
print("EXPLICIT UNSUPPORTED IMPLEMENTS")
print("-" * 130)

if N_UNSUPPORTED_ROWS > 0:

    display(
        UNSUPPORTED_IMPLEMENT_ROWS[
            [
                column
                for column
                in [
                    "aaAthleteId",
                    "result_date_dt",
                    "discipline",
                    "eventId",
                    "mark",
                    "sex",
                    "implement_weight_g",
                    "group_resolution_source",
                ]
                if column
                in UNSUPPORTED_IMPLEMENT_ROWS.columns
            ]
        ]
        .head(
            100
        )
    )

else:

    print(
        "None."
    )


print()
print("REMAINING UNRESOLVED NUMERIC IMPLEMENTS")
print("-" * 130)

if N_UNRESOLVED_NUMERIC_GROUP_ROWS > 0:

    display(
        UNRESOLVED_GROUP_ROWS[
            [
                column
                for column
                in [
                    "aaAthleteId",
                    "result_date_dt",
                    "DOB",
                    "age_years",
                    "age_lower_years",
                    "discipline",
                    "eventId",
                    "mark",
                    "sex",
                    "group_resolution_source",
                ]
                if column
                in UNRESOLVED_GROUP_ROWS.columns
            ]
        ]
        .head(
            200
        )
    )

else:

    print(
        "None."
    )


print()
print("CHECKPOINTS")
print("-" * 130)

print(
    f"Patched staging                : {STAGING_V2_PATH}"
)

print(
    f"Canonical candidates           : {CANONICAL_CANDIDATES_V2_PATH}"
)

print(
    f"Unsupported implements         : {UNSUPPORTED_IMPLEMENT_PATH}"
)

print(
    f"Unresolved implements          : {UNRESOLVED_GROUP_PATH}"
)


print()
print("=" * 130)


if CELL_11A_PASS:

    print(
        "CELL 11A STATUS: NORMALIZATION & CROSS-SOURCE QA PASSED"
    )

    print()
    print(
        "Cell 12 may now reconcile refreshed athlete-year slices "
        "against the historical canonical dataset."
    )


else:

    print(
        "CELL 11A STATUS: REVIEW REQUIRED"
    )

    print()
    print(
        "Do not replace canonical athlete-year slices yet."
    )


print("=" * 130)


WORLD ATHLETICS REFRESHED HISTORY — IMPLEMENT PATCH & CORRECTED QA

MODEL GROUP RESOLUTION — V2
----------------------------------------------------------------------------------------------------------------------------------


,group_resolution_source,model_group,implement_weight_g,rows,athletes
2,CANONICAL_EVENT_ID_MAP,M_800,800.0,1731,351
3,CANONICAL_EVENT_ID_MAP,W_600,600.0,1701,321
7,EXPLICIT_WEIGHT,W_500,500.0,421,135
6,EXPLICIT_WEIGHT,M_700,700.0,259,103
0,ADULT_GENERIC_AGE_FALLBACK,M_800,800.0,62,3
8,UNRESOLVED,NaN,NaN,7,1
1,ADULT_GENERIC_AGE_FALLBACK,W_600,600.0,2,2
4,EXPLICIT_UNSUPPORTED_WEIGHT,NaN,400.0,2,1
5,EXPLICIT_UNSUPPORTED_WEIGHT,NaN,600.0,2,2



CANONICAL / EXCLUSION COUNTS
----------------------------------------------------------------------------------------------------------------------------------
Raw javelin rows               : 4,187
Canonical candidate rows        : 4,146
Explicit unsupported implements : 4
Unresolved numeric group rows   : 7

SOURCE DUPLICATE CHECK
----------------------------------------------------------------------------------------------------------------------------------
Duplicate source identities     : 0
Conflicting source identities   : 0

CORRECTED COMPETITION ↔ ATHLETE-HISTORY QA
----------------------------------------------------------------------------------------------------------------------------------
Competition fingerprints        : 976
Matched in athlete history      : 976
Missing from athlete history    : 0
History-only fingerprints       : 0
Coverage                        : 100.0000%

EXPLICIT UNSUPPORTED IMPLEMENTS
-------------------------------------------------------------

,aaAthleteId,result_date_dt,discipline,eventId,mark,sex,implement_weight_g,group_resolution_source
0,15227826,2026-03-14,Javelin Throw (400gr),10229958,38.51,W,400.0,EXPLICIT_UNSUPPORTED_WEIGHT
1,15227826,2026-09-13,Javelin Throw (400gr),10229958,39.20,W,400.0,EXPLICIT_UNSUPPORTED_WEIGHT
2,15332265,2026-09-12,Javelin Throw (600gr),10229872,56.68,M,600.0,EXPLICIT_UNSUPPORTED_WEIGHT
3,15347866,2026-09-12,Javelin Throw (600gr),10229872,55.81,M,600.0,EXPLICIT_UNSUPPORTED_WEIGHT



REMAINING UNRESOLVED NUMERIC IMPLEMENTS
----------------------------------------------------------------------------------------------------------------------------------


,aaAthleteId,result_date_dt,DOB,age_years,age_lower_years,discipline,eventId,mark,sex,group_resolution_source
0,14800681,2019-06-02,NaT,NaN,NaN,Javelin Throw,None,37.05,W,UNRESOLVED
1,14800681,2021-06-09,NaT,NaN,NaN,Javelin Throw,None,37.29,W,UNRESOLVED
2,14800681,2021-06-20,NaT,NaN,NaN,Javelin Throw,None,37.56,W,UNRESOLVED
3,14800681,2022-06-12,NaT,NaN,NaN,Javelin Throw,None,37.12,W,UNRESOLVED
4,14800681,2022-07-16,NaT,NaN,NaN,Javelin Throw,None,36.46,W,UNRESOLVED
5,14800681,2022-08-13,NaT,NaN,NaN,Javelin Throw,None,33.42,W,UNRESOLVED
6,14800681,2022-09-10,NaT,NaN,NaN,Javelin Throw,None,34.68,W,UNRESOLVED



CHECKPOINTS
----------------------------------------------------------------------------------------------------------------------------------
Patched staging                : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_normalized_staging_v2.parquet
Canonical candidates           : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_canonical_candidates_v2.parquet
Unsupported implements         : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_unsupported_implements.parquet
Unresolved implements          : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_unresolved_groups_v2.parquet

CELL 11A STATUS: REVIEW REQUIRED

Do not replace canonical athlete-year slices yet.


In [14]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 11B — AUDITED RESOLUTION OF REMAINING HISTORICAL IMPLEMENTS
# ==================================================================================================
#
# Cell 11A left exactly seven numeric javelin results unresolved.
#
# All seven belong to:
#
#       Florence Baulk
#       aaAthleteId = 14800681
#
# External competition evidence establishes:
#
#       2019 -> U20 Women Javelin 600g
#       2021 -> Senior/U23 Women Javelin 600g
#       2022 -> Senior/U23 Women Javelin 600g
#
#
# IMPORTANT DESIGN PRINCIPLE
# --------------------------
#
# We do NOT broaden the generic implement inference rule.
#
# Instead we create an explicit, auditable override table containing only
# historically verified athlete/year cases.
#
# Future exceptions can be appended to this table deliberately.
#
# ==================================================================================================

from __future__ import annotations

import json

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "STAGING_V2_DEDUPED",
    "DISCOVERY_QA",

    "INCREMENTAL_START",
    "INCREMENTAL_END",

    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "normalize_wa_id",
    "build_result_fingerprint",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 Cell 11A first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


# ==================================================================================================
# PART B — AUDITED HISTORICAL IMPLEMENT OVERRIDES
# ==================================================================================================
#
# The table is deliberately explicit.
#
# Do not add an athlete/year here without documentary support.
#
# ==================================================================================================

HISTORICAL_IMPLEMENT_OVERRIDES = pd.DataFrame(
    [
        {
            "aaAthleteId": 14800681,
            "result_year": 2019,
            "sex": "W",
            "implement_weight_g": 600,
            "model_group": "W_600",
            "evidence_classification": "U20 Women Javelin 600g",
            "evidence_note": (
                "Florence Baulk 37.05 at 2019 Southern U20 Championships; "
                "Surrey historical profile classifies 2019 U20 result under "
                "Javelin 600g."
            ),
        },

        {
            "aaAthleteId": 14800681,
            "result_year": 2021,
            "sex": "W",
            "implement_weight_g": 600,
            "model_group": "W_600",
            "evidence_classification": "Senior/U23 Women Javelin 600g",
            "evidence_note": (
                "Surrey historical profile classifies Florence Baulk's "
                "2021 senior result under Javelin 600g."
            ),
        },

        {
            "aaAthleteId": 14800681,
            "result_year": 2022,
            "sex": "W",
            "implement_weight_g": 600,
            "model_group": "W_600",
            "evidence_classification": "Senior/U23 Women Javelin 600g",
            "evidence_note": (
                "Surrey historical profile classifies Florence Baulk's "
                "2022 senior result under Javelin 600g; Southern Athletics "
                "League also identifies her as U23."
            ),
        },
    ]
)


# ==================================================================================================
# PART C — VALIDATE OVERRIDE TABLE
# ==================================================================================================

allowed_groups = {
    "M_700",
    "M_800",
    "W_500",
    "W_600",
}


if HISTORICAL_IMPLEMENT_OVERRIDES[
    [
        "aaAthleteId",
        "result_year",
    ]
].duplicated().any():

    raise RuntimeError(
        "Historical implement override table contains duplicate athlete/year keys."
    )


if not HISTORICAL_IMPLEMENT_OVERRIDES[
    "model_group"
].isin(
    allowed_groups
).all():

    raise RuntimeError(
        "Historical implement override table contains unsupported model groups."
    )


# ==================================================================================================
# PART D — APPLY ONLY TO CURRENTLY UNRESOLVED ROWS
# ==================================================================================================

STAGING_V3 = (
    STAGING_V2_DEDUPED.copy()
)


STAGING_V3[
    "result_year"
] = pd.to_datetime(
    STAGING_V3[
        "result_date_dt"
    ],
    errors="coerce",
).dt.year


# ----------------------------------------------------------------------------------------------
# Create lookup
# ----------------------------------------------------------------------------------------------

override_lookup = {

    (
        int(row["aaAthleteId"]),
        int(row["result_year"]),
    ):
        {
            "sex":
                row["sex"],

            "implement_weight_g":
                int(
                    row[
                        "implement_weight_g"
                    ]
                ),

            "model_group":
                row[
                    "model_group"
                ],

            "evidence_classification":
                row[
                    "evidence_classification"
                ],

            "evidence_note":
                row[
                    "evidence_note"
                ],
        }

    for _, row
    in HISTORICAL_IMPLEMENT_OVERRIDES.iterrows()
}


# ----------------------------------------------------------------------------------------------
# Audit columns
# ----------------------------------------------------------------------------------------------

STAGING_V3[
    "historical_override_applied"
] = False


STAGING_V3[
    "historical_override_evidence"
] = None


STAGING_V3[
    "historical_override_note"
] = None


override_rows_applied = []


for index, row in STAGING_V3.iterrows():

    # Only touch unresolved rows.
    if row.get(
        "group_resolution_source"
    ) != "UNRESOLVED":

        continue


    athlete_id = normalize_wa_id(
        row.get(
            "aaAthleteId"
        )
    )


    result_year = row.get(
        "result_year"
    )


    if (
        athlete_id is None
        or
        pd.isna(
            result_year
        )
    ):

        continue


    key = (
        int(
            athlete_id
        ),
        int(
            result_year
        ),
    )


    override = override_lookup.get(
        key
    )


    if override is None:

        continue


    # ------------------------------------------------------------------------------------------
    # Sex consistency gate
    # ------------------------------------------------------------------------------------------

    existing_sex = row.get(
        "sex"
    )


    if (
        existing_sex is not None
        and
        not pd.isna(
            existing_sex
        )
        and
        str(
            existing_sex
        )
        != str(
            override[
                "sex"
            ]
        )
    ):

        raise RuntimeError(
            "Historical implement override sex conflict:\n"
            f"athlete={athlete_id}, year={result_year}, "
            f"row sex={existing_sex}, override sex={override['sex']}"
        )


    # ------------------------------------------------------------------------------------------
    # Apply audited implementation
    # ------------------------------------------------------------------------------------------

    STAGING_V3.at[
        index,
        "implement_weight_g"
    ] = override[
        "implement_weight_g"
    ]


    STAGING_V3.at[
        index,
        "model_group"
    ] = override[
        "model_group"
    ]


    STAGING_V3.at[
        index,
        "group_resolution_source"
    ] = "AUDITED_HISTORICAL_OVERRIDE"


    STAGING_V3.at[
        index,
        "historical_override_applied"
    ] = True


    STAGING_V3.at[
        index,
        "historical_override_evidence"
    ] = override[
        "evidence_classification"
    ]


    STAGING_V3.at[
        index,
        "historical_override_note"
    ] = override[
        "evidence_note"
    ]


    override_rows_applied.append(
        {
            "aaAthleteId":
                int(
                    athlete_id
                ),

            "result_date_dt":
                row.get(
                    "result_date_dt"
                ),

            "mark":
                row.get(
                    "mark"
                ),

            "result_year":
                int(
                    result_year
                ),

            "model_group":
                override[
                    "model_group"
                ],

            "implement_weight_g":
                override[
                    "implement_weight_g"
                ],

            "evidence_classification":
                override[
                    "evidence_classification"
                ],
        }
    )


HISTORICAL_OVERRIDE_ROWS_APPLIED = pd.DataFrame(
    override_rows_applied
)


# ==================================================================================================
# PART E — REBUILD ELIGIBILITY
# ==================================================================================================

STAGING_V3[
    "has_supported_model_group"
] = (

    STAGING_V3[
        "model_group"
    ]
    .isin(
        [
            "M_700",
            "M_800",
            "W_500",
            "W_600",
        ]
    )
)


STAGING_V3[
    "canonical_candidate"
] = (

    STAGING_V3[
        "has_valid_date"
    ]

    &
    STAGING_V3[
        "has_numeric_mark"
    ]

    &
    STAGING_V3[
        "has_resolved_sex"
    ]

    &
    STAGING_V3[
        "has_supported_model_group"
    ]
)


REFRESHED_CANONICAL_CANDIDATES_V3 = (

    STAGING_V3.loc[
        STAGING_V3[
            "canonical_candidate"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


REFRESHED_EXCLUDED_ROWS_V3 = (

    STAGING_V3.loc[
        ~STAGING_V3[
            "canonical_candidate"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART F — FINAL UNRESOLVED NUMERIC IMPLEMENT CHECK
# ==================================================================================================

FINAL_UNRESOLVED_NUMERIC_GROUP_ROWS = (

    STAGING_V3.loc[
        (
            STAGING_V3[
                "group_resolution_source"
            ]
            == "UNRESOLVED"
        )
        &
        (
            STAGING_V3[
                "has_numeric_mark"
            ]
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


N_FINAL_UNRESOLVED_NUMERIC = int(
    len(
        FINAL_UNRESOLVED_NUMERIC_GROUP_ROWS
    )
)


# ==================================================================================================
# PART G — FINAL UNSUPPORTED IMPLEMENT TABLE
# ==================================================================================================

FINAL_UNSUPPORTED_IMPLEMENT_ROWS = (

    STAGING_V3.loc[
        STAGING_V3[
            "group_resolution_source"
        ]
        == "EXPLICIT_UNSUPPORTED_WEIGHT"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART H — FINAL CROSS-SOURCE QA
# ==================================================================================================

ATHLETE_HISTORY_QA_V3 = (

    STAGING_V3.loc[
        (
            STAGING_V3[
                "has_valid_date"
            ]
        )
        &
        (
            STAGING_V3[
                "has_numeric_mark"
            ]
        )
        &
        (
            STAGING_V3[
                "result_date_dt"
            ]
            >= INCREMENTAL_START
        )
        &
        (
            STAGING_V3[
                "result_date_dt"
            ]
            <= INCREMENTAL_END
        )
    ]
    .copy()
)


ATHLETE_HISTORY_QA_V3[
    "_fingerprint"
] = [

    build_result_fingerprint(

        athlete_id=
            row.get(
                "aaAthleteId"
            ),

        result_date=
            row.get(
                "result_date_dt"
            ),

        competition_id=
            row.get(
                "competitionId"
            ),

        event_id=
            row.get(
                "eventId"
            ),

        mark=
            row.get(
                "mark_numeric_m"
            ),
    )

    for _, row
    in ATHLETE_HISTORY_QA_V3.iterrows()
]


ATHLETE_HISTORY_FP_V3 = set(

    ATHLETE_HISTORY_QA_V3[
        "_fingerprint"
    ]
    .dropna()
    .unique()
)


DISCOVERY_FP_V3 = set(

    DISCOVERY_QA[
        "_fingerprint"
    ]
    .dropna()
    .unique()
)


MATCHED_FP_V3 = (

    DISCOVERY_FP_V3
    &
    ATHLETE_HISTORY_FP_V3
)


MISSING_FP_V3 = (

    DISCOVERY_FP_V3
    -
    ATHLETE_HISTORY_FP_V3
)


CROSS_SOURCE_COVERAGE_V3 = (

    len(
        MATCHED_FP_V3
    )
    /
    len(
        DISCOVERY_FP_V3
    )

    if len(
        DISCOVERY_FP_V3
    ) > 0

    else np.nan
)


# ==================================================================================================
# PART I — FINAL SOURCE DUPLICATE / CONFLICT CHECK
# ==================================================================================================

SOURCE_IDENTITY_QA_V3 = (

    STAGING_V3.loc[
        STAGING_V3[
            "has_valid_date"
        ]
        &
        STAGING_V3[
            "has_numeric_mark"
        ]
    ]
    .groupby(
        [
            "aaAthleteId",
            "result_date_dt",
            "_competition_id_key",
            "_event_id_key",
            "_race_key",
        ],
        dropna=False,
    )
    .agg(
        n_rows=(
            "mark_numeric_m",
            "size",
        ),

        n_marks=(
            "mark_numeric_m",
            "nunique",
        ),
    )
    .reset_index()
)


FINAL_CONFLICTING_SOURCE_IDENTITIES = (

    SOURCE_IDENTITY_QA_V3.loc[
        SOURCE_IDENTITY_QA_V3[
            "n_marks"
        ]
        > 1
    ]
    .copy()
)


N_FINAL_CONFLICTS = int(
    len(
        FINAL_CONFLICTING_SOURCE_IDENTITIES
    )
)


# ==================================================================================================
# PART J — FINAL RELEASE GATE
# ==================================================================================================

CELL_11B_PASS = bool(

    N_FINAL_UNRESOLVED_NUMERIC == 0

    and
    N_FINAL_CONFLICTS == 0

    and
    len(
        MISSING_FP_V3
    )
    == 0

    and
    len(
        REFRESHED_CANONICAL_CANDIDATES_V3
    )
    > 0
)


# ==================================================================================================
# PART K — SAVE FINAL NORMALIZED STAGING
# ==================================================================================================

STAGING_V3_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_normalized_staging_final.parquet"
)


CANONICAL_CANDIDATES_V3_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_canonical_candidates_final.parquet"
)


EXCLUDED_V3_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_excluded_rows_final.parquet"
)


HISTORICAL_OVERRIDE_PATH = (
    RUN_CHECKPOINT_DIR
    / "historical_implement_overrides.parquet"
)


HISTORICAL_OVERRIDE_APPLIED_PATH = (
    RUN_CHECKPOINT_DIR
    / "historical_implement_overrides_applied.parquet"
)


FINAL_UNRESOLVED_GROUP_PATH = (
    RUN_CHECKPOINT_DIR
    / "refreshed_javelin_unresolved_groups_final.parquet"
)


STAGING_V3.to_parquet(
    STAGING_V3_PATH,
    index=False,
)


REFRESHED_CANONICAL_CANDIDATES_V3.to_parquet(
    CANONICAL_CANDIDATES_V3_PATH,
    index=False,
)


REFRESHED_EXCLUDED_ROWS_V3.to_parquet(
    EXCLUDED_V3_PATH,
    index=False,
)


HISTORICAL_IMPLEMENT_OVERRIDES.to_parquet(
    HISTORICAL_OVERRIDE_PATH,
    index=False,
)


HISTORICAL_OVERRIDE_ROWS_APPLIED.to_parquet(
    HISTORICAL_OVERRIDE_APPLIED_PATH,
    index=False,
)


FINAL_UNRESOLVED_NUMERIC_GROUP_ROWS.to_parquet(
    FINAL_UNRESOLVED_GROUP_PATH,
    index=False,
)


# ==================================================================================================
# PART L — PROMOTE FINAL VARIABLES FOR CELL 12
# ==================================================================================================

REFRESHED_STAGING_FINAL = (
    STAGING_V3
)


REFRESHED_CANONICAL_CANDIDATES_FINAL = (
    REFRESHED_CANONICAL_CANDIDATES_V3
)


REFRESHED_EXCLUDED_ROWS_FINAL = (
    REFRESHED_EXCLUDED_ROWS_V3
)


NORMALIZATION_FINAL_PASS = (
    CELL_11B_PASS
)


# ==================================================================================================
# PART M — UPDATE MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "historical_implement_adjudication"
] = {

    "status":
        (
            "VALIDATED"
            if CELL_11B_PASS
            else "REVIEW_REQUIRED"
        ),

    "override_definitions":
        int(
            len(
                HISTORICAL_IMPLEMENT_OVERRIDES
            )
        ),

    "rows_overridden":
        int(
            len(
                HISTORICAL_OVERRIDE_ROWS_APPLIED
            )
        ),

    "remaining_unresolved_numeric_groups":
        N_FINAL_UNRESOLVED_NUMERIC,

    "unsupported_implement_rows":
        int(
            len(
                FINAL_UNSUPPORTED_IMPLEMENT_ROWS
            )
        ),

    "cross_source_missing":
        int(
            len(
                MISSING_FP_V3
            )
        ),

    "cross_source_coverage":
        (
            float(
                CROSS_SOURCE_COVERAGE_V3
            )
            if np.isfinite(
                CROSS_SOURCE_COVERAGE_V3
            )
            else None
        ),

    "conflicting_source_identities":
        N_FINAL_CONFLICTS,

    "staging_path":
        str(
            STAGING_V3_PATH
        ),

    "canonical_candidate_path":
        str(
            CANONICAL_CANDIDATES_V3_PATH
        ),

    "override_definition_path":
        str(
            HISTORICAL_OVERRIDE_PATH
        ),

    "override_applied_path":
        str(
            HISTORICAL_OVERRIDE_APPLIED_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "REFRESHED_HISTORY_NORMALIZED"

    if CELL_11B_PASS

    else "REFRESHED_HISTORY_NORMALIZATION_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART N — OUTPUT
# ==================================================================================================

print()
print("=" * 130)
print("WORLD ATHLETICS FINAL HISTORICAL IMPLEMENT ADJUDICATION")
print("=" * 130)


print()
print("AUDITED OVERRIDES")
print("-" * 130)

display(
    HISTORICAL_IMPLEMENT_OVERRIDES
)


print()
print("ROWS RESOLVED BY OVERRIDE")
print("-" * 130)

display(
    HISTORICAL_OVERRIDE_ROWS_APPLIED
)


print()
print("FINAL IMPLEMENT RESOLUTION")
print("-" * 130)

display(
    STAGING_V3.groupby(
        [
            "group_resolution_source",
            "model_group",
            "implement_weight_g",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        rows=(
            "aaAthleteId",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),
    )
    .sort_values(
        "rows",
        ascending=False,
    )
)


print()
print("FINAL QA")
print("-" * 130)

print(
    f"Canonical candidate rows        : "
    f"{len(REFRESHED_CANONICAL_CANDIDATES_V3):,}"
)

print(
    f"Unsupported implement rows      : "
    f"{len(FINAL_UNSUPPORTED_IMPLEMENT_ROWS):,}"
)

print(
    f"Unresolved numeric group rows   : "
    f"{N_FINAL_UNRESOLVED_NUMERIC:,}"
)

print(
    f"Conflicting source identities   : "
    f"{N_FINAL_CONFLICTS:,}"
)

print(
    f"Competition fingerprints        : "
    f"{len(DISCOVERY_FP_V3):,}"
)

print(
    f"Matched fingerprints            : "
    f"{len(MATCHED_FP_V3):,}"
)

print(
    f"Missing fingerprints            : "
    f"{len(MISSING_FP_V3):,}"
)

print(
    f"Cross-source coverage           : "
    f"{CROSS_SOURCE_COVERAGE_V3:.4%}"
    if np.isfinite(
        CROSS_SOURCE_COVERAGE_V3
    )
    else "Cross-source coverage           : N/A"
)


print()
print("FINAL EXCLUSION SUMMARY")
print("-" * 130)

if len(
    REFRESHED_EXCLUDED_ROWS_V3
) > 0:

    display(
        REFRESHED_EXCLUDED_ROWS_V3.groupby(
            [
                "has_numeric_mark",
                "group_resolution_source",
            ],
            dropna=False,
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size":
                    "rows"
            }
        )
        .sort_values(
            "rows",
            ascending=False,
        )
    )


else:

    print(
        "None."
    )


print()
print("CHECKPOINTS")
print("-" * 130)

print(
    f"Final normalized staging       : {STAGING_V3_PATH}"
)

print(
    f"Final canonical candidates     : {CANONICAL_CANDIDATES_V3_PATH}"
)

print(
    f"Historical override table      : {HISTORICAL_OVERRIDE_PATH}"
)

print(
    f"Applied override audit         : {HISTORICAL_OVERRIDE_APPLIED_PATH}"
)


print()
print("=" * 130)


if CELL_11B_PASS:

    print(
        "CELL 11B STATUS: FINAL NORMALIZATION QA PASSED"
    )

    print()
    print(
        "All numeric javelin performances now have a resolved or explicitly "
        "unsupported implement classification."
    )

    print(
        "Cell 12 may build the reconciled candidate canonical dataset."
    )


else:

    print(
        "CELL 11B STATUS: REVIEW REQUIRED"
    )

    print()
    print(
        "Do not reconcile the production canonical dataset yet."
    )


print("=" * 130)


WORLD ATHLETICS FINAL HISTORICAL IMPLEMENT ADJUDICATION

AUDITED OVERRIDES
----------------------------------------------------------------------------------------------------------------------------------


,aaAthleteId,result_year,sex,implement_weight_g,model_group,evidence_classification,evidence_note
0,14800681,2019,W,600,W_600,U20 Women Javelin 600g,Florence Baulk 37.05 at 2019 Southern U20 Cham...
1,14800681,2021,W,600,W_600,Senior/U23 Women Javelin 600g,Surrey historical profile classifies Florence ...
2,14800681,2022,W,600,W_600,Senior/U23 Women Javelin 600g,Surrey historical profile classifies Florence ...



ROWS RESOLVED BY OVERRIDE
----------------------------------------------------------------------------------------------------------------------------------


,aaAthleteId,result_date_dt,mark,result_year,model_group,implement_weight_g,evidence_classification
0,14800681,2019-06-02,37.05,2019,W_600,600,U20 Women Javelin 600g
1,14800681,2021-06-09,37.29,2021,W_600,600,Senior/U23 Women Javelin 600g
2,14800681,2021-06-20,37.56,2021,W_600,600,Senior/U23 Women Javelin 600g
3,14800681,2022-06-12,37.12,2022,W_600,600,Senior/U23 Women Javelin 600g
4,14800681,2022-07-16,36.46,2022,W_600,600,Senior/U23 Women Javelin 600g
5,14800681,2022-08-13,33.42,2022,W_600,600,Senior/U23 Women Javelin 600g
6,14800681,2022-09-10,34.68,2022,W_600,600,Senior/U23 Women Javelin 600g



FINAL IMPLEMENT RESOLUTION
----------------------------------------------------------------------------------------------------------------------------------


,group_resolution_source,model_group,implement_weight_g,rows,athletes
3,CANONICAL_EVENT_ID_MAP,M_800,800.0,1731,351
4,CANONICAL_EVENT_ID_MAP,W_600,600.0,1701,321
8,EXPLICIT_WEIGHT,W_500,500.0,421,135
7,EXPLICIT_WEIGHT,M_700,700.0,259,103
0,ADULT_GENERIC_AGE_FALLBACK,M_800,800.0,62,3
2,AUDITED_HISTORICAL_OVERRIDE,W_600,600.0,7,1
1,ADULT_GENERIC_AGE_FALLBACK,W_600,600.0,2,2
5,EXPLICIT_UNSUPPORTED_WEIGHT,NaN,400.0,2,1
6,EXPLICIT_UNSUPPORTED_WEIGHT,NaN,600.0,2,2



FINAL QA
----------------------------------------------------------------------------------------------------------------------------------
Canonical candidate rows        : 4,153
Unsupported implement rows      : 4
Unresolved numeric group rows   : 0
Conflicting source identities   : 0
Competition fingerprints        : 976
Matched fingerprints            : 976
Missing fingerprints            : 0
Cross-source coverage           : 100.0000%

FINAL EXCLUSION SUMMARY
----------------------------------------------------------------------------------------------------------------------------------


,has_numeric_mark,group_resolution_source,rows
1,False,CANONICAL_EVENT_ID_MAP,20
0,False,ADULT_GENERIC_AGE_FALLBACK,6
2,False,EXPLICIT_WEIGHT,4
3,True,EXPLICIT_UNSUPPORTED_WEIGHT,4



CHECKPOINTS
----------------------------------------------------------------------------------------------------------------------------------
Final normalized staging       : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_normalized_staging_final.parquet
Final canonical candidates     : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/refreshed_javelin_canonical_candidates_final.parquet
Historical override table      : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/historical_implement_overrides.parquet
Applied override audit         : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/historical_implement_overrides_applied.parquet

CELL 11B STATUS: FINAL NORMALIZATION QA PASSED

All numeric javelin performances now have a r

In [35]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 12 — PRODUCTION CANONICAL RECONCILIATION
#            NORMALIZATION + AGE POLICY + GOLDEN-REFERENCE QA
# ==================================================================================================
#
# THIS CELL REPLACES THE EXPLORATORY:
#
#   12
#   12A
#   12B
#   12C
#   12D
#   12E
#   12F
#   13A
#   13B
#   13C
#   13D
#
# Cell 13 remains separate as the independent final release regression.
#
#
# INPUT
# -----
#
# Successful Cell 11B state:
#
#   REFRESHED_STAGING_FINAL
#   NORMALIZATION_FINAL_PASS
#
# plus the earlier refresh/download notebook state.
#
#
# OUTPUT
# ------
#
#   CANONICAL_REFRESHED_NORMALIZED_FINAL
#   CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH
#   CANONICAL_REFRESHED_NORMALIZED_FINAL_PASS
#
#
# IMPORTANT POLICIES
# ------------------
#
# 1. Production canonical parquet is NEVER overwritten here.
#
# 2. Successfully refreshed athlete-year slices replace the corresponding
#    production athlete-year slices.
#
# 3. Nonnumeric javelin results remain auditable staging rows but are excluded
#    from the canonical numeric-performance dataset.
#
# 4. Physical performance identity is:
#
#       athlete + date + mark + event_id
#
#    Classification metadata is NOT part of physical identity.
#
# 5. Existing physical observations preserve production age_years EXACTLY.
#
# 6. New observations:
#
#       exact DOB:
#           age_years = (result_date - DOB).days / 365.2425
#
#       YEAR-only:
#           age_years = NaN
#
#       NONE / UNKNOWN:
#           age_years = NaN
#
# 7. Current-spec vocabulary is canonicalized.
#
# 8. Valid non-core groups such as M_600, M_500 and W_400 remain in canonical
#    history but are not V4 core-group eligible.
#
# 9. Context-sensitive WA event IDs are never globally mapped merely from
#    their raw discipline label.
#
# 10. The known validated run 20260919_225312 is checked against its golden
#     reference before Cell 12 can pass.
#
# ==================================================================================================

from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "CONFIG",
    "RUN_ID",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",
    "REFRESHED_STAGING_FINAL",
    "NORMALIZATION_FINAL_PASS",
    "normalize_wa_id",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 through Cell 11B first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not NORMALIZATION_FINAL_PASS:

    raise RuntimeError(
        "Cell 11B normalization did not pass. "
        "Canonical reconciliation is blocked."
    )


# ==================================================================================================
# PART B — CANONICAL SCHEMA / CONSTANTS
# ==================================================================================================

EXPECTED_CANONICAL_COLUMNS = [
    "aaAthleteId",
    "result_date_dt",
    "mark_numeric_m",
    "gender_raw",
    "implement_weight_g",
    "canonical_birth_date",
    "canonical_birth_year",
    "birth_precision",
    "age_years",
    "spec_regime",
    "record_source",
    "reconciliation_status",
    "event_id",
    "model_group",
    "spec_model_eligible",
    "core_group_eligible",
    "age_available",
    "v4_base_eligible",
]


CORE_GROUPS = {
    "M_700",
    "M_800",
    "W_500",
    "W_600",
}


NONCURRENT_SPEC_LABELS = {
    "OLD_SPEC",
    "UNKNOWN_SPEC",
    "M800_1986_1991_SPEC_AMBIGUOUS",
}


# ==================================================================================================
# PART C — GENERIC HELPERS
# ==================================================================================================

def first_existing_column(df, candidates):

    for column in candidates:

        if column in df.columns:

            return column

    return None


def normalize_gender(value):

    if value is None:

        return None

    try:

        if pd.isna(value):

            return None

    except Exception:

        pass


    value = str(value).strip().upper()


    if value in {
        "M",
        "MALE",
        "MEN",
        "MAN",
    }:

        return "M"


    if value in {
        "W",
        "F",
        "FEMALE",
        "WOMEN",
        "WOMAN",
    }:

        return "W"


    return None


def parse_weight_grams(value):

    if value is None:

        return np.nan


    try:

        if pd.isna(value):

            return np.nan

    except Exception:

        pass


    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        value = float(value)

        if np.isfinite(value):

            return value

        return np.nan


    text = (
        str(value)
        .strip()
        .lower()
        .replace(",", ".")
    )


    match = re.search(
        r"(\d+(?:\.\d+)?)",
        text,
    )


    if match is None:

        return np.nan


    number = float(
        match.group(1)
    )


    # Defensive conversion for values such as 0.8kg.
    if (
        "kg" in text
        and number < 10
    ):

        number *= 1000.0


    return number


def derive_model_group(
    gender,
    weight,
):

    gender = normalize_gender(
        gender
    )


    weight = parse_weight_grams(
        weight
    )


    if (
        gender not in {"M", "W"}
        or pd.isna(weight)
    ):

        return None


    return (
        f"{gender}_"
        f"{int(round(float(weight)))}"
    )


def canonical_current_spec_label(
    model_group,
):

    if model_group is None:

        return None


    match = re.fullmatch(
        r"([MW])_(\d+)",
        str(model_group).strip(),
    )


    if match is None:

        return None


    return (
        f"{match.group(1)}_JAVELIN_"
        f"{match.group(2)}_CURRENT_SPEC"
    )


def bool_value(value):

    if value is None:

        return False


    try:

        if pd.isna(value):

            return False

    except Exception:

        pass


    return bool(value)


# ==================================================================================================
# PART D — PHYSICAL PERFORMANCE KEYS
# ==================================================================================================

def build_physical_key(row):

    athlete_id = normalize_wa_id(
        row.get("aaAthleteId")
    )


    result_date = pd.to_datetime(
        row.get("result_date_dt"),
        errors="coerce",
    )


    event_id = normalize_wa_id(
        row.get("event_id")
    )


    try:

        mark = float(
            row.get("mark_numeric_m")
        )

    except Exception:

        return None


    if (
        athlete_id is None
        or pd.isna(result_date)
        or not np.isfinite(mark)
    ):

        return None


    return "|".join(
        [
            str(int(athlete_id)),
            result_date.strftime("%Y-%m-%d"),
            f"{mark:.3f}",
            (
                str(int(event_id))
                if event_id is not None
                else ""
            ),
        ]
    )


def add_occurrence_keys(df):

    out = df.copy()


    out[
        "_physical_key"
    ] = out.apply(
        build_physical_key,
        axis=1,
    )


    if out[
        "_physical_key"
    ].isna().any():

        bad = (
            out.loc[
                out["_physical_key"].isna()
            ]
            .head(100)
        )

        display(bad)

        raise RuntimeError(
            "Dataset contains rows that cannot form "
            "a physical-performance key."
        )


    out[
        "_physical_occurrence"
    ] = (
        out.groupby(
            "_physical_key",
            dropna=False,
        )
        .cumcount()
    )


    out[
        "_match_key"
    ] = (
        out["_physical_key"]
        + "#"
        + out["_physical_occurrence"].astype(str)
    )


    return out


# ==================================================================================================
# PART E — LOAD FRESH PRODUCTION BASE
# ==================================================================================================

PRODUCTION_BASE = pd.read_parquet(
    CONFIG.canonical_dataset_path
)


missing_production_columns = [
    column
    for column in EXPECTED_CANONICAL_COLUMNS
    if column not in PRODUCTION_BASE.columns
]


if missing_production_columns:

    raise RuntimeError(
        "Production canonical schema is missing:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_production_columns
        )
    )


PRODUCTION_BASE = (
    PRODUCTION_BASE[
        EXPECTED_CANONICAL_COLUMNS
    ]
    .copy()
)


PRODUCTION_BASE[
    "result_date_dt"
] = pd.to_datetime(
    PRODUCTION_BASE["result_date_dt"],
    errors="coerce",
)


PRODUCTION_BASE[
    "canonical_birth_date"
] = pd.to_datetime(
    PRODUCTION_BASE["canonical_birth_date"],
    errors="coerce",
)


PRODUCTION_BASE[
    "canonical_birth_year"
] = pd.to_numeric(
    PRODUCTION_BASE["canonical_birth_year"],
    errors="coerce",
)


PRODUCTION_BASE[
    "age_years"
] = pd.to_numeric(
    PRODUCTION_BASE["age_years"],
    errors="coerce",
)


PRODUCTION_BASE[
    "implement_weight_g"
] = pd.to_numeric(
    PRODUCTION_BASE["implement_weight_g"],
    errors="coerce",
)


N_PRODUCTION_ROWS = int(
    len(PRODUCTION_BASE)
)


N_PRODUCTION_ATHLETES = int(
    PRODUCTION_BASE[
        "aaAthleteId"
    ].nunique()
)


PRODUCTION_ATHLETE_IDS = set(
    PRODUCTION_BASE[
        "aaAthleteId"
    ]
    .dropna()
    .map(normalize_wa_id)
)


# ==================================================================================================
# PART F — CONTEXT-SENSITIVE EVENT OVERRIDE REGISTRY
# ==================================================================================================
#
# WA event 10229685 is returned as "Javelin Throw (old)" by WA.
#
# Source verification established that these exact 2025 contexts correspond
# to men's current-spec 800g javelin.
#
# IMPORTANT:
#
# event 10229685 is NOT globally declared to be M800.
#
# ==================================================================================================

EVENT_CONTEXT_OVERRIDE_REGISTRY = pd.DataFrame(
    [
        {
            "event_id": 10229685,
            "competition_id": 7230449,
            "result_date": pd.Timestamp("2025-08-18"),
            "implement_weight_g": 800.0,
            "model_group": "M_800",
            "spec_regime": "M_JAVELIN_800_CURRENT_SPEC",
            "spec_model_eligible": True,
            "evidence": "SOURCE_VERIFIED_EVENT_CONTEXT",
        },

        {
            "event_id": 10229685,
            "competition_id": 7221680,
            "result_date": pd.Timestamp("2025-09-28"),
            "implement_weight_g": 800.0,
            "model_group": "M_800",
            "spec_regime": "M_JAVELIN_800_CURRENT_SPEC",
            "spec_model_eligible": True,
            "evidence": "SOURCE_VERIFIED_EVENT_CONTEXT",
        },

        {
            "event_id": 10229685,
            "competition_id": 7232497,
            "result_date": pd.Timestamp("2025-10-11"),
            "implement_weight_g": 800.0,
            "model_group": "M_800",
            "spec_regime": "M_JAVELIN_800_CURRENT_SPEC",
            "spec_model_eligible": True,
            "evidence": "SOURCE_VERIFIED_EVENT_CONTEXT",
        },
    ]
)


CONTEXT_SENSITIVE_EVENT_IDS = set(
    EVENT_CONTEXT_OVERRIDE_REGISTRY[
        "event_id"
    ].astype(int)
)


# ==================================================================================================
# PART G — BUILD SAFE HISTORICAL EVENT-ID SEMANTIC MAP
# ==================================================================================================

PRODUCTION_EVIDENCE = (
    PRODUCTION_BASE.copy()
)


PRODUCTION_EVIDENCE[
    "_derived_model_group"
] = [
    derive_model_group(
        gender,
        weight,
    )
    for gender, weight
    in zip(
        PRODUCTION_EVIDENCE["gender_raw"],
        PRODUCTION_EVIDENCE["implement_weight_g"],
    )
]


event_semantic_rows = []


for event_id, hist in PRODUCTION_EVIDENCE.groupby(
    "event_id",
    dropna=True,
):

    event_id_norm = normalize_wa_id(
        event_id
    )


    if event_id_norm is None:

        continue


    event_id_norm = int(
        event_id_norm
    )


    if event_id_norm in CONTEXT_SENSITIVE_EVENT_IDS:

        continue


    groups = sorted(
        hist[
            "_derived_model_group"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )


    eligibility_values = sorted(
        hist[
            "spec_model_eligible"
        ]
        .dropna()
        .astype(bool)
        .unique()
        .tolist()
    )


    if (
        len(groups) != 1
        or eligibility_values != [True]
    ):

        continue


    group = groups[0]


    weight = int(
        group.split("_")[1]
    )


    event_semantic_rows.append(
        {
            "event_id": event_id_norm,
            "model_group": group,
            "implement_weight_g": float(weight),
            "spec_regime":
                canonical_current_spec_label(group),
            "spec_model_eligible": True,
            "historical_rows": int(len(hist)),
        }
    )


EVENT_SEMANTIC_MAP = pd.DataFrame(
    event_semantic_rows
)


EVENT_SEMANTIC_LOOKUP = {
    int(row["event_id"]):
        row.to_dict()

    for _, row
    in EVENT_SEMANTIC_MAP.iterrows()
}


# ==================================================================================================
# PART H — EARLIEST ELIGIBLE DATE BY MODEL GROUP
# ==================================================================================================

eligible_history = (
    PRODUCTION_EVIDENCE.loc[
        PRODUCTION_EVIDENCE[
            "spec_model_eligible"
        ]
        .fillna(False)
        .astype(bool)

        &
        PRODUCTION_EVIDENCE[
            "_derived_model_group"
        ]
        .notna()

        &
        PRODUCTION_EVIDENCE[
            "result_date_dt"
        ]
        .notna()
    ]
    .copy()
)


EARLIEST_ELIGIBLE_DATE_BY_GROUP = (
    eligible_history.groupby(
        "_derived_model_group"
    )[
        "result_date_dt"
    ]
    .min()
    .to_dict()
)


# ==================================================================================================
# PART I — GLOBAL PRODUCTION VOCABULARY NORMALIZATION
# ==================================================================================================

PRODUCTION_NORMALIZED = (
    PRODUCTION_BASE.copy()
)


PRODUCTION_NORMALIZED[
    "_derived_group"
] = [
    derive_model_group(
        gender,
        weight,
    )
    for gender, weight
    in zip(
        PRODUCTION_NORMALIZED["gender_raw"],
        PRODUCTION_NORMALIZED["implement_weight_g"],
    )
]


production_fill_group_mask = (
    PRODUCTION_NORMALIZED[
        "model_group"
    ]
    .isna()

    &
    PRODUCTION_NORMALIZED[
        "_derived_group"
    ]
    .notna()
)


PRODUCTION_NORMALIZED.loc[
    production_fill_group_mask,
    "model_group",
] = (
    PRODUCTION_NORMALIZED.loc[
        production_fill_group_mask,
        "_derived_group",
    ]
)


production_eligible_group_mask = (
    PRODUCTION_NORMALIZED[
        "spec_model_eligible"
    ]
    .fillna(False)
    .astype(bool)

    &
    PRODUCTION_NORMALIZED[
        "model_group"
    ]
    .notna()
)


PRODUCTION_NORMALIZED.loc[
    production_eligible_group_mask,
    "spec_regime",
] = (
    PRODUCTION_NORMALIZED.loc[
        production_eligible_group_mask,
        "model_group",
    ]
    .map(
        canonical_current_spec_label
    )
)


# ==================================================================================================
# PART J — APPLY VERIFIED 10229685 PATCH TO PRODUCTION METADATA
# ==================================================================================================

for _, override in EVENT_CONTEXT_OVERRIDE_REGISTRY.iterrows():

    event_value = int(
        override["event_id"]
    )


    date_value = pd.Timestamp(
        override["result_date"]
    )


    event_series = (
        PRODUCTION_NORMALIZED[
            "event_id"
        ]
        .map(normalize_wa_id)
    )


    mask = (
        event_series.eq(event_value)

        &
        PRODUCTION_NORMALIZED[
            "result_date_dt"
        ]
        .eq(date_value)
    )


    PRODUCTION_NORMALIZED.loc[
        mask,
        "implement_weight_g",
    ] = float(
        override["implement_weight_g"]
    )


    PRODUCTION_NORMALIZED.loc[
        mask,
        "model_group",
    ] = override[
        "model_group"
    ]


    PRODUCTION_NORMALIZED.loc[
        mask,
        "spec_regime",
    ] = override[
        "spec_regime"
    ]


    PRODUCTION_NORMALIZED.loc[
        mask,
        "spec_model_eligible",
    ] = True


# ==================================================================================================
# PART K — REFRESHED STAGING SOURCE
# ==================================================================================================

SOURCE = (
    REFRESHED_STAGING_FINAL.copy()
)


N_STAGING_ROWS = int(
    len(SOURCE)
)


# ==================================================================================================
# PART L — SOURCE COLUMN RESOLUTION
# ==================================================================================================
#
# This directly incorporates the finding from exploratory Cell 12A.
#
# Actual current staging fields include:
#
#       DOB
#       birth_year
#       birth_date_precision
#       age_years
#
# We intentionally DO NOT copy source age_years into canonical.
#
# Canonical age is applied later according to the final age policy.
# ==================================================================================================

athlete_col = first_existing_column(
    SOURCE,
    [
        "aaAthleteId",
        "athlete_id",
        "athleteId",
    ],
)


date_col = first_existing_column(
    SOURCE,
    [
        "result_date_dt",
        "resultDate",
        "date",
        "date_dt",
    ],
)


mark_col = first_existing_column(
    SOURCE,
    [
        "mark_numeric_m",
        "mark_numeric",
        "result_numeric",
        "mark",
    ],
)


gender_col = first_existing_column(
    SOURCE,
    [
        "gender_raw",
        "gender",
        "sex",
    ],
)


weight_col = first_existing_column(
    SOURCE,
    [
        "implement_weight_g",
        "implement_g",
        "weight_g",
        "implement_weight",
    ],
)


event_col = first_existing_column(
    SOURCE,
    [
        "event_id",
        "eventId",
    ],
)


competition_col = first_existing_column(
    SOURCE,
    [
        "competitionId",
        "competition_id",
    ],
)


birth_date_col = first_existing_column(
    SOURCE,
    [
        "canonical_birth_date",
        "DOB",
        "birth_date",
        "dateOfBirth",
        "dob",
    ],
)


birth_year_col = first_existing_column(
    SOURCE,
    [
        "canonical_birth_year",
        "birth_year",
        "birthYear",
    ],
)


birth_precision_col = first_existing_column(
    SOURCE,
    [
        "birth_precision",
        "birth_date_precision",
        "birthPrecision",
    ],
)


source_age_col = first_existing_column(
    SOURCE,
    [
        "age_years",
        "ageYears",
    ],
)


print()
print("SOURCE COLUMN RESOLUTION")
print("-" * 145)

print(
    f"Athlete ID source column         : "
    f"{athlete_col}"
)

print(
    f"Result-date source column        : "
    f"{date_col}"
)

print(
    f"Numeric-mark source column       : "
    f"{mark_col}"
)

print(
    f"Gender source column             : "
    f"{gender_col}"
)

print(
    f"Implement source column          : "
    f"{weight_col}"
)

print(
    f"Event-ID source column           : "
    f"{event_col}"
)

print(
    f"Competition-ID source column     : "
    f"{competition_col}"
)

print(
    f"Birth-date source column         : "
    f"{birth_date_col}"
)

print(
    f"Birth-year source column         : "
    f"{birth_year_col}"
)

print(
    f"Birth-precision source column    : "
    f"{birth_precision_col}"
)

print(
    f"Upstream age source column       : "
    f"{source_age_col}  (diagnostic only; not copied)"
)


required_source_columns = {
    "athlete": athlete_col,
    "date": date_col,
    "mark": mark_col,
    "gender": gender_col,
    "weight": weight_col,
}


missing_source_fields = [
    name
    for name, column
    in required_source_columns.items()
    if column is None
]


if missing_source_fields:

    raise RuntimeError(
        "REFRESHED_STAGING_FINAL is missing required semantic fields:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_source_fields
        )
    )


# ==================================================================================================
# PART M — BUILD CANONICAL REPLACEMENT FRAME
# ==================================================================================================

REPLACEMENT = pd.DataFrame(
    index=SOURCE.index
)


REPLACEMENT[
    "aaAthleteId"
] = (
    SOURCE[
        athlete_col
    ]
    .map(
        normalize_wa_id
    )
)


REPLACEMENT[
    "result_date_dt"
] = pd.to_datetime(
    SOURCE[
        date_col
    ],
    errors="coerce",
)


REPLACEMENT[
    "mark_numeric_m"
] = pd.to_numeric(
    SOURCE[
        mark_col
    ],
    errors="coerce",
)


REPLACEMENT[
    "gender_raw"
] = (
    SOURCE[
        gender_col
    ]
    .map(
        normalize_gender
    )
)


REPLACEMENT[
    "implement_weight_g"
] = (
    SOURCE[
        weight_col
    ]
    .map(
        parse_weight_grams
    )
)


if event_col is not None:

    REPLACEMENT[
        "event_id"
    ] = (
        SOURCE[
            event_col
        ]
        .map(
            normalize_wa_id
        )
    )

else:

    REPLACEMENT[
        "event_id"
    ] = np.nan


if competition_col is not None:

    REPLACEMENT[
        "_competition_id"
    ] = (
        SOURCE[
            competition_col
        ]
        .map(
            normalize_wa_id
        )
    )

else:

    REPLACEMENT[
        "_competition_id"
    ] = np.nan


# ==================================================================================================
# PART N — DEMOGRAPHIC / BIRTH METADATA
# ==================================================================================================

if birth_date_col is not None:

    REPLACEMENT[
        "canonical_birth_date"
    ] = pd.to_datetime(
        SOURCE[
            birth_date_col
        ],
        errors="coerce",
    )

else:

    REPLACEMENT[
        "canonical_birth_date"
    ] = pd.NaT


if birth_year_col is not None:

    REPLACEMENT[
        "canonical_birth_year"
    ] = pd.to_numeric(
        SOURCE[
            birth_year_col
        ],
        errors="coerce",
    )

else:

    REPLACEMENT[
        "canonical_birth_year"
    ] = np.nan


if birth_precision_col is not None:

    REPLACEMENT[
        "birth_precision"
    ] = (
        SOURCE[
            birth_precision_col
        ]
        .copy()
    )

else:

    REPLACEMENT[
        "birth_precision"
    ] = None


# Exact DOB determines canonical birth year.
exact_dob_mask = (
    REPLACEMENT[
        "canonical_birth_date"
    ]
    .notna()
)


REPLACEMENT.loc[
    exact_dob_mask,
    "canonical_birth_year",
] = (
    REPLACEMENT.loc[
        exact_dob_mask,
        "canonical_birth_date",
    ]
    .dt.year
)


# Normalize birth precision.
REPLACEMENT.loc[
    exact_dob_mask,
    "birth_precision",
] = "EXACT"


year_only_birth_mask = (
    REPLACEMENT[
        "canonical_birth_date"
    ]
    .isna()

    &
    REPLACEMENT[
        "canonical_birth_year"
    ]
    .notna()
)


REPLACEMENT.loc[
    year_only_birth_mask,
    "birth_precision",
] = "YEAR"


# Preserve NONE / UNKNOWN where supplied.
REPLACEMENT[
    "birth_precision"
] = (
    REPLACEMENT[
        "birth_precision"
    ]
    .where(
        REPLACEMENT[
            "birth_precision"
        ]
        .notna(),
        "UNKNOWN",
    )
)


# ==================================================================================================
# PART O — INITIAL MODEL GROUP
# ==================================================================================================

REPLACEMENT[
    "model_group"
] = [
    derive_model_group(
        gender,
        weight,
    )
    for gender, weight
    in zip(
        REPLACEMENT[
            "gender_raw"
        ],
        REPLACEMENT[
            "implement_weight_g"
        ],
    )
]


# ==================================================================================================
# PART P — SOURCE SPECIFICATION EVIDENCE
# ==================================================================================================

if "spec_model_eligible" in SOURCE.columns:

    REPLACEMENT[
        "_source_spec_eligible"
    ] = (
        SOURCE[
            "spec_model_eligible"
        ]
        .fillna(False)
        .astype(bool)
    )

else:

    REPLACEMENT[
        "_source_spec_eligible"
    ] = False


if "spec_regime" in SOURCE.columns:

    REPLACEMENT[
        "_source_spec_regime"
    ] = SOURCE[
        "spec_regime"
    ].copy()

else:

    REPLACEMENT[
        "_source_spec_regime"
    ] = None


REPLACEMENT[
    "spec_model_eligible"
] = False


REPLACEMENT[
    "spec_regime"
] = None


REPLACEMENT[
    "_spec_resolution_method"
] = "UNRESOLVED"


# ==================================================================================================
# PART Q — APPLY SOURCE-VERIFIED CONTEXT OVERRIDES
# ==================================================================================================

for _, override in EVENT_CONTEXT_OVERRIDE_REGISTRY.iterrows():

    event_value = int(
        override["event_id"]
    )


    competition_value = int(
        override["competition_id"]
    )


    date_value = pd.Timestamp(
        override["result_date"]
    )


    event_match = (
        REPLACEMENT[
            "event_id"
        ]
        .map(normalize_wa_id)
        .eq(event_value)
    )


    date_match = (
        REPLACEMENT[
            "result_date_dt"
        ]
        .eq(date_value)
    )


    competition_match = (
        REPLACEMENT[
            "_competition_id"
        ]
        .map(normalize_wa_id)
        .eq(competition_value)
    )


    mask = (
        event_match
        &
        (
            competition_match
            |
            date_match
        )
    )


    REPLACEMENT.loc[
        mask,
        "implement_weight_g",
    ] = float(
        override["implement_weight_g"]
    )


    REPLACEMENT.loc[
        mask,
        "model_group",
    ] = override[
        "model_group"
    ]


    REPLACEMENT.loc[
        mask,
        "spec_model_eligible",
    ] = True


    REPLACEMENT.loc[
        mask,
        "spec_regime",
    ] = override[
        "spec_regime"
    ]


    REPLACEMENT.loc[
        mask,
        "_spec_resolution_method",
    ] = "CONTEXT_OVERRIDE"


# ==================================================================================================
# PART R — SAFE EVENT-ID RESOLUTION
# ==================================================================================================

for idx in REPLACEMENT.index:

    if bool_value(
        REPLACEMENT.at[
            idx,
            "spec_model_eligible",
        ]
    ):

        continue


    event_id = normalize_wa_id(
        REPLACEMENT.at[
            idx,
            "event_id",
        ]
    )


    if event_id is None:

        continue


    evidence = EVENT_SEMANTIC_LOOKUP.get(
        int(event_id)
    )


    if evidence is None:

        continue


    REPLACEMENT.at[
        idx,
        "implement_weight_g",
    ] = float(
        evidence[
            "implement_weight_g"
        ]
    )


    REPLACEMENT.at[
        idx,
        "model_group",
    ] = evidence[
        "model_group"
    ]


    REPLACEMENT.at[
        idx,
        "spec_model_eligible",
    ] = True


    REPLACEMENT.at[
        idx,
        "spec_regime",
    ] = evidence[
        "spec_regime"
    ]


    REPLACEMENT.at[
        idx,
        "_spec_resolution_method",
    ] = "SAFE_EVENT_ID"


# ==================================================================================================
# PART S — TRUST CELL 11B ELIGIBILITY WHERE GROUP IS KNOWN
# ==================================================================================================

source_eligible_mask = (
    ~REPLACEMENT[
        "spec_model_eligible"
    ]
    .astype(bool)

    &
    REPLACEMENT[
        "_source_spec_eligible"
    ]
    .astype(bool)

    &
    REPLACEMENT[
        "model_group"
    ]
    .notna()
)


REPLACEMENT.loc[
    source_eligible_mask,
    "spec_model_eligible",
] = True


REPLACEMENT.loc[
    source_eligible_mask,
    "spec_regime",
] = (
    REPLACEMENT.loc[
        source_eligible_mask,
        "model_group",
    ]
    .map(
        canonical_current_spec_label
    )
)


REPLACEMENT.loc[
    source_eligible_mask,
    "_spec_resolution_method",
] = "SOURCE_ELIGIBLE"


# ==================================================================================================
# PART T — CHRONOLOGICAL CURRENT-SPEC FALLBACK
# ==================================================================================================
#
# If group is known and the performance occurs on or after the earliest
# historically known model-eligible date for that group, current-spec
# classification is accepted.
#
# ==================================================================================================

for idx in REPLACEMENT.index:

    if bool_value(
        REPLACEMENT.at[
            idx,
            "spec_model_eligible",
        ]
    ):

        continue


    group = REPLACEMENT.at[
        idx,
        "model_group",
    ]


    result_date = REPLACEMENT.at[
        idx,
        "result_date_dt",
    ]


    if (
        group is None
        or pd.isna(result_date)
    ):

        continue


    threshold = (
        EARLIEST_ELIGIBLE_DATE_BY_GROUP
        .get(group)
    )


    if threshold is None:

        continue


    if (
        pd.Timestamp(result_date)
        >=
        pd.Timestamp(threshold)
    ):

        REPLACEMENT.at[
            idx,
            "spec_model_eligible",
        ] = True


        REPLACEMENT.at[
            idx,
            "spec_regime",
        ] = (
            canonical_current_spec_label(
                group
            )
        )


        REPLACEMENT.at[
            idx,
            "_spec_resolution_method",
        ] = "CHRONOLOGICAL_CURRENT_SPEC"


# ==================================================================================================
# PART U — PRESERVE EXPLICIT NONCURRENT / UNKNOWN SPECIFICATION
# ==================================================================================================

for idx in REPLACEMENT.index:

    if bool_value(
        REPLACEMENT.at[
            idx,
            "spec_model_eligible",
        ]
    ):

        continue


    source_label = REPLACEMENT.at[
        idx,
        "_source_spec_regime",
    ]


    if (
        source_label is not None
        and str(source_label)
        in NONCURRENT_SPEC_LABELS
    ):

        REPLACEMENT.at[
            idx,
            "spec_regime",
        ] = str(
            source_label
        )


        REPLACEMENT.at[
            idx,
            "_spec_resolution_method",
        ] = "SOURCE_NONCURRENT_SPEC"


    else:

        REPLACEMENT.at[
            idx,
            "spec_regime",
        ] = "UNKNOWN_SPEC"


        REPLACEMENT.at[
            idx,
            "_spec_resolution_method",
        ] = "UNKNOWN_SPEC"


# ==================================================================================================
# PART V — RECORD SOURCE / RECONCILIATION STATUS
# ==================================================================================================

if "record_source" in SOURCE.columns:

    REPLACEMENT[
        "record_source"
    ] = SOURCE[
        "record_source"
    ]

else:

    REPLACEMENT[
        "record_source"
    ] = "WORLD_ATHLETICS_INCREMENTAL"


replacement_athlete_ids = (
    REPLACEMENT[
        "aaAthleteId"
    ]
    .map(
        normalize_wa_id
    )
)


existing_athlete_mask = (
    replacement_athlete_ids
    .isin(
        PRODUCTION_ATHLETE_IDS
    )
)


if "reconciliation_status" in SOURCE.columns:

    REPLACEMENT[
        "reconciliation_status"
    ] = SOURCE[
        "reconciliation_status"
    ]

else:

    REPLACEMENT[
        "reconciliation_status"
    ] = np.where(
        existing_athlete_mask,
        "REFRESHED_EXISTING_ATHLETE_YEAR",
        "REFRESHED_NEW_ATHLETE_HISTORY",
    )


# Age is intentionally finalized AFTER reconciliation.
REPLACEMENT[
    "age_years"
] = np.nan


REPLACEMENT[
    "age_available"
] = False


REPLACEMENT[
    "core_group_eligible"
] = (
    REPLACEMENT[
        "model_group"
    ]
    .isin(
        CORE_GROUPS
    )
)


REPLACEMENT[
    "v4_base_eligible"
] = False


# ==================================================================================================
# PART W — REPLACEMENT ROW QA + NONNUMERIC RESULT EXCLUSION
# ==================================================================================================
#
# Nonnumeric javelin rows are legitimate staging observations but are not
# canonical numeric performances.
#
# Missing athlete/date/gender/implement is a HARD ERROR.
#
# Missing numeric mark is audited and excluded.
#
# ==================================================================================================

hard_required_mask = (
    REPLACEMENT[
        "aaAthleteId"
    ]
    .notna()

    &
    REPLACEMENT[
        "result_date_dt"
    ]
    .notna()

    &
    REPLACEMENT[
        "gender_raw"
    ]
    .notna()

    &
    REPLACEMENT[
        "implement_weight_g"
    ]
    .notna()
)


REPLACEMENT_STRUCTURAL_ERRORS = (
    REPLACEMENT.loc[
        ~hard_required_mask
    ]
    .copy()
)


if len(
    REPLACEMENT_STRUCTURAL_ERRORS
) > 0:

    print()
    print(
        "Hard structural errors detected in replacement rows:"
    )

    display(
        REPLACEMENT_STRUCTURAL_ERRORS.head(
            100
        )
    )

    raise RuntimeError(
        "Replacement rows contain missing athlete/date/gender/implement "
        "fields after Cell 11B normalization."
    )


nonnumeric_mark_mask = (
    REPLACEMENT[
        "mark_numeric_m"
    ]
    .isna()
)


NONNUMERIC_REPLACEMENT_EXCLUSIONS = (
    REPLACEMENT.loc[
        nonnumeric_mark_mask
    ]
    .copy()
)


N_NONNUMERIC_REPLACEMENT_EXCLUSIONS = int(
    len(
        NONNUMERIC_REPLACEMENT_EXCLUSIONS
    )
)


NONNUMERIC_REPLACEMENT_EXCLUSIONS_PATH = (
    RUN_CHECKPOINT_DIR
    /
    "canonical_reconciliation_nonnumeric_exclusions.parquet"
)


NONNUMERIC_REPLACEMENT_EXCLUSIONS.to_parquet(
    NONNUMERIC_REPLACEMENT_EXCLUSIONS_PATH,
    index=False,
)


REPLACEMENT = (
    REPLACEMENT.loc[
        ~nonnumeric_mark_mask
    ]
    .copy()
)


N_REPLACEMENT_ROWS = int(
    len(
        REPLACEMENT
    )
)


remaining_invalid_mask = (
    REPLACEMENT[
        "aaAthleteId"
    ]
    .isna()

    |
    REPLACEMENT[
        "result_date_dt"
    ]
    .isna()

    |
    REPLACEMENT[
        "mark_numeric_m"
    ]
    .isna()

    |
    REPLACEMENT[
        "gender_raw"
    ]
    .isna()

    |
    REPLACEMENT[
        "implement_weight_g"
    ]
    .isna()
)


REPLACEMENT_INVALID_ROWS = (
    REPLACEMENT.loc[
        remaining_invalid_mask
    ]
    .copy()
)


if len(
    REPLACEMENT_INVALID_ROWS
) > 0:

    display(
        REPLACEMENT_INVALID_ROWS.head(
            100
        )
    )

    raise RuntimeError(
        "Numeric replacement rows still contain missing required fields."
    )


print()
print("REPLACEMENT ROW QA")
print("-" * 145)

print(
    f"Staging rows received             : "
    f"{N_STAGING_ROWS:,}"
)

print(
    f"Nonnumeric result rows excluded   : "
    f"{N_NONNUMERIC_REPLACEMENT_EXCLUSIONS:,}"
)

print(
    f"Numeric replacement rows retained : "
    f"{N_REPLACEMENT_ROWS:,}"
)

print(
    f"Hard structural errors            : "
    f"{len(REPLACEMENT_STRUCTURAL_ERRORS):,}"
)

print(
    f"Nonnumeric audit                  : "
    f"{NONNUMERIC_REPLACEMENT_EXCLUSIONS_PATH}"
)


# ==================================================================================================
# PART X — DERIVE SUCCESSFULLY REFRESHED ATHLETE-YEAR KEYS
# ==================================================================================================

def derive_refresh_keys():

    # Preferred source.
    if "ATHLETE_YEAR_REFRESH_SUMMARY" in globals():

        try:

            summary = pd.DataFrame(
                ATHLETE_YEAR_REFRESH_SUMMARY
            ).copy()


            athlete_column = first_existing_column(
                summary,
                [
                    "aaAthleteId",
                    "athlete_id",
                    "athleteId",
                ],
            )


            year_column = first_existing_column(
                summary,
                [
                    "year",
                    "result_year",
                ],
            )


            if (
                athlete_column is not None
                and year_column is not None
            ):

                keys = set()


                for _, row in summary.iterrows():

                    athlete_id = normalize_wa_id(
                        row[
                            athlete_column
                        ]
                    )


                    try:

                        year = int(
                            row[
                                year_column
                            ]
                        )

                    except Exception:

                        continue


                    if athlete_id is None:

                        continue


                    keys.add(
                        (
                            int(athlete_id),
                            year,
                        )
                    )


                if len(keys) > 0:

                    return (
                        keys,
                        "ATHLETE_YEAR_REFRESH_SUMMARY",
                    )


        except Exception:

            pass


    # Secondary source.
    if "REFRESHED_ATHLETE_RESULTS_RAW" in globals():

        try:

            raw = pd.DataFrame(
                REFRESHED_ATHLETE_RESULTS_RAW
            ).copy()


            athlete_column = first_existing_column(
                raw,
                [
                    "aaAthleteId",
                    "athlete_id",
                    "athleteId",
                ],
            )


            date_column = first_existing_column(
                raw,
                [
                    "result_date_dt",
                    "date",
                    "resultDate",
                ],
            )


            if (
                athlete_column is not None
                and date_column is not None
            ):

                raw[
                    "_date"
                ] = pd.to_datetime(
                    raw[
                        date_column
                    ],
                    errors="coerce",
                )


                keys = set()


                for athlete, result_date in zip(
                    raw[
                        athlete_column
                    ],
                    raw[
                        "_date"
                    ],
                ):

                    athlete_id = normalize_wa_id(
                        athlete
                    )


                    if (
                        athlete_id is None
                        or pd.isna(result_date)
                    ):

                        continue


                    keys.add(
                        (
                            int(athlete_id),
                            int(result_date.year),
                        )
                    )


                if len(keys) > 0:

                    return (
                        keys,
                        "REFRESHED_ATHLETE_RESULTS_RAW",
                    )


        except Exception:

            pass


    # Final fallback.
    keys = set(
        zip(
            REPLACEMENT[
                "aaAthleteId"
            ]
            .astype(int),

            REPLACEMENT[
                "result_date_dt"
            ]
            .dt.year
            .astype(int),
        )
    )


    return (
        keys,
        "REFRESHED_STAGING_FINAL",
    )


REFRESH_KEYS, REFRESH_KEY_SOURCE = (
    derive_refresh_keys()
)


N_REFRESH_KEYS = int(
    len(
        REFRESH_KEYS
    )
)


# ==================================================================================================
# PART Y — REMOVE SUCCESSFULLY REFRESHED PRODUCTION SLICES
# ==================================================================================================

PRODUCTION_NORMALIZED[
    "_refresh_key"
] = list(
    zip(
        PRODUCTION_NORMALIZED[
            "aaAthleteId"
        ]
        .map(
            normalize_wa_id
        ),

        PRODUCTION_NORMALIZED[
            "result_date_dt"
        ]
        .dt.year,
    )
)


REMOVE_OLD_SLICE_MASK = (
    PRODUCTION_NORMALIZED[
        "_refresh_key"
    ]
    .isin(
        REFRESH_KEYS
    )
)


OLD_SLICE_ROWS = (
    PRODUCTION_NORMALIZED.loc[
        REMOVE_OLD_SLICE_MASK
    ]
    .copy()
)


UNCHANGED_PRODUCTION_ROWS = (
    PRODUCTION_NORMALIZED.loc[
        ~REMOVE_OLD_SLICE_MASK
    ]
    .copy()
)


N_OLD_SLICE_ROWS = int(
    len(
        OLD_SLICE_ROWS
    )
)


# ==================================================================================================
# PART Z — BUILD RAW RECONCILED CANDIDATE
# ==================================================================================================

REPLACEMENT_CANONICAL = (
    REPLACEMENT[
        EXPECTED_CANONICAL_COLUMNS
    ]
    .copy()
)


RECONCILED = pd.concat(
    [
        UNCHANGED_PRODUCTION_ROWS[
            EXPECTED_CANONICAL_COLUMNS
        ],

        REPLACEMENT_CANONICAL,
    ],

    ignore_index=True,
)


# ==================================================================================================
# PART AA — GLOBAL POST-RECONCILIATION VOCABULARY NORMALIZATION
# ==================================================================================================

RECONCILED[
    "_derived_group"
] = [
    derive_model_group(
        gender,
        weight,
    )
    for gender, weight
    in zip(
        RECONCILED[
            "gender_raw"
        ],
        RECONCILED[
            "implement_weight_g"
        ],
    )
]


fill_group_mask = (
    RECONCILED[
        "model_group"
    ]
    .isna()

    &
    RECONCILED[
        "_derived_group"
    ]
    .notna()
)


RECONCILED.loc[
    fill_group_mask,
    "model_group",
] = (
    RECONCILED.loc[
        fill_group_mask,
        "_derived_group",
    ]
)


eligible_group_mask = (
    RECONCILED[
        "spec_model_eligible"
    ]
    .fillna(False)
    .astype(bool)

    &
    RECONCILED[
        "model_group"
    ]
    .notna()
)


RECONCILED.loc[
    eligible_group_mask,
    "spec_regime",
] = (
    RECONCILED.loc[
        eligible_group_mask,
        "model_group",
    ]
    .map(
        canonical_current_spec_label
    )
)


# ==================================================================================================
# PART AB — FINAL DEFENSIVE 10229685 CONTEXT PATCH
# ==================================================================================================

for _, override in EVENT_CONTEXT_OVERRIDE_REGISTRY.iterrows():

    event_value = int(
        override["event_id"]
    )


    date_value = pd.Timestamp(
        override["result_date"]
    )


    mask = (
        RECONCILED[
            "event_id"
        ]
        .map(normalize_wa_id)
        .eq(event_value)

        &
        RECONCILED[
            "result_date_dt"
        ]
        .eq(date_value)
    )


    RECONCILED.loc[
        mask,
        "implement_weight_g",
    ] = float(
        override["implement_weight_g"]
    )


    RECONCILED.loc[
        mask,
        "model_group",
    ] = override[
        "model_group"
    ]


    RECONCILED.loc[
        mask,
        "spec_regime",
    ] = override[
        "spec_regime"
    ]


    RECONCILED.loc[
        mask,
        "spec_model_eligible",
    ] = True


# ==================================================================================================
# PART AC — FINAL BACKWARD-COMPATIBLE AGE POLICY
# ==================================================================================================
#
# EXISTING:
#
#       preserve production age exactly
#
# NEW:
#
#       exact DOB -> / 365.2425
#       YEAR only -> NaN
#       NONE      -> NaN
#       UNKNOWN   -> NaN
#
# ==================================================================================================

PRODUCTION_KEYED = add_occurrence_keys(
    PRODUCTION_BASE
)


RECONCILED_KEYED = add_occurrence_keys(
    RECONCILED
)


production_age_lookup = (
    PRODUCTION_KEYED
    .set_index(
        "_match_key"
    )[
        "age_years"
    ]
    .to_dict()
)


production_match_keys = set(
    PRODUCTION_KEYED[
        "_match_key"
    ]
)


RECONCILED_KEYED[
    "_existed_in_production"
] = (
    RECONCILED_KEYED[
        "_match_key"
    ]
    .isin(
        production_match_keys
    )
)


existing_mask = (
    RECONCILED_KEYED[
        "_existed_in_production"
    ]
)


new_mask = (
    ~existing_mask
)


# Existing physical observations retain production age exactly.
RECONCILED_KEYED.loc[
    existing_mask,
    "age_years",
] = (
    RECONCILED_KEYED.loc[
        existing_mask,
        "_match_key",
    ]
    .map(
        production_age_lookup
    )
)


# New observations start without age.
RECONCILED_KEYED.loc[
    new_mask,
    "age_years",
] = np.nan


new_exact_dob_mask = (
    new_mask

    &
    RECONCILED_KEYED[
        "canonical_birth_date"
    ]
    .notna()

    &
    RECONCILED_KEYED[
        "result_date_dt"
    ]
    .notna()
)


RECONCILED_KEYED.loc[
    new_exact_dob_mask,
    "age_years",
] = (
    (
        RECONCILED_KEYED.loc[
            new_exact_dob_mask,
            "result_date_dt",
        ]
        -
        RECONCILED_KEYED.loc[
            new_exact_dob_mask,
            "canonical_birth_date",
        ]
    )
    .dt.days
    /
    365.2425
)


new_year_only_mask = (
    new_mask

    &
    RECONCILED_KEYED[
        "canonical_birth_date"
    ]
    .isna()

    &
    RECONCILED_KEYED[
        "canonical_birth_year"
    ]
    .notna()

    &
    RECONCILED_KEYED[
        "birth_precision"
    ]
    .fillna("")
    .astype(str)
    .str.upper()
    .eq("YEAR")
)


# YEAR-only rows deliberately remain NaN.


# ==================================================================================================
# PART AD — FINAL ELIGIBILITY FLAGS
# ==================================================================================================

RECONCILED_KEYED[
    "core_group_eligible"
] = (
    RECONCILED_KEYED[
        "model_group"
    ]
    .isin(
        CORE_GROUPS
    )
)


RECONCILED_KEYED[
    "age_available"
] = (
    RECONCILED_KEYED[
        "age_years"
    ]
    .notna()
)


RECONCILED_KEYED[
    "v4_base_eligible"
] = (
    RECONCILED_KEYED[
        "spec_model_eligible"
    ]
    .fillna(False)
    .astype(bool)

    &
    RECONCILED_KEYED[
        "core_group_eligible"
    ]
    .astype(bool)

    &
    RECONCILED_KEYED[
        "age_available"
    ]
    .astype(bool)
)


CANONICAL_CANDIDATE = (
    RECONCILED_KEYED[
        EXPECTED_CANONICAL_COLUMNS
    ]
    .copy()
)


# ==================================================================================================
# PART AE — MODEL GROUP QA
# ==================================================================================================

expected_group = pd.Series(
    [
        derive_model_group(
            gender,
            weight,
        )
        for gender, weight
        in zip(
            CANONICAL_CANDIDATE[
                "gender_raw"
            ],
            CANONICAL_CANDIDATE[
                "implement_weight_g"
            ],
        )
    ],

    index=
        CANONICAL_CANDIDATE.index,
)


MODEL_GROUP_CONFLICT_MASK = (
    CANONICAL_CANDIDATE[
        "model_group"
    ]
    .notna()

    &
    expected_group
    .notna()

    &
    (
        CANONICAL_CANDIDATE[
            "model_group"
        ]
        .astype(str)

        !=

        expected_group
        .astype(str)
    )
)


N_MODEL_GROUP_CONFLICTS = int(
    MODEL_GROUP_CONFLICT_MASK.sum()
)


# ==================================================================================================
# PART AF — VOCABULARY QA
# ==================================================================================================

canonical_current_label_mask = (
    CANONICAL_CANDIDATE[
        "spec_regime"
    ]
    .astype(str)
    .str.match(
        r"^[MW]_JAVELIN_\d+_CURRENT_SPEC$",
        na=False,
    )
)


ELIGIBLE_MALFORMED_MASK = (
    CANONICAL_CANDIDATE[
        "spec_model_eligible"
    ]
    .fillna(False)
    .astype(bool)

    &
    ~canonical_current_label_mask
)


N_ELIGIBLE_MALFORMED = int(
    ELIGIBLE_MALFORMED_MASK.sum()
)


N_GENERIC_CURRENT_SPEC = int(
    CANONICAL_CANDIDATE[
        "spec_regime"
    ]
    .isin(
        [
            "CURRENT_SPEC",
            "CURRENT_SPEC_DIRECT",
            "M800_POST_1991_RECORD_REGIME",
            "W600_POST_1999_SPEC",
        ]
    )
    .sum()
)


FRESH_UNKNOWN_SPEC_ROWS = (
    REPLACEMENT.loc[
        REPLACEMENT[
            "spec_regime"
        ]
        .eq(
            "UNKNOWN_SPEC"
        )
    ]
    .copy()
)


N_FRESH_UNKNOWN_SPEC = int(
    len(
        FRESH_UNKNOWN_SPEC_ROWS
    )
)


# ==================================================================================================
# PART AG — PHYSICAL PERFORMANCE REGRESSION
# ==================================================================================================

FINAL_KEYED = add_occurrence_keys(
    CANONICAL_CANDIDATE
)


production_counts = (
    PRODUCTION_KEYED[
        "_physical_key"
    ]
    .value_counts()
)


candidate_counts = (
    FINAL_KEYED[
        "_physical_key"
    ]
    .value_counts()
)


PHYSICAL_DELTA = (
    pd.concat(
        [
            production_counts.rename(
                "production_rows"
            ),

            candidate_counts.rename(
                "candidate_rows"
            ),
        ],

        axis=1,
    )
    .fillna(0)
    .astype(int)
)


PHYSICAL_DELTA[
    "delta"
] = (
    PHYSICAL_DELTA[
        "candidate_rows"
    ]
    -
    PHYSICAL_DELTA[
        "production_rows"
    ]
)


N_PHYSICAL_REMOVED = int(
    (
        -PHYSICAL_DELTA.loc[
            PHYSICAL_DELTA[
                "delta"
            ]
            < 0,
            "delta",
        ]
    )
    .sum()
)


N_PHYSICAL_ADDED = int(
    PHYSICAL_DELTA.loc[
        PHYSICAL_DELTA[
            "delta"
        ]
        > 0,
        "delta",
    ]
    .sum()
)


# ==================================================================================================
# PART AH — EXISTING ROW TRANSITIONS
# ==================================================================================================

MATCHED_EXISTING = (
    PRODUCTION_KEYED.merge(
        FINAL_KEYED,

        on=
            "_match_key",

        how=
            "inner",

        suffixes=(
            "_old",
            "_new",
        ),

        validate=
            "one_to_one",
    )
)


old_v4 = (
    MATCHED_EXISTING[
        "v4_base_eligible_old"
    ]
    .fillna(False)
    .astype(bool)
)


new_v4 = (
    MATCHED_EXISTING[
        "v4_base_eligible_new"
    ]
    .fillna(False)
    .astype(bool)
)


N_EXISTING_V4_LOST = int(
    (
        old_v4
        &
        ~new_v4
    )
    .sum()
)


N_EXISTING_V4_GAINED = int(
    (
        ~old_v4
        &
        new_v4
    )
    .sum()
)


# ==================================================================================================
# PART AI — EXISTING AGE BACKWARD-COMPATIBILITY
# ==================================================================================================

old_age = pd.to_numeric(
    MATCHED_EXISTING[
        "age_years_old"
    ],
    errors="coerce",
)


new_age = pd.to_numeric(
    MATCHED_EXISTING[
        "age_years_new"
    ],
    errors="coerce",
)


EXISTING_AGE_CONFLICT_MASK = (
    (
        old_age.isna()
        ^
        new_age.isna()
    )

    |
    (
        old_age.notna()
        &
        new_age.notna()
        &
        (
            np.abs(
                old_age
                -
                new_age
            )
            >
            1e-12
        )
    )
)


N_EXISTING_AGE_CONFLICTS = int(
    EXISTING_AGE_CONFLICT_MASK.sum()
)


# ==================================================================================================
# PART AJ — NEW-ROW AGE POLICY QA
# ==================================================================================================

N_NEW_EXACT_DOB = int(
    new_exact_dob_mask.sum()
)


N_NEW_YEAR_ONLY = int(
    new_year_only_mask.sum()
)


N_NEW_YEAR_WITH_AGE = int(
    (
        new_year_only_mask

        &
        RECONCILED_KEYED[
            "age_years"
        ]
        .notna()
    )
    .sum()
)


N_NEW_EXACT_MISSING_AGE = int(
    (
        new_exact_dob_mask

        &
        RECONCILED_KEYED[
            "age_years"
        ]
        .isna()
    )
    .sum()
)


if N_NEW_EXACT_DOB > 0:

    exact_check = (
        RECONCILED_KEYED.loc[
            new_exact_dob_mask,
            [
                "result_date_dt",
                "canonical_birth_date",
                "age_years",
            ],
        ]
        .copy()
    )


    exact_check[
        "_expected_age"
    ] = (
        (
            exact_check[
                "result_date_dt"
            ]
            -
            exact_check[
                "canonical_birth_date"
            ]
        )
        .dt.days
        /
        365.2425
    )


    N_NEW_EXACT_FORMULA_ERRORS = int(
        (
            np.abs(
                exact_check[
                    "age_years"
                ]
                -
                exact_check[
                    "_expected_age"
                ]
            )
            >
            1e-12
        )
        .sum()
    )


else:

    N_NEW_EXACT_FORMULA_ERRORS = 0


# ==================================================================================================
# PART AK — DUPLICATE REGRESSION
# ==================================================================================================

N_PRODUCTION_DUPLICATE_ROWS = int(
    PRODUCTION_KEYED[
        "_physical_key"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)


N_CANDIDATE_DUPLICATE_ROWS = int(
    FINAL_KEYED[
        "_physical_key"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)


N_DUPLICATE_INCREASE = max(
    0,
    (
        N_CANDIDATE_DUPLICATE_ROWS
        -
        N_PRODUCTION_DUPLICATE_ROWS
    ),
)


# ==================================================================================================
# PART AL — POPULATION COUNTS
# ==================================================================================================

N_CANDIDATE_ROWS = int(
    len(
        CANONICAL_CANDIDATE
    )
)


N_ROW_DELTA = (
    N_CANDIDATE_ROWS
    -
    N_PRODUCTION_ROWS
)


N_CANDIDATE_ATHLETES = int(
    CANONICAL_CANDIDATE[
        "aaAthleteId"
    ]
    .nunique()
)


candidate_athletes = set(
    CANONICAL_CANDIDATE[
        "aaAthleteId"
    ]
    .dropna()
    .map(
        normalize_wa_id
    )
)


production_athletes = set(
    PRODUCTION_BASE[
        "aaAthleteId"
    ]
    .dropna()
    .map(
        normalize_wa_id
    )
)


NEW_CANONICAL_ATHLETE_IDS = (
    candidate_athletes
    -
    production_athletes
)


N_NEW_CANONICAL_ATHLETES = int(
    len(
        NEW_CANONICAL_ATHLETE_IDS
    )
)


N_PRODUCTION_V4 = int(
    PRODUCTION_BASE[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


N_CANDIDATE_V4 = int(
    CANONICAL_CANDIDATE[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


production_match_keys = set(
    PRODUCTION_KEYED[
        "_match_key"
    ]
)


NEW_PHYSICAL_ROWS = (
    FINAL_KEYED.loc[
        ~FINAL_KEYED[
            "_match_key"
        ]
        .isin(
            production_match_keys
        )
    ]
    .copy()
)


N_NEW_PHYSICAL_V4 = int(
    NEW_PHYSICAL_ROWS[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


# ==================================================================================================
# PART AM — INTERNAL ELIGIBILITY CONSISTENCY
# ==================================================================================================

EXPECTED_CORE = (
    CANONICAL_CANDIDATE[
        "model_group"
    ]
    .isin(
        CORE_GROUPS
    )
)


EXPECTED_AGE_AVAILABLE = (
    CANONICAL_CANDIDATE[
        "age_years"
    ]
    .notna()
)


EXPECTED_V4 = (
    CANONICAL_CANDIDATE[
        "spec_model_eligible"
    ]
    .fillna(False)
    .astype(bool)

    &
    EXPECTED_CORE

    &
    EXPECTED_AGE_AVAILABLE
)


N_CORE_FLAG_MISMATCH = int(
    (
        CANONICAL_CANDIDATE[
            "core_group_eligible"
        ]
        .fillna(False)
        .astype(bool)

        !=

        EXPECTED_CORE
    )
    .sum()
)


N_AGE_FLAG_MISMATCH = int(
    (
        CANONICAL_CANDIDATE[
            "age_available"
        ]
        .fillna(False)
        .astype(bool)

        !=

        EXPECTED_AGE_AVAILABLE
    )
    .sum()
)


N_V4_FLAG_MISMATCH = int(
    (
        CANONICAL_CANDIDATE[
            "v4_base_eligible"
        ]
        .fillna(False)
        .astype(bool)

        !=

        EXPECTED_V4
    )
    .sum()
)


# ==================================================================================================
# PART AN — STRUCTURAL RELEASE GATE
# ==================================================================================================

SCHEMA_PASS = bool(
    list(
        CANONICAL_CANDIDATE.columns
    )
    ==
    EXPECTED_CANONICAL_COLUMNS
)


ROW_ARITHMETIC_PASS = bool(
    N_ROW_DELTA
    ==
    N_PHYSICAL_ADDED
    -
    N_PHYSICAL_REMOVED
)


CELL_12_STRUCTURAL_PASS = bool(

    N_PHYSICAL_REMOVED == 0

    and N_EXISTING_V4_LOST == 0

    and N_EXISTING_AGE_CONFLICTS == 0

    and N_MODEL_GROUP_CONFLICTS == 0

    and N_ELIGIBLE_MALFORMED == 0

    and N_GENERIC_CURRENT_SPEC == 0

    and N_DUPLICATE_INCREASE == 0

    and N_CORE_FLAG_MISMATCH == 0

    and N_AGE_FLAG_MISMATCH == 0

    and N_V4_FLAG_MISMATCH == 0

    and N_NEW_YEAR_WITH_AGE == 0

    and N_NEW_EXACT_MISSING_AGE == 0

    and N_NEW_EXACT_FORMULA_ERRORS == 0

    and SCHEMA_PASS

    and ROW_ARITHMETIC_PASS
)


# ==================================================================================================
# PART AO — GOLDEN REFERENCE FOR THE VALIDATED REFACTOR RUN
# ==================================================================================================
#
# This is deliberately run-specific.
#
# Future update runs do not have to reproduce these fixed counts.
#
# ==================================================================================================

GOLDEN_REFERENCE_EXPECTATIONS = {

    "20260919_225312": {

        "candidate_rows":
            249233,

        "candidate_athletes":
            21901,

        "new_athletes":
            73,

        "physical_added":
            337,

        "physical_removed":
            0,

        "candidate_v4":
            192983,

        "existing_v4_gained":
            20,

        "existing_v4_lost":
            0,

        "new_physical_v4":
            261,

        "candidate_duplicate_rows":
            358,

        "new_exact_dob_rows":
            263,

        "new_year_precision_rows":
            46,
    },
}


reference = (
    GOLDEN_REFERENCE_EXPECTATIONS
    .get(
        str(RUN_ID)
    )
)


if reference is not None:

    GOLDEN_CHECKS = {

        "candidate_rows":
            N_CANDIDATE_ROWS
            ==
            reference[
                "candidate_rows"
            ],

        "candidate_athletes":
            N_CANDIDATE_ATHLETES
            ==
            reference[
                "candidate_athletes"
            ],

        "new_athletes":
            N_NEW_CANONICAL_ATHLETES
            ==
            reference[
                "new_athletes"
            ],

        "physical_added":
            N_PHYSICAL_ADDED
            ==
            reference[
                "physical_added"
            ],

        "physical_removed":
            N_PHYSICAL_REMOVED
            ==
            reference[
                "physical_removed"
            ],

        "candidate_v4":
            N_CANDIDATE_V4
            ==
            reference[
                "candidate_v4"
            ],

        "existing_v4_gained":
            N_EXISTING_V4_GAINED
            ==
            reference[
                "existing_v4_gained"
            ],

        "existing_v4_lost":
            N_EXISTING_V4_LOST
            ==
            reference[
                "existing_v4_lost"
            ],

        "new_physical_v4":
            N_NEW_PHYSICAL_V4
            ==
            reference[
                "new_physical_v4"
            ],

        "candidate_duplicate_rows":
            N_CANDIDATE_DUPLICATE_ROWS
            ==
            reference[
                "candidate_duplicate_rows"
            ],

        "new_exact_dob_rows":
            N_NEW_EXACT_DOB
            ==
            reference[
                "new_exact_dob_rows"
            ],

        "new_year_precision_rows":
            N_NEW_YEAR_ONLY
            ==
            reference[
                "new_year_precision_rows"
            ],
    }


    GOLDEN_REFERENCE_PASS = bool(
        all(
            GOLDEN_CHECKS.values()
        )
    )


else:

    GOLDEN_CHECKS = {}


    GOLDEN_REFERENCE_PASS = True


CELL_12_PASS = bool(
    CELL_12_STRUCTURAL_PASS
    and
    GOLDEN_REFERENCE_PASS
)


# ==================================================================================================
# PART AP — AUDIT TABLES
# ==================================================================================================

SLICE_RECONCILIATION_AUDIT = pd.DataFrame(
    [
        {
            "refresh_key_source":
                REFRESH_KEY_SOURCE,

            "athlete_year_refresh_keys":
                N_REFRESH_KEYS,

            "staging_rows":
                N_STAGING_ROWS,

            "nonnumeric_excluded":
                N_NONNUMERIC_REPLACEMENT_EXCLUSIONS,

            "old_rows_removed":
                N_OLD_SLICE_ROWS,

            "fresh_numeric_rows_inserted":
                N_REPLACEMENT_ROWS,

            "production_rows":
                N_PRODUCTION_ROWS,

            "candidate_rows":
                N_CANDIDATE_ROWS,

            "net_row_delta":
                N_ROW_DELTA,

            "physical_added":
                N_PHYSICAL_ADDED,

            "physical_removed":
                N_PHYSICAL_REMOVED,
        }
    ]
)


MODEL_POPULATION_AUDIT = pd.DataFrame(
    [
        {
            "production_v4":
                N_PRODUCTION_V4,

            "candidate_v4":
                N_CANDIDATE_V4,

            "existing_v4_gained":
                N_EXISTING_V4_GAINED,

            "existing_v4_lost":
                N_EXISTING_V4_LOST,

            "new_physical_v4":
                N_NEW_PHYSICAL_V4,

            "existing_age_conflicts":
                N_EXISTING_AGE_CONFLICTS,

            "new_exact_dob_rows":
                N_NEW_EXACT_DOB,

            "new_year_only_rows":
                N_NEW_YEAR_ONLY,

            "new_year_rows_with_age":
                N_NEW_YEAR_WITH_AGE,

            "new_exact_formula_errors":
                N_NEW_EXACT_FORMULA_ERRORS,
        }
    ]
)


SOURCE_SCHEMA_AUDIT = pd.DataFrame(
    [
        {
            "athlete_col":
                athlete_col,

            "date_col":
                date_col,

            "mark_col":
                mark_col,

            "gender_col":
                gender_col,

            "weight_col":
                weight_col,

            "event_col":
                event_col,

            "competition_col":
                competition_col,

            "birth_date_col":
                birth_date_col,

            "birth_year_col":
                birth_year_col,

            "birth_precision_col":
                birth_precision_col,

            "upstream_age_col":
                source_age_col,
        }
    ]
)


# ==================================================================================================
# PART AQ — SAVE VERSIONED CANDIDATE + AUDITS
# ==================================================================================================

FINAL_DATA_DIR = Path(
    CONFIG.canonical_dataset_path
).parent


CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH = (
    FINAL_DATA_DIR
    /
    (
        f"javelin_reconciled_base_1986_2026_"
        f"refreshed_{RUN_ID}_candidate.parquet"
    )
)


SLICE_AUDIT_PATH = (
    RUN_CHECKPOINT_DIR
    /
    "canonical_reconciliation_slice_audit.parquet"
)


MODEL_AUDIT_PATH = (
    RUN_CHECKPOINT_DIR
    /
    "canonical_reconciliation_model_audit.parquet"
)


SOURCE_SCHEMA_AUDIT_PATH = (
    RUN_CHECKPOINT_DIR
    /
    "canonical_reconciliation_source_schema_audit.parquet"
)


UNKNOWN_SPEC_PATH = (
    RUN_CHECKPOINT_DIR
    /
    "canonical_reconciliation_fresh_unknown_spec.parquet"
)


CONTEXT_OVERRIDE_PATH = (
    RUN_CHECKPOINT_DIR
    /
    "canonical_event_context_override_registry.parquet"
)


CANONICAL_CANDIDATE.to_parquet(
    CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH,
    index=False,
)


SLICE_RECONCILIATION_AUDIT.to_parquet(
    SLICE_AUDIT_PATH,
    index=False,
)


MODEL_POPULATION_AUDIT.to_parquet(
    MODEL_AUDIT_PATH,
    index=False,
)


SOURCE_SCHEMA_AUDIT.to_parquet(
    SOURCE_SCHEMA_AUDIT_PATH,
    index=False,
)


FRESH_UNKNOWN_SPEC_ROWS.to_parquet(
    UNKNOWN_SPEC_PATH,
    index=False,
)


EVENT_CONTEXT_OVERRIDE_REGISTRY.to_parquet(
    CONTEXT_OVERRIDE_PATH,
    index=False,
)


# ==================================================================================================
# PART AR — PROMOTE IN-MEMORY CANDIDATE FOR CELL 13
# ==================================================================================================

CANONICAL_REFRESHED_NORMALIZED_FINAL = (
    CANONICAL_CANDIDATE.copy()
)


CANONICAL_REFRESHED_NORMALIZED_FINAL_PASS = (
    CELL_12_PASS
)


# ==================================================================================================
# PART AS — MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "canonical_reconciliation"
] = {

    "status":
        (
            "VALIDATED"
            if CELL_12_PASS
            else "REVIEW_REQUIRED"
        ),

    "source_columns": {
        "birth_date":
            birth_date_col,

        "birth_year":
            birth_year_col,

        "birth_precision":
            birth_precision_col,

        "upstream_age":
            source_age_col,
    },

    "refresh_key_source":
        REFRESH_KEY_SOURCE,

    "athlete_year_refresh_keys":
        N_REFRESH_KEYS,

    "staging_rows":
        N_STAGING_ROWS,

    "nonnumeric_excluded":
        N_NONNUMERIC_REPLACEMENT_EXCLUSIONS,

    "production_rows":
        N_PRODUCTION_ROWS,

    "old_rows_removed":
        N_OLD_SLICE_ROWS,

    "replacement_rows":
        N_REPLACEMENT_ROWS,

    "candidate_rows":
        N_CANDIDATE_ROWS,

    "physical_added":
        N_PHYSICAL_ADDED,

    "physical_removed":
        N_PHYSICAL_REMOVED,

    "production_athletes":
        N_PRODUCTION_ATHLETES,

    "candidate_athletes":
        N_CANDIDATE_ATHLETES,

    "new_canonical_athletes":
        N_NEW_CANONICAL_ATHLETES,

    "production_v4":
        N_PRODUCTION_V4,

    "candidate_v4":
        N_CANDIDATE_V4,

    "existing_v4_gained":
        N_EXISTING_V4_GAINED,

    "existing_v4_lost":
        N_EXISTING_V4_LOST,

    "new_physical_v4":
        N_NEW_PHYSICAL_V4,

    "existing_age_conflicts":
        N_EXISTING_AGE_CONFLICTS,

    "new_exact_dob_rows":
        N_NEW_EXACT_DOB,

    "new_year_precision_rows":
        N_NEW_YEAR_ONLY,

    "new_year_rows_with_age":
        N_NEW_YEAR_WITH_AGE,

    "new_exact_formula_errors":
        N_NEW_EXACT_FORMULA_ERRORS,

    "model_group_conflicts":
        N_MODEL_GROUP_CONFLICTS,

    "eligible_malformed_spec":
        N_ELIGIBLE_MALFORMED,

    "generic_current_spec_labels":
        N_GENERIC_CURRENT_SPEC,

    "fresh_unknown_spec_rows":
        N_FRESH_UNKNOWN_SPEC,

    "production_duplicate_rows":
        N_PRODUCTION_DUPLICATE_ROWS,

    "candidate_duplicate_rows":
        N_CANDIDATE_DUPLICATE_ROWS,

    "duplicate_increase":
        N_DUPLICATE_INCREASE,

    "golden_reference_applied":
        reference is not None,

    "golden_reference_pass":
        GOLDEN_REFERENCE_PASS,

    "candidate_path":
        str(
            CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "CANONICAL_CANDIDATE_READY"

    if CELL_12_PASS

    else "CANONICAL_CANDIDATE_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART AT — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("WORLD ATHLETICS — PRODUCTION CANONICAL RECONCILIATION")
print("=" * 145)


print()
print("SOURCE DEMOGRAPHICS")
print("-" * 145)

print(
    f"Birth-date source                : "
    f"{birth_date_col}"
)

print(
    f"Birth-year source                : "
    f"{birth_year_col}"
)

print(
    f"Birth-precision source           : "
    f"{birth_precision_col}"
)

print(
    f"Upstream age source              : "
    f"{source_age_col}  (not copied)"
)


print()
print("REFRESH SLICES")
print("-" * 145)

print(
    f"Refresh-key source               : "
    f"{REFRESH_KEY_SOURCE}"
)

print(
    f"Athlete-year refresh keys        : "
    f"{N_REFRESH_KEYS:,}"
)

print(
    f"Old production rows removed      : "
    f"{N_OLD_SLICE_ROWS:,}"
)

print(
    f"Fresh numeric replacement rows   : "
    f"{N_REPLACEMENT_ROWS:,}"
)

print(
    f"Nonnumeric staging rows excluded : "
    f"{N_NONNUMERIC_REPLACEMENT_EXCLUSIONS:,}"
)


print()
print("DATASET")
print("-" * 145)

print(
    f"Production rows                  : "
    f"{N_PRODUCTION_ROWS:,}"
)

print(
    f"Candidate rows                   : "
    f"{N_CANDIDATE_ROWS:,}"
)

print(
    f"Net row delta                    : "
    f"{N_ROW_DELTA:+,}"
)

print(
    f"Production athletes              : "
    f"{N_PRODUCTION_ATHLETES:,}"
)

print(
    f"Candidate athletes               : "
    f"{N_CANDIDATE_ATHLETES:,}"
)

print(
    f"New canonical athletes           : "
    f"{N_NEW_CANONICAL_ATHLETES:,}"
)


print()
print("PHYSICAL PERFORMANCE RECONCILIATION")
print("-" * 145)

print(
    f"Physical performances added      : "
    f"{N_PHYSICAL_ADDED:,}"
)

print(
    f"Physical performances removed    : "
    f"{N_PHYSICAL_REMOVED:,}"
)

print(
    f"Row arithmetic pass              : "
    f"{ROW_ARITHMETIC_PASS}"
)


print()
print("VOCABULARY / SPECIFICATION")
print("-" * 145)

print(
    f"Eligible malformed spec labels   : "
    f"{N_ELIGIBLE_MALFORMED:,}"
)

print(
    f"Generic current-spec labels      : "
    f"{N_GENERIC_CURRENT_SPEC:,}"
)

print(
    f"Fresh UNKNOWN_SPEC rows          : "
    f"{N_FRESH_UNKNOWN_SPEC:,}"
)

print(
    f"Model-group conflicts            : "
    f"{N_MODEL_GROUP_CONFLICTS:,}"
)


print()
print("AGE POLICY")
print("-" * 145)

print(
    f"Existing age conflicts           : "
    f"{N_EXISTING_AGE_CONFLICTS:,}"
)

print(
    f"New exact-DOB rows               : "
    f"{N_NEW_EXACT_DOB:,}"
)

print(
    f"New YEAR-precision rows          : "
    f"{N_NEW_YEAR_ONLY:,}"
)

print(
    f"New YEAR rows with age           : "
    f"{N_NEW_YEAR_WITH_AGE:,}"
)

print(
    f"New exact-DOB formula errors     : "
    f"{N_NEW_EXACT_FORMULA_ERRORS:,}"
)

print(
    "New exact-DOB age rule           : "
    "(result_date - DOB).days / 365.2425"
)

print(
    "New YEAR-only age rule           : "
    "age_years = NaN"
)


print()
print("MODEL POPULATION")
print("-" * 145)

print(
    f"Production V4 eligible           : "
    f"{N_PRODUCTION_V4:,}"
)

print(
    f"Candidate V4 eligible            : "
    f"{N_CANDIDATE_V4:,}"
)

print(
    f"Existing rows newly eligible     : "
    f"{N_EXISTING_V4_GAINED:,}"
)

print(
    f"Existing rows losing eligibility : "
    f"{N_EXISTING_V4_LOST:,}"
)

print(
    f"New physical rows V4 eligible    : "
    f"{N_NEW_PHYSICAL_V4:,}"
)


print()
print("DUPLICATES")
print("-" * 145)

print(
    f"Production duplicate rows        : "
    f"{N_PRODUCTION_DUPLICATE_ROWS:,}"
)

print(
    f"Candidate duplicate rows         : "
    f"{N_CANDIDATE_DUPLICATE_ROWS:,}"
)

print(
    f"Duplicate increase               : "
    f"{N_DUPLICATE_INCREASE:,}"
)


print()
print("INTERNAL ELIGIBILITY QA")
print("-" * 145)

print(
    f"Core-group flag mismatches       : "
    f"{N_CORE_FLAG_MISMATCH:,}"
)

print(
    f"Age-available flag mismatches    : "
    f"{N_AGE_FLAG_MISMATCH:,}"
)

print(
    f"V4-base flag mismatches          : "
    f"{N_V4_FLAG_MISMATCH:,}"
)


if reference is not None:

    print()
    print("GOLDEN REFERENCE — VALIDATED RUN")
    print("-" * 145)


    for check_name, passed in GOLDEN_CHECKS.items():

        print(
            f"{check_name:<34}: "
            f"{passed}"
        )


    print()

    print(
        f"Golden reference pass            : "
        f"{GOLDEN_REFERENCE_PASS}"
    )


print()
print("OUTPUT")
print("-" * 145)

print(
    f"Candidate canonical dataset      : "
    f"{CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH}"
)

print(
    f"Slice audit                      : "
    f"{SLICE_AUDIT_PATH}"
)

print(
    f"Model audit                      : "
    f"{MODEL_AUDIT_PATH}"
)

print(
    f"Source-schema audit              : "
    f"{SOURCE_SCHEMA_AUDIT_PATH}"
)

print(
    f"Nonnumeric audit                 : "
    f"{NONNUMERIC_REPLACEMENT_EXCLUSIONS_PATH}"
)

print(
    f"Unknown-spec audit               : "
    f"{UNKNOWN_SPEC_PATH}"
)

print(
    f"Context override registry        : "
    f"{CONTEXT_OVERRIDE_PATH}"
)


print()
print("=" * 145)


if CELL_12_PASS:

    print(
        "CELL 12 STATUS: PRODUCTION CANONICAL CANDIDATE PASSED"
    )

    print()

    print(
        "Run Cell 13 next as the independent release regression."
    )

    print(
        "The production canonical parquet has NOT been overwritten."
    )


else:

    print(
        "CELL 12 STATUS: CANONICAL CANDIDATE REVIEW REQUIRED"
    )

    print()

    print(
        "Do NOT promote. Review failed structural or "
        "golden-reference checks."
    )


print("=" * 145)


SOURCE COLUMN RESOLUTION
-------------------------------------------------------------------------------------------------------------------------------------------------
Athlete ID source column         : aaAthleteId
Result-date source column        : result_date_dt
Numeric-mark source column       : mark_numeric_m
Gender source column             : sex
Implement source column          : implement_weight_g
Event-ID source column           : eventId
Competition-ID source column     : competitionId
Birth-date source column         : DOB
Birth-year source column         : birth_year
Birth-precision source column    : birth_date_precision
Upstream age source column       : age_years  (diagnostic only; not copied)

REPLACEMENT ROW QA
-------------------------------------------------------------------------------------------------------------------------------------------------
Staging rows received             : 4,187
Nonnumeric result rows excluded   : 30
Numeric replacement rows retaine

In [36]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 13 — FINAL PRODUCTION VS REFRESHED REGRESSION / MODEL-POPULATION QA
# ==================================================================================================
#
# Purpose
# -------
# Compare:
#
#       CURRENT PRODUCTION CANONICAL
#
# against:
#
#       FINAL REFRESHED + NORMALIZED CANONICAL
#
#
# We explicitly distinguish:
#
#   1. PHYSICAL PERFORMANCE CHANGES
#   2. METADATA / VOCABULARY CHANGES
#   3. MODEL-ELIGIBILITY CHANGES
#
#
# Hard release gates include:
#
#   • no historical physical performance disappears
#   • no unexpected duplicate increase
#   • no old V4-eligible observation becomes ineligible
#   • no conflicting demographic changes
#   • canonical eligibility flags are internally consistent
#   • schema and row arithmetic remain correct
#
#
# Production is NOT overwritten.
# ==================================================================================================

from __future__ import annotations

import json

import numpy as np
import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "CONFIG",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "EXPECTED_CANONICAL_COLUMNS",

    "CANONICAL_REFRESHED_NORMALIZED_FINAL",
    "CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH",
    "CANONICAL_REFRESHED_NORMALIZED_FINAL_PASS",

    "normalize_wa_id",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Run Notebook 1 through Cell 13D first. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not CANONICAL_REFRESHED_NORMALIZED_FINAL_PASS:

    raise RuntimeError(
        "The final refreshed candidate has not passed its prerequisite QA."
    )


# ==================================================================================================
# PART B — LOAD FRESH PRODUCTION COPY
# ==================================================================================================

PRODUCTION_REGRESSION_BASE = pd.read_parquet(
    CONFIG.canonical_dataset_path
)


REFRESHED_REGRESSION_CANDIDATE = (
    CANONICAL_REFRESHED_NORMALIZED_FINAL.copy()
)


PRODUCTION_REGRESSION_BASE = (
    PRODUCTION_REGRESSION_BASE[
        EXPECTED_CANONICAL_COLUMNS
    ]
    .copy()
)


REFRESHED_REGRESSION_CANDIDATE = (
    REFRESHED_REGRESSION_CANDIDATE[
        EXPECTED_CANONICAL_COLUMNS
    ]
    .copy()
)


for df in [
    PRODUCTION_REGRESSION_BASE,
    REFRESHED_REGRESSION_CANDIDATE,
]:

    df[
        "result_date_dt"
    ] = pd.to_datetime(
        df[
            "result_date_dt"
        ],
        errors="coerce",
    )


    df[
        "canonical_birth_date"
    ] = pd.to_datetime(
        df[
            "canonical_birth_date"
        ],
        errors="coerce",
    )


# ==================================================================================================
# PART C — PHYSICAL PERFORMANCE KEY
# ==================================================================================================
#
# This deliberately ignores:
#
#       implement classification
#       model_group
#       spec_regime
#       eligibility flags
#
# because those are metadata.
#
# It identifies the observed physical performance by:
#
#       athlete
#       date
#       mark
#       WA event ID
#
# ==================================================================================================

def build_physical_key(
    row,
):

    athlete_id = normalize_wa_id(
        row.get(
            "aaAthleteId"
        )
    )


    result_date = pd.to_datetime(
        row.get(
            "result_date_dt"
        ),
        errors="coerce",
    )


    event_id = normalize_wa_id(
        row.get(
            "event_id"
        )
    )


    try:

        mark = float(
            row.get(
                "mark_numeric_m"
            )
        )

    except Exception:

        return None


    if (
        athlete_id is None
        or
        pd.isna(
            result_date
        )
        or
        not np.isfinite(
            mark
        )
    ):

        return None


    return "|".join(
        [
            str(
                int(
                    athlete_id
                )
            ),

            result_date.strftime(
                "%Y-%m-%d"
            ),

            f"{mark:.3f}",

            (
                str(
                    int(
                        event_id
                    )
                )
                if event_id is not None
                else ""
            ),
        ]
    )


# ==================================================================================================
# PART D — ADD MULTISET MATCH KEYS
# ==================================================================================================
#
# cumcount() allows legitimate duplicate physical keys to be compared as a
# multiset rather than collapsing them into one set member.
# ==================================================================================================

def add_regression_keys(
    df,
):

    out = (
        df.copy()
    )


    out[
        "_physical_key"
    ] = (
        out.apply(
            build_physical_key,
            axis=1,
        )
    )


    if out[
        "_physical_key"
    ].isna().any():

        raise RuntimeError(
            "Regression dataset contains rows that cannot form "
            "a physical-performance key."
        )


    out[
        "_physical_occurrence"
    ] = (
        out.groupby(
            "_physical_key",
            dropna=False,
        )
        .cumcount()
    )


    out[
        "_match_key"
    ] = (
        out[
            "_physical_key"
        ]
        +
        "#"
        +
        out[
            "_physical_occurrence"
        ]
        .astype(str)
    )


    return out


PRODUCTION_QA = add_regression_keys(
    PRODUCTION_REGRESSION_BASE
)


REFRESHED_QA = add_regression_keys(
    REFRESHED_REGRESSION_CANDIDATE
)


# ==================================================================================================
# PART E — PHYSICAL PERFORMANCE MULTISET DELTA
# ==================================================================================================

production_counts = (
    PRODUCTION_QA[
        "_physical_key"
    ]
    .value_counts()
    .rename(
        "production_rows"
    )
)


refreshed_counts = (
    REFRESHED_QA[
        "_physical_key"
    ]
    .value_counts()
    .rename(
        "refreshed_rows"
    )
)


PHYSICAL_KEY_DELTA = (
    pd.concat(
        [
            production_counts,
            refreshed_counts,
        ],
        axis=1,
    )
    .fillna(
        0
    )
    .astype(
        int
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "physical_key"
        }
    )
)


PHYSICAL_KEY_DELTA[
    "delta"
] = (
    PHYSICAL_KEY_DELTA[
        "refreshed_rows"
    ]
    -
    PHYSICAL_KEY_DELTA[
        "production_rows"
    ]
)


PHYSICAL_REMOVED_KEYS = (
    PHYSICAL_KEY_DELTA.loc[
        PHYSICAL_KEY_DELTA[
            "delta"
        ]
        < 0
    ]
    .copy()
)


PHYSICAL_ADDED_KEYS = (
    PHYSICAL_KEY_DELTA.loc[
        PHYSICAL_KEY_DELTA[
            "delta"
        ]
        > 0
    ]
    .copy()
)


N_PHYSICAL_REMOVED = int(
    (
        -PHYSICAL_REMOVED_KEYS[
            "delta"
        ]
    )
    .sum()
)


N_PHYSICAL_ADDED = int(
    PHYSICAL_ADDED_KEYS[
        "delta"
    ]
    .sum()
)


# ==================================================================================================
# PART F — MATCH ALL PRE-EXISTING PHYSICAL OBSERVATIONS
# ==================================================================================================

MATCHED_OLD_ROWS = (
    PRODUCTION_QA.merge(
        REFRESHED_QA,

        on=
            "_match_key",

        how=
            "inner",

        suffixes=(
            "_old",
            "_new",
        ),

        validate=
            "one_to_one",
    )
)


N_OLD_ROWS_MATCHED = int(
    len(
        MATCHED_OLD_ROWS
    )
)


# ==================================================================================================
# PART G — IDENTIFY TRULY NEW PHYSICAL OBSERVATIONS
# ==================================================================================================

production_match_keys = set(
    PRODUCTION_QA[
        "_match_key"
    ]
)


NEW_PHYSICAL_ROWS = (
    REFRESHED_QA.loc[
        ~REFRESHED_QA[
            "_match_key"
        ]
        .isin(
            production_match_keys
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART H — OLD-ROW V4 ELIGIBILITY TRANSITIONS
# ==================================================================================================

old_v4 = (
    MATCHED_OLD_ROWS[
        "v4_base_eligible_old"
    ]
    .fillna(
        False
    )
    .astype(
        bool
    )
)


new_v4 = (
    MATCHED_OLD_ROWS[
        "v4_base_eligible_new"
    ]
    .fillna(
        False
    )
    .astype(
        bool
    )
)


MATCHED_OLD_ROWS[
    "_v4_transition"
] = np.select(
    [
        old_v4
        &
        new_v4,

        old_v4
        &
        ~new_v4,

        ~old_v4
        &
        new_v4,
    ],

    [
        "ELIGIBLE_UNCHANGED",
        "ELIGIBLE_LOST",
        "ELIGIBLE_GAINED",
    ],

    default=
        "INELIGIBLE_UNCHANGED",
)


OLD_V4_LOST = (
    MATCHED_OLD_ROWS.loc[
        MATCHED_OLD_ROWS[
            "_v4_transition"
        ]
        == "ELIGIBLE_LOST"
    ]
    .copy()
)


OLD_V4_GAINED = (
    MATCHED_OLD_ROWS.loc[
        MATCHED_OLD_ROWS[
            "_v4_transition"
        ]
        == "ELIGIBLE_GAINED"
    ]
    .copy()
)


N_OLD_V4_LOST = int(
    len(
        OLD_V4_LOST
    )
)


N_OLD_V4_GAINED = int(
    len(
        OLD_V4_GAINED
    )
)


# ==================================================================================================
# PART I — METADATA CHANGE FLAGS ON OLD PHYSICAL OBSERVATIONS
# ==================================================================================================

def nullable_text_changed(
    old_series,
    new_series,
):

    return (
        old_series
        .fillna(
            "<NULL>"
        )
        .astype(str)

        !=

        new_series
        .fillna(
            "<NULL>"
        )
        .astype(str)
    )


def nullable_numeric_changed(
    old_series,
    new_series,
    atol=1e-10,
):

    old_num = pd.to_numeric(
        old_series,
        errors="coerce",
    )


    new_num = pd.to_numeric(
        new_series,
        errors="coerce",
    )


    both_null = (
        old_num.isna()
        &
        new_num.isna()
    )


    one_null = (
        old_num.isna()
        ^
        new_num.isna()
    )


    both_present_different = (
        old_num.notna()
        &
        new_num.notna()
        &
        ~np.isclose(
            old_num,
            new_num,
            rtol=0.0,
            atol=atol,
            equal_nan=True,
        )
    )


    return (
        ~both_null
        &
        (
            one_null
            |
            both_present_different
        )
    )


MATCHED_OLD_ROWS[
    "_implement_changed"
] = nullable_numeric_changed(
    MATCHED_OLD_ROWS[
        "implement_weight_g_old"
    ],
    MATCHED_OLD_ROWS[
        "implement_weight_g_new"
    ],
)


MATCHED_OLD_ROWS[
    "_model_group_changed"
] = nullable_text_changed(
    MATCHED_OLD_ROWS[
        "model_group_old"
    ],
    MATCHED_OLD_ROWS[
        "model_group_new"
    ],
)


MATCHED_OLD_ROWS[
    "_spec_regime_changed"
] = nullable_text_changed(
    MATCHED_OLD_ROWS[
        "spec_regime_old"
    ],
    MATCHED_OLD_ROWS[
        "spec_regime_new"
    ],
)


MATCHED_OLD_ROWS[
    "_spec_eligibility_changed"
] = (
    MATCHED_OLD_ROWS[
        "spec_model_eligible_old"
    ]
    .fillna(False)
    .astype(bool)

    !=

    MATCHED_OLD_ROWS[
        "spec_model_eligible_new"
    ]
    .fillna(False)
    .astype(bool)
)


MATCHED_OLD_ROWS[
    "_core_eligibility_changed"
] = (
    MATCHED_OLD_ROWS[
        "core_group_eligible_old"
    ]
    .fillna(False)
    .astype(bool)

    !=

    MATCHED_OLD_ROWS[
        "core_group_eligible_new"
    ]
    .fillna(False)
    .astype(bool)
)


# ==================================================================================================
# PART J — DEMOGRAPHIC CONFLICT CHECKS
# ==================================================================================================
#
# Filling previously missing metadata is allowed.
#
# A conflicting non-null -> different non-null value is not.
# ==================================================================================================

old_gender = (
    MATCHED_OLD_ROWS[
        "gender_raw_old"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


new_gender = (
    MATCHED_OLD_ROWS[
        "gender_raw_new"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


GENDER_CONFLICT_MASK = (
    old_gender.ne("")
    &
    new_gender.ne("")
    &
    old_gender.ne(
        new_gender
    )
)


old_birth_date = pd.to_datetime(
    MATCHED_OLD_ROWS[
        "canonical_birth_date_old"
    ],
    errors="coerce",
)


new_birth_date = pd.to_datetime(
    MATCHED_OLD_ROWS[
        "canonical_birth_date_new"
    ],
    errors="coerce",
)


BIRTH_DATE_CONFLICT_MASK = (
    old_birth_date.notna()
    &
    new_birth_date.notna()
    &
    old_birth_date.ne(
        new_birth_date
    )
)


old_birth_year = pd.to_numeric(
    MATCHED_OLD_ROWS[
        "canonical_birth_year_old"
    ],
    errors="coerce",
)


new_birth_year = pd.to_numeric(
    MATCHED_OLD_ROWS[
        "canonical_birth_year_new"
    ],
    errors="coerce",
)


BIRTH_YEAR_CONFLICT_MASK = (
    old_birth_year.notna()
    &
    new_birth_year.notna()
    &
    old_birth_year.ne(
        new_birth_year
    )
)


old_age = pd.to_numeric(
    MATCHED_OLD_ROWS[
        "age_years_old"
    ],
    errors="coerce",
)


new_age = pd.to_numeric(
    MATCHED_OLD_ROWS[
        "age_years_new"
    ],
    errors="coerce",
)


AGE_CONFLICT_MASK = (
    (
        old_age.isna()
        ^
        new_age.isna()
    )
    |
    (
        old_age.notna()
        &
        new_age.notna()
        &
        (
            np.abs(
                old_age
                -
                new_age
            )
            >
            1e-10
        )
    )
)


N_GENDER_CONFLICTS = int(
    GENDER_CONFLICT_MASK.sum()
)


N_BIRTH_DATE_CONFLICTS = int(
    BIRTH_DATE_CONFLICT_MASK.sum()
)


N_BIRTH_YEAR_CONFLICTS = int(
    BIRTH_YEAR_CONFLICT_MASK.sum()
)


N_AGE_CONFLICTS = int(
    AGE_CONFLICT_MASK.sum()
)


# ==================================================================================================
# PART K — METADATA COMPLETION COUNTS
# ==================================================================================================

N_BIRTH_DATE_FILLED = int(
    (
        old_birth_date.isna()
        &
        new_birth_date.notna()
    )
    .sum()
)


N_BIRTH_YEAR_FILLED = int(
    (
        old_birth_year.isna()
        &
        new_birth_year.notna()
    )
    .sum()
)


N_AGE_FILLED = int(
    (
        old_age.isna()
        &
        new_age.notna()
    )
    .sum()
)


# ==================================================================================================
# PART L — INTERNAL ELIGIBILITY CONSISTENCY
# ==================================================================================================

CORE_GROUPS = {
    "M_700",
    "M_800",
    "W_500",
    "W_600",
}


EXPECTED_CORE_FLAG = (
    REFRESHED_QA[
        "model_group"
    ]
    .isin(
        CORE_GROUPS
    )
)


EXPECTED_AGE_FLAG = (
    REFRESHED_QA[
        "age_years"
    ]
    .notna()
)


EXPECTED_V4_FLAG = (
    REFRESHED_QA[
        "spec_model_eligible"
    ]
    .fillna(False)
    .astype(bool)

    &
    EXPECTED_CORE_FLAG

    &
    EXPECTED_AGE_FLAG
)


CORE_FLAG_MISMATCH = (
    REFRESHED_QA[
        "core_group_eligible"
    ]
    .fillna(False)
    .astype(bool)

    !=

    EXPECTED_CORE_FLAG
)


AGE_FLAG_MISMATCH = (
    REFRESHED_QA[
        "age_available"
    ]
    .fillna(False)
    .astype(bool)

    !=

    EXPECTED_AGE_FLAG
)


V4_FLAG_MISMATCH = (
    REFRESHED_QA[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)

    !=

    EXPECTED_V4_FLAG
)


N_CORE_FLAG_MISMATCH = int(
    CORE_FLAG_MISMATCH.sum()
)


N_AGE_FLAG_MISMATCH = int(
    AGE_FLAG_MISMATCH.sum()
)


N_V4_FLAG_MISMATCH = int(
    V4_FLAG_MISMATCH.sum()
)


# ==================================================================================================
# PART M — MODEL GROUP / SEX / WEIGHT CONSISTENCY
# ==================================================================================================

def expected_group_from_row(
    row,
):

    gender = row.get(
        "gender_raw"
    )


    if gender is None:

        return None


    try:

        if pd.isna(
            gender
        ):

            return None

    except Exception:

        pass


    gender = str(
        gender
    ).strip().upper()


    if gender in {
        "MALE",
        "MEN",
    }:

        gender = "M"


    elif gender in {
        "F",
        "FEMALE",
        "WOMEN",
    }:

        gender = "W"


    if gender not in {
        "M",
        "W",
    }:

        return None


    weight = pd.to_numeric(
        pd.Series(
            [
                row.get(
                    "implement_weight_g"
                )
            ]
        ),
        errors="coerce",
    ).iloc[
        0
    ]


    if pd.isna(
        weight
    ):

        return None


    return (
        f"{gender}_{int(round(float(weight)))}"
    )


REFRESHED_QA[
    "_expected_model_group"
] = (
    REFRESHED_QA.apply(
        expected_group_from_row,
        axis=1,
    )
)


MODEL_GROUP_CONFLICT_MASK = (
    REFRESHED_QA[
        "model_group"
    ]
    .notna()
    &
    REFRESHED_QA[
        "_expected_model_group"
    ]
    .notna()
    &
    (
        REFRESHED_QA[
            "model_group"
        ]
        .astype(str)

        !=

        REFRESHED_QA[
            "_expected_model_group"
        ]
        .astype(str)
    )
)


N_MODEL_GROUP_CONFLICTS = int(
    MODEL_GROUP_CONFLICT_MASK.sum()
)


# ==================================================================================================
# PART N — DUPLICATE REGRESSION
# ==================================================================================================

BASE_DUPLICATE_PHYSICAL_ROWS = int(
    PRODUCTION_QA[
        "_physical_key"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)


REFRESHED_DUPLICATE_PHYSICAL_ROWS = int(
    REFRESHED_QA[
        "_physical_key"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)


NEW_DUPLICATE_PHYSICAL_INCREASE = max(
    0,
    (
        REFRESHED_DUPLICATE_PHYSICAL_ROWS
        -
        BASE_DUPLICATE_PHYSICAL_ROWS
    ),
)


# ==================================================================================================
# PART O — POPULATION COUNTS
# ==================================================================================================

N_PRODUCTION_ROWS = int(
    len(
        PRODUCTION_QA
    )
)


N_REFRESHED_ROWS = int(
    len(
        REFRESHED_QA
    )
)


N_ROW_DELTA = (
    N_REFRESHED_ROWS
    -
    N_PRODUCTION_ROWS
)


N_PRODUCTION_ATHLETES = int(
    PRODUCTION_QA[
        "aaAthleteId"
    ]
    .nunique()
)


N_REFRESHED_ATHLETES = int(
    REFRESHED_QA[
        "aaAthleteId"
    ]
    .nunique()
)


PRODUCTION_ATHLETE_IDS = set(
    PRODUCTION_QA[
        "aaAthleteId"
    ]
    .dropna()
    .map(
        normalize_wa_id
    )
)


REFRESHED_ATHLETE_IDS = set(
    REFRESHED_QA[
        "aaAthleteId"
    ]
    .dropna()
    .map(
        normalize_wa_id
    )
)


NEW_ATHLETE_IDS = (
    REFRESHED_ATHLETE_IDS
    -
    PRODUCTION_ATHLETE_IDS
)


N_NEW_ATHLETES = int(
    len(
        NEW_ATHLETE_IDS
    )
)


N_PRODUCTION_V4 = int(
    PRODUCTION_QA[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


N_REFRESHED_V4 = int(
    REFRESHED_QA[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


N_NEW_PHYSICAL_V4 = int(
    NEW_PHYSICAL_ROWS[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


# ==================================================================================================
# PART P — MODEL POPULATION BY GROUP
# ==================================================================================================

def model_population_summary(
    df,
    label,
):

    work = (
        df.copy()
    )


    return (
        work.groupby(
            "model_group",
            dropna=False,
            as_index=False,
        )
        .agg(
            rows=(
                "aaAthleteId",
                "size",
            ),

            athletes=(
                "aaAthleteId",
                "nunique",
            ),

            spec_eligible=(
                "spec_model_eligible",
                lambda s:
                    int(
                        s.fillna(False)
                        .astype(bool)
                        .sum()
                    ),
            ),

            core_eligible=(
                "core_group_eligible",
                lambda s:
                    int(
                        s.fillna(False)
                        .astype(bool)
                        .sum()
                    ),
            ),

            age_available=(
                "age_available",
                lambda s:
                    int(
                        s.fillna(False)
                        .astype(bool)
                        .sum()
                    ),
            ),

            v4_eligible=(
                "v4_base_eligible",
                lambda s:
                    int(
                        s.fillna(False)
                        .astype(bool)
                        .sum()
                    ),
            ),
        )
        .assign(
            dataset=
                label
        )
    )


MODEL_POPULATION_COMPARISON = pd.concat(
    [
        model_population_summary(
            PRODUCTION_QA,
            "PRODUCTION",
        ),

        model_population_summary(
            REFRESHED_QA,
            "REFRESHED",
        ),
    ],

    ignore_index=True,
)


# ==================================================================================================
# PART Q — V4 POPULATION DELTA BY YEAR / GROUP
# ==================================================================================================

def v4_year_group_counts(
    df,
    label,
):

    work = (
        df.loc[
            df[
                "v4_base_eligible"
            ]
            .fillna(False)
            .astype(bool)
        ]
        .copy()
    )


    work[
        "year"
    ] = (
        work[
            "result_date_dt"
        ]
        .dt.year
    )


    out = (
        work.groupby(
            [
                "year",
                "model_group",
            ],
            dropna=False,
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size":
                    label
            }
        )
    )


    return out


V4_PRODUCTION_BY_YEAR = (
    v4_year_group_counts(
        PRODUCTION_QA,
        "production_v4",
    )
)


V4_REFRESHED_BY_YEAR = (
    v4_year_group_counts(
        REFRESHED_QA,
        "refreshed_v4",
    )
)


V4_YEAR_GROUP_DELTA = (
    V4_PRODUCTION_BY_YEAR.merge(
        V4_REFRESHED_BY_YEAR,

        on=[
            "year",
            "model_group",
        ],

        how="outer",
    )
    .fillna(
        {
            "production_v4":
                0,

            "refreshed_v4":
                0,
        }
    )
)


V4_YEAR_GROUP_DELTA[
    "production_v4"
] = (
    V4_YEAR_GROUP_DELTA[
        "production_v4"
    ]
    .astype(int)
)


V4_YEAR_GROUP_DELTA[
    "refreshed_v4"
] = (
    V4_YEAR_GROUP_DELTA[
        "refreshed_v4"
    ]
    .astype(int)
)


V4_YEAR_GROUP_DELTA[
    "delta"
] = (
    V4_YEAR_GROUP_DELTA[
        "refreshed_v4"
    ]
    -
    V4_YEAR_GROUP_DELTA[
        "production_v4"
    ]
)


V4_CHANGED_YEAR_GROUPS = (
    V4_YEAR_GROUP_DELTA.loc[
        V4_YEAR_GROUP_DELTA[
            "delta"
        ]
        != 0
    ]
    .sort_values(
        [
            "year",
            "model_group",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# PART R — NEW ROW SUMMARY
# ==================================================================================================

NEW_PHYSICAL_ROWS[
    "result_year"
] = (
    NEW_PHYSICAL_ROWS[
        "result_date_dt"
    ]
    .dt.year
)


NEW_ROW_SUMMARY = (
    NEW_PHYSICAL_ROWS.groupby(
        [
            "result_year",
            "model_group",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        rows=(
            "aaAthleteId",
            "size",
        ),

        athletes=(
            "aaAthleteId",
            "nunique",
        ),

        v4_eligible=(
            "v4_base_eligible",
            lambda s:
                int(
                    s.fillna(False)
                    .astype(bool)
                    .sum()
                ),
        ),
    )
    .sort_values(
        [
            "result_year",
            "model_group",
        ]
    )
)


# ==================================================================================================
# PART S — SCHEMA / DATE / ROW QA
# ==================================================================================================

SCHEMA_PASS = bool(
    list(
        REFRESHED_QA[
            EXPECTED_CANONICAL_COLUMNS
        ].columns
    )
    ==
    EXPECTED_CANONICAL_COLUMNS
)


ROW_DELTA_PASS = bool(
    N_ROW_DELTA
    ==
    N_PHYSICAL_ADDED
    -
    N_PHYSICAL_REMOVED
)


DATE_MAX_ADVANCED = bool(
    REFRESHED_QA[
        "result_date_dt"
    ].max()
    >=
    PRODUCTION_QA[
        "result_date_dt"
    ].max()
)


# ==================================================================================================
# PART T — RELEASE GATE
# ==================================================================================================

CELL_13_PASS = bool(

    # No existing physical result disappeared.
    N_PHYSICAL_REMOVED == 0

    # Every production physical row remains represented.
    and
    N_OLD_ROWS_MATCHED
    ==
    N_PRODUCTION_ROWS

    # No previous model-eligible result was lost.
    and
    N_OLD_V4_LOST == 0

    # No contradictory demographics were introduced.
    and
    N_GENDER_CONFLICTS == 0

    and
    N_BIRTH_DATE_CONFLICTS == 0

    and
    N_BIRTH_YEAR_CONFLICTS == 0

    and
    N_AGE_CONFLICTS == 0

    # Eligibility flags agree with their definitions.
    and
    N_CORE_FLAG_MISMATCH == 0

    and
    N_AGE_FLAG_MISMATCH == 0

    and
    N_V4_FLAG_MISMATCH == 0

    # Sex / implement / group are internally coherent.
    and
    N_MODEL_GROUP_CONFLICTS == 0

    # No duplicate inflation.
    and
    NEW_DUPLICATE_PHYSICAL_INCREASE == 0

    # Structural checks.
    and
    SCHEMA_PASS

    and
    ROW_DELTA_PASS

    and
    DATE_MAX_ADVANCED
)


# ==================================================================================================
# PART U — AUDIT OUTPUTS
# ==================================================================================================

REGRESSION_SUMMARY_PATH = (
    RUN_CHECKPOINT_DIR
    / "final_release_regression_summary.parquet"
)


NEW_PHYSICAL_ROWS_PATH = (
    RUN_CHECKPOINT_DIR
    / "final_release_new_physical_rows.parquet"
)


OLD_V4_GAINED_PATH = (
    RUN_CHECKPOINT_DIR
    / "final_release_old_rows_newly_v4_eligible.parquet"
)


OLD_V4_LOST_PATH = (
    RUN_CHECKPOINT_DIR
    / "final_release_old_rows_lost_v4_eligibility.parquet"
)


V4_YEAR_GROUP_DELTA_PATH = (
    RUN_CHECKPOINT_DIR
    / "final_release_v4_year_group_delta.parquet"
)


PHYSICAL_KEY_DELTA_PATH = (
    RUN_CHECKPOINT_DIR
    / "final_release_physical_key_delta.parquet"
)


REGRESSION_SUMMARY = pd.DataFrame(
    [
        {
            "production_rows":
                N_PRODUCTION_ROWS,

            "refreshed_rows":
                N_REFRESHED_ROWS,

            "row_delta":
                N_ROW_DELTA,

            "physical_added":
                N_PHYSICAL_ADDED,

            "physical_removed":
                N_PHYSICAL_REMOVED,

            "production_athletes":
                N_PRODUCTION_ATHLETES,

            "refreshed_athletes":
                N_REFRESHED_ATHLETES,

            "new_athletes":
                N_NEW_ATHLETES,

            "production_v4":
                N_PRODUCTION_V4,

            "refreshed_v4":
                N_REFRESHED_V4,

            "old_v4_gained":
                N_OLD_V4_GAINED,

            "old_v4_lost":
                N_OLD_V4_LOST,

            "new_physical_v4":
                N_NEW_PHYSICAL_V4,

            "gender_conflicts":
                N_GENDER_CONFLICTS,

            "birth_date_conflicts":
                N_BIRTH_DATE_CONFLICTS,

            "birth_year_conflicts":
                N_BIRTH_YEAR_CONFLICTS,

            "age_conflicts":
                N_AGE_CONFLICTS,

            "core_flag_mismatches":
                N_CORE_FLAG_MISMATCH,

            "age_flag_mismatches":
                N_AGE_FLAG_MISMATCH,

            "v4_flag_mismatches":
                N_V4_FLAG_MISMATCH,

            "model_group_conflicts":
                N_MODEL_GROUP_CONFLICTS,

            "base_duplicate_physical_rows":
                BASE_DUPLICATE_PHYSICAL_ROWS,

            "refreshed_duplicate_physical_rows":
                REFRESHED_DUPLICATE_PHYSICAL_ROWS,

            "new_duplicate_increase":
                NEW_DUPLICATE_PHYSICAL_INCREASE,

            "release_pass":
                CELL_13_PASS,
        }
    ]
)


REGRESSION_SUMMARY.to_parquet(
    REGRESSION_SUMMARY_PATH,
    index=False,
)


NEW_PHYSICAL_ROWS[
    EXPECTED_CANONICAL_COLUMNS
].to_parquet(
    NEW_PHYSICAL_ROWS_PATH,
    index=False,
)


OLD_V4_GAINED.to_parquet(
    OLD_V4_GAINED_PATH,
    index=False,
)


OLD_V4_LOST.to_parquet(
    OLD_V4_LOST_PATH,
    index=False,
)


V4_YEAR_GROUP_DELTA.to_parquet(
    V4_YEAR_GROUP_DELTA_PATH,
    index=False,
)


PHYSICAL_KEY_DELTA.to_parquet(
    PHYSICAL_KEY_DELTA_PATH,
    index=False,
)


# ==================================================================================================
# PART V — MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "final_release_regression"
] = {
    "status":
        (
            "PASSED"
            if CELL_13_PASS
            else "REVIEW_REQUIRED"
        ),

    "production_rows":
        N_PRODUCTION_ROWS,

    "refreshed_rows":
        N_REFRESHED_ROWS,

    "row_delta":
        N_ROW_DELTA,

    "physical_added":
        N_PHYSICAL_ADDED,

    "physical_removed":
        N_PHYSICAL_REMOVED,

    "old_rows_matched":
        N_OLD_ROWS_MATCHED,

    "new_athletes":
        N_NEW_ATHLETES,

    "production_v4_eligible":
        N_PRODUCTION_V4,

    "refreshed_v4_eligible":
        N_REFRESHED_V4,

    "old_v4_gained":
        N_OLD_V4_GAINED,

    "old_v4_lost":
        N_OLD_V4_LOST,

    "new_physical_v4_eligible":
        N_NEW_PHYSICAL_V4,

    "demographic_conflicts": {
        "gender":
            N_GENDER_CONFLICTS,

        "birth_date":
            N_BIRTH_DATE_CONFLICTS,

        "birth_year":
            N_BIRTH_YEAR_CONFLICTS,

        "age":
            N_AGE_CONFLICTS,
    },

    "eligibility_flag_mismatches": {
        "core_group":
            N_CORE_FLAG_MISMATCH,

        "age_available":
            N_AGE_FLAG_MISMATCH,

        "v4_base":
            N_V4_FLAG_MISMATCH,
    },

    "model_group_conflicts":
        N_MODEL_GROUP_CONFLICTS,

    "new_duplicate_increase":
        NEW_DUPLICATE_PHYSICAL_INCREASE,

    "refreshed_dataset_path":
        str(
            CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH
        ),

    "regression_summary_path":
        str(
            REGRESSION_SUMMARY_PATH
        ),
}


RUN_MANIFEST[
    "status"
] = (
    "CANONICAL_RELEASE_VALIDATED"

    if CELL_13_PASS

    else "CANONICAL_RELEASE_REVIEW_REQUIRED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART W — OUTPUT
# ==================================================================================================

print()
print("=" * 140)
print("WORLD ATHLETICS — FINAL PRODUCTION VS REFRESHED RELEASE REGRESSION")
print("=" * 140)


print()
print("DATASET")
print("-" * 140)

print(
    f"Production rows                  : "
    f"{N_PRODUCTION_ROWS:,}"
)

print(
    f"Refreshed rows                   : "
    f"{N_REFRESHED_ROWS:,}"
)

print(
    f"Net row delta                    : "
    f"{N_ROW_DELTA:+,}"
)

print(
    f"Production athletes              : "
    f"{N_PRODUCTION_ATHLETES:,}"
)

print(
    f"Refreshed athletes               : "
    f"{N_REFRESHED_ATHLETES:,}"
)

print(
    f"New athlete IDs                  : "
    f"{N_NEW_ATHLETES:,}"
)

print(
    f"Production max date              : "
    f"{PRODUCTION_QA['result_date_dt'].max().date()}"
)

print(
    f"Refreshed max date               : "
    f"{REFRESHED_QA['result_date_dt'].max().date()}"
)


print()
print("PHYSICAL PERFORMANCE REGRESSION")
print("-" * 140)

print(
    f"Old physical rows matched        : "
    f"{N_OLD_ROWS_MATCHED:,} / {N_PRODUCTION_ROWS:,}"
)

print(
    f"Physical performances added      : "
    f"{N_PHYSICAL_ADDED:,}"
)

print(
    f"Physical performances removed    : "
    f"{N_PHYSICAL_REMOVED:,}"
)

print(
    f"Row arithmetic consistent        : "
    f"{ROW_DELTA_PASS}"
)


print()
print("MODEL POPULATION")
print("-" * 140)

print(
    f"Production V4 eligible           : "
    f"{N_PRODUCTION_V4:,}"
)

print(
    f"Refreshed V4 eligible            : "
    f"{N_REFRESHED_V4:,}"
)

print(
    f"Existing rows newly V4 eligible  : "
    f"{N_OLD_V4_GAINED:,}"
)

print(
    f"Existing rows losing eligibility : "
    f"{N_OLD_V4_LOST:,}"
)

print(
    f"New physical rows V4 eligible    : "
    f"{N_NEW_PHYSICAL_V4:,}"
)


print()
print("METADATA COMPLETION / CONFLICTS")
print("-" * 140)

print(
    f"Birth dates newly filled         : "
    f"{N_BIRTH_DATE_FILLED:,}"
)

print(
    f"Birth years newly filled         : "
    f"{N_BIRTH_YEAR_FILLED:,}"
)

print(
    f"Exact ages newly filled          : "
    f"{N_AGE_FILLED:,}"
)

print(
    f"Gender conflicts                 : "
    f"{N_GENDER_CONFLICTS:,}"
)

print(
    f"Birth-date conflicts             : "
    f"{N_BIRTH_DATE_CONFLICTS:,}"
)

print(
    f"Birth-year conflicts             : "
    f"{N_BIRTH_YEAR_CONFLICTS:,}"
)

print(
    f"Age conflicts                    : "
    f"{N_AGE_CONFLICTS:,}"
)


print()
print("INTERNAL ELIGIBILITY QA")
print("-" * 140)

print(
    f"Core-group flag mismatches       : "
    f"{N_CORE_FLAG_MISMATCH:,}"
)

print(
    f"Age-available flag mismatches    : "
    f"{N_AGE_FLAG_MISMATCH:,}"
)

print(
    f"V4-base flag mismatches          : "
    f"{N_V4_FLAG_MISMATCH:,}"
)

print(
    f"Model-group conflicts            : "
    f"{N_MODEL_GROUP_CONFLICTS:,}"
)


print()
print("DUPLICATE REGRESSION")
print("-" * 140)

print(
    f"Production duplicate rows        : "
    f"{BASE_DUPLICATE_PHYSICAL_ROWS:,}"
)

print(
    f"Refreshed duplicate rows         : "
    f"{REFRESHED_DUPLICATE_PHYSICAL_ROWS:,}"
)

print(
    f"New duplicate increase           : "
    f"{NEW_DUPLICATE_PHYSICAL_INCREASE:,}"
)


print()
print("MODEL POPULATION BY GROUP")
print("-" * 140)

display(
    MODEL_POPULATION_COMPARISON
)


print()
print("V4 YEAR/GROUP CHANGES")
print("-" * 140)

if len(
    V4_CHANGED_YEAR_GROUPS
) > 0:

    display(
        V4_CHANGED_YEAR_GROUPS
    )

else:

    print(
        "No V4 population changes."
    )


print()
print("NEW PHYSICAL ROWS BY YEAR/GROUP")
print("-" * 140)

display(
    NEW_ROW_SUMMARY
)


print()
print("EXISTING ROWS NEWLY V4 ELIGIBLE")
print("-" * 140)

if N_OLD_V4_GAINED > 0:

    columns_to_show = [
        column
        for column in [
            "aaAthleteId_old",
            "result_date_dt_old",
            "mark_numeric_m_old",
            "event_id_old",
            "implement_weight_g_old",
            "implement_weight_g_new",
            "model_group_old",
            "model_group_new",
            "spec_regime_old",
            "spec_regime_new",
            "v4_base_eligible_old",
            "v4_base_eligible_new",
        ]
        if column in OLD_V4_GAINED.columns
    ]


    display(
        OLD_V4_GAINED[
            columns_to_show
        ]
        .head(
            200
        )
    )


else:

    print(
        "None."
    )


if N_OLD_V4_LOST > 0:

    print()
    print("WARNING — EXISTING ROWS THAT LOST V4 ELIGIBILITY")
    print("-" * 140)

    display(
        OLD_V4_LOST.head(
            200
        )
    )


print()
print("OUTPUT")
print("-" * 140)

print(
    f"Final refreshed dataset          : "
    f"{CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH}"
)

print(
    f"Regression summary               : "
    f"{REGRESSION_SUMMARY_PATH}"
)

print(
    f"New physical rows                : "
    f"{NEW_PHYSICAL_ROWS_PATH}"
)

print(
    f"V4 population delta              : "
    f"{V4_YEAR_GROUP_DELTA_PATH}"
)


print()
print("=" * 140)


if CELL_13_PASS:

    print(
        "CELL 13 STATUS: FINAL RELEASE REGRESSION PASSED"
    )

    print()
    print(
        "The refreshed normalized canonical dataset is eligible for promotion."
    )

    print(
        "The current production parquet has still NOT been overwritten."
    )


else:

    print(
        "CELL 13 STATUS: RELEASE REVIEW REQUIRED"
    )

    print()
    print(
        "Do not promote the refreshed dataset yet."
    )


print("=" * 140)


WORLD ATHLETICS — FINAL PRODUCTION VS REFRESHED RELEASE REGRESSION

DATASET
--------------------------------------------------------------------------------------------------------------------------------------------
Production rows                  : 248,896
Refreshed rows                   : 249,233
Net row delta                    : +337
Production athletes              : 21,828
Refreshed athletes               : 21,901
New athlete IDs                  : 73
Production max date              : 2026-09-13
Refreshed max date               : 2026-09-18

PHYSICAL PERFORMANCE REGRESSION
--------------------------------------------------------------------------------------------------------------------------------------------
Old physical rows matched        : 248,896 / 248,896
Physical performances added      : 337
Physical performances removed    : 0
Row arithmetic consistent        : True

MODEL POPULATION
---------------------------------------------------------------------------------

,model_group,rows,athletes,spec_eligible,core_eligible,age_available,v4_eligible,dataset
0,M_700,13945,4189,13894,13945,11603,11554,PRODUCTION
1,M_800,98858,9668,98857,98858,85651,85650,PRODUCTION
2,W_500,22088,5000,21987,22088,18819,18728,PRODUCTION
3,W_600,89082,8573,89063,89082,76789,76770,PRODUCTION
4,NaN,24923,5627,1219,0,21495,0,PRODUCTION
5,M_500,98,29,98,0,96,0,REFRESHED
6,M_600,617,237,617,0,581,0,REFRESHED
7,M_700,13973,4209,13922,13973,11629,11580,REFRESHED
8,M_800,99042,9687,99041,99042,85806,85805,REFRESHED
9,W_400,486,191,486,0,387,0,REFRESHED



V4 YEAR/GROUP CHANGES
--------------------------------------------------------------------------------------------------------------------------------------------


,year,model_group,production_v4,refreshed_v4,delta
0,2003,M_800,194,197,3
1,2004,M_800,192,194,2
2,2005,M_800,264,267,3
3,2006,M_800,351,356,5
4,2007,M_800,506,510,4
5,2008,M_800,506,510,4
6,2009,M_800,810,814,4
7,2010,M_800,1149,1155,6
8,2011,M_800,1343,1347,4
9,2012,M_800,1725,1741,16



NEW PHYSICAL ROWS BY YEAR/GROUP
--------------------------------------------------------------------------------------------------------------------------------------------


,result_year,model_group,rows,athletes,v4_eligible
0,2003,M_800,3,1,3
1,2004,M_800,2,1,2
2,2005,M_800,3,1,3
3,2006,M_800,5,1,5
4,2007,M_800,4,1,4
5,2008,M_800,4,1,4
6,2009,M_800,4,2,4
7,2010,M_800,6,1,6
8,2011,M_800,4,1,4
9,2012,M_800,16,3,16



EXISTING ROWS NEWLY V4 ELIGIBLE
--------------------------------------------------------------------------------------------------------------------------------------------


,aaAthleteId_old,result_date_dt_old,mark_numeric_m_old,event_id_old,implement_weight_g_old,implement_weight_g_new,model_group_old,model_group_new,spec_regime_old,spec_regime_new,v4_base_eligible_old,v4_base_eligible_new
69777,14596056,2025-08-18,64.91,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
85065,14671060,2025-08-18,79.49,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
85456,14672707,2025-09-28,52.48,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
95817,14715687,2025-08-18,66.19,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
96340,14717107,2025-09-28,48.69,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
125088,14813121,2025-09-28,58.22,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
125420,14813693,2025-10-11,74.26,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
126524,14817920,2025-09-28,57.1,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
133566,14843014,2025-08-18,63.64,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True
133603,14843032,2025-08-18,69.34,10229685,NaN,800.0,None,M_800,M_JAVELIN_<NA>_CURRENT_SPEC,M_JAVELIN_800_CURRENT_SPEC,False,True



OUTPUT
--------------------------------------------------------------------------------------------------------------------------------------------
Final refreshed dataset          : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/final/javelin_reconciled_base_1986_2026_refreshed_20260919_225312_candidate.parquet
Regression summary               : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/final_release_regression_summary.parquet
New physical rows                : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/final_release_new_physical_rows.parquet
V4 population delta              : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/incremental/checkpoints/20260919_225312/final_release_v4_year_group_delta.parquet

CELL 13 STATUS: FINAL RELEASE REGRESSION PASSED

The refreshed normalized canonical dataset is eligible

In [37]:
# ==================================================================================================
# NOTEBOOK 1 — WORLD ATHLETICS INCREMENTAL UPDATE
#
# CELL 14 — CONTROLLED CANONICAL PRODUCTION PROMOTION
# ==================================================================================================
#
# Purpose
# -------
# Promote the fully validated canonical candidate to the stable production path.
#
#
# REQUIRED:
#
#       CELL_12_PASS == True
#       CELL_13_PASS == True
#
#
# SAFETY DESIGN
# -------------
#
# 1. Refuses to run unless both release gates passed.
#
# 2. Computes SHA256 hashes for production and candidate.
#
# 3. Creates a versioned backup of the current production parquet.
#
# 4. Copies the candidate to a temporary file in the SAME directory.
#
# 5. Validates the temporary copy before touching production.
#
# 6. Uses os.replace() for an atomic filesystem replacement.
#
# 7. Reloads production and verifies:
#
#       • file hash
#       • row count
#       • schema
#       • athlete count
#       • max result date
#       • V4 population
#
# 8. Writes a release manifest.
#
# 9. Is idempotent:
#
#       if production already has the candidate SHA256,
#       no second promotion occurs.
#
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# ==================================================================================================
# PART A — PREREQUISITES
# ==================================================================================================

required_globals = [
    "CONFIG",
    "RUN_ID",
    "RUN_CHECKPOINT_DIR",
    "RUN_MANIFEST",
    "RUN_MANIFEST_PATH",

    "EXPECTED_CANONICAL_COLUMNS",

    "CANONICAL_REFRESHED_NORMALIZED_FINAL",
    "CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH",
    "CANONICAL_REFRESHED_NORMALIZED_FINAL_PASS",

    "CELL_12_PASS",
    "CELL_13_PASS",
]


missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]


if missing_globals:

    raise RuntimeError(
        "Missing required notebook state:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_globals
        )
    )


if not CELL_12_PASS:

    raise RuntimeError(
        "Cell 12 did not pass. Production promotion is blocked."
    )


if not CELL_13_PASS:

    raise RuntimeError(
        "Cell 13 did not pass. Production promotion is blocked."
    )


if not CANONICAL_REFRESHED_NORMALIZED_FINAL_PASS:

    raise RuntimeError(
        "Canonical candidate prerequisite flag is False. "
        "Production promotion is blocked."
    )


# ==================================================================================================
# PART B — PATHS
# ==================================================================================================

PRODUCTION_PATH = Path(
    CONFIG.canonical_dataset_path
)


CANDIDATE_PATH = Path(
    CANONICAL_REFRESHED_NORMALIZED_FINAL_PATH
)


if not PRODUCTION_PATH.exists():

    raise FileNotFoundError(
        f"Production canonical dataset does not exist:\n"
        f"{PRODUCTION_PATH}"
    )


if not CANDIDATE_PATH.exists():

    raise FileNotFoundError(
        f"Validated candidate does not exist:\n"
        f"{CANDIDATE_PATH}"
    )


BACKUP_PATH = (
    PRODUCTION_PATH.parent
    /
    (
        f"{PRODUCTION_PATH.stem}_"
        f"pre_{RUN_ID}"
        f"{PRODUCTION_PATH.suffix}"
    )
)


TEMP_PROMOTION_PATH = (
    PRODUCTION_PATH.parent
    /
    (
        f".{PRODUCTION_PATH.stem}_"
        f"promotion_{RUN_ID}.tmp"
        f"{PRODUCTION_PATH.suffix}"
    )
)


RELEASE_MANIFEST_PATH = (
    RUN_CHECKPOINT_DIR
    /
    "canonical_production_release_manifest.json"
)


ACTIVE_CANONICAL_PATH = (
    PRODUCTION_PATH.parent
    /
    "ACTIVE_CANONICAL.json"
)


# ==================================================================================================
# PART C — HASH HELPER
# ==================================================================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
):

    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


# ==================================================================================================
# PART D — HASH CURRENT FILES
# ==================================================================================================

CANDIDATE_SHA256 = sha256_file(
    CANDIDATE_PATH
)


PRODUCTION_SHA256_BEFORE = sha256_file(
    PRODUCTION_PATH
)


ALREADY_PROMOTED = bool(
    CANDIDATE_SHA256
    ==
    PRODUCTION_SHA256_BEFORE
)


# ==================================================================================================
# PART E — VALIDATE CANDIDATE FROM DISK
# ==================================================================================================

CANDIDATE_DISK = pd.read_parquet(
    CANDIDATE_PATH
)


CANDIDATE_DISK[
    "result_date_dt"
] = pd.to_datetime(
    CANDIDATE_DISK[
        "result_date_dt"
    ],
    errors="coerce",
)


CANDIDATE_ROWS = int(
    len(
        CANDIDATE_DISK
    )
)


CANDIDATE_ATHLETES = int(
    CANDIDATE_DISK[
        "aaAthleteId"
    ]
    .nunique()
)


CANDIDATE_MAX_DATE = (
    CANDIDATE_DISK[
        "result_date_dt"
    ]
    .max()
)


CANDIDATE_V4 = int(
    CANDIDATE_DISK[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


CANDIDATE_SCHEMA_PASS = bool(
    list(
        CANDIDATE_DISK.columns
    )
    ==
    EXPECTED_CANONICAL_COLUMNS
)


if not CANDIDATE_SCHEMA_PASS:

    raise RuntimeError(
        "Candidate parquet schema does not match the canonical schema."
    )


# ==================================================================================================
# PART F — VALIDATE IN-MEMORY VS DISK CANDIDATE
# ==================================================================================================

IN_MEMORY_ROWS = int(
    len(
        CANONICAL_REFRESHED_NORMALIZED_FINAL
    )
)


IN_MEMORY_V4 = int(
    CANONICAL_REFRESHED_NORMALIZED_FINAL[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


IN_MEMORY_DISK_PASS = bool(
    CANDIDATE_ROWS
    ==
    IN_MEMORY_ROWS

    and

    CANDIDATE_V4
    ==
    IN_MEMORY_V4
)


if not IN_MEMORY_DISK_PASS:

    raise RuntimeError(
        "Saved candidate does not agree with the validated "
        "in-memory candidate."
    )


# ==================================================================================================
# PART G — BACKUP + ATOMIC PROMOTION
# ==================================================================================================

PROMOTION_PERFORMED = False


if not ALREADY_PROMOTED:

    # ----------------------------------------------------------------------------------------------
    # G1 — BACKUP CURRENT PRODUCTION
    # ----------------------------------------------------------------------------------------------

    if BACKUP_PATH.exists():

        BACKUP_SHA256 = sha256_file(
            BACKUP_PATH
        )


        if (
            BACKUP_SHA256
            !=
            PRODUCTION_SHA256_BEFORE
        ):

            raise RuntimeError(
                "A backup already exists for this RUN_ID but its hash does "
                "not match the current production dataset.\n\n"
                f"Backup:     {BACKUP_PATH}\n"
                f"Production: {PRODUCTION_PATH}\n\n"
                "Review manually before promotion."
            )


    else:

        shutil.copy2(
            PRODUCTION_PATH,
            BACKUP_PATH,
        )


        BACKUP_SHA256 = sha256_file(
            BACKUP_PATH
        )


        if (
            BACKUP_SHA256
            !=
            PRODUCTION_SHA256_BEFORE
        ):

            raise RuntimeError(
                "Production backup hash verification failed."
            )


    # ----------------------------------------------------------------------------------------------
    # G2 — CLEAN STALE TEMP FILE
    # ----------------------------------------------------------------------------------------------

    if TEMP_PROMOTION_PATH.exists():

        TEMP_PROMOTION_PATH.unlink()


    # ----------------------------------------------------------------------------------------------
    # G3 — COPY CANDIDATE TO TEMPORARY PRODUCTION FILE
    # ----------------------------------------------------------------------------------------------

    shutil.copy2(
        CANDIDATE_PATH,
        TEMP_PROMOTION_PATH,
    )


    TEMP_SHA256 = sha256_file(
        TEMP_PROMOTION_PATH
    )


    if (
        TEMP_SHA256
        !=
        CANDIDATE_SHA256
    ):

        raise RuntimeError(
            "Temporary promotion copy does not match candidate SHA256."
        )


    # ----------------------------------------------------------------------------------------------
    # G4 — LOAD TEMP FILE BEFORE REPLACEMENT
    # ----------------------------------------------------------------------------------------------

    TEMP_DATA = pd.read_parquet(
        TEMP_PROMOTION_PATH
    )


    TEMP_DATA[
        "result_date_dt"
    ] = pd.to_datetime(
        TEMP_DATA[
            "result_date_dt"
        ],
        errors="coerce",
    )


    TEMP_VALIDATION_PASS = bool(

        len(
            TEMP_DATA
        )
        ==
        CANDIDATE_ROWS

        and

        list(
            TEMP_DATA.columns
        )
        ==
        EXPECTED_CANONICAL_COLUMNS

        and

        int(
            TEMP_DATA[
                "aaAthleteId"
            ]
            .nunique()
        )
        ==
        CANDIDATE_ATHLETES

        and

        TEMP_DATA[
            "result_date_dt"
        ]
        .max()
        ==
        CANDIDATE_MAX_DATE

        and

        int(
            TEMP_DATA[
                "v4_base_eligible"
            ]
            .fillna(False)
            .astype(bool)
            .sum()
        )
        ==
        CANDIDATE_V4
    )


    if not TEMP_VALIDATION_PASS:

        TEMP_PROMOTION_PATH.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "Temporary promotion dataset failed validation. "
            "Production was NOT modified."
        )


    # ----------------------------------------------------------------------------------------------
    # G5 — ATOMIC REPLACEMENT
    # ----------------------------------------------------------------------------------------------

    os.replace(
        TEMP_PROMOTION_PATH,
        PRODUCTION_PATH,
    )


    PROMOTION_PERFORMED = True


else:

    BACKUP_SHA256 = (
        sha256_file(
            BACKUP_PATH
        )
        if BACKUP_PATH.exists()
        else None
    )


# ==================================================================================================
# PART H — POST-PROMOTION VERIFICATION
# ==================================================================================================

PRODUCTION_SHA256_AFTER = sha256_file(
    PRODUCTION_PATH
)


PRODUCTION_AFTER = pd.read_parquet(
    PRODUCTION_PATH
)


PRODUCTION_AFTER[
    "result_date_dt"
] = pd.to_datetime(
    PRODUCTION_AFTER[
        "result_date_dt"
    ],
    errors="coerce",
)


PRODUCTION_ROWS_AFTER = int(
    len(
        PRODUCTION_AFTER
    )
)


PRODUCTION_ATHLETES_AFTER = int(
    PRODUCTION_AFTER[
        "aaAthleteId"
    ]
    .nunique()
)


PRODUCTION_MAX_DATE_AFTER = (
    PRODUCTION_AFTER[
        "result_date_dt"
    ]
    .max()
)


PRODUCTION_V4_AFTER = int(
    PRODUCTION_AFTER[
        "v4_base_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)


PRODUCTION_SCHEMA_AFTER_PASS = bool(
    list(
        PRODUCTION_AFTER.columns
    )
    ==
    EXPECTED_CANONICAL_COLUMNS
)


HASH_PASS = bool(
    PRODUCTION_SHA256_AFTER
    ==
    CANDIDATE_SHA256
)


ROW_PASS = bool(
    PRODUCTION_ROWS_AFTER
    ==
    CANDIDATE_ROWS
)


ATHLETE_PASS = bool(
    PRODUCTION_ATHLETES_AFTER
    ==
    CANDIDATE_ATHLETES
)


DATE_PASS = bool(
    PRODUCTION_MAX_DATE_AFTER
    ==
    CANDIDATE_MAX_DATE
)


V4_PASS = bool(
    PRODUCTION_V4_AFTER
    ==
    CANDIDATE_V4
)


CELL_14_PASS = bool(
    HASH_PASS
    and
    ROW_PASS
    and
    ATHLETE_PASS
    and
    DATE_PASS
    and
    V4_PASS
    and
    PRODUCTION_SCHEMA_AFTER_PASS
)


# ==================================================================================================
# PART I — EMERGENCY ROLLBACK IF POST-WRITE VERIFICATION FAILS
# ==================================================================================================

ROLLBACK_PERFORMED = False


if (
    PROMOTION_PERFORMED
    and
    not CELL_14_PASS
):

    if not BACKUP_PATH.exists():

        raise RuntimeError(
            "Post-promotion verification failed and backup is missing. "
            "Manual intervention is required."
        )


    rollback_temp = (
        PRODUCTION_PATH.parent
        /
        (
            f".{PRODUCTION_PATH.stem}_"
            f"rollback_{RUN_ID}.tmp"
            f"{PRODUCTION_PATH.suffix}"
        )
    )


    if rollback_temp.exists():

        rollback_temp.unlink()


    shutil.copy2(
        BACKUP_PATH,
        rollback_temp,
    )


    os.replace(
        rollback_temp,
        PRODUCTION_PATH,
    )


    ROLLBACK_PERFORMED = True


    raise RuntimeError(
        "Post-promotion verification failed. "
        "The previous production dataset was restored from backup."
    )


# ==================================================================================================
# PART J — RELEASE MANIFEST
# ==================================================================================================

PROMOTION_TIMESTAMP_UTC = (
    datetime.now(
        timezone.utc
    )
    .isoformat()
)


RELEASE_MANIFEST = {
    "status":
        (
            "PROMOTED"
            if CELL_14_PASS
            else "FAILED"
        ),

    "run_id":
        str(
            RUN_ID
        ),

    "promotion_timestamp_utc":
        PROMOTION_TIMESTAMP_UTC,

    "already_promoted_on_entry":
        ALREADY_PROMOTED,

    "promotion_performed_this_run":
        PROMOTION_PERFORMED,

    "rollback_performed":
        ROLLBACK_PERFORMED,

    "production_path":
        str(
            PRODUCTION_PATH
        ),

    "candidate_path":
        str(
            CANDIDATE_PATH
        ),

    "backup_path":
        (
            str(
                BACKUP_PATH
            )
            if BACKUP_PATH.exists()
            else None
        ),

    "production_sha256_before":
        PRODUCTION_SHA256_BEFORE,

    "candidate_sha256":
        CANDIDATE_SHA256,

    "production_sha256_after":
        PRODUCTION_SHA256_AFTER,

    "backup_sha256":
        BACKUP_SHA256,

    "rows":
        PRODUCTION_ROWS_AFTER,

    "athletes":
        PRODUCTION_ATHLETES_AFTER,

    "max_result_date":
        (
            PRODUCTION_MAX_DATE_AFTER.isoformat()
            if pd.notna(
                PRODUCTION_MAX_DATE_AFTER
            )
            else None
        ),

    "v4_eligible":
        PRODUCTION_V4_AFTER,

    "verification": {
        "hash":
            HASH_PASS,

        "rows":
            ROW_PASS,

        "athletes":
            ATHLETE_PASS,

        "max_date":
            DATE_PASS,

        "v4_population":
            V4_PASS,

        "schema":
            PRODUCTION_SCHEMA_AFTER_PASS,
    },
}


with RELEASE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RELEASE_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART K — ACTIVE CANONICAL POINTER
# ==================================================================================================

ACTIVE_CANONICAL = {
    "run_id":
        str(
            RUN_ID
        ),

    "production_path":
        str(
            PRODUCTION_PATH
        ),

    "source_candidate":
        str(
            CANDIDATE_PATH
        ),

    "production_sha256":
        PRODUCTION_SHA256_AFTER,

    "rows":
        PRODUCTION_ROWS_AFTER,

    "athletes":
        PRODUCTION_ATHLETES_AFTER,

    "v4_eligible":
        PRODUCTION_V4_AFTER,

    "max_result_date":
        (
            PRODUCTION_MAX_DATE_AFTER.isoformat()
            if pd.notna(
                PRODUCTION_MAX_DATE_AFTER
            )
            else None
        ),

    "promoted_at_utc":
        PROMOTION_TIMESTAMP_UTC,
}


with ACTIVE_CANONICAL_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        ACTIVE_CANONICAL,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART L — UPDATE NOTEBOOK MANIFEST
# ==================================================================================================

RUN_MANIFEST[
    "canonical_promotion"
] = RELEASE_MANIFEST


RUN_MANIFEST[
    "status"
] = (
    "CANONICAL_PRODUCTION_PROMOTED"
    if CELL_14_PASS
    else "CANONICAL_PROMOTION_FAILED"
)


with RUN_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        RUN_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# ==================================================================================================
# PART M — OUTPUT
# ==================================================================================================

print()
print("=" * 145)
print("WORLD ATHLETICS — CANONICAL PRODUCTION PROMOTION")
print("=" * 145)


print()
print("RELEASE GATES")
print("-" * 145)

print(
    f"Cell 12 candidate gate           : "
    f"{CELL_12_PASS}"
)

print(
    f"Cell 13 release regression       : "
    f"{CELL_13_PASS}"
)


print()
print("PROMOTION")
print("-" * 145)

print(
    f"Already promoted on entry        : "
    f"{ALREADY_PROMOTED}"
)

print(
    f"Promotion performed this run     : "
    f"{PROMOTION_PERFORMED}"
)

print(
    f"Production path                  : "
    f"{PRODUCTION_PATH}"
)

print(
    f"Candidate path                   : "
    f"{CANDIDATE_PATH}"
)

print(
    f"Backup path                      : "
    f"{BACKUP_PATH if BACKUP_PATH.exists() else 'N/A'}"
)


print()
print("POST-PROMOTION DATASET")
print("-" * 145)

print(
    f"Rows                             : "
    f"{PRODUCTION_ROWS_AFTER:,}"
)

print(
    f"Athletes                         : "
    f"{PRODUCTION_ATHLETES_AFTER:,}"
)

print(
    f"V4 eligible                      : "
    f"{PRODUCTION_V4_AFTER:,}"
)

print(
    f"Max result date                  : "
    f"{PRODUCTION_MAX_DATE_AFTER.date()}"
)


print()
print("VERIFICATION")
print("-" * 145)

print(
    f"SHA256 matches candidate         : "
    f"{HASH_PASS}"
)

print(
    f"Row count                        : "
    f"{ROW_PASS}"
)

print(
    f"Athlete count                    : "
    f"{ATHLETE_PASS}"
)

print(
    f"Max date                         : "
    f"{DATE_PASS}"
)

print(
    f"V4 population                    : "
    f"{V4_PASS}"
)

print(
    f"Canonical schema                 : "
    f"{PRODUCTION_SCHEMA_AFTER_PASS}"
)


print()
print("HASHES")
print("-" * 145)

print(
    f"Production SHA256 before         : "
    f"{PRODUCTION_SHA256_BEFORE}"
)

print(
    f"Candidate SHA256                 : "
    f"{CANDIDATE_SHA256}"
)

print(
    f"Production SHA256 after          : "
    f"{PRODUCTION_SHA256_AFTER}"
)


print()
print("RELEASE METADATA")
print("-" * 145)

print(
    f"Release manifest                 : "
    f"{RELEASE_MANIFEST_PATH}"
)

print(
    f"Active canonical pointer         : "
    f"{ACTIVE_CANONICAL_PATH}"
)


print()
print("=" * 145)


if CELL_14_PASS:

    print(
        "CELL 14 STATUS: CANONICAL PRODUCTION PROMOTION PASSED"
    )

    print()

    print(
        "The validated refreshed canonical dataset is now the "
        "active production dataset."
    )

    print(
        "The previous production dataset is preserved in the "
        "versioned backup."
    )


else:

    print(
        "CELL 14 STATUS: PRODUCTION PROMOTION FAILED"
    )


print("=" * 145)


WORLD ATHLETICS — CANONICAL PRODUCTION PROMOTION

RELEASE GATES
-------------------------------------------------------------------------------------------------------------------------------------------------
Cell 12 candidate gate           : True
Cell 13 release regression       : True

PROMOTION
-------------------------------------------------------------------------------------------------------------------------------------------------
Already promoted on entry        : False
Promotion performed this run     : True
Production path                  : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/final/javelin_reconciled_base_1986_2026.parquet
Candidate path                   : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/final/javelin_reconciled_base_1986_2026_refreshed_20260919_225312_candidate.parquet
Backup path                      : /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_global/final/javelin_reconciled